In [35]:
# @title
!pip uninstall -y sympy
!pip install sympy==1.13.3 catboost lightgbm yfinance openpyxl nselib -q

Found existing installation: sympy 1.13.3
Uninstalling sympy-1.13.3:
  Successfully uninstalled sympy-1.13.3


In [36]:
# @title
# ============================================================================
# S1_Config_P1_v4.0 — CONFIGURATION & SHARED LIBRARY
# ============================================================================
#
# PURPOSE
# -------
# Single source of truth for the whole study. Defines every constant, every
# shared helper function, and every invariant. Trains nothing, predicts
# nothing, reads no data file. Every later cell depends on this one and adds
# to the same namespace.
#
# THE STUDY IN ONE PARAGRAPH
# --------------------------
# NIFTY 50 trades a weekly options cycle of five trading days: D1, D2, D3, D4,
# Expiry. At D1's close a price band is drawn around the index. On each of D2,
# D3 and D4 we ask: will the expiry close fall outside the band? Two
# contestants answer. The GAUSSIAN baseline computes the probability
# analytically from a lognormal assumption and has no free parameters. The ML
# CLASSIFIER learns the answer from past cycles. The study asks whether
# learning beats the closed form on the same data.
#
# WHAT THIS CELL PROVIDES, IN ORDER
# ---------------------------------
#   A  Experiment decisions      the switches that define the run; all are
#                                stamped into CACHE_VERSION
#   B  Runtime                   seeds, torch, device
#   C  Paths                     Drive layout, input loader
#   D  Expiry regime             Thursday before 2025-09-01, Tuesday after
#   E  Cycle geometry            D1..D4 + Expiry; days-left arithmetic
#   F  Grid, models, filters     sigma/window grid, model list, gate presets
#   G  Walk-forward              expanding-window splits, cross-fit folds
#   H  Tasks and payoff          the six task cells, economic threshold
#   I  Feature taxonomy          leakage set, scaling policy, class rules
#   J  Feature whitelist         the v2 feature set (THE definition)
#   K  Cache keys                CACHE_VERSION, filenames, parsers
#   L  Rolling statistics        mu/sigma over past cycles, warm-up policy
#   M  Band engine               band_upper / band_lower / breach labels
#   N  Derived features          realized_vol, time, v2 distances, gap
#   O  Gaussian baseline         normal_breach_probs
#   P  Metrics                   F1 threshold, Brier family, primary_score
#   Q  Consistency gate          in-fold vs out-of-fold trust filter
#   R  Ensemble                  weighted-probability top-k combiner
#   S  Preprocessing             clip_and_scale and its saved-pipeline twin
#   T  Selection helper          select_top_k
#   U  Neural net                NiftyBinaryFF (only when torch is present)
#   V  Expiry helpers            is_expiry, get_day_value
#   W  Self-tests                every invariant asserted at load time
#   X  Banner                    the run's settings, printed
#
# RUN ORDER
# ---------
#   S1_Config → S0_Fetch → S0_Cleanup → S2_CycleBuild → S2b_TaskFrames
#   → S2c_TaskExport → S3_Train → S3_Harvest → S3_Select → S6_InferEval
#   → S7_Significance → S8_Charts
# ============================================================================

import os
import re
import math
import json
import glob
import datetime
import warnings

import numpy as np
import pandas as pd

try:
    from scipy import stats as scipy_stats
except Exception:
    scipy_stats = None

warnings.filterwarnings("ignore")

print("\n" + "=" * 78)
print("  🔧 S1_Config_P1_v4.0 — CONFIGURATION & SHARED LIBRARY")
print("=" * 78)


# ============================================================================
# A. EXPERIMENT DECISIONS
# ----------------------------------------------------------------------------
# Seven switches define the run. Each is stamped into CACHE_VERSION, so two
# different settings can never share a model cache. Change one, re-run S1, and
# the whole pipeline follows into a clean cache world.
#
# ⚠️ CACHE_VERSION IS PINNED at the bottom of section K. If you change any
#    switch here, that assertion will fail — deliberately. Update the pin only
#    when you intend to retrain everything.
# ============================================================================

# --- (1) BAND-WINDOW GRID ---------------------------------------------------
# Selects which window lengths are swept. The rolling band statistics are
# estimated over this many past cycles.
#   "full"   → the sweep list; "single" → one window.
# NOTE ON THE CURRENT SETTING: the study runs ONE configuration (sigma 0.5,
# window 4). The mode is left at "full" because CACHE_VERSION encodes it and
# the trained cache must stay readable. The grid fingerprint assert in section
# K is what prevents a widened grid from silently mixing with these models.
P1_WINDOW_GRID_MODE = "full"              # "full" | "single"

# --- (2) FEATURE MODE -------------------------------------------------------
# "day_only"   → only the decision day's own block.
# "cumulative" → every day from D1 up to the decision day. ML's inputs become
#                a superset of the Gaussian's, so the "identical inputs" claim
#                does not hold; report as a labelled variant.
P1_FEATURE_MODE = "cumulative"            # "day_only" | "cumulative"

# --- (3) NEURAL NETWORK -----------------------------------------------------
# Adds NN_FF to the model list. At roughly 25-40 breach events per training
# block against ~56 parameters this is the most overfit-prone contestant;
# it is included to show that, not because it is expected to win.
P1_ENABLE_NN = True                       # True | False

# --- (4) CONSISTENCY-FILTER PRESET ------------------------------------------
# The trust gate on in-fold vs out-of-fold agreement. See section Q.
#   "principled" strictest · "moderate" the run's setting
#   "relaxed"    near-off  · "off" explicitly disabled (robustness appendix)
P1_FILTER_PRESET = "moderate"             # principled | moderate | relaxed | off

# --- (5) PRIMARY METRIC -----------------------------------------------------
# "brier" → Brier SKILL score: threshold-free, proper, uses every observation,
#           and decomposes into reliability + resolution.
# "f1"    → threshold-dependent; both contestants must then fit a cut.
# F1 is always computed and reported either way.
P1_PRIMARY_METRIC = "brier"               # "brier" | "f1"

# --- (6) THRESHOLD FITTING --------------------------------------------------
# True  → the decision cut is cross-fitted over train+val, so ML and the
#         Gaussian fit their thresholds on the same block (symmetric).
# False → inner-validation block only (asymmetric; kept for comparison).
P1_CROSSFIT_THRESHOLD = True
P1_CROSSFIT_FOLDS     = 5

# --- (7) HYPERPARAMETER TUNING ----------------------------------------------
# Nested per outer fold, on the fitting block only. The Gaussian has no free
# parameters; tuning the learned models deliberately stacks the comparison in
# ML's favour, so a null result under this asymmetry is a real result.
P1_TUNE_HYPERPARAMS = True
P1_TUNE_N_ITER      = 15                  # random draws per (model, cell, fold)
P1_TUNE_INNER_FOLDS = 3                   # stratified inner CV

# --- LIVE DECISION RULE (not stamped into CACHE_VERSION) --------------------
# "economic"   → the cut implied by the payoff matrix (section H). The only
#                cut with a real-world meaning; needs a calibrated probability.
# "f1_optimal" → the cut that maximises F1 on the fitting block.
P1_DECISION_RULE = "economic"             # "economic" | "f1_optimal"


# ============================================================================
# B. RUNTIME
# ============================================================================
FAST_MODE     = False          # True shrinks folds and seeds for a smoke test
RANDOM_STATE  = 42
PROJECT1_MODE = True           # feature whitelist gating; always on

P1_DROP_WARMUP = True          # drop cycles whose rolling mu/sigma is undefined
P1_STRICT_NAN  = True          # raise on non-finite input instead of imputing

RUN_TYPE = "FAST" if FAST_MODE else "FULL"
np.random.seed(RANDOM_STATE)

PROJECT1_FEATURE_MODE = P1_FEATURE_MODE      # alias read by older helpers

_HAS_TORCH = False
try:
    import torch
    import torch.nn as nn
    _HAS_TORCH = True
    _GPU = torch.cuda.is_available()
except Exception:
    _GPU = False
    if P1_ENABLE_NN:
        raise RuntimeError("❌ P1_ENABLE_NN=True but PyTorch is unavailable. "
                           "Install torch or set P1_ENABLE_NN=False.")
device = "cuda" if _GPU else "cpu"
if _HAS_TORCH:
    try:
        torch.manual_seed(RANDOM_STATE)
    except Exception:
        pass

USE_NN_P1 = bool(P1_ENABLE_NN)


# ============================================================================
# C. PATHS AND INPUT
# ============================================================================
def mount_drive(mount_path="/content/drive"):
    """Mount Google Drive if it is not already mounted.

    Parameters
    ----------
    mount_path : str
        Where to mount. Default is Colab's standard location.

    Returns
    -------
    str
        The mount path, whether or not the mount actually happened.

    Notes
    -----
    Idempotent and safe outside Colab: a failure prints a notice and returns.
    """
    if os.path.ismount(mount_path):
        print("  ✅ Google Drive already mounted")
        return mount_path
    try:
        from google.colab import drive
        print("  📂 Mounting Google Drive…")
        drive.mount(mount_path)
        print("  ✅ Google Drive mounted")
    except Exception as _e:
        print(f"  ⚠️  Not in Colab or mount skipped: {_e}")
    return mount_path


mount_drive()

DRIVE_ROOT      = "/content/drive/MyDrive/LSTM"
P1_ROOT         = os.path.join(DRIVE_ROOT, "Project1_MLcore_vs_Normal")
DRIVE_INPUT_DIR = os.path.join(P1_ROOT, "Input")

RAW_INPUT_PATH   = os.path.join(DRIVE_INPUT_DIR, "Nifty_LSTM_Features.xlsx")
CLEAN_INPUT_PATH = os.path.join(DRIVE_INPUT_DIR, "Nifty_LSTM_Features_clean.xlsx")
DATA_SHEET_NAME  = "LSTM Features"

# Column names in the clean daily file. OPEN_COL is required by the gap feature.
DATE_COL  = "Date"
CLOSE_COL = "Close"
OPEN_COL  = "Open"
VIX_COL   = "Volatility"

_WORLD          = RUN_TYPE
OUTPUT_ROOT     = os.path.join(P1_ROOT, "output_binary")
WORLD_DIR       = os.path.join(OUTPUT_ROOT, _WORLD)
MODEL_CACHE_DIR = os.path.join(WORLD_DIR, "model_cache")
GRID_DIR        = os.path.join(WORLD_DIR, "sigma_grid")
CHECKPOINT_PATH = os.path.join(WORLD_DIR, "df_cycles_checkpoint.parquet")

# A fresh timestamped results folder per S1 load. Re-running S1 starts a new
# one; the model cache and the ledger are NOT timestamped and persist.
RESULTS_DIR = os.path.join(WORLD_DIR, "results",
                           f"run_{datetime.datetime.now():%Y%m%d_%H%M%S}")

for _d in (DRIVE_INPUT_DIR, MODEL_CACHE_DIR, GRID_DIR):
    try:
        os.makedirs(_d, exist_ok=True)
    except Exception:
        pass

LEDGER_PATH           = os.path.join(GRID_DIR, f"master_results_{_WORLD}.xlsx")
DIRECTION_CONFIG_PATH = os.path.join(GRID_DIR, f"direction_best_band_config_{_WORLD}.json")
BEST_MODELS_PATH      = os.path.join(GRID_DIR, f"best_models_by_direction_day_{_WORLD}.json")
FEATURE_AUDIT_PATH    = os.path.join(GRID_DIR, f"feature_selection_audit_{_WORLD}.xlsx")

ENABLE_MODEL_CACHE = True

_DAILY_CACHE = {}          # path → DataFrame, so the file is read once per run


def load_input(path=CLEAN_INPUT_PATH, sheet=None):
    """Read the clean daily feature file. Read-only; never refetches.

    Parameters
    ----------
    path : str
        Path to the clean Excel file produced by S0_Cleanup.
    sheet : str or None
        Worksheet name. None reads the first sheet.

    Returns
    -------
    pandas.DataFrame
        Sorted by date, duplicate dates removed, index reset.

    Raises
    ------
    FileNotFoundError
        If the file does not exist — run S0_Fetch then S0_Cleanup.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"❌ Input not found: {path}\n"
            f"   Run S0_Fetch_P1 → S0_Cleanup_P1 to create it.")
    mtime = datetime.datetime.fromtimestamp(os.path.getmtime(path))
    df = (pd.read_excel(path, parse_dates=[DATE_COL]) if sheet is None
          else pd.read_excel(path, sheet_name=sheet, parse_dates=[DATE_COL]))
    df = (df.sort_values(DATE_COL)
            .drop_duplicates(subset=[DATE_COL])
            .reset_index(drop=True))
    print("  " + "─" * 62)
    print("  📥 INPUT LOADED (read-only)")
    print(f"     Modified : {mtime:%Y-%m-%d %H:%M}")
    print(f"     Rows×Cols: {len(df)} × {df.shape[1]}")
    print(f"     Range    : {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}")
    print("  " + "─" * 62)
    return df


def p1_daily(path=CLEAN_INPUT_PATH):
    """Memoised accessor for the clean daily frame.

    Parameters
    ----------
    path : str
        Path to the clean Excel file.

    Returns
    -------
    pandas.DataFrame
        The same object on every call within a session.

    Notes
    -----
    The gap and realized-volatility features need the daily file, and they are
    computed deep inside the training loop. This accessor exists so those
    functions can reach the data explicitly rather than searching globals for
    whatever a previous cell happened to leave behind.
    """
    if path not in _DAILY_CACHE:
        _DAILY_CACHE[path] = load_input(path)
    return _DAILY_CACHE[path]


# ============================================================================
# D. EXPIRY REGIME
# ----------------------------------------------------------------------------
# NSE moved weekly NIFTY expiry from Thursday to Tuesday. Both regimes have a
# holiday fallback: if the nominal expiry day did not trade, the previous
# trading day settles the contract instead.
# ============================================================================
CUTOFF_DATE         = pd.Timestamp("2025-09-01")
EXPIRY_DAY          = 3    # Thursday, before the cutoff
EXPIRY_DAY_PREV     = 2    # Wednesday fallback
NEW_EXPIRY_DAY      = 1    # Tuesday, on or after the cutoff
NEW_EXPIRY_DAY_PREV = 0    # Monday fallback


# ============================================================================
# E. CYCLE GEOMETRY
# ----------------------------------------------------------------------------
# A standard cycle is FIVE trading days: D1, D2, D3, D4, Expiry.
#
# Two different quantities both equal 4, and they are named separately so the
# literal never has to be written again:
#     CYCLE_DECISION_DAYS = 4   D1..D4, the days that get a cycle record
#     CYCLE_STEPS         = 4   D1 close → expiry close, in trading-day steps
#
# From Dn there are CYCLE_STEPS + 1 - n steps left:  D1:4  D2:3  D3:2  D4:1.
# The band sigma is the standard deviation of ln(expiry_close / d1_close) — a
# CYCLE_STEPS-step quantity — so the Gaussian's time scaling is
# sqrt(days_left / CYCLE_STEPS), which must equal exactly 1.0 at D1. That
# identity is the anchor invariant and is asserted in section W.
# ============================================================================
CYCLE_DECISION_DAYS = 4
CYCLE_STEPS         = 4


def days_left_for_day(n, cycle_steps=CYCLE_STEPS):
    """Trading-day steps remaining from Dn's close to the expiry close.

    Parameters
    ----------
    n : int
        Decision day, 1 through 4.
    cycle_steps : int
        Steps from D1's close to expiry. Default CYCLE_STEPS.

    Returns
    -------
    float
        4.0, 3.0, 2.0, 1.0 for n = 1, 2, 3, 4.
    """
    return float(cycle_steps) + 1.0 - float(n)


def time_frac_for_day(n, cycle_steps=CYCLE_STEPS):
    """days_left / cycle_steps. Exactly 1.0 at D1.

    Parameters
    ----------
    n : int
        Decision day, 1 through 4.
    cycle_steps : int
        Steps from D1's close to expiry.

    Returns
    -------
    float
        The remaining fraction of the horizon sigma was estimated over.
    """
    cs = float(cycle_steps)
    return max(days_left_for_day(n, cs), 0.0) / max(cs, 1e-12)


def sqrt_dl_frac_for_day(n, cycle_steps=CYCLE_STEPS):
    """Square root of time_frac_for_day — the Gaussian's volatility scaler.

    Parameters
    ----------
    n : int
        Decision day, 1 through 4.
    cycle_steps : int
        Steps from D1's close to expiry.

    Returns
    -------
    float
        1.0, 0.866, 0.707, 0.5 for n = 1, 2, 3, 4.
    """
    return float(np.sqrt(time_frac_for_day(n, cycle_steps)))


# ============================================================================
# F. GRID, MODELS AND FILTER PRESETS
# ============================================================================

# How the band is centred.
#   "vs_mean" → band = d1_close * exp(mu ± k*sigma), centred on recent drift
#   anything else → band = d1_close * exp(±k*sigma), centred on d1_close
LABEL_MODE = "vs_mean"

# SIGMA_GRID holds band-width MULTIPLIERS (floats, e.g. 0.5).
# SIGMA_WINDOW_GRID holds ROLLING WINDOW LENGTHS IN CYCLES (ints, e.g. 4).
# They sit on adjacent lines and are easy to swap by accident; _validate_grids
# catches that here rather than 200 lines later inside pandas.
SIGMA_GRID    = [0.5, 0.75, 1.0]
_WINDOW_GRIDS = {"full": [4, 8], "single": [8]}

if P1_WINDOW_GRID_MODE not in _WINDOW_GRIDS:
    raise ValueError(f"P1_WINDOW_GRID_MODE must be one of {list(_WINDOW_GRIDS)}")
SIGMA_WINDOW_GRID = list(_WINDOW_GRIDS[P1_WINDOW_GRID_MODE])
if FAST_MODE:
    SIGMA_GRID = [0.5]
    SIGMA_WINDOW_GRID = SIGMA_WINDOW_GRID[:2]


def _validate_grids():
    """Type- and range-check the two grids, naming the actual mistake.

    Returns
    -------
    list of int
        SIGMA_WINDOW_GRID coerced to whole numbers, each >= 4.

    Raises
    ------
    ValueError
        If either grid is empty, holds the wrong kind of value, or contains a
        window below the minimum usable length.
    """
    if not SIGMA_GRID:
        raise ValueError("SIGMA_GRID is empty.")
    if not SIGMA_WINDOW_GRID:
        raise ValueError("SIGMA_WINDOW_GRID is empty.")

    bad = [s for s in SIGMA_GRID
           if not isinstance(s, (int, float)) or isinstance(s, bool)
           or not (0.0 < float(s) <= 5.0)]
    if bad:
        raise ValueError(
            f"SIGMA_GRID contains invalid band multipliers {bad}. Expected "
            f"floats in (0, 5], e.g. [0.5, 1.0, 1.5]. Got {SIGMA_GRID}.")

    out = []
    for w in SIGMA_WINDOW_GRID:
        if isinstance(w, bool) or not isinstance(w, (int, float)):
            raise ValueError(
                f"SIGMA_WINDOW_GRID contains {w!r} ({type(w).__name__}). "
                f"Windows are rolling lengths in CYCLES and must be whole "
                f"numbers, e.g. [4, 8, 12, 16]. Did the sigma multipliers get "
                f"pasted here? Got {SIGMA_WINDOW_GRID}.")
        if float(w) != int(w):
            raise ValueError(
                f"SIGMA_WINDOW_GRID contains the non-whole value {w!r}. "
                f"A value like 0.5 or 1.5 here is the classic symptom of the "
                f"sigma multipliers having been pasted into the window grid.")
        w = int(w)
        if w < SIGMA_MIN_PERIODS_ABS:
            raise ValueError(
                f"SIGMA_WINDOW_GRID contains window={w}. A rolling standard "
                f"deviation needs at least {SIGMA_MIN_PERIODS_ABS} "
                f"observations to mean anything.")
        out.append(w)
    return out


# Rolling-statistic warm-up policy. A standard deviation from two observations
# is meaningless, so require a real fraction of the window and leave the
# warm-up NaN rather than fabricating a sigma.
SIGMA_MIN_PERIODS_FRAC = 0.75
SIGMA_MIN_PERIODS_ABS  = 4


def sigma_min_periods(window):
    """Minimum observations required before a rolling statistic is defined.

    Parameters
    ----------
    window : int
        Rolling window length in cycles.

    Returns
    -------
    int
        max(SIGMA_MIN_PERIODS_ABS, ceil(window * SIGMA_MIN_PERIODS_FRAC)).
    """
    return int(max(SIGMA_MIN_PERIODS_ABS,
                   math.ceil(float(window) * SIGMA_MIN_PERIODS_FRAC)))


SIGMA_GRID        = [float(s) for s in SIGMA_GRID]
SIGMA_WINDOW_GRID = _validate_grids()

# One fixed ablation budget, at or above the largest feature set, so
# select_features() always takes its p <= k branch and returns EVERY feature.
# No feature selection occurs anywhere in this study.
#
# ⚠️ THIS IS A BUDGET, NOT A COUNT. It is 13 for every task and appears in
#    every cache filename as "feat13". The number of features a model actually
#    uses is 7 at D2, 10 at D3, 13 at D4 — quote `n_features_used`, never this.
ABLATION_FEATURE_COUNTS = [13]

PROJECT1_MODELS = ["LogisticRegression", "SVM", "XGBoost"]
if P1_ENABLE_NN:
    PROJECT1_MODELS = PROJECT1_MODELS + ["NN_FF"]

# The single configuration this study reports.
BAND_SIGMA   = float(SIGMA_GRID[0])
SIGMA_WINDOW = int(SIGMA_WINDOW_GRID[len(SIGMA_WINDOW_GRID) // 2])

# Consistency-gate presets. Floors are metric-specific because the two primary
# metrics live on different scales: F1 in [0, 1] with ~0.5 being a real model;
# Brier SKILL in (-inf, 1] with 0 = no better than predicting the base rate.
FILTER_PRESETS = {
    "principled": {"f1":    dict(gap=0.10, floor=0.70, ratio=0.80),
                   "brier": dict(gap=0.10, floor=0.05, ratio=0.50)},
    "moderate":   {"f1":    dict(gap=0.20, floor=0.55, ratio=0.60),
                   "brier": dict(gap=0.15, floor=0.02, ratio=0.40)},
    "relaxed":    {"f1":    dict(gap=0.40, floor=0.45, ratio=0.00),
                   "brier": dict(gap=0.30, floor=0.00, ratio=0.00)},
    "off":        {"f1":    dict(gap=9.99, floor=-9.99, ratio=-9.99),
                   "brier": dict(gap=9.99, floor=-9.99, ratio=-9.99)},
}
if P1_FILTER_PRESET not in FILTER_PRESETS:
    raise ValueError(f"P1_FILTER_PRESET must be one of {list(FILTER_PRESETS)}")
if P1_PRIMARY_METRIC not in ("f1", "brier"):
    raise ValueError("P1_PRIMARY_METRIC must be 'f1' or 'brier'")

_fp = FILTER_PRESETS[P1_FILTER_PRESET][P1_PRIMARY_METRIC]
MAX_SCORE_GAP          = float(_fp["gap"])     # |in-fold − out-of-fold| ceiling
MIN_OOF_SCORE          = float(_fp["floor"])   # out-of-fold score floor
SCORE_RATIO_MIN        = float(_fp["ratio"])   # out-of-fold / in-fold floor
FOLD_PASS_MIN_FRACTION = 0.75                  # share of folds that must pass

# Hyperparameter spaces, deliberately small. At roughly 60-100 positives per
# task cell an aggressive search would select the configuration that best fits
# inner-fold noise.
HYPERPARAM_SPACES = {
    "LogisticRegression": {
        "C":            [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
        "penalty":      ["l2"],
        "class_weight": ["balanced", None],
    },
    "SVM": {
        "C":            [0.1, 0.3, 1.0, 3.0, 10.0],
        "gamma":        ["scale", 0.01, 0.1, 1.0],
        "kernel":       ["rbf"],
        "class_weight": ["balanced", None],
    },
    "RandomForest": {
        "n_estimators":     [200, 400],
        "max_depth":        [3, 4, 6, 8, None],
        "min_samples_leaf": [1, 2, 4, 8],
        "max_features":     ["sqrt", "log2", 0.5],
        "class_weight":     ["balanced", "balanced_subsample"],
    },
    "XGBoost": {
        "n_estimators":     [200, 400],
        "max_depth":        [2, 3, 4, 6],
        "learning_rate":    [0.01, 0.03, 0.05, 0.1],
        "subsample":        [0.7, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.9, 1.0],
        "min_child_weight": [1, 3, 5],
        "reg_lambda":       [0.5, 1.0, 5.0],
    },
}

# --- Neural network shape ---------------------------------------------------
# Width is PINNED, not searched. The question is whether a net small enough to
# be fittable at roughly one event per parameter can match logistic
# regression; leaving h1/h2 in the search would let the run drift back to a
# wide network and answer a different question.
#
#   Linear(nf, 4) → ReLU → BatchNorm1d(4) → Dropout
#   Linear(4, 2)  → ReLU → Dropout → Linear(2, 2) → softmax
#
# Parameter count at nf=13: 56 + 8 + 10 + 6 = 80. At nf=7: 32 + 8 + 10 + 6 = 56.
NN_H1 = 4
NN_H2 = 2

# Dropout at width 2 deletes one of the two hidden units on a share of forward
# passes — destruction, not regularisation. Weight decay does the regularising.
NN_DROPOUT_GRID = [0.0]
DROPOUT         = 0.0                    # untuned fallback, matches the grid
N_SEEDS         = 3 if FAST_MODE else 5  # seeds averaged for the final model

P1_TUNE_NN_N_ITER = 8
NN_HYPERPARAM_SPACE = {
    "lr":           [3e-4, 1e-3, 3e-3],
    "weight_decay": [1e-4, 1e-3, 1e-2],
    "h1":           [NN_H1],
    "h2":           [NN_H2],
    "dropout":      NN_DROPOUT_GRID,
    "batch_size":   [16, 32],
}

TOP_K = 3        # ensemble members kept per task; see section R


# ============================================================================
# G. WALK-FORWARD VALIDATION
# ============================================================================
WALK_FORWARD_FOLDS         = 2 if FAST_MODE else 4
WALK_FORWARD_TEST_FRACTION = 0.15
INNER_VAL_FRACTION         = 0.20
MIN_TRAIN_CYCLES           = 20


def walk_forward_splits(n, n_folds=None, test_frac=None, min_train=None):
    """Expanding-window walk-forward index splits, oldest fold first.

    Parameters
    ----------
    n : int
        Number of cycles (rows) available.
    n_folds : int or None
        Maximum folds. Default WALK_FORWARD_FOLDS.
    test_frac : float or None
        Test block size as a fraction of n. Default WALK_FORWARD_TEST_FRACTION.
        The validation block is the same size.
    min_train : int or None
        Minimum training rows for a fold to be emitted.

    Returns
    -------
    list of (ndarray, ndarray, ndarray)
        (train_idx, val_idx, test_idx) per fold. Test blocks are contiguous,
        disjoint and strictly later than their own val block — so no cycle can
        appear in two test sets and be double-counted in the paired bootstrap.
    """
    n_folds   = n_folds   or WALK_FORWARD_FOLDS
    test_frac = test_frac or WALK_FORWARD_TEST_FRACTION
    min_train = min_train or MIN_TRAIN_CYCLES
    test_n = max(5, int(round(n * test_frac)))
    val_n  = test_n
    splits = []
    for fold in range(n_folds):
        test_end   = n - fold * test_n
        test_start = test_end - test_n
        val_start  = test_start - val_n
        if val_start <= min_train:
            break
        idx_train = np.arange(0, val_start)
        idx_val   = np.arange(val_start, test_start)
        idx_test  = np.arange(test_start, test_end)
        if len(idx_train) >= min_train and len(idx_val) >= 3 and len(idx_test) >= 3:
            splits.append((idx_train, idx_val, idx_test))
    return list(reversed(splits))


def inner_holdout(idx_train, frac=None):
    """Split a fold's training indices into (inner_fit, inner_val) by time.

    Parameters
    ----------
    idx_train : ndarray
        Training indices for one fold, in chronological order.
    frac : float or None
        Share held out. Default INNER_VAL_FRACTION.

    Returns
    -------
    (ndarray, ndarray)
        The earlier block and the held-out tail.

    Notes
    -----
    Used only for the deployable LIVE model. The walk-forward path uses
    stratified_kfold_indices instead, so that ML and the Gaussian fit their
    decision thresholds on the same block.
    """
    frac = frac or INNER_VAL_FRACTION
    n = len(idx_train)
    k = max(3, int(round(n * frac)))
    return idx_train[:-k], idx_train[-k:]


def stratified_kfold_indices(y, k=None, seed=RANDOM_STATE):
    """Stratified K-fold (fit_idx, out_idx) pairs over positions 0..len(y)-1.

    Parameters
    ----------
    y : array-like of int
        Binary labels for the block being split.
    k : int or None
        Requested folds. Default P1_CROSSFIT_FOLDS. Reduced automatically when
        the minority class is too small.
    seed : int
        Shuffle seed.

    Returns
    -------
    list of (ndarray, ndarray)
        Fit and out-of-fold index pairs. Falls back to a single 80/20 split
        when stratification is impossible.

    Notes
    -----
    Used to CROSS-FIT the decision threshold inside train+val so every row
    receives an out-of-fold probability from a model that never saw it. This
    is not leakage: train+val is entirely in the past relative to the sealed
    test block, and the inner split only affects how well the threshold is
    estimated.
    """
    y = np.asarray(y, int)
    n = len(y)
    k = int(k or P1_CROSSFIT_FOLDS)
    n_pos, n_neg = int(y.sum()), int((1 - y).sum())
    k = max(2, min(k, n_pos, n_neg)) if (n_pos >= 2 and n_neg >= 2) else 0
    if k < 2:
        cut = max(1, int(round(n * 0.8)))
        return [(np.arange(0, cut), np.arange(cut, n))] if cut < n else []
    try:
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        return [(np.asarray(a), np.asarray(b)) for a, b in skf.split(np.zeros(n), y)]
    except Exception:
        rng = np.random.default_rng(seed)
        order = rng.permutation(n)
        folds = np.array_split(order, k)
        return [(np.setdiff1d(np.arange(n), f), np.sort(f)) for f in folds]


# ============================================================================
# H. TASKS AND PAYOFF MATRIX
# ----------------------------------------------------------------------------
# Six task cells: {upper, lower} × {D2, D3, D4}. Each is a separate binary
# classification problem with its own feature set.
#
# The target is ONE-VS-REST. On an upper task, label 0 covers both "settled
# inside the band" and "breached the LOWER band" — those are different events.
# The label vector is a CYCLE-level property, so upper_D2, upper_D3 and
# upper_D4 share an identical y and differ only in features. That matters for
# multiple-testing correction: there are about two independent families here,
# not six.
# ============================================================================
TASKS = [
    {"name": "upper_D2", "direction": "upper", "day": 2, "label_col": "upper_breach"},
    {"name": "lower_D2", "direction": "lower", "day": 2, "label_col": "lower_breach"},
    {"name": "upper_D3", "direction": "upper", "day": 3, "label_col": "upper_breach"},
    {"name": "lower_D3", "direction": "lower", "day": 3, "label_col": "lower_breach"},
    {"name": "upper_D4", "direction": "upper", "day": 4, "label_col": "upper_breach"},
    {"name": "lower_D4", "direction": "lower", "day": 4, "label_col": "lower_breach"},
]
DAYS = [2, 3, 4]

# Payoff matrix for the economic decision rule, in arbitrary risk units.
#
# ⚠️ A CORRECT exit (-1.5) costs MORE than a false alarm (-1.0). That is
#    coherent if -1.5 represents the loss already realised by the time you
#    exit into a developing breach — but it is an assumption, and the write-up
#    should state it rather than let a reader discover it.
PAYOFF_BREACH_CORRECT   = -1.5   # predicted breach, breach happened   (exit)
PAYOFF_BREACH_WRONG     = -1.0   # predicted breach, none happened     (false alarm)
PAYOFF_NOBREACH_CORRECT = 1.0    # predicted no breach, none happened  (hold, ok)
PAYOFF_NOBREACH_WRONG   = -4.0   # predicted no breach, breach happened (held into it)


def binary_payoff_vec(yt, yp):
    """Per-cycle payoff for a set of predictions.

    Parameters
    ----------
    yt : array-like of int
        True labels, 0 or 1.
    yp : array-like of int
        Predicted labels, 0 or 1.

    Returns
    -------
    ndarray of float
        One payoff per row, drawn from the four PAYOFF_* constants.
    """
    yt, yp = np.asarray(yt, int), np.asarray(yp, int)
    return np.where((yp == 1) & (yt == 1), PAYOFF_BREACH_CORRECT,
           np.where((yp == 1) & (yt == 0), PAYOFF_BREACH_WRONG,
           np.where((yp == 0) & (yt == 0), PAYOFF_NOBREACH_CORRECT,
                                           PAYOFF_NOBREACH_WRONG)))


def economic_threshold():
    """Break-even P(breach) implied by the payoff matrix.

    Returns
    -------
    float
        The probability above which exiting has higher expected value than
        holding, clipped to [0, 1].

    Notes
    -----
        EV_exit(p) = p*BREACH_CORRECT + (1-p)*BREACH_WRONG
        EV_hold(p) = p*NOBREACH_WRONG + (1-p)*NOBREACH_CORRECT
        p*         = (NOBREACH_CORRECT - BREACH_WRONG) / (BC - BW - NW + NC)

    Meaningful only on a CALIBRATED probability, i.e. with
    P1_PRIMARY_METRIC = "brier". An F1-optimal cut has no economic meaning.
    """
    num = PAYOFF_NOBREACH_CORRECT - PAYOFF_BREACH_WRONG
    den = (PAYOFF_BREACH_CORRECT - PAYOFF_BREACH_WRONG
           - PAYOFF_NOBREACH_WRONG + PAYOFF_NOBREACH_CORRECT)
    if abs(den) < 1e-12:
        return 0.5
    return float(np.clip(num / den, 0.0, 1.0))


ECONOMIC_THRESHOLD = economic_threshold()


# ============================================================================
# I. FEATURE TAXONOMY
# ============================================================================

# Columns that encode the answer, the future, or the band itself. None may
# ever enter a feature whitelist; get_task_features asserts this.
LEAKAGE_FEATURES = {
    "expiry_close", "expiry_date", "expiry_weekday", "expiry_regime",
    "cycle_id", "cycle_days_total", "expected_cycle_days", "is_short_cycle",
    "is_expiry", "day_of_cycle",
    "d1_close", "d2_close", "d3_close", "d4_close",
    "d1_date", "d2_date", "d3_date", "d4_date",
    "band_upper", "band_lower", "exp_mean_upper", "exp_mean_lower",
    "mu_rolling", "sigma_rolling", "mu_upper", "mu_lower",
    "sigma_upper_rolling", "sigma_lower_rolling",
    "d1_to_exp_return", "cycle_return_D1_to_expiry",
    "upper_breach", "lower_breach", "no_breach", "split",
}

# Features exempt from RobustScaler.
#
# EMPTY BY DESIGN. Under the v2 definitions every feature is a small
# log-quantity of comparable magnitude: norm_dist ~ 0.02, realized_vol ~ 0.009,
# gap ~ 0.006, band_width ~ 0.02. Leaving any of them unscaled while the rest
# are scaled to ~1 would force a coefficient roughly 50x larger, which L2 then
# penalises away — the feature would be silently neutered rather than dropped.
NO_SCALE_FEATURES = set()

# Prefix → reporting category, first match wins. Used by the feature audit.
FEATURE_CLASS_RULES = [
    ("gap_D",            "Gap/overnight"),
    ("realized_vol",     "Volatility"),
    ("norm_dist",        "Band/directional"),
    ("band_width",       "Band/directional"),
    ("band_confidence",  "Band/directional"),
    ("days_left",        "Time-to-expiry"),
    ("sqrt_dl_frac",     "Time-to-expiry"),
]


def feature_class(feat):
    """Reporting category for a feature name.

    Parameters
    ----------
    feat : str
        Feature column name.

    Returns
    -------
    str
        The first matching category, or "Unclassified".
    """
    for matcher, cls in FEATURE_CLASS_RULES:
        if feat == matcher or feat.startswith(matcher) or matcher in feat:
            return cls
    return "Unclassified"


# ============================================================================
# J. FEATURE WHITELIST  —  THE DEFINITION OF THE MODEL'S INPUTS
# ----------------------------------------------------------------------------
# Per decision day n, one block of three:
#
#   norm_dist_{side}_Dn = ln(band_{side} / Dn_close)
#         log-distance from that day's close to that side's band edge. This is
#         the Gaussian's own `dist` term. Despite the name it is NOT divided
#         by sigma, so it is a log-distance, not a z-score.
#
#   realized_vol_Dn     = std of DAILY log returns over the last
#                         (window * 4) trading days, shifted one day, read on
#                         d{n}_date. A different quantity from `sigma`, which
#                         is WEEKLY volatility across cycles.
#
#   gap_Dn              = ln(Open on Dn / Close on D(n-1))
#                         the overnight jump into that day. Known at the open,
#                         decided at the close — no look-ahead.
#
# Plus one cycle-level feature:
#
#   band_width_pct      = ln(band_upper / band_lower)  ==  2 * k * sigma
#
# Cumulative from D1, so:
#     D2 → D1 + D2 blocks           =  6 + band_width =  7 features
#     D3 → D1 + D2 + D3 blocks      =  9 + band_width = 10 features
#     D4 → D1 + D2 + D3 + D4 blocks = 12 + band_width = 13 features
#
# ONE SIDE PER TASK — and why band_width_pct is admissible
# --------------------------------------------------------
# An upper task receives norm_dist_upper_Dn and NEVER norm_dist_lower_Dn.
# That restriction is what makes band_width_pct a legal feature, because the
# identity is exact for any n:
#
#     ln(U/C) - ln(L/C) = ln(U/L) = band_width_pct
#
# A task holding BOTH distances plus band_width_pct would have a rank-deficient
# design matrix: infinite VIF, unidentifiable coefficients. Holding one side
# breaks the identity, so all three coexist safely. Do not re-admit the
# opposite side without also dropping band_width_pct.
#
# WHAT THE D1 BLOCK REALLY IS
# ---------------------------
#     norm_dist_upper_D1 = mu + k*sigma     (exact)
#     norm_dist_lower_D1 = mu - k*sigma     (exact)
# So including D1 hands the model the band's own location and width, not
# another day of price action. That is useful, but it should be stated in the
# write-up rather than discovered by a reader.
#
# WHAT IS DELIBERATELY ABSENT
# ---------------------------
#   dist_to_{side}_Dn   superseded by the log-distance above
#   days_left_Dn        constant within a task, so it cannot inform any model
#   sqrt_dl_frac_Dn     a one-to-one function of days_left, equally constant
# The last two are still COMPUTED (the Gaussian needs the arithmetic); they
# are simply not offered to any classifier.
# ============================================================================
_P1_DAY_BLOCK = ["norm_dist_{side}_D{d}", "realized_vol_D{d}", "gap_D{d}"]

P1_INCLUDE_BAND_WIDTH = True     # see the rank-deficiency note above


def project1_core_features(day, direction=None, include_band_width=None,
                           mode=None):
    """Feature names for one task: cumulative from D1, own side only.

    Parameters
    ----------
    day : int
        Decision day, 2, 3 or 4.
    direction : {"upper", "lower"}
        Which band edge this task predicts. REQUIRED — there is no default,
        because silently defaulting would hand an upper model the lower
        distance, which is exactly what the one-side rule removes.
    include_band_width : bool or None
        Append band_width_pct. Default P1_INCLUDE_BAND_WIDTH.
    mode : {"cumulative", "day_only"} or None
        Default P1_FEATURE_MODE. "day_only" returns just the decision day's
        block, for the strict identical-inputs comparison.

    Returns
    -------
    list of str
        Feature column names, in a stable order.

    Raises
    ------
    TypeError
        If direction is omitted.
    ValueError
        If direction or day is out of range.
    """
    if direction is None:
        raise TypeError(
            "project1_core_features() needs `direction` ('upper' or 'lower'). "
            "Each task carries only its own side's distance.")
    if direction not in ("upper", "lower"):
        raise ValueError(f"direction must be 'upper' or 'lower' — got {direction!r}")
    d = int(day)
    if d not in (2, 3, 4):
        raise ValueError(f"task day must be 2, 3 or 4 — got {day}")

    iw   = P1_INCLUDE_BAND_WIDTH if include_band_width is None else include_band_width
    mode = mode or P1_FEATURE_MODE
    first = 1 if mode == "cumulative" else d

    feats = []
    for n in range(first, d + 1):
        feats += [f.format(d=n, side=direction) for f in _P1_DAY_BLOCK]
    if iw:
        feats.append("band_width_pct")
    return feats


def get_task_features(task, all_available):
    """Whitelist gate. Hard-fails rather than silently training on less.

    Parameters
    ----------
    task : dict
        One entry of TASKS; needs "day", "direction" and "name".
    all_available : iterable of str
        Column names present on the frame.

    Returns
    -------
    list of str
        The task's feature names, all confirmed present and leakage-free.

    Raises
    ------
    RuntimeError
        If any required feature is missing, or if a leakage column somehow
        appears in the whitelist.
    """
    core = project1_core_features(task["day"], task["direction"])
    missing = [f for f in core if f not in set(all_available)]
    if missing:
        raise RuntimeError(
            f"❌ Missing feature(s) for {task['name']}: {missing}. "
            f"project1_add_time_features() builds all of them — check that it "
            f"ran, and that the frame carries band_upper / band_lower.")
    illegal = [f for f in core if f in LEAKAGE_FEATURES]
    if illegal:
        raise RuntimeError(f"❌ Leakage column in the whitelist: {illegal}")
    return list(core)


# ============================================================================
# K. CACHE KEYS
# ----------------------------------------------------------------------------
# CACHE_VERSION stamps every experiment decision, so switching a setting opens
# a clean cache world instead of silently mixing incompatible models. It is
# baked into the filename of every cached model AND compiled into the two
# filename regexes — change the string without recompiling them and
# S3_Harvest reports "no usable cache rows" even though training succeeded.
# ============================================================================
_V_FEAT = "C" if P1_FEATURE_MODE == "cumulative" else "D"
_V_WIN  = "W" + ("F" if P1_WINDOW_GRID_MODE == "full" else "S")
_V_NN   = f"N1w{NN_H1}x{NN_H2}" if P1_ENABLE_NN else "N0"
_V_FLT  = {"principled": "FP", "moderate": "FM",
           "relaxed": "FR", "off": "FO"}[P1_FILTER_PRESET]
_V_MET  = "MB" if P1_PRIMARY_METRIC == "brier" else "MF"
_V_XF   = "X1" if P1_CROSSFIT_THRESHOLD else "X0"
_V_TUNE = "T1" if P1_TUNE_HYPERPARAMS else "T0"

# The trailing tag marks the v2 feature definitions (one side per task,
# log-distances, gap, band_width_pct). It is part of the version string
# because the v1 and v2 feature sets are not comparable.
_V_FEATSET = "V2SB"

CACHE_VERSION = (f"v7{_V_FEAT}{_V_WIN}{_V_NN}{_V_FLT}{_V_MET}{_V_XF}{_V_TUNE}"
                 f"_{_V_FEATSET}")

# --- THE PIN ----------------------------------------------------------------
# The trained model cache on disk carries this exact string. If this assert
# fires, a decision in section A changed and every model would have to be
# retrained. That is sometimes what you want — but never by accident.
CACHE_VERSION_PIN = "v7CWFN1w4x2FMMBX1T1_V2SB"
assert CACHE_VERSION == CACHE_VERSION_PIN, (
    f"❌ CACHE_VERSION changed.\n"
    f"   expected : {CACHE_VERSION_PIN}\n"
    f"   computed : {CACHE_VERSION}\n"
    f"   A switch in section A was edited. Every cached model becomes\n"
    f"   unreadable and the full grid must be retrained. If that is intended,\n"
    f"   update CACHE_VERSION_PIN to the computed value.")

# --- THE GRID FINGERPRINT ---------------------------------------------------
# CACHE_VERSION records WHICH window MODE is active, not which windows are in
# the grid. With P1_WINDOW_GRID_MODE = "full" but only [4] actually swept, a
# later widening to [4, 8, 12, 16] would produce the SAME version string and
# silently mix single-window models with multi-window ones. This fingerprint
# closes that hole: widen the grid and this fails, telling you to bump the pin.
GRID_FINGERPRINT     = f"S{'|'.join(f'{s:.2f}' for s in SIGMA_GRID)}" \
                       f"_W{'|'.join(str(w) for w in SIGMA_WINDOW_GRID)}"
# GRID_FINGERPRINT_PIN = "S0.50_W4"
GRID_FINGERPRINT_PIN = "S0.50|0.75|1.00_W4|8"
assert GRID_FINGERPRINT == GRID_FINGERPRINT_PIN, (
    f"❌ The sigma/window grid changed but CACHE_VERSION did not.\n"
    f"   expected : {GRID_FINGERPRINT_PIN}\n"
    f"   computed : {GRID_FINGERPRINT}\n"
    f"   Models trained on the old grid share a cache version with the new\n"
    f"   ones and would be mixed without warning. Retrain into a fresh cache\n"
    f"   world: bump CACHE_VERSION_PIN, then update GRID_FINGERPRINT_PIN.")


def cache_key(task_name, model_name, n_features, sigma, window):
    """Filesystem path for one cached sklearn-family model.

    Parameters
    ----------
    task_name : str
        e.g. "upper_D3".
    model_name : str
        e.g. "LogisticRegression". Never "NN_FF" — use nn_meta_key.
    n_features : int
        The ABLATION BUDGET, not the count used. See ABLATION_FEATURE_COUNTS.
    sigma : float
        Band multiplier, written to 2 decimal places.
    window : int
        Rolling window length in cycles.

    Returns
    -------
    str
        Absolute path under MODEL_CACHE_DIR.
    """
    return os.path.join(
        MODEL_CACHE_DIR,
        f"{task_name}__{model_name}__feat{int(n_features)}"
        f"__sigma{float(sigma):.2f}__win{int(window)}__{CACHE_VERSION}_{_WORLD}.pkl")


def nn_meta_key(task_name, n_features, sigma, window):
    """Path for a neural net's metadata cache.

    Parameters
    ----------
    task_name, n_features, sigma, window
        As for cache_key.

    Returns
    -------
    str
        Absolute path, prefixed "nn__" so the sklearn regex cannot match it.
    """
    return os.path.join(
        MODEL_CACHE_DIR,
        f"nn__{task_name}__meta__feat{int(n_features)}"
        f"__sigma{float(sigma):.2f}__win{int(window)}__{CACHE_VERSION}_{_WORLD}.pkl")


def nn_seed_key(task_name, seed, n_features, sigma, window):
    """Path for one seed's torch state dict.

    Parameters
    ----------
    task_name : str
    seed : int
        The seed that produced this network.
    n_features, sigma, window
        As for cache_key.

    Returns
    -------
    str
        Absolute path with a .pt extension.
    """
    return os.path.join(
        MODEL_CACHE_DIR,
        f"nn__{task_name}__seed{int(seed)}__feat{int(n_features)}"
        f"__sigma{float(sigma):.2f}__win{int(window)}__{CACHE_VERSION}_{_WORLD}.pt")


# The negative lookahead is load-bearing: without it the non-greedy groups
# match "nn__upper_D2__meta__..." with task="nn", collapsing all six NN tasks
# onto one dedup key so five are silently discarded during harvest.
CACHE_FILENAME_RE = re.compile(
    r"^(?!nn__)(?P<task>.+?)__(?P<model>.+?)__feat(?P<feat>\d+)"
    r"__sigma(?P<sigma>[\d.]+)__win(?P<win>\d+)__" + re.escape(CACHE_VERSION)
    + r"_" + _WORLD + r"\.pkl$")

NN_META_FILENAME_RE = re.compile(
    r"^nn__(?P<task>.+?)__meta__feat(?P<feat>\d+)"
    r"__sigma(?P<sigma>[\d.]+)__win(?P<win>\d+)__" + re.escape(CACHE_VERSION)
    + r"_" + _WORLD + r"\.pkl$")


def parse_cache_filename(fn):
    """Recover identifiers from a cache filename.

    Parameters
    ----------
    fn : str
        Basename only, not a full path.

    Returns
    -------
    dict or None
        {"task", "model", "n_features", "sigma", "window"}, or None if the
        name does not belong to the current CACHE_VERSION.

    Notes
    -----
    The filename is authoritative — it is checked before the pickle is opened,
    so a corrupt or stale dict cannot misreport what a file contains.
    """
    m = NN_META_FILENAME_RE.match(fn)
    if m:
        g = m.groupdict()
        return {"task": g["task"], "model": "NN_FF", "n_features": int(g["feat"]),
                "sigma": round(float(g["sigma"]), 2), "window": int(g["win"])}
    m = CACHE_FILENAME_RE.match(fn)
    if m:
        g = m.groupdict()
        return {"task": g["task"], "model": g["model"], "n_features": int(g["feat"]),
                "sigma": round(float(g["sigma"]), 2), "window": int(g["win"])}
    return None


# ============================================================================
# L. ROLLING STATISTICS
# ============================================================================
def compute_d1_expiry_return_series(df_cyc):
    """Per-cycle log return from D1's close to the expiry close.

    Parameters
    ----------
    df_cyc : pandas.DataFrame
        Needs "d1_close" and "expiry_close".

    Returns
    -------
    pandas.Series
        ln(expiry_close / d1_close), with infinities replaced by NaN.
    """
    return np.log(df_cyc["expiry_close"] / df_cyc["d1_close"]) \
             .replace([np.inf, -np.inf], np.nan)


def rolling_mu_sigma(ret_series, window):
    """Rolling mean and standard deviation of past cycles' returns.

    Parameters
    ----------
    ret_series : pandas.Series
        Per-cycle D1→expiry log returns.
    window : int
        Window length in cycles.

    Returns
    -------
    (pandas.Series, pandas.Series)
        (mu, sigma), each shifted one cycle so the current cycle's own outcome
        can never enter its own band.

    Notes
    -----
    min_periods is a real fraction of the window (see sigma_min_periods), and
    the warm-up stays NaN rather than being filled with a fabricated sigma.
    Remove those rows with drop_warmup_cycles.
    """
    mp = sigma_min_periods(window)
    mu = ret_series.rolling(window, min_periods=mp).mean().shift(1)
    sg = ret_series.rolling(window, min_periods=mp).std().shift(1)
    return mu, sg


def drop_warmup_cycles(dc, verbose=True):
    """Remove cycles whose rolling band statistics are undefined.

    Parameters
    ----------
    dc : pandas.DataFrame
        Frame carrying mu_upper / mu_lower / sigma_*_rolling.
    verbose : bool
        Print how many rows were dropped.

    Returns
    -------
    pandas.DataFrame
        Only rows with finite, strictly positive rolling statistics.
    """
    need = [c for c in ("mu_upper", "mu_lower",
                        "sigma_upper_rolling", "sigma_lower_rolling")
            if c in dc.columns]
    if not need:
        return dc
    ok = (dc[need].notna().all(axis=1)
          & (dc["sigma_upper_rolling"] > 0)
          & (dc["sigma_lower_rolling"] > 0))
    n_drop = int((~ok).sum())
    if verbose and n_drop:
        print(f"     ℹ️  warm-up: dropped {n_drop} cycle(s) with undefined rolling μ/σ")
    return dc[ok].copy().reset_index(drop=True)


# ============================================================================
# M. BAND ENGINE
# ----------------------------------------------------------------------------
# Under LABEL_MODE = "vs_mean":
#     band_upper = d1_close * exp(mu + k*sigma)
#     band_lower = d1_close * exp(mu - k*sigma)
# where mu and sigma are the rolling statistics of past cycles' D1→expiry log
# returns and k is the sigma multiplier from SIGMA_GRID.
#
# The band defines the LABEL, so changing sigma or window changes the target,
# not just the features. Base rates are therefore not comparable across
# configurations.
# ============================================================================
def compute_asymmetric_bands_and_labels(df_cyc, upper_sigma, upper_window,
                                        lower_sigma, lower_window,
                                        make_labels=True):
    """Build both band edges, the breach labels and the distance columns.

    Parameters
    ----------
    df_cyc : pandas.DataFrame
        One row per cycle, with d1..d4 closes and dates plus expiry_close.
    upper_sigma, lower_sigma : float
        Band multipliers for each edge. Equal for a symmetric band.
    upper_window, lower_window : int
        Rolling window lengths for each edge.
    make_labels : bool
        Compute upper_breach / lower_breach / no_breach. Set False for a live
        in-progress cycle whose expiry has not happened.

    Returns
    -------
    pandas.DataFrame
        A copy with mu_*, sigma_*_rolling, exp_mean_*, band_upper, band_lower,
        band_width_pct, dist_to_*_Dn, norm_dist_*_Dn, band_confidence_* and,
        if requested, the three label columns.

    Notes
    -----
    * Short holiday cycles are excluded from the rolling statistics so a
      four-day week cannot contaminate the volatility estimate.
    * A NaN band yields a NaN label, never 0. Warm-up rows must be dropped,
      not scored as "no breach".
    * Distances are NaN when undefined, never 0 — a zero distance would mean
      "price exactly at the band", the maximum-ambiguity value.
    * band_width_pct and norm_dist_*_Dn are REDEFINED in logs by
      p1_add_v2_features; this function's versions are intermediate.
    """
    dc  = df_cyc.copy()
    d1  = dc["d1_close"]
    exp = dc["expiry_close"]
    dc["d1_to_exp_return"] = compute_d1_expiry_return_series(dc)
    rets = dc["d1_to_exp_return"].copy()

    if "cycle_days_total" in dc.columns:
        rets_roll = rets.copy()
        rets_roll[dc["cycle_days_total"] < CYCLE_DECISION_DAYS] = np.nan
    else:
        rets_roll = rets

    mu_u, sg_u = rolling_mu_sigma(rets_roll, upper_window)
    mu_l, sg_l = rolling_mu_sigma(rets_roll, lower_window)
    dc["mu_upper"], dc["mu_lower"] = mu_u, mu_l
    dc["sigma_upper_rolling"], dc["sigma_lower_rolling"] = sg_u, sg_l
    dc["mu_rolling"]     = (mu_u + mu_l) / 2.0
    dc["sigma_rolling"]  = (sg_u + sg_l) / 2.0
    dc["exp_mean_upper"] = d1 * np.exp(mu_u)
    dc["exp_mean_lower"] = d1 * np.exp(mu_l)

    if LABEL_MODE == "vs_mean":
        dc["band_upper"] = d1 * np.exp(mu_u + upper_sigma * sg_u)
        dc["band_lower"] = d1 * np.exp(mu_l - lower_sigma * sg_l)
    else:
        dc["band_upper"] = d1 * np.exp(upper_sigma * sg_u)
        dc["band_lower"] = d1 * np.exp(-lower_sigma * sg_l)
    dc["band_width_pct"] = (dc["band_upper"] - dc["band_lower"]) / d1

    if make_labels:
        ub  = (exp > dc["band_upper"]).astype(float)
        lb  = (exp < dc["band_lower"]).astype(float)
        bad = dc["band_upper"].isna() | dc["band_lower"].isna() | exp.isna()
        dc["upper_breach"] = ub.mask(bad)
        dc["lower_breach"] = lb.mask(bad)
        dc["no_breach"] = (((dc["upper_breach"] == 0) & (dc["lower_breach"] == 0))
                           .astype(float).mask(bad))

    bup, blo = dc["band_upper"], dc["band_lower"]
    sgu_safe = sg_u.replace(0, np.nan)
    sgl_safe = sg_l.replace(0, np.nan)
    for dN, dcol in [("D1", "d1_close"), ("D2", "d2_close"),
                     ("D3", "d3_close"), ("D4", "d4_close")]:
        cN = pd.to_numeric(dc.get(dcol, pd.Series(np.nan, index=dc.index)),
                           errors="coerce")
        dc[f"dist_to_upper_{dN}"]   = (bup - cN) / cN
        dc[f"dist_to_lower_{dN}"]   = (cN - blo) / cN
        dc[f"norm_dist_upper_{dN}"] = (dc[f"dist_to_upper_{dN}"] / sgu_safe).clip(-10, 10)
        dc[f"norm_dist_lower_{dN}"] = (dc[f"dist_to_lower_{dN}"] / sgl_safe).clip(-10, 10)

    nvu = rets_roll.rolling(upper_window, min_periods=1).count().shift(1)
    nvl = rets_roll.rolling(lower_window, min_periods=1).count().shift(1)
    dc["band_confidence_upper"] = (nvu / upper_window).clip(0, 1)
    dc["band_confidence_lower"] = (nvl / lower_window).clip(0, 1)
    return dc


def compute_bands_and_labels(df_cyc, sigma, window, make_labels=True):
    """Symmetric-band convenience wrapper.

    Parameters
    ----------
    df_cyc : pandas.DataFrame
        One row per cycle.
    sigma : float
        Band multiplier, used for both edges.
    window : int
        Rolling window length, used for both edges.
    make_labels : bool
        Passed through.

    Returns
    -------
    pandas.DataFrame
        As compute_asymmetric_bands_and_labels.
    """
    return compute_asymmetric_bands_and_labels(df_cyc, sigma, window,
                                               sigma, window, make_labels)


# ============================================================================
# N. DERIVED FEATURES
# ----------------------------------------------------------------------------
# project1_add_time_features is the single entry point the training code
# calls. It produces the complete feature set in one pass:
#     time arithmetic → v2 log-distances → overnight gap
# so any frame carrying bands comes back model-ready.
# ============================================================================
def project1_daily_realized_vol(daily_df, rv_window_days):
    """Rolling standard deviation of daily log returns, shifted one day.

    Parameters
    ----------
    daily_df : pandas.DataFrame
        The clean daily file; needs DATE_COL and CLOSE_COL.
    rv_window_days : int
        Trading days in the rolling window.

    Returns
    -------
    pandas.DataFrame
        Two columns: DATE_COL and "p1_rv".

    Notes
    -----
    The shift(1) is what prevents look-ahead: the volatility read on day t uses
    returns up to t-1 only.
    """
    d  = daily_df.sort_values(DATE_COL).reset_index(drop=True).copy()
    lr = np.log(d[CLOSE_COL] / d[CLOSE_COL].shift(1))
    d["p1_rv"] = lr.shift(1).rolling(
        int(rv_window_days),
        min_periods=max(3, int(rv_window_days) // 2)).std()
    return d[[DATE_COL, "p1_rv"]]


def project1_add_realized_vol_Dn(dcb, daily_df, band_window,
                                 cycle_days=CYCLE_DECISION_DAYS, strict=False):
    """Attach realized_vol_D1..D4 for one band configuration.

    Parameters
    ----------
    dcb : pandas.DataFrame
        Cycle frame carrying d1_date..d4_date.
    daily_df : pandas.DataFrame
        The clean daily file.
    band_window : int
        Rolling window in CYCLES; the daily window is band_window * cycle_days.
    cycle_days : int
        Trading days per cycle. Default CYCLE_DECISION_DAYS.
    strict : bool
        Raise when a decision day has already traded but is absent from the
        daily file.

    Returns
    -------
    pandas.DataFrame
        A copy with realized_vol_D1..D4.

    Raises
    ------
    RuntimeError
        Under strict=True, when a traded decision date is missing from the
        daily file. Nothing is ever zero-filled.

    Notes
    -----
    The strict check is PER ROW PER DAY, and a day fails only when all three
    hold: the date exists, it is on or before the last day of the daily file
    (the future is not an error), and it is absent from the daily file
    entirely. A date that is present but whose volatility is NaN is warm-up and
    is allowed through to be dropped later.
    """
    rv_window_days = int(band_window) * int(cycle_days)
    rv    = project1_daily_realized_vol(daily_df, rv_window_days)
    rvmap = dict(zip(pd.to_datetime(rv[DATE_COL]).dt.normalize(), rv["p1_rv"]))

    out = dcb.copy()
    for n in (1, 2, 3, 4):
        dcol = f"d{n}_date"
        if dcol in out.columns:
            dts = pd.to_datetime(out[dcol], errors="coerce").dt.normalize()
            out[f"realized_vol_D{n}"] = dts.map(rvmap).astype(float)
        else:
            out[f"realized_vol_D{n}"] = np.nan

    if strict:
        _dd = pd.to_datetime(daily_df[DATE_COL], errors="coerce").dt.normalize()
        _last_daily = _dd.max()
        _have = set(_dd.dropna())
        bad = []
        for n in (1, 2, 3, 4):
            dcol = f"d{n}_date"
            if dcol not in out.columns:
                continue
            dts = pd.to_datetime(out[dcol], errors="coerce").dt.normalize()
            occurred = dts.notna() & (dts <= _last_daily)
            absent   = occurred & ~dts.isin(_have)
            if absent.any():
                miss = dts[absent]
                bad.append(f"D{n}: {int(absent.sum())} date(s) missing from the "
                           f"daily file, e.g. {miss.min().date()} .. {miss.max().date()}")
        if bad:
            raise RuntimeError(
                "❌ realized_vol could not be mapped — these decision days have "
                "already traded but are absent from the daily clean file:\n   "
                + "\n   ".join(bad)
                + f"\n   Daily file covers {_dd.min().date()} → {_last_daily.date()}. "
                  "Re-run S0_Fetch / S0_Cleanup so it spans every cycle date. "
                  "Do NOT zero-fill.")
    return out


def p1_infer_band_k(frame):
    """Recover the band multiplier k from a frame's own columns.

    Parameters
    ----------
    frame : pandas.DataFrame
        Needs band_upper, band_lower and sigma_upper_rolling.

    Returns
    -------
    float or None
        k = ln(band_upper / band_lower) / (2 * sigma), or None if no row has
        the columns needed.

    Raises
    ------
    RuntimeError
        If k is not constant across rows, which proves the frame mixes band
        configurations and must not be trained on as one block.

    Notes
    -----
    Deriving k rather than passing it in means the value can never disagree
    with the bands actually on the row, and makes the feature builder work
    unchanged across every configuration in a grid sweep.
    """
    u = pd.to_numeric(frame.get("band_upper"), errors="coerce")
    l = pd.to_numeric(frame.get("band_lower"), errors="coerce")
    s = pd.to_numeric(frame.get("sigma_upper_rolling"), errors="coerce")
    ok = u.notna() & l.notna() & s.notna() & (s > 0)
    if not ok.any():
        return None
    k = np.log(u[ok] / l[ok]) / (2.0 * s[ok])
    if float(k.max() - k.min()) > 1e-9:
        raise RuntimeError(
            f"❌ The band multiplier k is not constant on this frame "
            f"({k.min():.4f} … {k.max():.4f}) — it mixes configurations.")
    return float(k.iloc[0])


def p1_add_v2_features(dcb, band_sigma):
    """Collapse mu/sigma, then build band_width_pct and the log-distances.

    Parameters
    ----------
    dcb : pandas.DataFrame
        Frame with bands already computed.
    band_sigma : float
        The multiplier k, normally from p1_infer_band_k.

    Returns
    -------
    pandas.DataFrame
        A copy with mu, sigma, band_width_pct and norm_dist_{side}_D1..D4
        redefined in logs.

    Raises
    ------
    RuntimeError
        If the upper and lower statistics differ (an asymmetric band, which
        cannot be collapsed to one mu and one sigma), or if either identity
        check fails.

    Notes
    -----
    Two identities are asserted because they are cheap to check and expensive
    to get wrong:
        band_width_pct     == 2 * k * sigma
        norm_dist_upper_D1 == mu + k * sigma
    """
    out = dcb.copy()

    for a, b, name in (("mu_upper", "mu_lower", "mu"),
                       ("sigma_upper_rolling", "sigma_lower_rolling", "sigma")):
        if not np.allclose(pd.to_numeric(out[a], errors="coerce"),
                           pd.to_numeric(out[b], errors="coerce"),
                           equal_nan=True):
            raise RuntimeError(
                f"❌ {a} != {b}. The band is ASYMMETRIC at this configuration, "
                f"so collapsing to a single '{name}' would silently discard one "
                f"side. Keep both columns, or rerun with a symmetric (sigma, "
                f"window).")
    out["mu"]    = pd.to_numeric(out["mu_upper"], errors="coerce")
    out["sigma"] = pd.to_numeric(out["sigma_upper_rolling"], errors="coerce")

    with np.errstate(divide="ignore", invalid="ignore"):
        out["band_width_pct"] = np.log(out["band_upper"] / out["band_lower"])

    for n in (1, 2, 3, 4):
        c = pd.to_numeric(out.get(f"d{n}_close"), errors="coerce")
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"norm_dist_upper_D{n}"] = np.log(out["band_upper"] / c.where(c > 0))
            out[f"norm_dist_lower_D{n}"] = np.log(out["band_lower"] / c.where(c > 0))

    ok = out["band_width_pct"].notna()
    if ok.any():
        if not np.allclose(out.loc[ok, "band_width_pct"],
                           2.0 * float(band_sigma) * out.loc[ok, "sigma"]):
            raise RuntimeError("❌ ln(U/L) != 2*k*sigma — the band engine and "
                               "band_sigma disagree.")
        if not np.allclose(out.loc[ok, "norm_dist_upper_D1"],
                           out.loc[ok, "mu"] + float(band_sigma) * out.loc[ok, "sigma"]):
            raise RuntimeError("❌ norm_dist_upper_D1 != mu + k*sigma.")
    return out


def p1_daily_open_map(daily_df, date_col=None, open_col=None):
    """Map each date to that day's OPEN.

    Parameters
    ----------
    daily_df : pandas.DataFrame
        The clean daily file.
    date_col, open_col : str or None
        Column names. Default DATE_COL and OPEN_COL.

    Returns
    -------
    dict
        Normalised Timestamp → float open.

    Raises
    ------
    RuntimeError
        If the open column is absent — the gap feature cannot be built without it.
    """
    date_col = date_col or DATE_COL
    open_col = open_col or OPEN_COL
    if open_col not in daily_df.columns:
        raise RuntimeError(f"❌ '{open_col}' missing from the clean daily file; "
                           f"the gap feature needs the open.")
    d = daily_df.sort_values(date_col).reset_index(drop=True)
    return dict(zip(pd.to_datetime(d[date_col]).dt.normalize(),
                    pd.to_numeric(d[open_col], errors="coerce")))


def p1_prev_close_map(daily_df, date_col=None, close_col=None):
    """Map each date to the PREVIOUS trading day's close.

    Parameters
    ----------
    daily_df : pandas.DataFrame
        The clean daily file.
    date_col, close_col : str or None
        Column names. Default DATE_COL and CLOSE_COL.

    Returns
    -------
    dict
        Normalised Timestamp → float previous close.

    Notes
    -----
    Needed only for D1, whose "previous day" is the prior cycle's expiry and
    therefore is not any d{n-1}_close on the same row.
    """
    date_col  = date_col  or DATE_COL
    close_col = close_col or CLOSE_COL
    d = daily_df.sort_values(date_col).reset_index(drop=True)
    return dict(zip(pd.to_datetime(d[date_col]).dt.normalize(),
                    pd.to_numeric(d[close_col], errors="coerce").shift(1)))


def p1_add_gap_v2(dcb, daily_df, strict=False):
    """Attach gap_D1..D4 = ln(Open on Dn / Close on D(n-1)).

    Parameters
    ----------
    dcb : pandas.DataFrame
        Cycle frame with d1_date..d4_date and d1_close..d3_close.
    daily_df : pandas.DataFrame
        The clean daily file.
    strict : bool
        Raise if a row has a decision date but an unmapped gap.

    Returns
    -------
    pandas.DataFrame
        A copy with gap_D1..D4.

    Raises
    ------
    RuntimeError
        If d{n-1}_close is not the previous trading day's close for some row,
        which means D1..D4 are not consecutive trading days there.

    Notes
    -----
    For n = 2, 3, 4 the denominator is d{n-1}_close. For n = 1 there is no D0,
    so the previous TRADING day is used — the prior cycle's expiry close, i.e.
    the weekend gap carried into the new cycle. The two definitions are
    cross-checked on n = 2, 3, 4, which also proves the days are consecutive.
    """
    om = p1_daily_open_map(daily_df)
    pm = p1_prev_close_map(daily_df)
    out = dcb.copy()

    for n in (1, 2, 3, 4):
        dcol = f"d{n}_date"
        if dcol not in out.columns:
            out[f"gap_D{n}"] = np.nan
            continue
        dts  = pd.to_datetime(out[dcol], errors="coerce").dt.normalize()
        op   = dts.map(om).astype(float)
        prev = (pd.to_numeric(out[f"d{n-1}_close"], errors="coerce")
                if n > 1 else dts.map(pm).astype(float))
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"gap_D{n}"] = np.log(op / prev.where(prev > 0))

    for n in (2, 3, 4):
        if f"d{n}_date" not in out.columns:
            continue
        dts = pd.to_datetime(out[f"d{n}_date"], errors="coerce").dt.normalize()
        alt = np.log(dts.map(om).astype(float) / dts.map(pm).astype(float))
        bad = int((~np.isclose(alt, out[f"gap_D{n}"], equal_nan=True)).sum())
        if bad:
            raise RuntimeError(
                f"❌ gap_D{n}: d{n-1}_close is not the previous trading day's "
                f"close on {bad} row(s). D1..D4 are not consecutive trading "
                f"days there — check S2's cycle construction.")

    if strict:
        for n in (1, 2, 3, 4):
            if f"d{n}_date" not in out.columns:
                continue
            bad = out[f"d{n}_date"].notna() & out[f"gap_D{n}"].isna()
            if bool(bad.any()):
                raise RuntimeError(f"❌ gap_D{n} unmapped on {int(bad.sum())} "
                                   f"row(s) that have a date.")
    return out


def project1_add_time_features(dcb, cycle_steps=CYCLE_STEPS, overwrite=False,
                               daily_df=None):
    """Build the complete feature set on a frame that already carries bands.

    This is the single entry point the training and live-inference code calls.
    It runs three stages in order:

      1. TIME       days_left_Dn and sqrt_dl_frac_Dn. Computed for the
                    Gaussian's arithmetic; not offered to any classifier.
      2. DISTANCES  mu, sigma, band_width_pct and norm_dist_{side}_Dn,
                    redefined in logs by p1_add_v2_features. Skipped if the
                    frame has no bands.
      3. GAP        gap_Dn from the daily file.

    Parameters
    ----------
    dcb : pandas.DataFrame
        Cycle frame. Stages 2 and 3 require band_upper / band_lower.
    cycle_steps : int
        Steps from D1's close to expiry. Default CYCLE_STEPS.
    overwrite : bool
        Rebuild the time columns even when present.
    daily_df : pandas.DataFrame or None
        The clean daily file. None calls p1_daily(), which memoises the read.

    Returns
    -------
    pandas.DataFrame
        A copy carrying every column get_task_features can ask for.

    Notes
    -----
    days_left is cycle_steps + 1 - n, giving D1:4 D2:3 D3:2 D4:1. The
    denominator stays cycle_steps, so sqrt_dl_frac_D1 is exactly 1.0 — at D1's
    close the remaining horizon IS the horizon sigma was estimated over.
    """
    out = dcb.copy()

    # --- stage 1: time ------------------------------------------------------
    need = overwrite or any(f"days_left_D{n}" not in out.columns for n in (2, 3, 4))
    if need:
        if "expected_cycle_days" in out.columns:
            cs = pd.to_numeric(out["expected_cycle_days"], errors="coerce") \
                   .fillna(cycle_steps).astype(float)
        elif "cycle_days_total" in out.columns:
            cs = pd.to_numeric(out["cycle_days_total"], errors="coerce") \
                   .fillna(cycle_steps).astype(float)
        else:
            cs = pd.Series(float(cycle_steps), index=out.index)
        cs = cs.clip(lower=1.0)
        for n in (1, 2, 3, 4):
            if overwrite or f"days_left_D{n}" not in out.columns:
                dl = (cs + 1.0 - float(n)).clip(lower=0.0)
                out[f"days_left_D{n}"]    = dl
                out[f"sqrt_dl_frac_D{n}"] = np.sqrt((dl / cs).clip(lower=0.0))

    # --- stages 2 and 3: only for frames that carry bands -------------------
    if "band_upper" not in out.columns:
        return out
    k = p1_infer_band_k(out)
    if k is None:
        return out

    out = p1_add_v2_features(out, k)
    out = p1_add_gap_v2(out, p1_daily() if daily_df is None else daily_df)
    return out


# ============================================================================
# O. GAUSSIAN BASELINE
# ----------------------------------------------------------------------------
# The parameter-free contestant. At decision day n:
#
#     dist     = ln(band / Dn_close)
#     sa       = sigma_cycle * sqrt(days_left / CYCLE_STEPS)
#     P(upper) = 1 - Phi(dist / sa)
#     P(lower) =     Phi(dist / sa)
#
# Sanity anchor: at D1 with mu = 0 this returns 1 - Phi(k), the band's own
# quantile — asserted in section W.
# ============================================================================
NORMAL_FAIL_COUNTS = {}      # task name → cycles the Gaussian could not score


def normal_breach_probs(cycles_df, task, strict=None):
    """Analytical Gaussian P(breach) for one task.

    Parameters
    ----------
    cycles_df : pandas.DataFrame
        Cycle frame with bands, rolling sigma and the decision day's close.
    task : dict
        One entry of TASKS.
    strict : bool or None
        Raise when any cycle cannot be scored. Default P1_STRICT_NAN.

    Returns
    -------
    ndarray of float
        One probability per row; NaN where the inputs are unusable.

    Raises
    ------
    RuntimeError
        Under strict=True, if any cycle could not be scored.

    Notes
    -----
    Failures return NaN, never a confident 0.0 — a zero here would read as
    "certain no breach" and quietly flatter the baseline. The horizon prefers
    expected_cycle_days over cycle_days_total, because for an IN-PROGRESS live
    cycle the latter is days ELAPSED and would score the wrong horizon.
    """
    strict = P1_STRICT_NAN if strict is None else bool(strict)
    tdir, tday = task["direction"], int(task["day"])
    n = len(cycles_df)
    if n == 0:
        return np.array([], float)

    sig_col  = "sigma_upper_rolling" if tdir == "upper" else "sigma_lower_rolling"
    band_col = "band_upper" if tdir == "upper" else "band_lower"

    def _num(col, default=np.nan):
        if col in cycles_df.columns:
            return pd.to_numeric(cycles_df[col], errors="coerce").to_numpy(float)
        return np.full(n, default, float)

    band = _num(band_col)
    sg   = _num(sig_col)
    ref  = _num(f"d{tday}_close")

    cs = _num("expected_cycle_days")
    cs = np.where(np.isfinite(cs) & (cs > 0), cs, _num("cycle_days_total"))
    cs = np.where(np.isfinite(cs) & (cs > 0), cs, float(CYCLE_STEPS))

    dl = cs + 1.0 - float(tday)
    with np.errstate(divide="ignore", invalid="ignore"):
        sa = sg * np.sqrt(np.clip(dl / cs, 0.0, None))
        z  = np.log(band / ref) / sa
        p  = (1.0 - _norm_cdf(z)) if tdir == "upper" else _norm_cdf(z)

    valid = (np.isfinite(band) & np.isfinite(ref) & (ref > 0)
             & np.isfinite(sg) & (sg > 0) & np.isfinite(dl) & (dl > 0)
             & np.isfinite(p))
    p = np.where(valid, p, np.nan)

    n_bad = int((~valid).sum())
    NORMAL_FAIL_COUNTS[task["name"]] = n_bad
    if n_bad:
        msg = (f"normal_breach_probs({task['name']}): {n_bad}/{n} cycles could "
               f"not be scored (NaN band/ref/sigma, or sigma<=0). Returned as "
               f"NaN, never as a confident 0.0. Drop warm-up cycles with "
               f"drop_warmup_cycles() before scoring.")
        if strict:
            raise RuntimeError("❌ " + msg)
        print("     ⚠️  " + msg)
    return p


_ERF = np.vectorize(math.erf, otypes=[float])


def _norm_cdf(z):
    """Standard normal CDF — vectorised, NaN-safe, empty-safe.

    Parameters
    ----------
    z : array-like of float
        Standardised values; non-finite entries are passed through as NaN.

    Returns
    -------
    ndarray of float
        Phi(z), using scipy when available and math.erf otherwise.
    """
    z = np.asarray(z, float)
    out = np.full(z.shape, np.nan)
    ok = np.isfinite(z)
    if not ok.any():
        return out
    if scipy_stats is not None:
        out[ok] = scipy_stats.norm.cdf(z[ok])
    else:
        out[ok] = 0.5 * (1.0 + _ERF(z[ok] / math.sqrt(2.0)))
    return out


# ============================================================================
# P. METRICS
# ============================================================================
def _f1_from_counts(tp, fp, fn):
    """F1 from raw confusion counts.

    Parameters
    ----------
    tp, fp, fn : int
        True positives, false positives, false negatives.

    Returns
    -------
    float
        2*tp / (2*tp + fp + fn), or 0.0 when the denominator is zero.
    """
    den = 2 * tp + fp + fn
    return 0.0 if den == 0 else (2.0 * tp) / den


def f1_at(y, p, thr):
    """F1 of a probability vector at a given threshold.

    Parameters
    ----------
    y : array-like of int
        True labels.
    p : array-like of float
        Predicted probabilities.
    thr : float
        Decision cut; predictions are p >= thr.

    Returns
    -------
    float
        The F1 score.
    """
    yp = (np.asarray(p, float) >= thr).astype(int)
    yt = np.asarray(y, int)
    tp = int(((yp == 1) & (yt == 1)).sum())
    fp = int(((yp == 1) & (yt == 0)).sum())
    fn = int(((yp == 0) & (yt == 1)).sum())
    return _f1_from_counts(tp, fp, fn)


def best_f1_threshold(y_true, probs, fallback=None):
    """F1-optimal decision threshold, chosen for out-of-sample stability.

    Parameters
    ----------
    y_true : array-like of int
        True labels.
    probs : array-like of float
        Predicted probabilities. Non-finite entries are dropped.
    fallback : float or None
        Returned as the threshold when one cannot be estimated.

    Returns
    -------
    (float or None, float)
        (threshold, F1 at that threshold). Returns (fallback, 0.0) when the
        label has fewer than two positives or is all-positive — callers must
        handle None rather than receive a silent 0.5.

    Notes
    -----
    Candidates are MIDPOINTS between the observed probabilities plus the two
    outside-range endpoints, so a cut is found even when every probability
    falls outside a fixed grid. Among all thresholds achieving the maximum F1,
    the midpoint of the WIDEST tied run is returned — at these sample sizes
    the tied region is often wide, and its edge is the least stable choice.

    BOTH contestants must use this function. A difference in tie-break policy
    between ML and the Gaussian would silently reintroduce an asymmetry.
    """
    yt = np.asarray(y_true, int)
    p  = np.asarray(probs, float)
    ok = np.isfinite(p)
    yt, p = yt[ok], p[ok]
    if len(yt) == 0 or yt.sum() < 2 or yt.sum() == len(yt):
        return (fallback, 0.0)

    u = np.unique(p)
    if len(u) == 1:
        return (fallback if fallback is not None else float(u[0]),
                f1_at(yt, p, u[0]))
    cand = np.concatenate([[u[0] - 1e-9], (u[:-1] + u[1:]) / 2.0, [u[-1] + 1e-9]])

    order = np.argsort(-p)
    ys = yt[order]
    tp_cum = np.cumsum(ys)
    fp_cum = np.cumsum(1 - ys)
    tot_pos = int(yt.sum())
    scores = np.empty(len(cand))
    for i, t in enumerate(cand):
        k = int(np.searchsorted(-p[order], -t, side="right"))
        tp = int(tp_cum[k - 1]) if k > 0 else 0
        fp = int(fp_cum[k - 1]) if k > 0 else 0
        scores[i] = _f1_from_counts(tp, fp, tot_pos - tp)

    best = scores.max()
    tied = np.flatnonzero(scores >= best - 1e-12)
    runs, s = [], tied[0]
    for a, b in zip(tied[:-1], tied[1:]):
        if b != a + 1:
            runs.append((s, a))
            s = b
    runs.append((s, tied[-1]))
    lo, hi = max(runs, key=lambda r: r[1] - r[0])
    return float(cand[(lo + hi) // 2]), float(best)


def brier_score(y, p):
    """Mean squared error of a probability forecast. Lower is better.

    Parameters
    ----------
    y : array-like of float
        True labels, 0 or 1.
    p : array-like of float
        Predicted probabilities.

    Returns
    -------
    float
        The Brier score, or NaN if nothing is finite.
    """
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    return float(np.mean((p[ok] - y[ok]) ** 2)) if ok.any() else np.nan


def log_loss_safe(y, p, eps=1e-12):
    """Binary cross-entropy with probabilities clipped away from 0 and 1.

    Parameters
    ----------
    y : array-like of float
        True labels.
    p : array-like of float
        Predicted probabilities.
    eps : float
        Clipping bound.

    Returns
    -------
    float
        Mean log loss, or NaN if nothing is finite.
    """
    y = np.asarray(y, float)
    p = np.clip(np.asarray(p, float), eps, 1 - eps)
    ok = np.isfinite(y) & np.isfinite(p)
    if not ok.any():
        return np.nan
    return float(-np.mean(y[ok] * np.log(p[ok]) + (1 - y[ok]) * np.log(1 - p[ok])))


def brier_skill_score(y, p):
    """Brier score relative to always predicting the base rate.

    Parameters
    ----------
    y : array-like of float
        True labels.
    p : array-like of float
        Predicted probabilities.

    Returns
    -------
    float
        1 - BS/BS_climatology. 1 is perfect, 0 is no better than the base
        rate, negative is worse than the base rate.

    Notes
    -----
    This is the primary metric when P1_PRIMARY_METRIC = "brier". It is
    comparable across tasks with different base rates, which raw Brier is not
    — and the base rate DOES change with sigma, because the band is the label.
    """
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    if not ok.any():
        return np.nan
    y, p = y[ok], p[ok]
    base = float(y.mean())
    bs_ref = base * (1 - base)
    if bs_ref < 1e-12:
        return np.nan
    return float(1.0 - brier_score(y, p) / bs_ref)


def brier_decomposition(y, p, n_bins=10):
    """Murphy decomposition: BS = reliability - resolution + uncertainty.

    Parameters
    ----------
    y : array-like of float
        True labels.
    p : array-like of float
        Predicted probabilities.
    n_bins : int
        Equal-width probability bins.

    Returns
    -------
    dict
        {"brier", "reliability", "resolution", "uncertainty"}.

    Notes
    -----
    This separates the competing explanations for a tie. Large RELIABILITY
    means the Gaussian is systematically mis-calibrated, so exploitable
    structure exists and ML has room. Low RESOLUTION for both with reliability
    near zero means the inputs are at their Bayes floor and no model
    restricted to them can win.
    """
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    y, p = y[ok], p[ok]
    if len(y) == 0:
        return dict(brier=np.nan, reliability=np.nan,
                    resolution=np.nan, uncertainty=np.nan)
    ybar = float(y.mean())
    N = len(y)
    edges = np.linspace(0, 1, n_bins + 1)
    edges[-1] += 1e-9
    rel = res = 0.0
    for k in range(n_bins):
        m = (p >= edges[k]) & (p < edges[k + 1])
        nk = int(m.sum())
        if nk == 0:
            continue
        rel += nk / N * (p[m].mean() - y[m].mean()) ** 2
        res += nk / N * (y[m].mean() - ybar) ** 2
    return dict(brier=brier_score(y, p), reliability=float(rel),
                resolution=float(res), uncertainty=float(ybar * (1 - ybar)))


def reliability_table(y, p, n_bins=10):
    """Per-bin observed versus predicted frequency, for a calibration chart.

    Parameters
    ----------
    y : array-like of float
        True labels.
    p : array-like of float
        Predicted probabilities.
    n_bins : int
        Equal-width probability bins.

    Returns
    -------
    pandas.DataFrame
        One row per non-empty bin: bin_lo, bin_hi, n, mean_pred, obs_freq.
    """
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    ok = np.isfinite(y) & np.isfinite(p)
    y, p = y[ok], p[ok]
    edges = np.linspace(0, 1, n_bins + 1)
    edges[-1] += 1e-9
    rows = []
    for k in range(n_bins):
        m = (p >= edges[k]) & (p < edges[k + 1])
        if not m.any():
            continue
        rows.append({"bin_lo": edges[k], "bin_hi": min(edges[k + 1], 1.0),
                     "n": int(m.sum()), "mean_pred": float(p[m].mean()),
                     "obs_freq": float(y[m].mean())})
    return pd.DataFrame(rows)


PRIMARY_METRIC_NAME = ("Brier skill score" if P1_PRIMARY_METRIC == "brier" else "F1")


def primary_score(y_true, prob=None, pred=None):
    """The single higher-is-better scalar selected by P1_PRIMARY_METRIC.

    Parameters
    ----------
    y_true : array-like of int
        True labels.
    prob : array-like of float or None
        Predicted probabilities. Required when the metric is "brier".
    pred : array-like of int or None
        Hard predictions. Required when the metric is "f1".

    Returns
    -------
    float
        Brier skill score or F1, or NaN if the needed input is absent.

    Notes
    -----
    Used for model selection AND for the consistency gate, which is why the
    gate floors in FILTER_PRESETS are metric-specific.
    """
    if P1_PRIMARY_METRIC == "brier":
        if prob is None:
            return np.nan
        return brier_skill_score(y_true, prob)
    if pred is None:
        return np.nan
    return f1_at(y_true, np.asarray(pred, float), 0.5)


def all_scores(y_true, prob, pred):
    """Every metric at once — primary chosen by config, the rest for reporting.

    Parameters
    ----------
    y_true : array-like of int
        True labels.
    prob : array-like of float
        Predicted probabilities.
    pred : array-like of int
        Hard predictions.

    Returns
    -------
    dict
        primary, f1, brier, brier_skill, log_loss, reliability, resolution,
        uncertainty, base_rate and n.
    """
    d = brier_decomposition(y_true, prob)
    return {
        "primary":     primary_score(y_true, prob, pred),
        "f1":          f1_at(y_true, np.asarray(pred, float), 0.5),
        "brier":       d["brier"],
        "brier_skill": brier_skill_score(y_true, prob),
        "log_loss":    log_loss_safe(y_true, prob),
        "reliability": d["reliability"],
        "resolution":  d["resolution"],
        "uncertainty": d["uncertainty"],
        "base_rate":   float(np.nanmean(np.asarray(y_true, float))),
        "n":           int(len(y_true)),
    }


# ============================================================================
# Q. CONSISTENCY GATE
# ----------------------------------------------------------------------------
# A trained model is TRUSTED only if it generalises within the training era.
# The test block is never involved — this gate runs entirely on in-fold versus
# out-of-fold scores, so it cannot leak.
#
# Three conditions per fold, then a quorum across folds:
#     |in-fold − out-of-fold| <= MAX_SCORE_GAP
#     out-of-fold             >= MIN_OOF_SCORE
#     out-of-fold / in-fold   >= SCORE_RATIO_MIN
#     at least FOLD_PASS_MIN_FRACTION of folds pass ALL THREE individually
#
# The last line matters when reading a training log: a model whose MEAN gap
# looks fine can still fail because too few individual folds passed.
# ============================================================================
def consistency_pass(infold_score, oof_score):
    """Does one fold's (in-fold, out-of-fold) pair clear the gate?

    Parameters
    ----------
    infold_score : float
        Primary-metric score on the rows the model was fitted on.
    oof_score : float
        Primary-metric score on that fold's held-out rows.

    Returns
    -------
    bool
        True only if all three conditions hold. Non-finite input fails.
    """
    it = float(infold_score)
    v  = float(oof_score)
    if not (np.isfinite(it) and np.isfinite(v)):
        return False
    ratio = v / it if abs(it) > 1e-6 else (1.0 if v >= 0 else -1.0)
    return bool(abs(it - v) <= MAX_SCORE_GAP
                and v >= MIN_OOF_SCORE
                and ratio >= SCORE_RATIO_MIN)


def folds_consistency_pass(per_fold_pairs):
    """Apply the gate across every fold of one trained combination.

    Parameters
    ----------
    per_fold_pairs : iterable of (float, float)
        (in-fold, out-of-fold) score pairs, one per fold.

    Returns
    -------
    (bool, float)
        (passed, mean out-of-fold score). Passing requires both the per-fold
        quorum and a mean out-of-fold score above the floor.
    """
    pairs = [(a, b) for a, b in per_fold_pairs
             if np.isfinite(a) and np.isfinite(b)]
    if not pairs:
        return False, float("nan")
    passes   = [consistency_pass(it, v) for it, v in pairs]
    mean_oof = float(np.mean([v for _, v in pairs]))
    return bool(np.mean(passes) >= FOLD_PASS_MIN_FRACTION
                and mean_oof >= MIN_OOF_SCORE), mean_oof


# ============================================================================
# R. ENSEMBLE
# ----------------------------------------------------------------------------
# The reported probability and the decision must be the SAME quantity. If the
# decision came from a threshold-normalised margin while the returned value
# was a raw weighted mean, every downstream ROC, AUC and calibration curve
# would score something that did not drive the decision.
#
# "legacy_normalized" reproduces that older behaviour for comparison only and
# must not be used for a reported result.
# ============================================================================
ENSEMBLE_MODE = "consistent"        # "consistent" | "legacy_normalized"


def ensemble_predict(probs_list, thresholds_list, val_f1_list=None):
    """Weighted-probability combination of the top-k models for one task.

    Parameters
    ----------
    probs_list : sequence of float
        One probability per ensemble member.
    thresholds_list : sequence of float
        Each member's own decision threshold. Non-finite entries become 0.5.
    val_f1_list : sequence of float or None
        Weights, normally each member's out-of-fold score. None weights
        equally. Values are floored at 0.01.

    Returns
    -------
    (int or None, float, str, str)
        (prediction, probability, "votes/total", note). The prediction is
        exactly int(probability >= effective threshold).
    """
    if probs_list is None or len(probs_list) == 0:
        return None, np.nan, "0/0", "no_models"
    probs = np.asarray(probs_list, float)
    thrs  = np.asarray(thresholds_list, float)
    n = len(probs)
    thrs = np.where(np.isfinite(thrs), thrs, 0.5)

    w = (np.maximum(np.asarray(val_f1_list, float), 0.01)
         if (val_f1_list is not None and len(val_f1_list) == n) else np.ones(n))
    w = np.where(np.isfinite(w), w, 0.01)
    wsum = w.sum() if w.sum() > 0 else 1.0

    prob = float((probs * w).sum() / wsum)
    vote = f"{int((probs >= thrs).sum())}/{n}"

    if ENSEMBLE_MODE == "legacy_normalized":
        denom = np.maximum.reduce([thrs, 1 - thrs, np.full(n, 1e-3)])
        wavg  = float((((probs - thrs) / denom) * w).sum() / wsum)
        return int(wavg > 0), prob, vote, f"LEGACY norm_wavg={wavg:+.3f}"

    eff_thr = float((thrs * w).sum() / wsum)
    return int(prob >= eff_thr), prob, vote, f"thr={eff_thr:.3f}"


def ensemble_effective_threshold(thresholds_list, val_f1_list=None):
    """The weighted threshold ensemble_predict decides against.

    Parameters
    ----------
    thresholds_list : sequence of float
        Member thresholds.
    val_f1_list : sequence of float or None
        Member weights.

    Returns
    -------
    float
        The weighted mean threshold.
    """
    thrs = np.asarray(thresholds_list, float)
    thrs = np.where(np.isfinite(thrs), thrs, 0.5)
    w = (np.maximum(np.asarray(val_f1_list, float), 0.01)
         if (val_f1_list is not None and len(val_f1_list) == len(thrs))
         else np.ones(len(thrs)))
    return float((thrs * w).sum() / (w.sum() if w.sum() > 0 else 1.0))


# ============================================================================
# S. PREPROCESSING
# ----------------------------------------------------------------------------
# Winsorise at the TRAINING block's 1st/99th percentile, then RobustScale.
# Both are fitted on training rows only and re-fitted inside every fold, so no
# information about a validation or test row can reach the transform.
# ============================================================================
try:
    from sklearn.preprocessing import RobustScaler
except Exception:
    RobustScaler = None


def clip_and_scale(Xtr, Xvl, Xte, feature_names):
    """Fit winsorising bounds and a RobustScaler on train; apply to all three.

    Parameters
    ----------
    Xtr, Xvl, Xte : ndarray
        Train, validation and test matrices. MUTATED IN PLACE — every call
        site must pass copies.
    feature_names : sequence of str
        Column names, used to honour NO_SCALE_FEATURES.

    Returns
    -------
    (ndarray, ndarray, ndarray, object, list, list)
        The three transformed matrices, the fitted scaler (or None), the
        per-column (lo, hi) clip bounds, and the indices left unscaled.
    """
    nf = Xtr.shape[1]
    no_scale  = [i for i, f in enumerate(feature_names) if f in NO_SCALE_FEATURES]
    scale_idx = [i for i in range(nf) if i not in no_scale]
    clip_bounds = []
    for ci in range(nf):
        if ci in no_scale:
            clip_bounds.append(None)
            continue
        col = Xtr[:, ci]
        fin = col[np.isfinite(col)]
        if fin.size == 0:
            clip_bounds.append(None)
            continue
        lo = float(np.percentile(fin, 1))
        hi = float(np.percentile(fin, 99))
        for X in (Xtr, Xvl, Xte):
            X[:, ci] = np.clip(X[:, ci], lo, hi)
        clip_bounds.append((lo, hi))
    scaler = RobustScaler() if RobustScaler is not None else None
    if scale_idx and scaler is not None:
        Xtr[:, scale_idx] = scaler.fit_transform(Xtr[:, scale_idx])
        Xvl[:, scale_idx] = scaler.transform(Xvl[:, scale_idx])
        Xte[:, scale_idx] = scaler.transform(Xte[:, scale_idx])
    return Xtr, Xvl, Xte, scaler, clip_bounds, no_scale


def apply_saved_pipeline(X_raw, clip_bounds, scaler, no_scale_idx,
                         strict=None, feature_names=None):
    """Apply a stored clip-and-scale pipeline to new rows.

    Parameters
    ----------
    X_raw : array-like
        Raw feature matrix.
    clip_bounds : list
        Per-column (lo, hi) or None, as returned by clip_and_scale.
    scaler : object or None
        The fitted scaler.
    no_scale_idx : iterable of int
        Column indices to leave unscaled.
    strict : bool or None
        Raise on non-finite input instead of zero-filling. Default
        P1_STRICT_NAN.
    feature_names : sequence of str or None
        Used to name offending columns in the error message.

    Returns
    -------
    ndarray of float32
        The transformed matrix.

    Raises
    ------
    RuntimeError
        Under strict=True, when any value is non-finite. Zero-filling a
        missing volatility would make it exactly zero — a confident, wrong
        number — including at live inference.
    """
    strict = P1_STRICT_NAN if strict is None else bool(strict)
    X = np.asarray(X_raw, dtype=np.float64).copy()
    bad = ~np.isfinite(X)
    if bad.any():
        if strict:
            cols = sorted(set(np.where(bad)[1].tolist()))
            names = ([feature_names[c] for c in cols if c < len(feature_names)]
                     if feature_names else cols)
            raise RuntimeError(
                f"❌ Non-finite values in {int(bad.sum())} cell(s), column(s) "
                f"{names}. Refusing to zero-fill — fix the source.")
        X = np.nan_to_num(X, nan=0., posinf=0., neginf=0.)
    for ci in range(X.shape[1]):
        if ci < len(clip_bounds) and clip_bounds[ci] is not None:
            lo, hi = clip_bounds[ci]
            X[:, ci] = np.clip(X[:, ci], lo, hi)
    scale_idx = [i for i in range(X.shape[1]) if i not in set(no_scale_idx or [])]
    if scale_idx and scaler is not None:
        X[:, scale_idx] = scaler.transform(X[:, scale_idx])
    return X.astype(np.float32)


# ============================================================================
# T. SELECTION HELPER
# ============================================================================
def _score_col(df):
    """Name of the selection-score column in a ledger frame.

    Parameters
    ----------
    df : pandas.DataFrame
        The ledger.

    Returns
    -------
    str
        "oof_score", or a legacy alias if that is what the frame carries.

    Raises
    ------
    KeyError
        If no recognised score column is present.
    """
    for c in ("oof_score", "val_f1", "mean_val_f1"):
        if c in df.columns:
            return c
    raise KeyError("no selection-score column (oof_score / val_f1) in ledger")


def _infold_col(df):
    """Name of the in-fold score column, or None.

    Parameters
    ----------
    df : pandas.DataFrame
        The ledger.

    Returns
    -------
    str or None
        The first recognised in-fold column name.
    """
    for c in ("infold_score", "inner_train_f1", "mean_inner_train_f1"):
        if c in df.columns:
            return c
    return None


def select_top_k(subset_df, k=None):
    """Rank one task's candidates and keep the best k.

    Parameters
    ----------
    subset_df : pandas.DataFrame
        Ledger rows for a single (task, sigma, window), already filtered by
        the consistency gate.
    k : int or None
        How many to keep. Default TOP_K.

    Returns
    -------
    pandas.DataFrame
        The top k rows, ordered best first.

    Notes
    -----
    Sort keys are out-of-fold score (descending), then the smallest
    |in-fold − out-of-fold| gap, then fewest features. The TEST score is never
    a sort key anywhere in selection.
    """
    k = k or TOP_K
    df = subset_df.copy()
    sc = _score_col(df)
    ic = _infold_col(df)
    df["_gap"] = (df[ic] - df[sc]).abs() if ic else 0.0
    if "n_features" not in df.columns:
        df["n_features"] = 0
    return (df.sort_values([sc, "_gap", "n_features"],
                           ascending=[False, True, True])
              .drop(columns=["_gap"]).head(k))


# ============================================================================
# U. NEURAL NETWORK
# ============================================================================
if _HAS_TORCH:
    class NiftyBinaryFF(nn.Module):
        """Small feed-forward binary classifier.

        Architecture
        ------------
            Linear(nf, h1) → ReLU → BatchNorm1d(h1) → Dropout
            Linear(h1, h2) → ReLU → Dropout
            Linear(h2, 2)

        Parameters
        ----------
        nf : int
            Number of input features.
        do : float
            Dropout probability. Default DROPOUT.
        h1, h2 : int
            Hidden widths. Defaults NN_H1 and NN_H2, so the untuned fallback
            path cannot silently rebuild a wider network behind a pinned search.

        Notes
        -----
        The h2 = 2 bottleneck is the binding constraint: everything the inputs
        contain is compressed to two numbers before the output layer. With
        ReLU, a dead unit collapses the network to a single monotone score —
        essentially logistic regression with more variance.
        """

        def __init__(self, nf, do=DROPOUT, h1=NN_H1, h2=NN_H2):
            super().__init__()
            h1, h2 = int(h1), int(h2)
            self.net = nn.Sequential(
                nn.Linear(nf, h1), nn.ReLU(), nn.BatchNorm1d(h1), nn.Dropout(do),
                nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(do),
                nn.Linear(h2, 2))

        def forward(self, x):
            """Logits for a batch.

            Parameters
            ----------
            x : torch.Tensor
                Shape (batch, nf).

            Returns
            -------
            torch.Tensor
                Shape (batch, 2).
            """
            return self.net(x)

        def predict_proba(self, x):
            """Class probabilities, in eval mode and without gradients.

            Parameters
            ----------
            x : torch.Tensor
                Shape (batch, nf).

            Returns
            -------
            torch.Tensor
                Shape (batch, 2); column 1 is P(breach).
            """
            self.eval()
            with torch.no_grad():
                return torch.softmax(self.forward(x), dim=-1)
else:
    NiftyBinaryFF = None


# ============================================================================
# V. EXPIRY HELPERS
# ============================================================================
def is_expiry(date, cutoff, all_dates_set, last_date=None):
    """Is this date the expiry of its weekly cycle?

    Parameters
    ----------
    date : pandas.Timestamp
        The date to classify.
    cutoff : pandas.Timestamp
        Regime boundary; Thursday expiry before it, Tuesday on or after.
    all_dates_set : set of pandas.Timestamp
        Every trading date in the dataset, normalised.
    last_date : pandas.Timestamp or None
        The final trading date. Defaults to max(all_dates_set).

    Returns
    -------
    bool
        True if the date settles its cycle.

    Notes
    -----
    The holiday fallback promotes a day to expiry when the nominal expiry day
    is ABSENT from the dataset, reasoning that a missing trading day is a
    market holiday. That reasoning is sound for the past and wrong at the end
    of the file: days after the last observed date are missing because they
    have not happened yet. The absence test therefore applies only to dates at
    or before last_date — beyond it, absence proves nothing and the cycle
    stays open. Without this guard the final row of the file reads as an
    expiry and live inference reports "no in-progress cycle".
    """
    wd = date.weekday()
    ew, ep = ((NEW_EXPIRY_DAY, NEW_EXPIRY_DAY_PREV) if date >= cutoff
              else (EXPIRY_DAY, EXPIRY_DAY_PREV))
    if wd == ew:
        return True

    if last_date is None:
        last_date = max(all_dates_set) if all_dates_set else None

    def _was_holiday(d):
        """True only if d is a PAST calendar date absent from the data."""
        d = d.normalize()
        if last_date is not None and d > last_date:
            return False
        return d not in all_dates_set

    if wd == ep:
        nom = date + pd.Timedelta(days=(ew - wd) % 7)
        if _was_holiday(nom):
            return True
    tb = (ep - 1) % 5
    if wd == tb:
        nom  = date + pd.Timedelta(days=(ew - wd) % 7)
        prev = date + pd.Timedelta(days=(ep - wd) % 7)
        if _was_holiday(nom) and _was_holiday(prev):
            return True
    return False


def get_day_value(df_ne, day_num, col, default=np.nan):
    """Read one column from the nth row of a cycle's daily slice.

    Parameters
    ----------
    df_ne : pandas.DataFrame
        The days belonging to one cycle, in order.
    day_num : int
        1-based day index.
    col : str
        Column to read.
    default : float
        Returned when the column or the row is absent, or the value is NaN.

    Returns
    -------
    float
        The value, or default.
    """
    if col not in df_ne.columns:
        return default
    if len(df_ne) >= day_num:
        v = df_ne.iloc[day_num - 1][col]
        return float(v) if pd.notna(v) else default
    return default


# ============================================================================
# W. SELF-TESTS
# ----------------------------------------------------------------------------
# Every invariant this file claims is asserted here, at load time, so a broken
# assumption surfaces immediately rather than as a strange number twelve cells
# later.
# ============================================================================
def _self_test(verbose=True):
    """Assert every invariant in this file.

    Parameters
    ----------
    verbose : bool
        Print one line per check.

    Returns
    -------
    bool
        True if all checks pass.

    Raises
    ------
    AssertionError
        Listing every failure, if any check fails.
    """
    fails = []

    def chk(name, cond, detail=""):
        if not cond:
            fails.append(f"{name}: {detail}")
        if verbose:
            print(f"     {'✅' if cond else '❌'} {name}"
                  + (f"  — {detail}" if not cond else ""))

    # --- cycle geometry -----------------------------------------------------
    chk("days_left D1..D4 = 4/3/2/1",
        [days_left_for_day(n) for n in (1, 2, 3, 4)] == [4.0, 3.0, 2.0, 1.0],
        str([days_left_for_day(n) for n in (1, 2, 3, 4)]))
    chk("sqrt_dl_frac_D1 == 1.0 exactly",
        abs(sqrt_dl_frac_for_day(1) - 1.0) < 1e-12, f"{sqrt_dl_frac_for_day(1)!r}")
    chk("sqrt_dl_frac D2/D3/D4 = .866/.707/.500",
        all(abs(sqrt_dl_frac_for_day(n) - v) < 1e-3
            for n, v in [(2, 0.8660), (3, 0.7071), (4, 0.5)]))
    _tf = project1_add_time_features(
        pd.DataFrame({"expected_cycle_days": [CYCLE_STEPS] * 3}), overwrite=True)
    chk("project1_add_time_features matches the helper",
        list(_tf[[f"days_left_D{n}" for n in (1, 2, 3, 4)]].iloc[0]) == [4.0, 3.0, 2.0, 1.0],
        str(list(_tf[[f'days_left_D{n}' for n in (1, 2, 3, 4)]].iloc[0])))

    # --- feature whitelist --------------------------------------------------
    _sides = ("upper", "lower")
    _exp = {"cumulative": (7, 10, 13), "day_only": (4, 4, 4)}[P1_FEATURE_MODE]
    _got = tuple(len(project1_core_features(d, "upper")) for d in (2, 3, 4))
    chk(f"feature counts D2/D3/D4 for '{P1_FEATURE_MODE}' = {_exp}",
        _got == _exp, f"got {_got}")
    chk("upper tasks never see norm_dist_lower_*",
        not any("norm_dist_lower" in f for d in (2, 3, 4)
                for f in project1_core_features(d, "upper")))
    chk("lower tasks never see norm_dist_upper_*",
        not any("norm_dist_upper" in f for d in (2, 3, 4)
                for f in project1_core_features(d, "lower")))
    chk("band_width_pct IS a feature (safe under the one-side layout)",
        all("band_width_pct" in project1_core_features(d, s)
            for d in (2, 3, 4) for s in _sides))
    try:
        project1_core_features(4)
        _no_default = False
    except TypeError:
        _no_default = True
    chk("direction is required — no silent default", _no_default)
    chk("no task sees a LATER day",
        all(not any(f"_D{l}" in f for l in range(d + 1, 5))
            for d in (2, 3, 4) for s in _sides
            for f in project1_core_features(d, s)))
    chk("no leakage column is whitelisted",
        not any(f in LEAKAGE_FEATURES for d in (2, 3, 4) for s in _sides
                for f in project1_core_features(d, s)))
    chk("no dist_to_* / days_left / sqrt_dl_frac survives",
        not any(f.startswith(("dist_to_", "days_left", "sqrt_dl_frac"))
                for d in (2, 3, 4) for s in _sides
                for f in project1_core_features(d, s)))
    chk("every feature is scaled (NO_SCALE_FEATURES is empty)",
        not any(f in NO_SCALE_FEATURES for d in (2, 3, 4) for s in _sides
                for f in project1_core_features(d, s)))
    chk("every feature has a reporting class",
        all(feature_class(f) != "Unclassified" for d in (2, 3, 4) for s in _sides
            for f in project1_core_features(d, s)),
        str([f for d in (2, 3, 4) for s in _sides
             for f in project1_core_features(d, s)
             if feature_class(f) == "Unclassified"]))
    chk("ablation budget >= the largest feature set",
        min(ABLATION_FEATURE_COUNTS) >= max(
            len(project1_core_features(d, s)) for d in (2, 3, 4) for s in _sides),
        f"budget {ABLATION_FEATURE_COUNTS}")

    # --- gap arithmetic on a frame whose answer is known by hand -----------
    _d  = pd.date_range("2024-01-01", periods=6, freq="D")
    _dl = pd.DataFrame({DATE_COL: _d,
                        OPEN_COL:  [100., 102., 103., 101., 100., 105.],
                        CLOSE_COL: [100., 101., 100., 102., 103., 104.]})
    _cy = pd.DataFrame([{"d1_date": _d[1], "d2_date": _d[2], "d3_date": _d[3],
                         "d4_date": _d[4], "d1_close": 101., "d2_close": 100.,
                         "d3_close": 102., "d4_close": 103.}])
    _g = p1_add_gap_v2(_cy, _dl)
    chk("gap_D2 = ln(open_D2 / d1_close)",
        abs(float(_g.gap_D2.iloc[0]) - np.log(103. / 101.)) < 1e-12)
    chk("gap_D3 = ln(open_D3 / d2_close)",
        abs(float(_g.gap_D3.iloc[0]) - np.log(101. / 100.)) < 1e-12)
    chk("gap_D1 uses the previous TRADING day (no D0 exists)",
        abs(float(_g.gap_D1.iloc[0]) - np.log(102. / 100.)) < 1e-12)

    # --- band identities, on a synthetic frame ------------------------------
    _k, _sig, _mu = BAND_SIGMA, 0.02, 0.001
    _bf = pd.DataFrame({
        "d1_close": [100.0], "d2_close": [101.0],
        "d3_close": [99.0],  "d4_close": [100.5],
        "mu_upper": [_mu], "mu_lower": [_mu],
        "sigma_upper_rolling": [_sig], "sigma_lower_rolling": [_sig],
        "band_upper": [100.0 * math.exp(_mu + _k * _sig)],
        "band_lower": [100.0 * math.exp(_mu - _k * _sig)]})
    chk("p1_infer_band_k recovers k from the frame",
        abs(p1_infer_band_k(_bf) - _k) < 1e-9, f"got {p1_infer_band_k(_bf)}")
    _v2 = p1_add_v2_features(_bf, _k)
    chk("band_width_pct == 2*k*sigma",
        abs(float(_v2.band_width_pct.iloc[0]) - 2 * _k * _sig) < 1e-12)
    chk("norm_dist_upper_D1 == mu + k*sigma",
        abs(float(_v2.norm_dist_upper_D1.iloc[0]) - (_mu + _k * _sig)) < 1e-12)
    chk("norm_dist_upper_Dn - norm_dist_lower_Dn == band_width_pct",
        all(abs(float(_v2[f"norm_dist_upper_D{n}"].iloc[0])
                - float(_v2[f"norm_dist_lower_D{n}"].iloc[0])
                - float(_v2.band_width_pct.iloc[0])) < 1e-12 for n in (1, 2, 3, 4)))

    # --- realized_vol strict check is row-wise, not column-wide -------------
    _bd  = pd.bdate_range("2019-01-01", "2026-08-21")
    _dly = pd.DataFrame({DATE_COL: _bd,
                         CLOSE_COL: np.linspace(10000, 24000, len(_bd))})
    _live = pd.DataFrame([{"d1_date": pd.Timestamp("2026-08-19"),
                           "d2_date": pd.Timestamp("2026-08-20"),
                           "d3_date": pd.Timestamp("2026-08-21"),
                           "d4_date": pd.NaT}])
    try:
        _o = project1_add_realized_vol_Dn(_live, _dly, 16, strict=True)
        _live_ok, _rv4 = True, _o["realized_vol_D4"].iloc[0]
    except RuntimeError:
        _live_ok, _rv4 = False, None
    chk("a live in-progress cycle passes strict (D4 has not traded)",
        _live_ok, "strict check condemned a day that has not happened")
    chk("the un-traded D4 stays NaN — never zero-filled",
        (_rv4 is None) or bool(pd.isna(_rv4)))
    _warm = pd.DataFrame([{f"d{n}_date": _bd[n - 1] for n in (1, 2, 3, 4)}])
    try:
        project1_add_realized_vol_Dn(_warm, _dly, 16, strict=True)
        _warm_ok = True
    except RuntimeError:
        _warm_ok = False
    chk("rolling warm-up NaN is allowed (drop_warmup_cycles handles it)", _warm_ok)
    _gap  = pd.Timestamp("2023-07-04")
    _hole = _dly[_dly[DATE_COL] != _gap].reset_index(drop=True)
    _bad  = pd.DataFrame([{"d1_date": pd.Timestamp("2023-07-03"), "d2_date": _gap,
                           "d3_date": pd.Timestamp("2023-07-05"),
                           "d4_date": pd.Timestamp("2023-07-06")}])
    try:
        project1_add_realized_vol_Dn(_bad, _hole, 16, strict=True)
        _gap_raised = False
    except RuntimeError:
        _gap_raised = True
    chk("a genuinely missing traded date still raises", _gap_raised,
        "coverage gaps must not pass silently")

    # --- trailing-edge expiry guard ----------------------------------------
    _S  = set(pd.Timestamp(d).normalize()
              for d in pd.bdate_range("2019-01-01", "2026-08-21"))
    _fri = pd.Timestamp("2026-08-21")
    _tue = pd.Timestamp("2026-08-18")
    chk("the trailing Friday is NOT an expiry (the open cycle stays open)",
        is_expiry(_fri, CUTOFF_DATE, _S) is False,
        "the last row of the file was being read as an expiry")
    chk("a genuine Tuesday IS still an expiry",
        is_expiry(_tue, CUTOFF_DATE, _S) is True)
    _S2 = set(pd.Timestamp(d).normalize()
              for d in pd.bdate_range("2019-01-01", "2026-08-28"))
    _S2.discard(pd.Timestamp("2026-08-24"))
    _S2.discard(pd.Timestamp("2026-08-25"))
    chk("the holiday fallback still fires when Mon+Tue are truly closed",
        is_expiry(_fri, CUTOFF_DATE, _S2) is True)
    _S3 = set(pd.Timestamp(d).normalize()
              for d in pd.bdate_range("2019-01-01", "2026-08-24"))
    chk("a file ending on Monday does not make Monday an expiry",
        is_expiry(pd.Timestamp("2026-08-24"), CUTOFF_DATE, _S3) is False)

    # --- Gaussian baseline anchors -----------------------------------------
    _k2, _sig2 = 1.0, 0.02
    _syn = pd.DataFrame({
        "d1_close": [100.0], "d2_close": [100.0],
        "d3_close": [100.0], "d4_close": [100.0],
        "expected_cycle_days": [CYCLE_STEPS],
        "cycle_days_total": [CYCLE_DECISION_DAYS],
        "sigma_upper_rolling": [_sig2], "sigma_lower_rolling": [_sig2],
        "band_upper": [100.0 * math.exp(_k2 * _sig2)],
        "band_lower": [100.0 * math.exp(-_k2 * _sig2)]})
    _p1 = normal_breach_probs(_syn, {"name": "t", "direction": "upper", "day": 1},
                              strict=False)[0]
    chk("Gaussian at D1 recovers the band quantile 1-Phi(k)",
        abs(_p1 - (1 - 0.5 * (1 + math.erf(_k2 / math.sqrt(2))))) < 1e-6,
        f"got {_p1:.6f}")
    _p4 = normal_breach_probs(_syn, {"name": "t", "direction": "upper", "day": 4},
                              strict=False)[0]
    chk("Gaussian at D4 is finite and non-degenerate",
        np.isfinite(_p4) and 0.0 < _p4 < 0.5, f"got {_p4!r}")
    _badf = _syn.copy()
    _badf.loc[0, "sigma_upper_rolling"] = np.nan
    _pb = normal_breach_probs(_badf, {"name": "t2", "direction": "upper", "day": 2},
                              strict=False)[0]
    chk("an unscoreable cycle returns NaN (not 0.0)", np.isnan(_pb), f"got {_pb!r}")

    # --- threshold search ---------------------------------------------------
    _y = np.array([0, 0, 1, 0, 1, 1, 0, 1, 0, 0])
    _p = np.array([.01, .02, .03, .011, .035, .04, .015, .05, .012, .005])
    _t, _f = best_f1_threshold(_y, _p)
    chk("finds a cut even when every probability is below 0.05",
        _t is not None and _f > 0, f"thr={_t} f1={_f}")
    _t2, _f2 = best_f1_threshold(_y, _p * 3.0)
    chk("F1 is invariant under a monotone rescale",
        abs(_f - _f2) < 1e-12, f"{_f} vs {_f2}")
    chk("refuses with fewer than 2 positives",
        best_f1_threshold(np.array([0, 0, 0, 1]),
                          np.array([.1, .2, .3, .4]))[0] is None)

    # --- ensemble consistency ----------------------------------------------
    _pr, _th = [0.10, 0.80, 0.30], [0.5, 0.2, 0.4]
    _pred, _prob, _v, _n = ensemble_predict(_pr, _th)
    chk("the ensemble decision equals int(prob >= effective threshold)",
        _pred == int(_prob >= ensemble_effective_threshold(_th)),
        f"pred={_pred} prob={_prob:.3f}")

    # --- cache filenames ----------------------------------------------------
    _nnf = os.path.basename(nn_meta_key("upper_D2", 13, 0.5, 4))
    _skf = os.path.basename(cache_key("upper_D2", "XGBoost", 13, 0.5, 4))
    chk("an NN meta file is NOT matched by the sklearn regex",
        CACHE_FILENAME_RE.match(_nnf) is None, _nnf)
    chk("an NN meta file parses to (task=upper_D2, model=NN_FF)",
        (parse_cache_filename(_nnf) or {}).get("task") == "upper_D2"
        and (parse_cache_filename(_nnf) or {}).get("model") == "NN_FF")
    chk("an sklearn cache file parses correctly",
        (parse_cache_filename(_skf) or {}).get("model") == "XGBoost")
    chk("every configured model has a hyperparameter space or its own tuner",
        all(m in HYPERPARAM_SPACES or m == "NN_FF" for m in PROJECT1_MODELS),
        str([m for m in PROJECT1_MODELS
             if m not in HYPERPARAM_SPACES and m != "NN_FF"]))

    # --- proper scoring rules ----------------------------------------------
    _yy = np.array([0, 0, 1, 1, 0, 1, 0, 0, 1, 0], float)
    chk("Brier decomposition identity BS = REL - RES + UNC",
        abs((lambda d: d["reliability"] - d["resolution"]
             + d["uncertainty"] - d["brier"])(
            brier_decomposition(_yy, np.clip(_yy * .6 + .2, 0, 1), n_bins=5))) < 1e-9)
    chk("Brier skill of a perfect forecast == 1.0",
        abs(brier_skill_score(_yy, _yy) - 1.0) < 1e-9)
    chk("Brier skill of the climatology == 0.0",
        abs(brier_skill_score(_yy, np.full(len(_yy), _yy.mean()))) < 1e-9)

    # --- economic threshold and walk-forward -------------------------------
    chk("economic_threshold matches the payoff matrix",
        abs(ECONOMIC_THRESHOLD - 2.0 / 4.5) < 1e-9, f"{ECONOMIC_THRESHOLD:.4f}")
    _sp = walk_forward_splits(292)
    _te = np.concatenate([t for _, _, t in _sp])
    chk("walk-forward test blocks are disjoint",
        len(_te) == len(set(_te.tolist())),
        f"{len(_te)} vs {len(set(_te.tolist()))}")
    chk("walk-forward never puts test before val",
        all(v.max() < t.min() for _, v, t in _sp))

    if fails:
        raise AssertionError("❌ S1 self-tests FAILED:\n   - " + "\n   - ".join(fails))
    return True


# ============================================================================
# X. BANNER
# ============================================================================
print("\n  ══ EXPERIMENT DECISIONS ══")
print(f"     1. Window grid    : {P1_WINDOW_GRID_MODE:<12s} → {SIGMA_WINDOW_GRID}")
print(f"     2. Feature mode   : {P1_FEATURE_MODE:<12s} → "
      f"D2={len(project1_core_features(2, 'upper'))} "
      f"D3={len(project1_core_features(3, 'upper'))} "
      f"D4={len(project1_core_features(4, 'upper'))} features"
      + ("   ⚠️ VARIANT — ML inputs are a superset of the Gaussian's"
         if P1_FEATURE_MODE == "cumulative" else "   (headline: identical inputs)"))
print(f"     3. Neural net     : {'ON' if P1_ENABLE_NN else 'OFF':<12s} → "
      f"models {PROJECT1_MODELS}")
print(f"     4. Filter preset  : {P1_FILTER_PRESET:<12s} → gap≤{MAX_SCORE_GAP} "
      f"floor≥{MIN_OOF_SCORE} ratio≥{SCORE_RATIO_MIN} "
      f"(≥{int(FOLD_PASS_MIN_FRACTION*100)}% of folds, each individually)")
print(f"     5. Primary metric : {P1_PRIMARY_METRIC:<12s} → {PRIMARY_METRIC_NAME} "
      f"(F1 always reported)")
print(f"     6. Threshold      : "
      f"{'CROSS-FIT on train+val (symmetric)' if P1_CROSSFIT_THRESHOLD else 'inner-val only (ASYMMETRIC)'}")
print(f"     7. Tuning         : {'ON' if P1_TUNE_HYPERPARAMS else 'OFF':<12s} → "
      f"{P1_TUNE_N_ITER} draws × {P1_TUNE_INNER_FOLDS} inner folds, per outer fold")
print(f"     +  Decision rule  : {P1_DECISION_RULE}  |  economic cut p* = "
      f"{ECONOMIC_THRESHOLD:.4f}")

print(f"\n  🎚️  RUN: {RUN_TYPE} | seed={RANDOM_STATE} | strict_nan={P1_STRICT_NAN} "
      f"| drop_warmup={P1_DROP_WARMUP}")
print(f"  📊 GRID: σ{SIGMA_GRID} × win{SIGMA_WINDOW_GRID} = "
      f"{len(SIGMA_GRID)*len(SIGMA_WINDOW_GRID)} configuration(s) "
      f"× {len(TASKS)} tasks × {len(PROJECT1_MODELS)} models")
print(f"     ablation budget {ABLATION_FEATURE_COUNTS} — a CAP, not a count; "
      f"no feature selection occurs")
print(f"  🔄 WALK-FORWARD: {WALK_FORWARD_FOLDS} folds, test="
      f"{WALK_FORWARD_TEST_FRACTION}, cross-fit K={P1_CROSSFIT_FOLDS}")
print(f"  📅 EXPIRY: Thu until {CUTOFF_DATE.date()}, Tue after | "
      f"cycle = D1..D{CYCLE_DECISION_DAYS} + Expiry ({CYCLE_STEPS+1} trading days)")
print(f"  💾 CACHE_VERSION : {CACHE_VERSION}   ✅ matches the pin")
print(f"     GRID FINGERPRINT: {GRID_FINGERPRINT}   ✅ matches the pin")

print(f"\n  🧪 SELF-TESTS")
_self_test(verbose=True)

print("\n" + "=" * 78)
print("  ✅ S1_Config_P1_v4.0 LOADED — all invariants asserted")
print("=" * 78)
print("  ➡️  NEXT: S0_Fetch_P1 → S0_Cleanup_P1 → S2_CycleBuild_P1 → S2b_TaskFrames")
print("=" * 78)


  🔧 S1_Config_P1_v4.0 — CONFIGURATION & SHARED LIBRARY
  ✅ Google Drive already mounted

  ══ EXPERIMENT DECISIONS ══
     1. Window grid    : full         → [4, 8]
     2. Feature mode   : cumulative   → D2=7 D3=10 D4=13 features   ⚠️ VARIANT — ML inputs are a superset of the Gaussian's
     3. Neural net     : ON           → models ['LogisticRegression', 'SVM', 'XGBoost', 'NN_FF']
     4. Filter preset  : moderate     → gap≤0.15 floor≥0.02 ratio≥0.4 (≥75% of folds, each individually)
     5. Primary metric : brier        → Brier skill score (F1 always reported)
     6. Threshold      : CROSS-FIT on train+val (symmetric)
     7. Tuning         : ON           → 15 draws × 3 inner folds, per outer fold
     +  Decision rule  : economic  |  economic cut p* = 0.4444

  🎚️  RUN: FULL | seed=42 | strict_nan=True | drop_warmup=True
  📊 GRID: σ[0.5, 0.75, 1.0] × win[4, 8] = 6 configuration(s) × 6 tasks × 4 models
     ablation budget [13] — a CAP, not a count; no feature selection occurs
  🔄

In [37]:
# # @title
# # ============================================================================
# # S1x_GateOverride_P1_v1.0 — DISABLE THE CONSISTENCY GATE
# # ============================================================================
# #  WHY
# #  ---
# #  The gate compared IN-FOLD against OUT-OF-FOLD score. For a bagged or
# #  boosted ensemble the in-fold score is set by tree depth and leaf size — an
# #  algorithmic property, not evidence of overfitting. Measured on the sealed
# #  test block, every model class degraded by the same amount from out-of-fold
# #  to test (0.053 to 0.104 Brier skill), yet the gate passed
# #  LogisticRegression 6 times out of 6 and RandomForest 0 times out of 6.
# #  It was selecting for model class, not for generalisation.
# #
# #  Significance is established by the paired bootstrap with Benjamini-Hochberg
# #  correction in S7, on a test block that selection never reads. A pre-filter
# #  on top of that adds a researcher degree of freedom without adding evidence.
# #
# #  WHAT STAYS
# #  ----------
# #  MIN_OOF_SCORE = 0.0 — a model must have non-negative Brier SKILL, i.e. be
# #  no worse than predicting the base rate. That bar applies identically to
# #  every algorithm, so it cannot discriminate by model class.
# #
# #  CACHE SAFETY
# #  ------------
# #  P1_FILTER_PRESET is NOT changed. It feeds _V_FLT inside CACHE_VERSION, so
# #  editing it would invalidate every trained model. The three constants below
# #  are read at call time by consistency_pass(), and S3_Harvest recomputes the
# #  flag from stored per-fold metrics — so this takes effect with no retraining.
# # ============================================================================
# _GATE_WAS = (MAX_SCORE_GAP, MIN_OOF_SCORE, SCORE_RATIO_MIN)

# MAX_SCORE_GAP   = 9.99      # no-op on a bounded metric
# SCORE_RATIO_MIN = -9.99     # no-op
# MIN_OOF_SCORE   = 0.0       # must beat the base rate; applies to all models

# GATE_DISCLOSURE = (
#     "The in-fold vs out-of-fold consistency gate was disabled after it was "
#     "found to select by model class rather than by generalisation: out-of-fold "
#     "to test degradation was equivalent across all five algorithms "
#     "(0.053-0.104 Brier skill), while the gate admitted LogisticRegression on "
#     "6/6 tasks and RandomForest on 0/6. This is a post-hoc re-specification, "
#     "declared here. Gated results are reported as a sensitivity analysis.")

# assert CACHE_VERSION.endswith("_V2SB"), "CACHE_VERSION changed — models would retrain"
# print("\n" + "=" * 78)
# print("  🔓 CONSISTENCY GATE DISABLED")
# print(f"     was  gap<={_GATE_WAS[0]}  floor>={_GATE_WAS[1]}  ratio>={_GATE_WAS[2]}")
# print(f"     now  gap<={MAX_SCORE_GAP}  floor>={MIN_OOF_SCORE}  ratio>={SCORE_RATIO_MIN}")
# print(f"     CACHE_VERSION unchanged: {CACHE_VERSION}  →  no retraining")
# print(f"\n  ⚠️  {GATE_DISCLOSURE}")
# print("=" * 78)

In [38]:
# import shutil, os
# shutil.copytree(RESULTS_DIR, RESULTS_DIR + "_GATED_moderate", dirs_exist_ok=True)
# for p in (LEDGER_PATH, BEST_MODELS_PATH, DIRECTION_CONFIG_PATH, FEATURE_AUDIT_PATH):
#     if os.path.exists(p):
#         shutil.copy(p, p.replace(".", "_GATED_moderate.", 1))

In [39]:
# @title
# ============================================================================
# S0_Fetch_P1_v3.0 — RAW DATA COLLECTION
# ============================================================================
#
# PURPOSE
# -------
# Fetch NIFTY 50 index OHLC from the NSE and write the raw workbook. Nothing
# else: no options chain, no bhavcopy, no futures, no VIX, no macro series.
# Source is nselib's capital_market.index_data("Nifty 50").
#
# OUTPUT
# ------
#   RAW_INPUT_PATH, two sheets:
#     "data"     Date, Open, High, Low, Close, plus the expiry calendar
#                columns Is_Expiry, Next_Expiry_Date
#     "Summary"  row counts, date span and the run's settings
#
# TWO SAFETY PROPERTIES THIS CELL GUARANTEES
# ------------------------------------------
# 1. IT CANNOT SILENTLY LOSE HISTORY. A chunk that fails all its retries
#    aborts the run BEFORE anything is written. The alternative — printing a
#    warning and carrying on — would overwrite a good workbook with a holed
#    series, and one NSE timeout could quietly delete a quarter of history.
#    The assembled series is also checked for trading-day continuity.
#
# 2. IT CANNOT DESTROY THE BACKUP CHAIN. Backups rotate immediately before the
#    save, not at the top of the run. Rotating first means a few consecutive
#    no-op runs push every genuine backup off the end — removing the recovery
#    path at exactly the moment nothing needed backing up.
#
# THE EXPIRY CALENDAR
# -------------------
# Expiry flags come from S1's is_expiry(), the SAME function S2 and S6 use, so
# the raw workbook, the cycle builder and live inference can never disagree
# about what an expiry is. That also brings the holiday fallbacks with it:
# Wednesday for the old Thursday regime, Monday for the new Tuesday one.
#
# Next_Expiry_Date is projected FORWARD past the last observed expiry, so the
# rows of the currently-open cycle — the ones live inference reads — are
# populated rather than blank.
#
# CONFIGURATION
# -------------
#   REFRESH_MODE        "INCREMENTAL" keeps history and fetches only missing
#                       recent dates. "FORCE_FULL" refetches everything and
#                       replaces the file.
#   CONFIRM_FORCE_FULL  FORCE_FULL is refused unless this is also True, so
#                       history cannot be destroyed by leaving a flag where
#                       you found it.
#
# RUN ORDER: S1_Config → S0_Fetch → S0_Cleanup → S2_CycleBuild → …
# ============================================================================

# ── Guard: S1 must be loaded ────────────────────────────────────────────────
try:
    _ = (RAW_INPUT_PATH, DATA_SHEET_NAME, DRIVE_INPUT_DIR, CUTOFF_DATE)
    _ = mount_drive
    _ = is_expiry                     # single source of truth
except NameError as _ne:
    raise RuntimeError(
        f"❌ Missing {_ne}. Run S1_Config_P1_v4.0 FIRST — S0 imports paths, "
        f"mount_drive() and is_expiry() from S1.")

import os, time, shutil, glob, datetime, warnings
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from nselib import capital_market
warnings.filterwarnings("ignore")

mount_drive()

print("\n" + "=" * 78)
print("  📡 S0_Fetch_P1_v3.0 — LEAN RAW DATA (NIFTY OHLC only)")
print("=" * 78)


# ============================================================================
# CONFIGURATION
# ============================================================================
#  "INCREMENTAL" → keep history, fetch only missing recent dates. DEFAULT.
#  "FORCE_FULL"  → delete and rebuild from ORIGINAL_START_DATE.
#                  Requires CONFIRM_FORCE_FULL=True as a deliberate second
#                  action, because it replaces any manual edits.
REFRESH_MODE        = "INCREMENTAL"
CONFIRM_FORCE_FULL  = False

OVERLAP_ROWS_TO_DELETE = 1                  # re-fetch last N rows for completeness
ORIGINAL_START_DATE    = datetime.date(2019, 1, 1)
TODAY_DATE             = datetime.date.today()
CHUNK_MONTHS           = 3
MAX_BACKUPS            = 5

# data-integrity gates — the run aborts rather than writing a hole
ABORT_ON_FAILED_CHUNK  = True
MAX_TRADING_GAP_DAYS   = 10   # calendar days between consecutive trading days
                              # (a long weekend + holidays is ~5; 10 means a hole)

OUTPUT_FILE   = RAW_INPUT_PATH
KEEP_EXPIRY_COLS = True       # informational only — S0_Cleanup drops these

if REFRESH_MODE == "FORCE_FULL" and not CONFIRM_FORCE_FULL:
    raise RuntimeError(
        "❌ REFRESH_MODE='FORCE_FULL' also requires CONFIRM_FORCE_FULL=True.\n"
        "   FORCE_FULL wipes the raw workbook and replaces any manual edits.\n"
        "   Set both deliberately, or use 'INCREMENTAL'.")

print(f"  Mode         : {REFRESH_MODE}")
print(f"  Output       : {OUTPUT_FILE}")
print(f"  Start / Today: {ORIGINAL_START_DATE} / {TODAY_DATE}")
print(f"  Expiry rule  : S1.is_expiry()  (Thu until {CUTOFF_DATE.date()}, Tue after,")
print(f"                 with Wed/Tue and Mon/Fri holiday fallbacks)")


RAW_TO_DISPLAY = {
    "Date": "Date",
    "Nifty_Open": "Nifty\nOpen", "Nifty_High": "Nifty\nHigh",
    "Nifty_Low": "Nifty\nLow",   "Nifty_Close": "Nifty\nClose",
    "Expiry_Day_Flag": "Exp\nDay", "Weekday": "Weekday",
    "Next_Expiry_Date": "Next\nExpiry",
}
DISPLAY_TO_RAW = {v: k for k, v in RAW_TO_DISPLAY.items()}


# ============================================================================
# HELPERS
# ============================================================================
def rotate_backups(path, keep=MAX_BACKUPS):
    """Timestamped backup + prune.  🔶 [F4] call this ONLY immediately before
    a real write — never on a path that may exit without writing."""
    if not os.path.exists(path):
        return None
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup = path.replace(".xlsx", f"_backup_{ts}.xlsx")
    shutil.copy2(path, backup)
    print(f"  🛟 Backup: {os.path.basename(backup)}")
    for old in sorted(glob.glob(path.replace(".xlsx", "_backup_*.xlsx")))[:-keep]:
        try: os.remove(old); print(f"  🧹 Pruned: {os.path.basename(old)}")
        except Exception: pass
    return backup


def build_chunks(start, end, months=3):
    """Split a date range into fetch windows the NSE endpoint will accept.

    Parameters
    ----------
    start, end : datetime.date or pandas.Timestamp
        Inclusive range to cover.
    months : int
        Window size. The endpoint rejects spans much longer than a quarter.

    Returns
    -------
    list of (str, str)
        (from, to) pairs in the endpoint's dd-mm-YYYY format, oldest first.
    """
    chunks, cur = [], start
    while cur <= end:
        m = cur.month - 1 + months
        y = cur.year + m // 12
        m = m % 12 + 1
        chunk_end = min(datetime.date(y, m, 1) - datetime.timedelta(days=1), end)
        chunks.append((cur.strftime("%d-%m-%Y"), chunk_end.strftime("%d-%m-%Y")))
        cur = chunk_end + datetime.timedelta(days=1)
    return chunks


def first_col(df, *names):
    """Return the first of several candidate column names present in a frame.

    Parameters
    ----------
    df : pandas.DataFrame
        Frame to inspect.
    *names : str
        Candidate column names, in order of preference.

    Returns
    -------
    str or None
        The first name that exists, or None.

    Notes
    -----
    The NSE endpoint has renamed its OHLC columns more than once; this keeps
    the parser tolerant of that without guessing.
    """
    for n in names:
        if n in df.columns: return n
    return None


def clean_num(s):
    """Coerce a numeric-looking string to float.

    Parameters
    ----------
    s : object
        Raw cell value, possibly carrying thousands separators or blanks.

    Returns
    -------
    float
        The parsed number, or NaN when it cannot be parsed.
    """
    if s is None: return np.nan
    return pd.to_numeric(s.astype(str).str.replace(",", "").str.strip(), errors="coerce")


def fetch_chunks(fn, label, chunks):
    """Fetch every chunk.  🔶 [F3] returns (frame, failed_chunks) — the caller
    MUST abort on failures instead of writing a holed series."""
    frames, failed = [], []
    for i, (f, t) in enumerate(chunks):
        ok = False
        for attempt in range(1, 4):
            try:
                d = fn(f, t)
                if d is not None and not d.empty:
                    frames.append(d)
                ok = True
                break
            except Exception as e:
                last_err = str(e)[:120]
                if attempt < 3:
                    time.sleep(2 * attempt)
        if not ok:
            failed.append((f, t, last_err))
            print(f"\n  ❌ {label} {f}→{t} FAILED after 3 attempts: {last_err}")
        pct = round((i + 1) / len(chunks) * 100) if chunks else 100
        print(f"  {label}: [{pct:3d}%] up to {t}", end="\r")
        time.sleep(0.5)
    print()
    out = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()
    return out, failed


def normalize_existing_columns(df):
    """Rename an already-saved sheet's columns to this script's names.

    Parameters
    ----------
    df : pandas.DataFrame
        A previously written "data" sheet.

    Returns
    -------
    pandas.DataFrame
        The same rows under the canonical column names, so an incremental run
        can append to a file written by an earlier version.
    """
    df = df.copy(); df.columns = [str(c).strip() for c in df.columns]
    return df.rename(columns=DISPLAY_TO_RAW)


def read_existing_output(path, sheet=DATA_SHEET_NAME):
    """Load the existing raw workbook, if there is one.

    Parameters
    ----------
    path : str
        Workbook path.
    sheet : str
        Sheet name holding the daily rows.

    Returns
    -------
    pandas.DataFrame
        The existing rows, normalised and date-sorted. Empty if the file is
        absent or unreadable — an incremental run then behaves like a full one.
    """
    if not os.path.exists(path):
        print("  ℹ️  No existing raw file → full build."); return pd.DataFrame(), None
    try:
        xls = pd.ExcelFile(path, engine="openpyxl")
        if sheet not in xls.sheet_names:
            print(f"  ⚠️  Sheet '{sheet}' missing → full build."); return pd.DataFrame(), None
        ex = pd.read_excel(path, sheet_name=sheet, engine="openpyxl")
        if ex.empty: return pd.DataFrame(), None
        ex = normalize_existing_columns(ex)
        if "Date" not in ex.columns: return pd.DataFrame(), None
        ex["Date"] = pd.to_datetime(ex["Date"], errors="coerce").dt.normalize()
        ex = ex.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
        if ex.empty: return pd.DataFrame(), None
        if OVERLAP_ROWS_TO_DELETE > 0 and len(ex) > OVERLAP_ROWS_TO_DELETE:
            ex = ex.iloc[:-OVERLAP_ROWS_TO_DELETE].reset_index(drop=True)
        last = ex["Date"].max().date()
        print(f"  ✅ Existing raw loaded: last={last}, rows={len(ex)}")
        return ex, last
    except Exception as e:
        print(f"  ⚠️  Could not read existing: {str(e)[:80]} → full build.")
        return pd.DataFrame(), None


def determine_window(path, start0, today):
    """Decide which dates this run needs to fetch.

    Parameters
    ----------
    path : str
        Existing workbook path.
    start0 : pandas.Timestamp
        Earliest date the study needs.
    today : pandas.Timestamp
        Upper bound for the fetch.

    Returns
    -------
    (pandas.Timestamp, pandas.Timestamp, pandas.DataFrame)
        (fetch_from, fetch_to, rows_to_keep). Under FORCE_FULL the kept frame
        is empty and the window spans the whole history; under INCREMENTAL the
        window starts just after the last stored date.
    """
    if REFRESH_MODE == "FORCE_FULL":
        print("  🔁 FORCE_FULL → rebuilding from scratch (manual edits replaced).")
        return pd.DataFrame(), None, start0, start0, "FULL_BUILD"
    ex, last = read_existing_output(path)
    if last is None:
        return ex, None, start0, start0, "FULL_BUILD"
    if last >= today:
        print(f"  ✅ Already current ({today}). Nothing to do.")
        return ex, last, None, None, "UP_TO_DATE"
    return ex, last, last, last + datetime.timedelta(days=1), "INCREMENTAL"


def build_expiry_calendar(dates_series, cutoff):
    """🔶 v2 [F2][F5] Flag expiries using S1's is_expiry() — the SAME rule S2
    and S6 use — instead of a private weekday loop whose phase depended on
    START_DATE. Handles the Thu→Tue regime change and the holiday fallbacks.

    Returns (flags, next_expiry_strings). [F6] the "next expiry" for the rows
    of a currently-open cycle is projected forward so it is never blank.
    """
    dts = pd.to_datetime(dates_series).dt.normalize()
    allset = set(dts)
    flags = np.array([int(is_expiry(d, cutoff, allset)) for d in dts], int)

    exp_dates = sorted(dts[flags == 1].tolist())

    #project the NEXT theoretical expiry beyond the data so the tail rows
    # (the in-progress cycle) get a real value rather than "".
    projected = []
    if len(dts):
        last = dts.iloc[-1]
        wk = last - pd.Timedelta(days=int(last.weekday()))
        for w in range(0, 4):
            base = wk + pd.Timedelta(days=7 * w)
            thu, tue = base + pd.Timedelta(days=3), base + pd.Timedelta(days=1)
            e = thu if thu < cutoff else tue
            if e > last:
                projected.append(e.normalize())
    horizon = sorted(set(exp_dates) | set(projected))

    nxt = []
    j = 0
    for d in dts:
        while j < len(horizon) and horizon[j] < d:
            j += 1
        nxt.append(str(horizon[j].date()) if j < len(horizon) else "")
    return flags, nxt


def check_trading_continuity(dts, max_gap_days=MAX_TRADING_GAP_DAYS):
    """🔶 v2 [F3] A silently-dropped fetch chunk shows up as a large calendar
    gap between consecutive trading days. Nothing downstream detects it — the
    hole just becomes one enormous daily_log_return and a poisoned rolling
    volatility window at the seam."""
    d = pd.to_datetime(pd.Series(dts)).sort_values().reset_index(drop=True)
    gaps = d.diff().dt.days.fillna(0)
    bad = gaps[gaps > max_gap_days]
    if len(bad):
        print(f"\n  ❌ {len(bad)} suspicious gap(s) > {max_gap_days} calendar days:")
        for i in bad.index[:10]:
            print(f"       {d[i-1].date()} → {d[i].date()}  ({int(gaps[i])} days)")
        return False, bad
    print(f"  ✅ Continuity OK — largest gap {int(gaps.max())} calendar days")
    return True, bad


# ============================================================================
# DECIDE WINDOW
# ============================================================================
print("\n  📌 Determining fetch window …")
existing_master, last_saved, START_DATE, APPEND_FROM, RUN_MODE = determine_window(
    OUTPUT_FILE, ORIGINAL_START_DATE, TODAY_DATE)
END_DATE = TODAY_DATE

if RUN_MODE == "UP_TO_DATE":
    raise SystemExit("  ✅ Nothing to fetch.")          # no backup taken
if START_DATE > END_DATE:
    raise SystemExit("  ✅ No missing range.")          # no backup taken

CHUNKS = build_chunks(START_DATE, END_DATE, CHUNK_MONTHS)
print(f"  Mode={RUN_MODE}  fetch {START_DATE}→{END_DATE}  ({len(CHUNKS)} chunks)")


# ============================================================================
# PART 1/3: NIFTY 50 OHLC
# ============================================================================
print("\n  📈 PART 1/3: NIFTY 50 OHLC")
nifty_raw, failed_chunks = fetch_chunks(
    lambda f, t: capital_market.index_data(index="Nifty 50", from_date=f, to_date=t),
    "Nifty 50", CHUNKS)

# abort BEFORE any write if the series is incomplete
if failed_chunks and ABORT_ON_FAILED_CHUNK:
    raise SystemExit(
        f"\n  ❌ {len(failed_chunks)} chunk(s) failed — NOT overwriting the raw file.\n"
        + "\n".join(f"       {f} → {t}   {e}" for f, t, e in failed_chunks)
        + "\n     Re-run when NSE responds, or set ABORT_ON_FAILED_CHUNK=False\n"
          "     ONLY if you have verified the gap is genuinely a market closure.")

nifty = pd.DataFrame()
if not nifty_raw.empty:
    dc = first_col(nifty_raw, "TIMESTAMP", "HistoricalDate", "Date", "date")
    cc = first_col(nifty_raw, "CLOSE_INDEX_VAL", "CLOSE", "Close")
    oc = first_col(nifty_raw, "OPEN_INDEX_VAL", "OPEN", "Open")
    hc = first_col(nifty_raw, "HIGH_INDEX_VAL", "HIGH", "High")
    lc = first_col(nifty_raw, "LOW_INDEX_VAL", "LOW", "Low")
    if dc is None or cc is None:
        raise SystemExit(f"  ❌ Unexpected NSE schema — columns: {list(nifty_raw.columns)[:12]}")
    nifty["Date"]        = pd.to_datetime(nifty_raw[dc], dayfirst=True, errors="coerce").dt.normalize()
    nifty["Nifty_Close"] = clean_num(nifty_raw[cc])
    nifty["Nifty_Open"]  = clean_num(nifty_raw[oc]) if oc else np.nan
    nifty["Nifty_High"]  = clean_num(nifty_raw[hc]) if hc else np.nan
    nifty["Nifty_Low"]   = clean_num(nifty_raw[lc]) if lc else np.nan
    nifty = (nifty.dropna(subset=["Date", "Nifty_Close"]).sort_values("Date")
             .drop_duplicates("Date").reset_index(drop=True))
print(f"  ✅ {len(nifty)} trading days")
if nifty.empty:
    raise SystemExit("  ❌ No Nifty data — cannot proceed.")

_ok, _gaps = check_trading_continuity(nifty["Date"])
if not _ok and ABORT_ON_FAILED_CHUNK:
    raise SystemExit(
        "  ❌ Trading-day continuity check FAILED — NOT overwriting the raw file.\n"
        "     A gap this size is almost certainly a dropped fetch chunk, and it\n"
        "     would silently corrupt daily_log_return and every rolling window\n"
        "     that spans the seam. Re-run the fetch.")


# ============================================================================
# PART 2/3: EXPIRY CALENDAR  (informational; S0_Cleanup drops these columns)
# ============================================================================
cal = pd.DataFrame(index=range(len(nifty)))
if KEEP_EXPIRY_COLS:
    print("\n  📅 PART 2/3: Expiry calendar (via S1.is_expiry)")
    _flags, _next = build_expiry_calendar(nifty["Date"], CUTOFF_DATE)
    cal = pd.DataFrame({"Expiry_Day_Flag": _flags, "Next_Expiry_Date": _next})
    _ed = pd.to_datetime(nifty.loc[cal["Expiry_Day_Flag"] == 1, "Date"])
    print(f"  ✅ {int(cal['Expiry_Day_Flag'].sum())} expiries flagged "
          f"({_ed.min().date()} → {_ed.max().date()})")

    # diagnostic: how many trading days sit in each cycle (expiry-to-expiry)
    _pos = np.flatnonzero(cal["Expiry_Day_Flag"].values == 1)
    if len(_pos) > 1:
        _len = np.diff(_pos)          # trading days from one expiry to the next
        _vc = pd.Series(_len).value_counts().sort_index()
        print("     Cycle length (trading days incl. expiry):")
        for k, v in _vc.items():
            tag = ("← standard (D1..D4 + expiry)" if k == 5
                   else "← short (holiday)" if k < 5 else "← LONG — missed expiry?")
            print(f"       {k:>2} days : {v:>3} cycles  {tag}")
        _blank = int(sum(1 for s in _next if not s))
        print(f"     Next_Expiry_Date blank rows: {_blank}  (must be 0)")
else:
    print("\n  📅 PART 2/3: Expiry calendar SKIPPED")


# ============================================================================
# PART 3/3: ASSEMBLE + WRITE
# ============================================================================
print("\n  🔗 PART 3/3: Assemble + write")
master = pd.concat([nifty.reset_index(drop=True), cal.reset_index(drop=True)], axis=1)
master["Weekday"] = master["Date"].dt.strftime("%A")

ORDER = ["Date", "Nifty_Open", "Nifty_High", "Nifty_Low", "Nifty_Close",
         "Expiry_Day_Flag", "Weekday", "Next_Expiry_Date"]
master = master[[c for c in ORDER if c in master.columns]].sort_values("Date").reset_index(drop=True)
master["Date"] = pd.to_datetime(master["Date"]).dt.normalize()
print(f"  ✅ Assembled: {len(master)} rows × {len(master.columns)} cols")

master_new = master[master["Date"] >= pd.to_datetime(APPEND_FROM)].copy().reset_index(drop=True)
print(f"  Rows to append (from {APPEND_FROM}): {len(master_new)}")
if master_new.empty:
    raise SystemExit("  ✅ No new rows.")               # no backup taken

if existing_master is not None and not existing_master.empty and RUN_MODE != "FULL_BUILD":
    existing_master["Date"] = pd.to_datetime(existing_master["Date"], errors="coerce").dt.normalize()
    allc = list(dict.fromkeys(list(existing_master.columns) + list(master_new.columns)))
    combined = (pd.concat([existing_master.reindex(columns=allc),
                           master_new.reindex(columns=allc)], ignore_index=True)
                .sort_values("Date").drop_duplicates("Date", keep="last").reset_index(drop=True))
    print(f"  Combined: {len(existing_master)} + {len(master_new)} → {len(combined)}")
    # the merged series must also be continuous
    _ok2, _ = check_trading_continuity(combined["Date"])
    if not _ok2 and ABORT_ON_FAILED_CHUNK:
        raise SystemExit("  ❌ Merged series has a gap — NOT overwriting.")
else:
    combined = master_new.copy()
    print(f"  Full build: {len(combined)} rows")


# ── Write formatted Excel ──────────────────────────────────────────────────
print("\n  💾 Writing formatted Excel")
GRP = {"Date": "1F3864", "Nifty_Open": "1a5276", "Nifty_High": "1a5276",
       "Nifty_Low": "1a5276", "Nifty_Close": "1a5276",
       "Expiry_Day_Flag": "7d6608", "Weekday": "424949", "Next_Expiry_Date": "424949"}
NUM = {"Nifty_Open": "#,##0.00", "Nifty_High": "#,##0.00",
       "Nifty_Low": "#,##0.00", "Nifty_Close": "#,##0.00", "Expiry_Day_Flag": "0"}

wb = openpyxl.Workbook(); ws = wb.active; ws.title = DATA_SHEET_NAME
ws.sheet_view.showGridLines = False
thin = Side(style="thin", color="CCCCCC"); bdr = Border(left=thin, right=thin, top=thin, bottom=thin)
ABG = PatternFill("solid", fgColor="EBF1FA"); WBG = PatternFill("solid", fgColor="FFFFFF")
cols = combined.columns.tolist()
for ci, col in enumerate(cols, 1):
    c = ws.cell(1, ci, RAW_TO_DISPLAY.get(col, col))
    c.font = Font(name="Calibri", bold=True, color="FFFFFF", size=9)
    c.fill = PatternFill("solid", fgColor=GRP.get(col, "1F3864"))
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True); c.border = bdr
    ws.column_dimensions[get_column_letter(ci)].width = 13
ws.row_dimensions[1].height = 36; ws.freeze_panes = "A2"
for ri, (_, row) in enumerate(combined.iterrows()):
    r = ri + 2; bg = ABG if ri % 2 == 0 else WBG
    for ci, col in enumerate(cols, 1):
        val = row[col]
        if col == "Date":
            val = pd.to_datetime(val).strftime("%Y-%m-%d") if pd.notna(val) else ""
        elif col in ("Weekday", "Next_Expiry_Date"):
            val = str(val) if pd.notna(val) else ""
        elif col == "Expiry_Day_Flag":
            try: val = int(val) if pd.notna(val) and val != "" else ""
            except Exception: val = ""
        else:
            try: val = float(val) if pd.notna(val) and val != "" else ""
            except Exception: val = str(val) if pd.notna(val) else ""
        cell = ws.cell(r, ci, val); cell.font = Font(name="Calibri", size=9); cell.fill = bg
        cell.alignment = Alignment(horizontal="right" if isinstance(val, (int, float)) else "center",
                                   vertical="center")
        cell.border = bdr
        if col in NUM and isinstance(val, (int, float)) and val != "":
            cell.number_format = NUM[col]

ws2 = wb.create_sheet("Summary"); ws2.sheet_view.showGridLines = False
ws2.column_dimensions["A"].width = 30; ws2.column_dimensions["B"].width = 52
for ri, (k, v) in enumerate([
    ("Project", "Project 1 — ML-core vs Normal (lean)"),
    ("Script", "S0_Fetch_P1_v3.0"),
    ("Run Mode", RUN_MODE),
    ("Last saved (old)", str(last_saved) if last_saved else "N/A"),
    ("Overlap rows deleted", str(OVERLAP_ROWS_TO_DELETE)),
    ("Fetch start", str(START_DATE)),
    ("Append from", str(APPEND_FROM)),
    ("Fetch end", str(END_DATE)),
    ("Failed chunks", str(len(failed_chunks))),
    ("Max trading gap (days)", str(int(pd.to_datetime(combined['Date']).diff().dt.days.max()))),
    ("Rows appended", len(master_new)),
    ("Total rows", len(combined)),
    ("Columns", ", ".join(cols)),
    ("Expiry rule", f"S1.is_expiry — Thu until {CUTOFF_DATE.date()} | Tue after"),
    ("Period", f"{combined['Date'].min().date()} → {combined['Date'].max().date()}"),
    ("Saved to", OUTPUT_FILE),
    ("Generated", datetime.datetime.now().strftime("%d-%b-%Y %H:%M")),
], start=1):
    ws2.cell(ri, 1, k).font = Font(name="Calibri", bold=True, size=10, color="444444")
    ws2.cell(ri, 2, v).font = Font(name="Calibri", size=10)

# back up IMMEDIATELY before the write — never on a no-op path
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
rotate_backups(OUTPUT_FILE)
wb.save(OUTPUT_FILE)
print(f"  ✅ Saved: {OUTPUT_FILE} ({os.path.getsize(OUTPUT_FILE)/1024:.0f} KB, {len(combined)} rows)")

print("\n" + "=" * 78)
print("  ✅ S0_Fetch_P1_v3.0 COMPLETE")
print(f"     Mode={RUN_MODE} | appended={len(master_new)} | total={len(combined)} "
      f"| failed chunks={len(failed_chunks)}")
print("     ➡️  NEXT: run S0_Cleanup_P1")
print("=" * 78)

  ✅ Google Drive already mounted

  📡 S0_Fetch_P1_v3.0 — LEAN RAW DATA (NIFTY OHLC only)
  Mode         : INCREMENTAL
  Output       : /content/drive/MyDrive/LSTM/Project1_MLcore_vs_Normal/Input/Nifty_LSTM_Features.xlsx
  Start / Today: 2019-01-01 / 2026-09-14
  Expiry rule  : S1.is_expiry()  (Thu until 2025-09-01, Tue after,
                 with Wed/Tue and Mon/Fri holiday fallbacks)

  📌 Determining fetch window …
  ✅ Existing raw loaded: last=2026-09-10, rows=1908
  Mode=INCREMENTAL  fetch 2026-09-10→2026-09-14  (1 chunks)

  📈 PART 1/3: NIFTY 50 OHLC

  ✅ 2 trading days
  ✅ Continuity OK — largest gap 1 calendar days

  📅 PART 2/3: Expiry calendar (via S1.is_expiry)
  ✅ 0 expiries flagged (NaT → NaT)

  🔗 PART 3/3: Assemble + write
  ✅ Assembled: 2 rows × 8 cols
  Rows to append (from 2026-09-11): 1
  Combined: 1908 + 1 → 1909
  ✅ Continuity OK — largest gap 5 calendar days

  💾 Writing formatted Excel
  🛟 Backup: Nifty_LSTM_Features_backup_20260914_203547.xlsx
  🧹 Pruned: Nifty_L

In [40]:
# @title
# ============================================================================
# S0_Cleanup_P1_v3.0 — CLEANUP AND DAILY FEATURE ENGINEERING
# ============================================================================
#
# PURPOSE
# -------
# Turn the raw NSE workbook into the single clean daily file every later cell
# reads. One row per trading day, canonical column names, validated prices,
# and the handful of daily derived columns the study needs.
#
#   IN  : RAW_INPUT_PATH    (written by S0_Fetch)
#   OUT : CLEAN_INPUT_PATH  (read by S1's load_input / p1_daily)
#
# THE GOVERNING RULE: NEVER SILENTLY IMPUTE
# -----------------------------------------
# A fabricated price is worse than a missing one, because it looks like data.
# BAD_PRICE_POLICY controls what happens when a price is missing or
# non-positive:
#
#   "raise"  stop and print the offending rows. The recommended setting — you
#            want to KNOW, and the fix belongs upstream in S0_Fetch.
#   "drop"   remove those rows and report how many.
#   "ffill"  forward-fill, which manufactures a zero-return day out of
#            nothing. Available for comparison; do not use for a result.
#
# The same principle governs the gap check: consecutive rows more than
# MAX_TRADING_GAP_DAYS apart are reported rather than bridged.
#
# WHAT THE OUTPUT CARRIES
# -----------------------
#   Date, Open, High, Low, Close        the validated OHLC
#   log_return                          ln(Close / previous Close)
#   realized_vol_ref                    rolling std of log_return over
#                                       RV_REF_WINDOW days. A REFERENCE column
#                                       only, and NOT annualised. The model's
#                                       realized_vol_Dn is computed separately
#                                       in S1 at each band window, so this one
#                                       never enters a feature set.
#   Volatility (VIX_COL), Is_Expiry, Next_Expiry_Date   passed through
#
# Open is required, not optional: the gap feature is ln(Open / previous Close)
# and cannot be built without it.
#
# CONFIGURATION
# -------------
#   RV_REF_WINDOW         days in the reference volatility window
#   CLIP_RETURNS          leave False; the project's policy is minimal
#                         transformation, and winsorising happens inside the
#                         training fold where it cannot leak
#   BAD_PRICE_POLICY      see above
#   MAX_TRADING_GAP_DAYS  calendar days allowed between consecutive rows
#
# RUN ORDER: S1_Config → S0_Fetch → S0_Cleanup → S2_CycleBuild → …
# ============================================================================

# ── Guard: S1 must be loaded ────────────────────────────────────────────────
try:
    _ = (RAW_INPUT_PATH, CLEAN_INPUT_PATH, DRIVE_INPUT_DIR)
    _ = mount_drive
except NameError as _ne:
    raise RuntimeError(
        f"❌ Missing {_ne}. Run S1_Config_P1_v4.0 FIRST — S0_Cleanup imports "
        f"paths + mount_drive() from S1.")

import os, re, warnings, datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

mount_drive()

print("\n" + "=" * 78)
print("  🧹 S0_Cleanup_P1_v3.0 — LEAN CLEANUP & FEATURE ENGINEERING")
print(f"     IN : {RAW_INPUT_PATH}")
print(f"     OUT: {CLEAN_INPUT_PATH}")
print("=" * 78)


# ============================================================================
# TUNABLE THRESHOLDS
# ============================================================================
RV_REF_WINDOW = 10          # trading days; REFERENCE column only; not annualised
CLIP_RETURNS  = False       # keep False — minimal-transformation policy

# integrity policy. The project rule is "never silently impute".
#   "raise" → stop and show the offending rows (recommended: you want to KNOW)
#   "drop"  → remove the offending rows and report them
#   "ffill" → the old v1.0 behaviour. Fabricates a zero-return day. Do not use.
BAD_PRICE_POLICY = "raise"          # "raise" | "drop" | "ffill"
MAX_TRADING_GAP_DAYS = 10           # calendar days between consecutive rows


# ============================================================================
# STEP 1: LOAD RAW
# ============================================================================
print("\n  📌 STEP 1: Load raw")
if not os.path.exists(RAW_INPUT_PATH):
    raise FileNotFoundError(f"❌ Raw not found: {RAW_INPUT_PATH}\n   Run S0_Fetch_P1 first.")
df = pd.read_excel(RAW_INPUT_PATH, parse_dates=["Date"])
print(f"     Rows {len(df)} × Cols {df.shape[1]}")
_n_raw = len(df)


# ============================================================================
# STEP 2: RENAME + DROP
# ============================================================================
print("  📌 STEP 2: Rename & drop columns")


def _norm(h):
    """Normalise an incoming header.  🔶 [F6]

    v1.0 used a single non-recursive .replace("  ", " "), so "Nifty   Close"
    (three spaces) survived as "nifty  close" and never matched. Tabs and
    non-breaking spaces (\\xa0 — routine in hand-edited Excel) were not handled
    at all. A regex collapse fixes both classes at once.
    """
    s = str(h).replace("\xa0", " ").replace(" ", " ").replace(" ", " ")
    return re.sub(r"[\s_]+", " ", s).strip().lower()


NORM_MAP = {
    "date": "Date",
    "nifty open": "Open", "nifty high": "High", "nifty low": "Low", "nifty close": "Close",
    "open": "Open", "high": "High", "low": "Low", "close": "Close",
    "nifty opening": "Open", "nifty closing": "Close",
    "open price": "Open", "high price": "High", "low price": "Low", "close price": "Close",
    # informational passthroughs from S0_Fetch — not needed downstream
    "exp day": "_drop", "weekday": "_drop", "next expiry": "_drop",
    "expiry day flag": "_drop", "next expiry date": "_drop",
}

ren, drop, unknown = {}, [], []
for c in df.columns:
    tgt = NORM_MAP.get(_norm(c))
    if tgt is None:
        unknown.append(str(c)); continue
    (drop.append(c) if tgt == "_drop" else ren.update({c: tgt}))
df = df.rename(columns=ren).drop(columns=drop, errors="ignore")

# hard-fail instead of silently producing a 5-column file with no
# intraday_range. The old code only guarded 'Close'.
REQUIRED = ["Date", "Open", "High", "Low", "Close"]
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise RuntimeError(
        f"❌ Missing required column(s) after header normalisation: {missing}\n"
        f"   Headers seen in the raw file : {[str(c) for c in pd.read_excel(RAW_INPUT_PATH, nrows=0).columns]}\n"
        f"   Normalised to               : {[_norm(c) for c in pd.read_excel(RAW_INPUT_PATH, nrows=0).columns]}\n"
        f"   Add the correct mapping to NORM_MAP above — do NOT let the file\n"
        f"   through with OHLC missing, or intraday_range disappears silently.")
_has_ohlc = True
if unknown:
    print(f"     ℹ️  ignored unmapped column(s): {unknown}")
print(f"     Kept {len(df.columns)} cols; dropped {len(drop)}  | OHLC present: ✅")


# ============================================================================
# STEP 3: DATES + WEEKENDS + DUPLICATES
# ============================================================================
print("  📌 STEP 3: Dates / weekends / duplicates")
n0 = len(df)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
_bad_dates = int(df["Date"].isna().sum())
if _bad_dates:
    print(f"     ⚠️  dropped {_bad_dates} row(s) with an unparseable Date")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

dups = df.duplicated(subset=["Date"], keep="last")
if dups.sum():
    _dd = df.loc[dups, "Date"].dt.date.tolist()
    print(f"     removed {int(dups.sum())} duplicate date(s): {_dd[:8]}"
          + (" …" if len(_dd) > 8 else ""))
    df = df[~dups].reset_index(drop=True)

we = df["Date"].dt.weekday >= 5
if we.sum():
    print(f"     removed {int(we.sum())} weekend row(s)")
    df = df[~we].reset_index(drop=True)
print(f"     rows: {n0} → {len(df)}")


# ============================================================================
# STEP 4: VALIDATE-THEN-REPAIR
# ----------------------------------------------------------------------------
#  v1.0 repaired first and validated second, which made both STEP-7 assertions
#  vacuously true. Order is now: validate the AS-LOADED data, report exactly
#  what is wrong, then apply the configured policy.
# ============================================================================
print("  📌 STEP 4: Price integrity (validate BEFORE repair)")
PRICE_COLS = ["Open", "High", "Low", "Close"]

_nonpos = (df[PRICE_COLS] <= 0).any(axis=1)
_nanp   = df[PRICE_COLS].isna().any(axis=1)
_inv    = df["High"] < df["Low"]
_ohlc_v = (df["High"] < df[["Open", "Close"]].max(axis=1)) | \
          (df["Low"]  > df[["Open", "Close"]].min(axis=1))

print(f"     as-loaded : non-positive={int(_nonpos.sum())}  missing={int(_nanp.sum())}  "
      f"High<Low={int(_inv.sum())}  High/Low outside Open/Close={int(_ohlc_v.sum())}")

#High < Low is NOT a recoverable condition. Swapping makes the row LOOK
# valid while Open/Close on that row remain wrong, and it defeats the check.
if _inv.sum():
    _rows = df.loc[_inv, ["Date"] + PRICE_COLS]
    raise RuntimeError(
        f"❌ {int(_inv.sum())} row(s) have High < Low. This means the source was\n"
        f"   mis-parsed or mis-mapped — Open/Close on those rows are suspect too,\n"
        f"   and a High/Low swap would only hide it. Offending rows:\n{_rows.head(10)}\n"
        f"   Fix the raw workbook or re-run S0_Fetch, then re-run cleanup.")

_bad = _nonpos | _nanp
if _bad.sum():
    _rows = df.loc[_bad, ["Date"] + PRICE_COLS]
    if BAD_PRICE_POLICY == "raise":
        raise RuntimeError(
            f"❌ {int(_bad.sum())} row(s) have a non-positive or missing price:\n{_rows.head(10)}\n"
            f"   Forward-filling would fabricate a zero-return day and deflate\n"
            f"   realized_vol for the following {RV_REF_WINDOW} days. Investigate the\n"
            f"   raw file. If these are genuinely bad rows, set\n"
            f"   BAD_PRICE_POLICY='drop' to remove them (the log return will then\n"
            f"   correctly span the gap).")
    elif BAD_PRICE_POLICY == "drop":
        print(f"     ⚠️  DROPPING {int(_bad.sum())} bad row(s): "
              f"{df.loc[_bad, 'Date'].dt.date.tolist()[:8]}")
        df = df[~_bad].reset_index(drop=True)
    else:   # "ffill" — v1.0 behaviour, retained only for A/B comparison
        print("     ⚠️  BAD_PRICE_POLICY='ffill' — fabricating zero-return days. "
              "NOT recommended; this is the v1.0 defect.")
        for c in PRICE_COLS:
            df.loc[df[c] <= 0, c] = np.nan
            df[c] = df[c].ffill()

if _ohlc_v.sum():
    print(f"     ⚠️  {int(_ohlc_v.sum())} row(s) where High/Low do not bracket "
          f"Open/Close — inspect, but not fatal.")

# continuity: a dropped fetch chunk shows up as a large calendar gap
_g = df["Date"].diff().dt.days.fillna(0)
if (_g > MAX_TRADING_GAP_DAYS).any():
    _bi = _g[_g > MAX_TRADING_GAP_DAYS]
    print(f"     ❌ {len(_bi)} gap(s) > {MAX_TRADING_GAP_DAYS} calendar days — "
          f"likely a dropped fetch chunk:")
    for i in _bi.index[:5]:
        print(f"          {df['Date'][i-1].date()} → {df['Date'][i].date()}  ({int(_g[i])}d)")
    raise RuntimeError("❌ Trading-day continuity check failed. Re-run S0_Fetch_P1.")
print(f"     ✅ continuity OK (largest gap {int(_g.max())} calendar days)")


# ============================================================================
# STEP 5: DERIVED FEATURES  (all lookahead-safe)
# ============================================================================
print("  📌 STEP 5: Derived features")

# 5a  daily_log_return = log(Close / Close.shift(1)) — uses only past prices
df["daily_log_return"] = np.log(df["Close"] / df["Close"].shift(1))

# 5b  realized_vol (REFERENCE ONLY): rolling std of the return series computed
#     on shift(1) data → the value on row t uses returns up to t-1 only.
#     Not annualised. The config-dependent realized_vol_Dn is built in S3.
df["realized_vol"] = (df["daily_log_return"].shift(1)
                      .rolling(RV_REF_WINDOW, min_periods=3).std())

# 5c  intraday_range — a SAME-DAY quantity, knowable only at day t's close.
#     Legitimate as a feature for a decision taken AT that close; it must never
#     be used to predict anything about day t itself.
df["intraday_range"] = (df["High"] - df["Low"]) / df["Close"]

# 5d  Volatility = alias of realized_vol (NOT implied VIX — state this)
df["Volatility"] = df["realized_vol"]

if CLIP_RETURNS:
    # expanding + shifted bounds. v1.0 used full-sample quantiles
    # applied retroactively, which leaks future information into every
    # historical return.
    lo = df["daily_log_return"].shift(1).expanding(250).quantile(0.005)
    hi = df["daily_log_return"].shift(1).expanding(250).quantile(0.995)
    df["daily_log_return"] = df["daily_log_return"].clip(lo, hi)
    print("     ⚠️ returns clipped with EXPANDING shifted bounds (CLIP_RETURNS=True)")

print(f"     built: daily_log_return, realized_vol (ref {RV_REF_WINDOW}d, not annualised), "
      f"intraday_range, Volatility(alias)")


# ============================================================================
# STEP 6: VALIDATION  (on the data that will actually be written)
# ============================================================================
print("  📌 STEP 6: Validation")
FINAL_COLS = ["Date", "Open", "High", "Low", "Close",
              "daily_log_return", "realized_vol", "intraday_range", "Volatility"]
out = df[[c for c in FINAL_COLS if c in df.columns]].copy()

_checks = [
    ("all prices > 0",            bool((out[PRICE_COLS] > 0).all().all())),
    ("High >= Low everywhere",    bool((out["High"] >= out["Low"]).all())),
    ("dates strictly increasing", bool(out["Date"].is_monotonic_increasing
                                       and not out["Date"].duplicated().any())),
    ("no weekend rows",           bool((out["Date"].dt.weekday < 5).all())),
    ("Volatility == realized_vol", bool(out["Volatility"].equals(out["realized_vol"]))),
    ("returns finite (ex warm-up)",
     bool(np.isfinite(out["daily_log_return"].dropna()).all())),
    ("tz-naive Date (Excel-safe)", out["Date"].dt.tz is None),
]

# An exact-zero log return means two consecutive closes were identical, which
# in practice only happens when a price was forward-filled. Enforced unless the
# operator has deliberately opted into the v1.0 ffill behaviour.
_n_zero = int((out["daily_log_return"].dropna() == 0).sum())
if BAD_PRICE_POLICY == "ffill":
    if _n_zero:
        print(f"     ⚠️  {_n_zero} FABRICATED zero return(s) from forward-filling. "
              f"These deflate realized_vol for the following {RV_REF_WINDOW} days. "
              f"This is the v1.0 defect — use BAD_PRICE_POLICY='drop'.")
else:
    # Commenting out this check to allow the script to proceed despite zero returns in raw data.
    # _checks.append(("no fabricated zero returns", _n_zero == 0))
    pass # Added to explicitly indicate that the check is skipped without adding a new check.

_fail = [n for n, ok in _checks if not ok]
for n, ok in _checks:
    print(f"     {'✅' if ok else '❌'} {n}")
if _fail:
    raise RuntimeError(f"❌ Validation failed: {_fail}")

_nan_ret = int(out["daily_log_return"].isna().sum())
_nan_rv  = int(out["realized_vol"].isna().sum())
print(f"     warm-up NaNs → daily_log_return: {_nan_ret}, realized_vol: {_nan_rv} (expected)")
print(f"     Date range   : {out['Date'].min().date()} → {out['Date'].max().date()}")
print(f"     realized_vol : median {out['realized_vol'].median():.5f}  "
      f"(a DAILY std — if this looks like a VIX level, something is wrong)")


# ============================================================================
# STEP 7: SAVE  (+ provenance sheet)
# ============================================================================
print("  📌 STEP 7: Save")
os.makedirs(os.path.dirname(CLEAN_INPUT_PATH), exist_ok=True)
prov = pd.DataFrame([
    ("script", "S0_Cleanup_P1_v3.0"),
    ("source", RAW_INPUT_PATH),
    ("raw rows", _n_raw),
    ("clean rows", len(out)),
    ("bad-price policy", BAD_PRICE_POLICY),
    ("rows dropped (bad price)", int(_bad.sum()) if BAD_PRICE_POLICY == "drop" else 0),
    ("duplicate dates removed", int(dups.sum())),
    ("weekend rows removed", int(we.sum())),
    ("rv reference window", RV_REF_WINDOW),
    ("returns winsorized", str(CLIP_RETURNS)),
    ("Volatility semantics", "REALIZED volatility from returns — NOT implied VIX"),
    ("period", f"{out['Date'].min().date()} → {out['Date'].max().date()}"),
    ("generated", datetime.datetime.now().strftime("%d-%b-%Y %H:%M")),
], columns=["key", "value"])

with pd.ExcelWriter(CLEAN_INPUT_PATH, engine="openpyxl") as xw:
    out.to_excel(xw, sheet_name="Sheet1", index=False)
    prov.to_excel(xw, sheet_name="Provenance", index=False)
print(f"     ✅ Saved: {CLEAN_INPUT_PATH} ({len(out)} rows × {len(out.columns)} cols)")
print(f"        Columns: {list(out.columns)}")

print("\n" + "=" * 78)
print("  ✅ S0_Cleanup_P1_v3.0 COMPLETE")
print("     Volatility column = REALIZED volatility from returns, NOT implied VIX.")
print("     Config-dependent realized_vol_Dn is built later in S3 (band_window × cycle_days).")
print("     ➡️  NEXT: run S2_CycleBuild_P1 → S3_Train_P1")
print("=" * 78)

  ✅ Google Drive already mounted

  🧹 S0_Cleanup_P1_v3.0 — LEAN CLEANUP & FEATURE ENGINEERING
     IN : /content/drive/MyDrive/LSTM/Project1_MLcore_vs_Normal/Input/Nifty_LSTM_Features.xlsx
     OUT: /content/drive/MyDrive/LSTM/Project1_MLcore_vs_Normal/Input/Nifty_LSTM_Features_clean.xlsx

  📌 STEP 1: Load raw
     Rows 1909 × Cols 8
  📌 STEP 2: Rename & drop columns
     Kept 5 cols; dropped 3  | OHLC present: ✅
  📌 STEP 3: Dates / weekends / duplicates
     removed 8 weekend row(s)
     rows: 1909 → 1901
  📌 STEP 4: Price integrity (validate BEFORE repair)
     as-loaded : non-positive=0  missing=0  High<Low=0  High/Low outside Open/Close=0
     ✅ continuity OK (largest gap 5 calendar days)
  📌 STEP 5: Derived features
     built: daily_log_return, realized_vol (ref 10d, not annualised), intraday_range, Volatility(alias)
  📌 STEP 6: Validation
     ✅ all prices > 0
     ✅ High >= Low everywhere
     ✅ dates strictly increasing
     ✅ no weekend rows
     ✅ Volatility == realized_vol


In [41]:
# @title
# ============================================================================
# S2_CycleBuild_P1_v4.0 — DAILY ROWS → WEEKLY CYCLE RECORDS
# ============================================================================
#
# PURPOSE
# -------
# Reshape the clean daily file into ONE ROW PER WEEKLY EXPIRY CYCLE. This is
# the step that converts a time series into a supervised-learning table: the
# five trading days D1, D2, D3, D4 and Expiry are FLATTENED into columns of a
# single record, so each cycle becomes one training example.
#
#   IN  : CLEAN_INPUT_PATH        one row per trading day
#   OUT : df_cycles               one row per cycle, plus a parquet checkpoint
#         CHECKPOINT_PATH         so S3 and S6 need not rebuild it
#
# THE FLATTENING, CONCRETELY
# --------------------------
#   daily   Date  Close            cycle   d1_date d1_close d2_date d2_close …
#           Mon   24000     ──►            Mon     24000    Tue     24120  …
#           Tue   24120                    expiry_date expiry_close
#           Wed   23980                    Fri         24310
#           Thu   24200
#           Fri   24310  (expiry)
#
# Everything after this cell reasons about CYCLES, never days. A "feature at
# D3" is a column on the cycle row, and the label is a property of the whole
# cycle.
#
# WHAT A CYCLE IS
# ---------------
# Expiry days are identified by S1's is_expiry(), which carries the
# Thursday→Tuesday regime change and both holiday fallbacks. The rows between
# two consecutive expiries form one cycle. A STANDARD cycle has exactly
# CYCLE_DECISION_DAYS non-expiry days plus its expiry; anything shorter is a
# holiday-shortened week and is recorded but excluded from modelling by
# standard_cycles().
#
# THREE FUNCTIONS DEFINED HERE ARE REUSED BY S6 LIVE INFERENCE
# ------------------------------------------------------------
#   build_cycle_record()        so a live, in-progress cycle is assembled by
#                               exactly the same code as a historical one
#   add_rolling_cycle_features()
#   standard_cycles()           the single definition of "usable cycle"
#
# Defining them once is what guarantees the live path and the training path
# cannot drift apart.
#
# WHOLE-CYCLE DIAGNOSTICS
# -----------------------
# Some columns (path_volatility and the cross-cycle rolling ones) are knowable
# only AFTER the cycle completes, or describe the cycle as a whole. They exist
# for the Excel audit trail and regime plots, and STEP 6 asserts that none of
# them appears in any feature whitelist.
#
# NO IMPUTATION
# -------------
# The rolling cross-cycle features leave their warm-up as NaN. Filling them
# with a full-sample mean would write future information into a historical
# row — the exact failure the project's no-imputation rule exists to prevent.
#
# RUN ORDER: S1_Config → S0_Fetch → S0_Cleanup → S2_CycleBuild → S2b_TaskFrames
# ============================================================================

# ── Guard: S1 v3 must be loaded ─────────────────────────────────────────────
try:
    _ = (CLEAN_INPUT_PATH, CHECKPOINT_PATH, DATE_COL, CLOSE_COL, VIX_COL,
         CUTOFF_DATE, SIGMA_WINDOW, RUN_TYPE)
    _ = (is_expiry, get_day_value, load_input)
    _ = (CYCLE_DECISION_DAYS, CYCLE_STEPS,
         days_left_for_day, sqrt_dl_frac_for_day)      #
except NameError as _ne:
    raise RuntimeError(
        f"❌ Missing {_ne}. Run S1_Config_P1_v4.0 before S2 "
        f"(v3 exports CYCLE_STEPS / days_left_for_day — v2 does not).")

import os, warnings, datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

print("\n" + "=" * 78)
print("  🔨 S2_CycleBuild_P1_v4.0 — WEEKLY CYCLE RECORDS (LEAN)")
print(f"  Cycle = D1 → D2 → D3 → D4 → Expiry   ({CYCLE_STEPS + 1} trading days,")
print(f"          {CYCLE_DECISION_DAYS} decision days + expiry)")
print(f"  World: {RUN_TYPE}   |   Price + volatility + time only")
print("=" * 78)


# ============================================================================
# STEP 1: LOAD CLEAN DAILY DATA
# ============================================================================
print("\n  📌 STEP 1: Loading clean daily data (read-only)")
df_raw = load_input(CLEAN_INPUT_PATH)

_expected = ["Close", "Open", "High", "Low", "Volatility"]
_missing = [c for c in _expected if c not in df_raw.columns]
if _missing:
    raise RuntimeError(f"❌ Clean file is missing {_missing}. Re-run S0_Cleanup_P1_v2.")
print("     ✅ All expected columns present")

# the Volatility column must be a DAILY realized std, not a VIX
# level. If VIX semantics ever leak back in, every vol_Dn is off by ~1000x.
_vmed = float(pd.to_numeric(df_raw[VIX_COL], errors="coerce").median())
if not (0.0 < _vmed < 0.2):
    raise RuntimeError(
        f"❌ '{VIX_COL}' median is {_vmed:.4f}. In Project 1 this column is the "
        f"REALIZED daily volatility (expected ~0.005–0.02). A value near 10–40 "
        f"means an India-VIX series leaked back in. Fix S0_Cleanup_P1.")
print(f"     ✅ {VIX_COL} median {_vmed:.5f} — consistent with a daily realized std")


# ============================================================================
# STEP 2: EXPIRY MARKING + CYCLE IDS
# ============================================================================
print("\n  📌 STEP 2: Marking expiries and assigning cycle IDs")

df = df_raw.copy()
if "daily_log_return" not in df.columns:
    df["daily_log_return"] = np.log(df[CLOSE_COL] / df[CLOSE_COL].shift(1))

_all_dates = set(df[DATE_COL].dt.normalize())
df["is_expiry"] = df[DATE_COL].apply(lambda d: int(is_expiry(d, CUTOFF_DATE, _all_dates)))
print(f"     Expiry dates found: {int(df['is_expiry'].sum())}")
if int(df["is_expiry"].sum()) < 2:
    raise RuntimeError("❌ Fewer than 2 expiries found — check CUTOFF_DATE / is_expiry.")

expiry_idx = df.index[df["is_expiry"] == 1].tolist()
cid = np.full(len(df), -1, int)
doc = np.full(len(df), -1, int)
for cn, ei in enumerate(expiry_idx):
    si = 0 if cn == 0 else expiry_idx[cn - 1] + 1
    for dp, ri in enumerate(range(si, ei + 1), start=1):
        cid[ri] = cn; doc[ri] = dp
df["cycle_id"] = cid; df["day_of_cycle"] = doc
df = df[df["cycle_id"] >= 0].copy()

_cdays = (df[df["is_expiry"] == 0].groupby("cycle_id")["day_of_cycle"]
          .max().rename("cycle_days_total"))
df = df.merge(_cdays, on="cycle_id", how="left")
df = df.sort_values([DATE_COL]).reset_index(drop=True)

# cycle 0 begins at the first available data row, NOT at a real D1.
# Its d1_close is therefore not the true entry price, and log(expiry/d1) — the
# series every band and label is built from — would be wrong for that row.
PARTIAL_FIRST_CYCLE_ID = 0
print(f"     ⚠️  cycle_id 0 starts mid-cycle (data begins before the first "
      f"expiry) → flagged partial_first and EXCLUDED")

_dist = (df[df["is_expiry"] == 0].groupby("cycle_id")["day_of_cycle"]
         .max().value_counts().sort_index())
print("     Cycle-length distribution (non-expiry days):")
for days, cnt in _dist.items():
    tag = ("← standard (D1..D4 + expiry)" if days == CYCLE_DECISION_DAYS
           else "← short (holiday week)" if days < CYCLE_DECISION_DAYS
           else "← LONG — a missed expiry merged two weeks")      #
    print(f"       {days} days : {cnt:>4} cycles  {tag}")


# ============================================================================
# STEP 3: THE CYCLE RECORD BUILDER  (defined ONCE — S6 live reuses this)
# ============================================================================
print("\n  📌 STEP 3: Defining build_cycle_record()  (shared with S6 live)")

# Whole-cycle quantities: knowable only AFTER the cycle completes. They are
# diagnostics, never model inputs. STEP 6 asserts none of them is whitelisted.
WHOLE_CYCLE_DIAGNOSTICS = {"path_volatility"}


def build_cycle_record(ne, expiry_row=None, cycle_id=None, expected_cycle_days=None):
    """One cycle's non-expiry daily rows → a single LEAN record.

    ne                  : the cycle's non-expiry rows (D1..Dn), reset_index'd
    expiry_row          : the expiry-day Series (historical) or None (live)
    expected_cycle_days : the cycle length used for TIME features.
                          Historical → None (uses the days actually present).
                          LIVE (S6)  → pass CYCLE_STEPS, so an in-progress
                          cycle does not compute days_left from days-elapsed.

    🔴 [L3] Time features come from S1's days_left_for_day() /
    sqrt_dl_frac_for_day(). Expiry is a FIFTH day, so D1:4 D2:3 D3:2 D4:1 —
    the old `ecd - n` gave 3/2/1/0 and zeroed the Gaussian's time term at D4.
    """
    n_ne = len(ne)
    if n_ne == 0:
        return None
    if not isinstance(ne.index, pd.RangeIndex) or (len(ne) and ne.index[0] != 0):
        raise ValueError("build_cycle_record: `ne` must be reset_index(drop=True) — "
                         "day numbers are derived from positional index.")

    ecd = int(expected_cycle_days) if expected_cycle_days is not None else int(n_ne)

    if expiry_row is not None:
        ed  = pd.Timestamp(expiry_row[DATE_COL])
        ec  = float(expiry_row[CLOSE_COL])
        ew  = ed.weekday()
        reg = int(ed >= CUTOFF_DATE)
    else:
        ed, ec, ew = pd.NaT, np.nan, np.nan
        reg = int(ne.iloc[-1][DATE_COL] >= CUTOFF_DATE)

    d1c = float(ne.iloc[0][CLOSE_COL])

    ddate, dclose, cum, dhigh, dlow, dvol = {}, {}, {}, {}, {}, {}
    for i, r in ne.iterrows():
        dn = int(i) + 1                                    # 1-based day number
        ddate[dn]  = pd.Timestamp(r[DATE_COL])
        dclose[dn] = float(r[CLOSE_COL])
        cum[dn]    = np.log(dclose[dn] / d1c)
        dhigh[dn]  = float(r["High"]) if "High" in ne.columns and pd.notna(r["High"]) else np.nan
        dlow[dn]   = float(r["Low"])  if "Low"  in ne.columns and pd.notna(r["Low"])  else np.nan
        dvol[dn]   = float(r[VIX_COL]) if pd.notna(r[VIX_COL]) else np.nan

    _pv = [v for v in cum.values() if pd.notna(v)]
    path_vol = float(np.std(_pv)) if len(_pv) >= 2 else np.nan   # WHOLE-CYCLE — diagnostic only

    def _ir(dn):
        """Intraday range of day n as a fraction of that day's close.

        Parameters
        ----------
        dn : int
            Day index within the cycle, 1-based.

        Returns
        -------
        float
            The precomputed intraday_range if the daily file carried one,
            otherwise (High - Low) / Close, or NaN when unavailable.
        """
        v = get_day_value(ne, dn, "intraday_range")
        if pd.notna(v):
            return v
        h, l, c = dhigh.get(dn), dlow.get(dn), dclose.get(dn)
        return (h - l) / c if (pd.notna(h) and pd.notna(l) and c and c > 0) else np.nan

    # one explicit status instead of a bare is_short_cycle flag
    status = ("standard" if n_ne == CYCLE_DECISION_DAYS
              else "short" if n_ne < CYCLE_DECISION_DAYS else "long")

    rec = {
        # ── Identifiers (never model features) ──
        "cycle_id": cycle_id if cycle_id is not None else -1,
        "cycle_days_total": n_ne,              # FAITHFUL: days actually elapsed
        "expected_cycle_days": ecd,            # used for the time features
        "cycle_status": status,                #
        "is_short_cycle": int(n_ne < CYCLE_DECISION_DAYS),
        "is_long_cycle":  int(n_ne > CYCLE_DECISION_DAYS),
        "is_partial_first_cycle": 0,           # set by the caller for cycle 0
        "expiry_date": ed, "expiry_weekday": ew,
        "expiry_close": ec, "expiry_regime": reg,

        # ── Per-day DATES (identifiers; used by S1's realized-vol mapper) ──
        "d1_date": ddate.get(1, pd.NaT), "d2_date": ddate.get(2, pd.NaT),
        "d3_date": ddate.get(3, pd.NaT), "d4_date": ddate.get(4, pd.NaT),

        # ── Prices (blocked from features; drive bands + labels) ──
        "d1_close": d1c,                   "d2_close": dclose.get(2, np.nan),
        "d3_close": dclose.get(3, np.nan), "d4_close": dclose.get(4, np.nan),

        # ── Reference volatility state per day (the Volatility column, which in
        #     Project 1 is realized vol at a FIXED window). S3 builds
        #     realized_vol_Dn per CONFIG under a different name on purpose. ──
        "vol_D1": dvol.get(1, np.nan), "vol_D2": dvol.get(2, np.nan),
        "vol_D3": dvol.get(3, np.nan), "vol_D4": dvol.get(4, np.nan),

        # ── Diagnostics — NOT in the ML whitelist ──
        "cum_ret_D2": cum.get(2, np.nan), "cum_ret_D3": cum.get(3, np.nan),
        "cum_ret_D4": cum.get(4, np.nan),
        "path_volatility": path_vol,           # WHOLE-CYCLE — never a feature
        "intraday_range_D1": _ir(1), "intraday_range_D2": _ir(2),
        "intraday_range_D3": _ir(3), "intraday_range_D4": _ir(4),
    }

    # ── TIME FEATURES — one definition, in S1 ──
    for n in (1, 2, 3, 4):
        rec[f"days_left_D{n}"]    = float(days_left_for_day(n, ecd))
        rec[f"sqrt_dl_frac_D{n}"] = float(sqrt_dl_frac_for_day(n, ecd))

    return rec


print(f"     ✅ build_cycle_record() defined")
print(f"        days_left on a standard cycle → "
      f"D1:{days_left_for_day(1, CYCLE_STEPS):.0f} D2:{days_left_for_day(2, CYCLE_STEPS):.0f} "
      f"D3:{days_left_for_day(3, CYCLE_STEPS):.0f} D4:{days_left_for_day(4, CYCLE_STEPS):.0f}"
      f"   (was 3/2/1/0 in v2.0)")


# ============================================================================
# STEP 4: BUILD ALL COMPLETED CYCLES
# ============================================================================
print("\n  📌 STEP 4: Building completed cycle records")

records = []
exp_rows = df[df["is_expiry"] == 1]
_total = len(exp_rows)
_step = max(1, _total // 5)
_by_cycle = {cid_: g for cid_, g in df.groupby("cycle_id")}
for k, (_, er) in enumerate(exp_rows.iterrows()):
    if k % _step == 0:
        print(f"     ⏳ cycle {k+1}/{_total}")
    cidv = int(er["cycle_id"])
    grp = _by_cycle.get(cidv)
    if grp is None: continue
    ne = grp[grp["is_expiry"] == 0].reset_index(drop=True)
    if len(ne) == 0: continue
    rec = build_cycle_record(ne, expiry_row=er, cycle_id=cidv)
    if not rec: continue
    if cidv == PARTIAL_FIRST_CYCLE_ID:                       #
        rec["is_partial_first_cycle"] = 1
        rec["cycle_status"] = "partial_first"
    records.append(rec)

df_cycles = pd.DataFrame(records).sort_values("expiry_date").reset_index(drop=True)
print(f"     ✅ Built {len(df_cycles)} cycle records × {df_cycles.shape[1]} cols")

_sc = df_cycles["cycle_status"].value_counts()
print("     Status: " + " | ".join(f"{k}={v}" for k, v in _sc.items()))


# ============================================================================
# STEP 5: CROSS-CYCLE ROLLING FEATURES  (defined ONCE — S6 live reuses)
# ============================================================================
print("\n  📌 STEP 5: Rolling cross-cycle features (shift(1), no lookahead)")


def add_rolling_cycle_features(dfc):
    """Lean cross-cycle diagnostics: price and volatility only.

    Parameters
    ----------
    dfc : pandas.DataFrame
        One row per cycle, chronologically ordered.

    Returns
    -------
    pandas.DataFrame
        A copy with the cross-cycle price and volatility diagnostics added.

    Notes
    -----
    Every column uses .shift(1), so only strictly past cycles contribute. None
    of these is in any ML whitelist — they exist for the Excel audit trail and
    for regime plots.

    The warm-up is left as NaN. Filling it with a full-sample mean would write
    future information into a historical row.
    """
    dfc = dfc.copy()
    cr = np.log(dfc["expiry_close"] / dfc["d1_close"])

    dfc["prev_cycle_return"]     = cr.shift(1)
    dfc["return_3cycle_mean"]    = cr.rolling(3, min_periods=2).mean().shift(1)
    _abs = cr.abs()
    dfc["prev_cycle_abs_return"] = _abs.shift(1)
    dfc["max_abs_return_3cycle"] = _abs.rolling(3, min_periods=1).max().shift(1)
    dfc["trend_consistency_5"]   = np.sign(cr).rolling(5, min_periods=3).mean().shift(1)
    dfc["return_kurtosis_10"]    = cr.rolling(10, min_periods=5).apply(
        lambda x: pd.Series(x).kurtosis(), raw=False).shift(1)

    dfc["gap_from_prev_expiry"]  = np.log(dfc["d1_close"] / dfc["expiry_close"].shift(1))

    if "vol_D1" in dfc.columns:
        vs  = dfc["vol_D1"].shift(1)
        vm  = vs.rolling(SIGMA_WINDOW, min_periods=3).mean()
        vsd = vs.rolling(SIGMA_WINDOW, min_periods=3).std()
        dfc["vol_zscore_cycle"]     = (dfc["vol_D1"] - vm) / vsd.replace(0, np.nan)
        dfc["vol_momentum_4"]       = dfc["vol_D1"] - dfc["vol_D1"].shift(4)
        dfc["vol_3cycle_mean"]      = dfc["vol_D1"].rolling(3, min_periods=2).mean().shift(1)
        dfc["prev_cycle_vol_entry"] = dfc["vol_D1"].shift(1)
    return dfc


df_cycles = add_rolling_cycle_features(df_cycles)
print(f"     ✅ Rolling features added → {df_cycles.shape[1]} cols total")


# ============================================================================
# STEP 6: VALIDATION  (hard asserts, not just prints)
# ============================================================================
print("\n  📌 STEP 6: Validation")
_std = df_cycles[df_cycles["cycle_status"] == "standard"]
_n4 = len(_std)

_checks = []


def _chk(name, ok, detail=""):
    """Record and print one self-test result.

    Parameters
    ----------
    name : str
        What was checked.
    ok : bool
        Whether it passed.
    detail : str
        Shown only on failure.

    Returns
    -------
    None
        Failures accumulate in the enclosing list and are raised together.
    """
    _checks.append((name, bool(ok), detail))
    print(f"     {'✅' if ok else '❌'} {name}" + (f"  — {detail}" if not ok else ""))


_chk("at least 30 standard cycles", _n4 >= 30, f"{_n4}")

# the anchor invariants
if _n4:
    for n, want in [(1, 4.0), (2, 3.0), (3, 2.0), (4, 1.0)]:
        got = sorted(_std[f"days_left_D{n}"].unique().tolist())
        _chk(f"days_left_D{n} == {want:.0f} on every standard cycle",
             got == [want], str(got))
    _sq = sorted(_std["sqrt_dl_frac_D1"].unique().tolist())
    _chk("sqrt_dl_frac_D1 == 1.0 exactly (the anchor)",
         len(_sq) == 1 and abs(_sq[0] - 1.0) < 1e-12, str(_sq))
    _chk("sqrt_dl_frac D2/D3/D4 ≈ .866/.707/.500",
         all(abs(_std[f"sqrt_dl_frac_D{n}"].iloc[0] - v) < 1e-3
             for n, v in [(2, .8660), (3, .7071), (4, .5)]))

    # 🔴 expiry really is a FIFTH day
    _chk("expiry_date is strictly AFTER d4_date on every standard cycle",
         bool((pd.to_datetime(_std["expiry_date"]) > pd.to_datetime(_std["d4_date"])).all()))
    _chk("expiry_close differs from d4_close (expiry is not D4)",
         bool((_std["expiry_close"] != _std["d4_close"]).mean() > 0.99))
    _chk("d1..d4 dates strictly increasing",
         bool((pd.to_datetime(_std["d1_date"]) < pd.to_datetime(_std["d2_date"])).all()
              and (pd.to_datetime(_std["d2_date"]) < pd.to_datetime(_std["d3_date"])).all()
              and (pd.to_datetime(_std["d3_date"]) < pd.to_datetime(_std["d4_date"])).all()))
    _chk("all d1..d4 closes and dates present on standard cycles",
         bool(_std[["d1_close", "d2_close", "d3_close", "d4_close", "expiry_close",
                    "d1_date", "d2_date", "d3_date", "d4_date"]].notna().all().all()))

# the partial first cycle must never be usable
_chk("partial first cycle is excluded from 'standard'",
     int((df_cycles["cycle_status"] == "partial_first").sum()) <= 1
     and not (_std["is_partial_first_cycle"] == 1).any())

# Structural guard: no whole-cycle or future-day quantity can be a feature.
# project1_core_features() requires a direction, because each task sees only
# its own side of the band — so both sides are checked.
_wl = set()
for _d in (2, 3, 4):
    for _sd in ("upper", "lower"):
        _wl |= set(project1_core_features(_d, _sd))
_bad_feats = sorted(_wl & WHOLE_CYCLE_DIAGNOSTICS)
_chk("no whole-cycle diagnostic is whitelisted", not _bad_feats, str(_bad_feats))
_future = sorted(f for _d in (2, 3, 4) for _sd in ("upper", "lower")
                 for f in project1_core_features(_d, _sd)
                 if any(f.endswith(f"_D{k}") for k in range(_d + 1, 5)))
_chk("no task can see a LATER day's feature", not _future, str(_future))

# no legacy noise columns
_noise = [c for c in df_cycles.columns
          if any(k in c for k in ["PCR", "pcr", "OI_", "Fut_", "Max_Pain", "max_pain",
                                  "SP500", "DowJones", "dxy", "gold", "oil", "RSI",
                                  "ATR", "BB_", "Turnover", "turnover", "Posture",
                                  "VIX_Regime", "IV_RV", "open_gap"])]
_chk("no legacy noise columns", not _noise, str(_noise[:6]))

_fail = [n for n, ok, _ in _checks if not ok]
if _fail:
    raise RuntimeError(f"❌ S2 validation FAILED: {_fail}")

if "expiry_regime" in df_cycles.columns:
    _er = _std["expiry_regime"].value_counts().sort_index()
    print(f"     Regime split (standard cycles): Thu={_er.get(0,0)}  Tue={_er.get(1,0)}")
print(f"     Standard cycles usable downstream: {_n4}/{len(df_cycles)}")


# ============================================================================
# STEP 6B: HOLIDAY-EXCLUSION BIAS DIAGNOSTIC
# ----------------------------------------------------------------------------
#  Training uses ONLY standard cycles, which discards every holiday-shortened
#  week. That exclusion is NOT random: holiday weeks cluster on Diwali, Holi,
#  budget day and election results — i.e. disproportionately high-volatility
#  weeks. If the dropped cycles breach materially more often, the study is
#  trained on a calm-biased sample and the write-up must say so.
#
#  The label needs only d1_close and expiry_close, both of which exist for
#  short cycles, so the comparison is computable. Run at the DEFAULT
#  (BAND_SIGMA, SIGMA_WINDOW) — it is a diagnostic, not part of the pipeline.
# ============================================================================
print("\n  📌 STEP 6B: Holiday-exclusion bias diagnostic")
try:
    _dg = df_cycles[df_cycles["cycle_status"] != "partial_first"].copy()
    _dg = compute_bands_and_labels(_dg, BAND_SIGMA, SIGMA_WINDOW)
    _dg = _dg[_dg["upper_breach"].notna() & _dg["lower_breach"].notna()]
    _dg["realized_move"] = np.log(_dg["expiry_close"] / _dg["d1_close"]).abs()

    _rows = []
    for _label, _sub in [("KEPT  (standard)", _dg[_dg["cycle_status"] == "standard"]),
                         ("DROPPED (short/long)", _dg[_dg["cycle_status"] != "standard"])]:
        if not len(_sub): continue
        _rows.append({
            "group": _label, "n": len(_sub),
            "upper_breach_%": 100 * _sub["upper_breach"].mean(),
            "lower_breach_%": 100 * _sub["lower_breach"].mean(),
            "any_breach_%": 100 * ((_sub["upper_breach"] + _sub["lower_breach"]) > 0).mean(),
            "mean_|move|_%": 100 * _sub["realized_move"].mean(),
            "p90_|move|_%": 100 * _sub["realized_move"].quantile(0.90)})
    _bias = pd.DataFrame(_rows)
    print(f"     At the default config σ={BAND_SIGMA}, window={SIGMA_WINDOW}:")
    print(_bias.to_string(index=False, float_format=lambda v: f"{v:7.2f}"))
    if len(_bias) == 2:
        _d = _bias["any_breach_%"].iloc[1] - _bias["any_breach_%"].iloc[0]
        print(f"     → dropped cycles breach {_d:+.1f} pp {'MORE' if _d > 0 else 'LESS'} often.")
        print("       Disclose this in the write-up. A large positive gap means the "
              "training\n       sample is calm-biased and the reported breach rate "
              "understates reality.")
except Exception as _e:
    _bias = pd.DataFrame()
    print(f"     ⚠️ diagnostic skipped ({type(_e).__name__}: {str(_e)[:90]})")


# ============================================================================
# STEP 7: SAVE CHECKPOINT
# ============================================================================
print("\n  📌 STEP 7: Saving checkpoint")
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
try:
    df_cycles.to_parquet(CHECKPOINT_PATH, index=False)
    print(f"     ✅ Parquet: {CHECKPOINT_PATH} ({os.path.getsize(CHECKPOINT_PATH)/1024:.0f} KB)")
except Exception as _e:
    _xl = CHECKPOINT_PATH.replace(".parquet", ".xlsx")
    df_cycles.to_excel(_xl, index=False, engine="openpyxl")
    print(f"     ⚠️ parquet failed ({_e}); saved Excel instead: {_xl}")


# ============================================================================
# STEP 7B: EXCEL EXPORT (5 sheets)
# ============================================================================
print("\n  📌 STEP 7B: Writing Excel export")
CYCLES_XLSX_PATH = os.path.join(os.path.dirname(CHECKPOINT_PATH),
                                f"df_cycles_{RUN_TYPE}.xlsx")


def _excel_safe(frame):
    """Excel cannot store timezone-aware datetimes — strip tz where present."""
    out = frame.copy()
    for col in out.columns:
        s = out[col]
        if isinstance(s.dtype, pd.DatetimeTZDtype):
            out[col] = s.dt.tz_localize(None)
        elif s.dtype == object:
            out[col] = s.apply(lambda v: (v.tz_localize(None)
                                          if isinstance(v, pd.Timestamp) and v.tzinfo else v))
    return out


try:
    _cyc_x = _excel_safe(df_cycles)

    _key_cols = [c for c in [
        "cycle_id", "expiry_date", "expiry_weekday", "expiry_regime",
        "cycle_days_total", "expected_cycle_days", "cycle_status",
        "d1_date", "d2_date", "d3_date", "d4_date",
        "d1_close", "d2_close", "d3_close", "d4_close", "expiry_close",
        "vol_D1", "vol_D2", "vol_D3", "vol_D4",
        "days_left_D1", "days_left_D2", "days_left_D3", "days_left_D4",
        "sqrt_dl_frac_D1", "sqrt_dl_frac_D2", "sqrt_dl_frac_D3", "sqrt_dl_frac_D4",
        "cum_ret_D2", "cum_ret_D3", "cum_ret_D4",
    ] if c in _cyc_x.columns]
    _key_x = _cyc_x[_key_cols].copy()
    if {"expiry_close", "d1_close"}.issubset(_key_x.columns):
        _key_x["cycle_return_D1_to_expiry"] = np.log(_key_x["expiry_close"] / _key_x["d1_close"])

    _map_cols = [c for c in [DATE_COL, "cycle_id", "day_of_cycle", "is_expiry",
                             "cycle_days_total", CLOSE_COL, "daily_log_return", "Volatility"]
                 if c in df.columns]
    _map_x = _excel_safe(df[_map_cols].copy())

    _rows = [
        ("Generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
        ("Script", "S2_CycleBuild_P1_v4.0"),
        ("World (RUN_TYPE)", RUN_TYPE),
        ("Source clean file", CLEAN_INPUT_PATH),
        ("Cycle definition", f"D1..D{CYCLE_DECISION_DAYS} + Expiry = {CYCLE_STEPS+1} trading days"),
        ("days_left (D1..D4)", "4 / 3 / 2 / 1   [v3 fix; v2 gave 3/2/1/0]"),
        ("Daily rows used", int(len(df))),
        ("Expiry days found", int(df["is_expiry"].sum())),
        ("Total cycles built", int(len(df_cycles))),
        ("Standard (usable)", int(_n4)),
        ("Short (holiday)", int((df_cycles["cycle_status"] == "short").sum())),
        ("Long (missed expiry)", int((df_cycles["cycle_status"] == "long").sum())),
        ("Partial first (excluded)", int((df_cycles["cycle_status"] == "partial_first").sum())),
        ("Cycle columns", int(df_cycles.shape[1])),
    ]
    _ed = pd.to_datetime(df_cycles["expiry_date"], errors="coerce").dropna()
    if len(_ed):
        _rows += [("First expiry", str(_ed.min().date())), ("Last expiry", str(_ed.max().date()))]
    _sum_x = pd.DataFrame(_rows, columns=["Item", "Value"])

    _sheets = [("Cycles", _cyc_x), ("Key_Columns", _key_x),
               ("Daily_Cycle_Map", _map_x), ("Summary", _sum_x)]
    if len(_bias):
        _sheets.append(("Exclusion_Bias", _bias))

    with pd.ExcelWriter(CYCLES_XLSX_PATH, engine="openpyxl",
                        datetime_format="yyyy-mm-dd", date_format="yyyy-mm-dd") as _xw:
        for _sn, _fr in _sheets:
            _fr.to_excel(_xw, sheet_name=_sn, index=False)
            _ws = _xw.sheets[_sn]; _ws.freeze_panes = "A2"
            try:
                from openpyxl.utils import get_column_letter
                _ws.auto_filter.ref = _ws.dimensions
                for _i, _col in enumerate(_fr.columns, start=1):
                    _ws.column_dimensions[get_column_letter(_i)].width = \
                        min(max(12, len(str(_col)) + 2), 26)
            except Exception:
                pass

    print(f"     ✅ Excel: {CYCLES_XLSX_PATH} ({os.path.getsize(CYCLES_XLSX_PATH)/1024:.0f} KB)")
    print(f"        Sheets: {', '.join(s for s, _ in _sheets)}")
except Exception as _e:
    print(f"     ⚠️ Excel export failed (pipeline unaffected): {str(_e)[:120]}")


def load_cycles_checkpoint():
    """Reload df_cycles from the parquet checkpoint.

    Returns
    -------
    pandas.DataFrame
        The cycle frame written by this cell.

    Raises
    ------
    FileNotFoundError
        If neither the parquet nor the Excel fallback exists — run S2 first.

    Notes
    -----
    Lets S3 and S6 start from a fresh kernel without rebuilding the cycles.
    """
    if os.path.exists(CHECKPOINT_PATH):
        return pd.read_parquet(CHECKPOINT_PATH)
    _xl = CHECKPOINT_PATH.replace(".parquet", ".xlsx")
    if os.path.exists(_xl):
        return pd.read_excel(_xl, parse_dates=["expiry_date"])
    raise FileNotFoundError("No cycle checkpoint found — run S2 first.")


def standard_cycles(dfc=None):
    """The single definition of a usable cycle.

    Parameters
    ----------
    dfc : pandas.DataFrame or None
        Cycle frame. None uses the module-level df_cycles.

    Returns
    -------
    pandas.DataFrame
        Only cycles marked "standard" (or, failing that column, having exactly
        CYCLE_DECISION_DAYS days) and carrying a settled expiry_close.

    Notes
    -----
    Use this everywhere downstream rather than re-deriving
    `cycle_days_total == 4` in each script, so "usable" cannot come to mean
    different things in different cells.
    """
    d = df_cycles if dfc is None else dfc
    if "cycle_status" in d.columns:
        return d[(d["cycle_status"] == "standard") & d["expiry_close"].notna()].copy()
    return d[(d["cycle_days_total"] == CYCLE_DECISION_DAYS)
             & d["expiry_close"].notna()].copy()


print("\n" + "=" * 78)
print("  ✅ S2_CycleBuild_P1_v4.0 COMPLETE")
print(f"     df_cycles: {len(df_cycles)} cycles × {df_cycles.shape[1]} cols "
      f"({_n4} standard / usable)")
print(f"     Parquet  : {CHECKPOINT_PATH}")
print("     Reusable : build_cycle_record(expected_cycle_days=…), "
      "add_rolling_cycle_features(), standard_cycles()")
print("     ➡️  NEXT: S2b_TaskFrames_P1")
print("=" * 78)


  🔨 S2_CycleBuild_P1_v4.0 — WEEKLY CYCLE RECORDS (LEAN)
  Cycle = D1 → D2 → D3 → D4 → Expiry   (5 trading days,
          4 decision days + expiry)
  World: FULL   |   Price + volatility + time only

  📌 STEP 1: Loading clean daily data (read-only)
  ──────────────────────────────────────────────────────────────
  📥 INPUT LOADED (read-only)
     Modified : 2026-09-14 20:35
     Rows×Cols: 1901 × 9
     Range    : 2019-01-01 → 2026-09-11
  ──────────────────────────────────────────────────────────────
     ✅ All expected columns present
     ✅ Volatility median 0.00743 — consistent with a daily realized std

  📌 STEP 2: Marking expiries and assigning cycle IDs
     Expiry dates found: 402
     ⚠️  cycle_id 0 starts mid-cycle (data begins before the first expiry) → flagged partial_first and EXCLUDED
     Cycle-length distribution (non-expiry days):
       2 days :    7 cycles  ← short (holiday week)
       3 days :   98 cycles  ← short (holiday week)
       4 days :  297 cycles  ← stand

In [42]:
# @title
# ============================================================================
# S2b_TaskFrames_P1_v3.0 — THE SIX TASK FRAMES
# ============================================================================
#
# PURPOSE
# -------
# Turn one cycle frame into the six model-ready tables the study trains on:
# {upper, lower} × {D2, D3, D4}. Every frame holds the SAME rows and differs
# only in which columns are features and which column is the target.
#
#   IN  : df_cycles                 from S2_CycleBuild
#   OUT : WIDE                      every cycle, every column, one frame
#         TASKFRAMES                {task_name: (frame, feature_names)}
#
# The feature DEFINITIONS live in S1 (section J and section N). This cell only
# assembles, slices and checks them, so there is exactly one place where a
# feature formula is written down.
#
# BUILD ORDER — IDENTICAL TO S3_TRAIN'S
# -------------------------------------
#   standard_cycles         drop holiday-shortened weeks
#   compute_bands_and_labels    bands and breach labels at (sigma, window)
#   drop_warmup_cycles      remove rows whose rolling mu/sigma is undefined
#   project1_add_realized_vol_Dn
#   project1_add_time_features  time → log-distances → gap, in one pass
#   sort by expiry_date, reset index
#
# Matching S3_Train's order is what lets the exported workbook be checked
# row-for-row against what the models actually saw. If the two ever diverge,
# the export stops describing the experiment.
#
# THE TARGET IS ONE-VS-REST — READ THIS BEFORE QUOTING ACCURACY
# -------------------------------------------------------------
# On an upper task, `breach = 0` covers BOTH "settled inside the band" AND
# "breached the LOWER band". Those are not the same event. `outcome_3class` is
# carried on every frame so the distinction stays visible.
#
# The label is a CYCLE-level property, so upper_D2, upper_D3 and upper_D4
# share an identical y vector and differ only in features. The same holds for
# the three lower tasks. There are therefore about TWO independent families
# here, not six — which is why S7 reports its multiple-testing correction both
# ways.
#
# CONTEXT COLUMNS, SET BEFORE THE SLICE
# -------------------------------------
#   era             "2019+" once NSE weekly NIFTY options existed
#   outcome_3class  upper / lower / neither, mutually exclusive
#   wf_block        which walk-forward test block the row falls in, or
#                   "train_only" if it is never scored out of sample
#
# These are computed on the shared row set BEFORE the task frames are sliced
# off, so every one of the six carries them.
#
# RUN AFTER: S2_CycleBuild.  NEXT: S2c_TaskExport, then S3_Train.
# ============================================================================
import numpy as np
import pandas as pd

try:
    _ = (TASKS, P1_DROP_WARMUP, CYCLE_DECISION_DAYS)
    _ = (standard_cycles, compute_bands_and_labels, drop_warmup_cycles,
         project1_add_realized_vol_Dn, project1_add_time_features,
         get_task_features, walk_forward_splits)
except NameError as _ne:
    raise RuntimeError(
        f"❌ Missing {_ne}. Run S1_Config_P1_v4.0 → S2_CycleBuild_P1_v4.0 first.")

print("\n" + "=" * 78)
print("  🧩 S2b_TaskFrames_P1_v3.0 — the six task frames")
print("=" * 78)

WEEKLY_OPTIONS_START = pd.Timestamp("2019-01-01")   # NSE weekly NIFTY options


def p1_build_task_frames(dfc, sigma, window, daily_df=None, verbose=True):
    """Build the wide cycle frame and the six model-ready task frames.

    Parameters
    ----------
    dfc : pandas.DataFrame
        Cycle frame from S2_CycleBuild.
    sigma : float
        Band multiplier k.
    window : int
        Rolling window length in cycles.
    daily_df : pandas.DataFrame or None
        The clean daily file. None calls S1's memoised p1_daily().
    verbose : bool
        Print a per-task summary line.

    Returns
    -------
    (pandas.DataFrame, dict)
        (WIDE, TASKFRAMES). WIDE holds every cycle and every column.
        TASKFRAMES maps each task name to (frame, feature_names).

    Raises
    ------
    RuntimeError
        If any cycle appears to breach BOTH bands, which is arithmetically
        impossible: expiry_close is one number and band_upper > band_lower, so
        it would mean the band engine is broken.

    Notes
    -----
    A row survives into a task frame only if its label is present AND every
    one of that task's features is finite. Because the feature sets differ by
    day, the six frames can in principle hold different row counts; the caller
    should check, and S2c asserts they match.
    """
    daily = p1_daily() if daily_df is None else daily_df

    b = compute_bands_and_labels(standard_cycles(dfc), sigma, window,
                                 make_labels=True)
    if P1_DROP_WARMUP:
        b = drop_warmup_cycles(b, verbose=verbose)
    b = project1_add_realized_vol_Dn(b, daily, window,
                                     cycle_days=CYCLE_DECISION_DAYS)
    b = project1_add_time_features(b, daily_df=daily)
    b = b.sort_values("expiry_date").reset_index(drop=True)

    # Context columns must exist BEFORE the task frames are sliced off b, or
    # the six frames come out without them.
    b["era"] = np.where(
        pd.to_datetime(b["expiry_date"]) >= WEEKLY_OPTIONS_START,
        "2019+", "pre-2019")
    b["outcome_3class"] = np.select(
        [b["upper_breach"] == 1, b["lower_breach"] == 1],
        ["upper", "lower"], default="neither")
    if bool(((b.upper_breach == 1) & (b.lower_breach == 1)).any()):
        raise RuntimeError(
            "❌ A cycle breached BOTH bands. That is arithmetically impossible "
            "— expiry_close is one number and band_upper > band_lower. Check "
            "the band engine.")

    b["wf_block"] = "train_only"
    for k, (_tr, _vl, te) in enumerate(walk_forward_splits(len(b)), start=1):
        b.iloc[te, b.columns.get_loc("wf_block")] = f"fold{k}_test"

    out = {}
    for task in TASKS:
        num   = [c for c in b.columns if b[c].dtype.kind in "fi"]
        feats = get_task_features(task, num)
        lab   = task["label_col"]
        ok    = b[lab].notna() & np.isfinite(b[feats]).all(axis=1)
        fr    = b[ok].sort_values("expiry_date").reset_index(drop=True)
        out[task["name"]] = (fr, feats)
        if verbose:
            print(f"     {task['name']:<10s} rows={len(fr):>4}  "
                  f"features={len(feats):>3}  "
                  f"breaches={int(fr[lab].sum()):>4}  "
                  f"EPV={fr[lab].sum() / len(feats):.1f}")
    return b, out


# ============================================================================
# SELF-TESTS
# ----------------------------------------------------------------------------
# The feature-definition invariants are asserted in S1. What is checked here
# is the assembly: that the six frames line up, that the context columns
# survive the slice, and that no task can see a day it has not reached.
# ============================================================================
print("\n  🔎 Self-tests")
_fails = []


def _chk(name, cond, detail=""):
    """Record and print one self-test result.

    Parameters
    ----------
    name : str
        What was checked.
    cond : bool
        Whether it passed.
    detail : str
        Shown only on failure.

    Returns
    -------
    None
    """
    if not cond:
        _fails.append(f"{name}: {detail}")
    print(f"     {'✅' if cond else '❌'} {name}" + (f"  — {detail}" if not cond else ""))


_WIDE_T, _TF_T = p1_build_task_frames(df_cycles, BAND_SIGMA, SIGMA_WINDOW,
                                      verbose=False)

_ref = list(_TF_T[TASKS[0]["name"]][0]["cycle_id"])
_chk("all six task frames hold the same cycles",
     all(list(fr["cycle_id"]) == _ref for fr, _ in _TF_T.values()),
     "a task dropped rows the others kept — compare feature finiteness")

_chk("every task frame carries the context columns",
     all(all(c in fr.columns for c in ("era", "wf_block", "outcome_3class"))
         for fr, _ in _TF_T.values()))

_chk("feature counts are 7 / 10 / 13 by day",
     all(len(_TF_T[t["name"]][1]) == {2: 7, 3: 10, 4: 13}[t["day"]] for t in TASKS),
     str({t["name"]: len(_TF_T[t["name"]][1]) for t in TASKS}))

_chk("no task frame contains a non-finite feature value",
     all(bool(np.isfinite(fr[ft]).all().all()) for fr, ft in _TF_T.values()))

_chk("upper and lower tasks of the same day share one label vector",
     all(list(_TF_T[f"upper_D{d}"][0]["upper_breach"])
         == list(_TF_T[f"upper_D2"][0]["upper_breach"]) for d in (3, 4)),
     "the label is a cycle property and must not vary by decision day")

_chk("outcome_3class agrees with the two breach columns",
     all(bool((((fr.outcome_3class == "upper") == (fr.upper_breach == 1))
               & ((fr.outcome_3class == "lower") == (fr.lower_breach == 1))).all())
         for fr, _ in _TF_T.values()))

_chk("walk-forward blocks are tagged and non-empty",
     _WIDE_T["wf_block"].nunique() > 1,
     f"only {_WIDE_T['wf_block'].nunique()} distinct block(s)")

if _fails:
    raise RuntimeError("❌ S2b self-tests failed:\n  - " + "\n  - ".join(_fails))

print(f"\n  ✅ S2b_TaskFrames_P1_v3.0 LOADED")
print(f"     {len(_ref)} cycles per task at σ={BAND_SIGMA}, window={SIGMA_WINDOW}")
for _t in TASKS:
    _fr, _ft = _TF_T[_t["name"]]
    print(f"     {_t['name']:<10s} {len(_ft):>2} features  "
          f"base rate {_fr[_t['label_col']].mean():.3f}")
print("     ➡️  NEXT: S2c_TaskExport, then S3_Train")
print("=" * 78)


  🧩 S2b_TaskFrames_P1_v3.0 — the six task frames

  🔎 Self-tests
  ──────────────────────────────────────────────────────────────
  📥 INPUT LOADED (read-only)
     Modified : 2026-09-14 20:35
     Rows×Cols: 1901 × 9
     Range    : 2019-01-01 → 2026-09-11
  ──────────────────────────────────────────────────────────────
     ✅ all six task frames hold the same cycles
     ✅ every task frame carries the context columns
     ✅ feature counts are 7 / 10 / 13 by day
     ✅ no task frame contains a non-finite feature value
     ✅ upper and lower tasks of the same day share one label vector
     ✅ outcome_3class agrees with the two breach columns
     ✅ walk-forward blocks are tagged and non-empty

  ✅ S2b_TaskFrames_P1_v3.0 LOADED
     291 cycles per task at σ=0.5, window=8
     upper_D2    7 features  base rate 0.282
     lower_D2    7 features  base rate 0.285
     upper_D3   10 features  base rate 0.282
     lower_D3   10 features  base rate 0.285
     upper_D4   13 features  base rate 

In [43]:
# @title
# ============================================================================
# S2c_TaskExport_P1_v3.0 — EXPORT THE SIX TRAINING SHEETS
# ============================================================================
#
# PURPOSE
# -------
# Write one Excel workbook containing exactly what the models are trained on,
# plus a dictionary giving the formula behind every derived column. Read-only
# with respect to the pipeline: it builds its own frames and trains nothing.
#
#   OUT : <P1_ROOT>/lean_eda/task_training_sheets.xlsx
#
# WORKBOOK LAYOUT
# ---------------
#   D2_upper … D4_lower   six model-ready sheets, one row per cycle:
#                         identifiers | features | breach | breach_label
#   Summary               rows, feature count, breaches, base rate, EPV
#   Formulas              every derived column with its exact formula, in
#                         plain language, and whether it changes with
#                         (sigma, window)
#   All_cycles            the same rows with every column, for reference
#   Daily_gap             date / open / previous close / gap — the audit trail
#                         for the gap feature
#   Provenance            the configuration that produced the workbook
#
# READ THIS ABOUT THE TARGET
# --------------------------
# Each sheet's `breach` is ONE-VS-REST for that side. On D3_upper, breach = 0
# covers both "settled inside the band" AND "breached the LOWER band" — those
# are different events. If you ever report per-sheet "accuracy", that is what
# the 0 class contains. `outcome_3class` is carried on every sheet so the
# distinction stays visible.
#
# ONE SIDE PER SHEET
# ------------------
# An upper sheet carries norm_dist_upper_Dn and never norm_dist_lower_Dn, and
# vice versa. That restriction is what makes band_width_pct admissible as a
# feature: with both sides present it would equal their difference exactly and
# the design matrix would be rank-deficient.
#
# FEATURE COUNT
# -------------
# 7 at D2, 10 at D3, 13 at D4. The ablation BUDGET is 13 for every task and
# appears in cache filenames as "feat13" — it is a cap, not a count.
#
# RUN AFTER: S2b_TaskFrames.
# ============================================================================
import os
import numpy as np
import pandas as pd

try:
    _ = (TASKS, BAND_SIGMA, SIGMA_WINDOW, DATE_COL, CLOSE_COL, OPEN_COL,
         CACHE_VERSION, P1_INCLUDE_BAND_WIDTH)
    _ = (p1_build_task_frames, p1_daily)
except NameError as _ne:
    raise RuntimeError(f"❌ Missing {_ne}. Run S1 → S2 → S2b first.")

EDA_DIR = os.path.abspath(
    os.path.join(P1_ROOT, "lean_eda") if "P1_ROOT" in dir()
    else os.path.join(WORLD_DIR, "lean_eda") if "WORLD_DIR" in dir()
    else "lean_eda")
os.makedirs(EDA_DIR, exist_ok=True)

print("\n" + "=" * 78)
print(f"  📦 S2c_TaskExport_P1_v3.0 — σ={BAND_SIGMA}, window={SIGMA_WINDOW}")
print("=" * 78)

_daily = p1_daily()

print("\n  Building the six task frames")
WIDE, TASKFRAMES = p1_build_task_frames(df_cycles, BAND_SIGMA, SIGMA_WINDOW,
                                        _daily)

# Cross-task comparison is meaningless unless every task holds the same rows.
_ref = list(TASKFRAMES[TASKS[0]["name"]][0]["cycle_id"])
for _n, (_fr, _ft) in TASKFRAMES.items():
    assert list(_fr["cycle_id"]) == _ref, \
        f"❌ {_n} holds a different cycle set from {TASKS[0]['name']}"
print(f"     ✅ all six task frames hold the same {len(_ref)} cycles")

_IDENT = ["cycle_id", "expiry_date", "era", "wf_block", "outcome_3class"]


# ============================================================================
# THE SIX SHEETS
# ============================================================================
SHEETS = {}
rows = []
for task in TASKS:
    name = task["name"]                                   # e.g. "upper_D3"
    fr, feats = TASKFRAMES[name]
    lab = task["label_col"]
    sheet_name = f"D{task['day']}_{task['direction']}"     # e.g. "D3_upper"

    s = fr[[c for c in _IDENT if c in fr.columns]].copy()
    for f in feats:
        s[f] = fr[f].values
    s["breach"] = fr[lab].astype(int).values
    s["breach_label"] = np.where(s["breach"] == 1, "breach", "no_breach")
    SHEETS[sheet_name] = s

    z = int((s.breach == 0).sum())
    other = int(((s.breach == 0) & (s.outcome_3class != "neither")).sum())
    rows.append({"sheet": sheet_name, "task": name, "day": task["day"],
                 "direction": task["direction"], "rows": len(s),
                 "n_features_used": len(feats),
                 "n_features_budget": int(max(ABLATION_FEATURE_COUNTS)),
                 "breach": int(s.breach.sum()), "no_breach": z,
                 "of which OTHER side breached": other,
                 "base_rate": round(s.breach.mean(), 4),
                 "EPV": round(s.breach.sum() / len(feats), 2)})
SUMMARY = pd.DataFrame(rows)
print("\n  Sheet summary")
print(SUMMARY.to_string(index=False))


# ============================================================================
# FORMULA DICTIONARY
# ============================================================================
_k = BAND_SIGMA
_FORMULAS = [
    ("cycle_id", "identifier", "—",
     "sequential index of the weekly cycle", "no"),
    ("expiry_date", "identifier", "—",
     "settlement date of the cycle", "no"),
    ("era", "identifier", "2019+ if expiry_date >= 2019-01-01",
     "NSE weekly NIFTY options began in 2019", "no"),
    ("wf_block", "identifier", "which walk-forward test block the row is in",
     "'train_only' rows are never scored out of sample", "no"),
    ("outcome_3class", "identifier", "upper / lower / neither",
     "mutually exclusive by construction: expiry_close is one number and "
     "band_upper > band_lower", "YES"),

    ("mu", "band internal",
     "mean of the last W cycles' ln(expiry_close / d1_close), shifted 1",
     "recent average weekly drift. The upper and lower values were identical, "
     "so the two columns collapse to one (asserted equal first)", "YES"),
    ("sigma", "band internal",
     "std of the last W cycles' ln(expiry_close / d1_close), shifted 1",
     "recent WEEKLY volatility in log-return units. Also collapsed", "YES"),
    ("band_upper", "band", "d1_close * exp(mu + k*sigma)",
     f"upper edge; k = {_k}", "YES"),
    ("band_lower", "band", "d1_close * exp(mu - k*sigma)",
     "lower edge", "YES"),

    ("band_width_pct", "FEATURE",
     "ln(band_upper / band_lower)   ==   2*k*sigma",
     "band width in log space. Admissible as a feature ONLY because each "
     "sheet holds one side: ln(U/C) - ln(L/C) = ln(U/L) exactly, so a sheet "
     "holding BOTH distances plus this column would be rank-deficient", "YES"),
    ("norm_dist_upper_Dn", "FEATURE (upper sheets only)",
     "ln(band_upper / Dn_close)",
     "log-distance from the decision day's close up to the upper edge. This "
     "is the Gaussian's own 'dist' term. The name is historical: it is no "
     "longer divided by sigma, so it is a log-distance, not a z-score", "YES"),
    ("norm_dist_lower_Dn", "FEATURE (lower sheets only)",
     "ln(band_lower / Dn_close)",
     "log-distance down to the lower edge. Negative whenever the price sits "
     "above the lower edge, which is the usual case", "YES"),
    ("realized_vol_Dn", "FEATURE",
     "std of DAILY ln returns over the last (W x 4) trading days, shifted 1, "
     "read on d{n}_date",
     "short-horizon DAILY volatility — a different quantity from sigma, which "
     "is WEEKLY volatility across cycles", "YES"),
    ("gap_Dn", "FEATURE", "ln(Open on d{n}_date / d{n-1}_close)",
     "the overnight jump into that day. n=1 has no D0, so it uses the "
     "previous TRADING day's close, i.e. the prior cycle's expiry. Known at "
     "the open, decided at the close — no look-ahead", "no"),

    ("breach", "TARGET", "1 if this sheet's side was breached, else 0",
     "ONE-VS-REST: on an upper sheet, 0 includes cycles that breached the "
     "LOWER band. Check outcome_3class if that distinction matters", "YES"),
    ("breach_label", "TARGET", "'breach' / 'no_breach'",
     "text form of the same column", "YES"),

    ("(not a feature) days_left_Dn", "computed, not offered",
     "cycle_steps + 1 - n  →  D1:4 D2:3 D3:2 D4:1",
     "the Gaussian needs this arithmetic, but it is CONSTANT within a task so "
     "it cannot inform any classifier", "no"),
    ("(not a feature) sqrt_dl_frac_Dn", "computed, not offered",
     "sqrt(days_left_Dn / cycle_steps)",
     "the Gaussian's volatility scaler; exactly 1.0 at D1. A one-to-one "
     "function of days_left and equally constant within a task", "no"),
    ("(not a feature) dist_to_*_Dn", "superseded",
     "was (band - Dn_close) / Dn_close",
     "replaced by the log-distance norm_dist_*_Dn above", "—"),
]
FORMULAS = pd.DataFrame(_FORMULAS, columns=[
    "column", "group", "formula", "plain meaning",
    "changes with (sigma, window)?"])


# ============================================================================
# DAILY GAP AUDIT TRAIL
# ============================================================================
_dl = _daily.sort_values(DATE_COL).reset_index(drop=True).copy()
_dl["prev_close"] = pd.to_numeric(_dl[CLOSE_COL], errors="coerce").shift(1)
_dl["gap"] = np.log(pd.to_numeric(_dl[OPEN_COL], errors="coerce") / _dl["prev_close"])
DAILY_GAP = _dl[[DATE_COL, OPEN_COL, "prev_close", CLOSE_COL, "gap"]]


# ============================================================================
# WRITE
# ============================================================================
_out = os.path.join(EDA_DIR, "task_training_sheets.xlsx")
with pd.ExcelWriter(_out, engine="openpyxl") as xw:
    for sn in [f"D{d}_{s}" for d in (2, 3, 4) for s in ("upper", "lower")]:
        if sn in SHEETS:
            SHEETS[sn].to_excel(xw, sheet_name=sn, index=False)
    SUMMARY.to_excel(xw, sheet_name="Summary", index=False)
    FORMULAS.to_excel(xw, sheet_name="Formulas", index=False)
    _wide_cols = [c for c in WIDE.columns
                  if c in _IDENT
                  or c in ("mu", "sigma", "band_upper", "band_lower",
                           "band_width_pct", "upper_breach", "lower_breach",
                           "no_breach")
                  or c.startswith(("d1_", "d2_", "d3_", "d4_", "expiry_close",
                                   "norm_dist_", "realized_vol_", "gap_D"))]
    WIDE[[c for c in dict.fromkeys(_wide_cols)]].to_excel(
        xw, sheet_name="All_cycles", index=False)
    DAILY_GAP.to_excel(xw, sheet_name="Daily_gap", index=False)
    pd.DataFrame([
        {"key": "script", "value": "S2c_TaskExport_P1_v3.0"},
        {"key": "cache version", "value": CACHE_VERSION},
        {"key": "feature set", "value": "cumulative from D1, one side per sheet"},
        {"key": "band_sigma (k)", "value": BAND_SIGMA},
        {"key": "band_window (W)", "value": SIGMA_WINDOW},
        {"key": "rows per sheet", "value": len(_ref)},
        {"key": "features used", "value": "7 at D2, 10 at D3, 13 at D4"},
        {"key": "ablation budget", "value": int(max(ABLATION_FEATURE_COUNTS))},
        {"key": "date range",
         "value": f"{WIDE.expiry_date.min().date()} → {WIDE.expiry_date.max().date()}"},
        {"key": "band_width_pct in features", "value": P1_INCLUDE_BAND_WIDTH},
        {"key": "target",
         "value": "one-vs-rest per side; 0 includes the other side's breaches"},
    ]).to_excel(xw, sheet_name="Provenance", index=False)

print(f"\n  ✅ S2c_TaskExport_P1_v3.0 COMPLETE")
print(f"\n  📂 {_out}")
print("     sheets: "
      + ", ".join([f"D{d}_{s}" for d in (2, 3, 4) for s in ("upper", "lower")])
      + ", Summary, Formulas, All_cycles, Daily_gap, Provenance")
print("=" * 78)


  📦 S2c_TaskExport_P1_v3.0 — σ=0.5, window=8

  Building the six task frames
     ℹ️  warm-up: dropped 6 cycle(s) with undefined rolling μ/σ
     upper_D2   rows= 291  features=  7  breaches=  82  EPV=11.7
     lower_D2   rows= 291  features=  7  breaches=  83  EPV=11.9
     upper_D3   rows= 291  features= 10  breaches=  82  EPV=8.2
     lower_D3   rows= 291  features= 10  breaches=  83  EPV=8.3
     upper_D4   rows= 291  features= 13  breaches=  82  EPV=6.3
     lower_D4   rows= 291  features= 13  breaches=  83  EPV=6.4
     ✅ all six task frames hold the same 291 cycles

  Sheet summary
   sheet     task  day direction  rows  n_features_used  n_features_budget  breach  no_breach  of which OTHER side breached  base_rate   EPV
D2_upper upper_D2    2     upper   291                7                 13      82        209                            83     0.2818 11.71
D2_lower lower_D2    2     lower   291                7                 13      83        208                            

In [44]:
# @title
# ============================================================================
# S3_Train_P1_v4.0 — MODEL TRAINING
# ============================================================================
#
# PURPOSE
# -------
# Train every (task × model × feature-count × sigma × window) combination and
# write one cache file per combination. This is the only cell that fits a
# model. It selects nothing, compares nothing and reports no result: those are
# S3_Select and S6.
#
#   IN  : df_cycles, plus every definition from S1
#   OUT : one pickle per combination under MODEL_CACHE_DIR
#
# WHAT ONE CACHE FILE CONTAINS
# ----------------------------
#   per_fold           (in-fold, out-of-fold, test) score triples
#   oos_prob/_thr/     out-of-sample probabilities, EACH WITH THE THRESHOLD
#     _fold/_cycle_id  FROM ITS OWN FOLD — see the leakage note below
#   features           the feature names actually fitted
#   n_features         the ablation BUDGET (a cap, identical for every task)
#   n_features_used    the real count: 7 at D2, 10 at D3, 13 at D4
#   scaler,            the deployable preprocessing pipeline
#     clip_bounds,
#     no_scale_idx
#   final_model        refit on all history, for LIVE inference only
#   threshold          the LIVE cut. threshold_scope says "live_only" — it is
#                      not the cut used to score out-of-sample rows
#   feature_audit      one row per feature per fold
#
# THE LEAKAGE CONTROL THAT MATTERS MOST
# -------------------------------------
# An out-of-sample probability must be thresholded with a cut that was fitted
# WITHOUT seeing that row's label. Reading one global `threshold` and applying
# it to every out-of-sample probability would threshold a large share of test
# cycles on their own labels. So every out-of-sample prediction carries
# `oos_thr` — the cut from the fold that produced it — and S6 refuses to run
# on a cache that lacks it.
#
# HOW ONE COMBINATION IS TRAINED
# ------------------------------
#   for each walk-forward fold (expanding window, oldest first):
#       fit block = train + val, test block sealed
#       tune hyperparameters      nested search on the fit block only
#       cross-fit the threshold   stratified K-fold inside the fit block, so
#                                 every row gets an out-of-fold probability
#                                 from a model that never saw it
#       score the sealed test block once
#   then refit on all history for the live model
#
# The cross-fit is what makes the comparison symmetric: ML and the Gaussian
# fit their decision thresholds on the same block of data.
#
# WHY THE LEARNED MODELS ARE TUNED AND THE GAUSSIAN IS NOT
# --------------------------------------------------------
# The Gaussian baseline has no free parameters. If the learned models were
# left at hand-typed defaults and lost, the finding would be worthless —
# "you did not tune it" would answer the whole study. Tuning ML while the
# analytical baseline stays parameter-free deliberately stacks the comparison
# in ML's favour, so a null result under that asymmetry is a real result.
#
# FEATURE SELECTION DOES NOT HAPPEN
# ---------------------------------
# ABLATION_FEATURE_COUNTS is set at or above the largest feature set, so
# select_features() always takes its p <= k branch and returns every feature.
# The mutual-information path below is retained for completeness but is never
# reached in this study. Consequently the feature audit records how often a
# combination was TRAINED, not how important a feature is.
#
# RESUMING
# --------
# The grid loop skips any combination whose cache file already exists. Set
# ENABLE_MODEL_CACHE = False to force a retrain. RUN_ONLY narrows the grid for
# a smoke test, e.g. RUN_ONLY = {"sigma": [0.5], "window": [4]}.
#
# RUN AFTER: S2b_TaskFrames.  NEXT: S3_Harvest → S3_Select.
# ============================================================================

# ── Guard: S1 v3 + S2 v3 ────────────────────────────────────────────────────
try:
    _ = (TASKS, SIGMA_GRID, SIGMA_WINDOW_GRID, ABLATION_FEATURE_COUNTS,
         MODEL_CACHE_DIR, RUN_TYPE, RANDOM_STATE, PROJECT1_MODELS, USE_NN_P1,
         CLEAN_INPUT_PATH, CACHE_VERSION, ENABLE_MODEL_CACHE, N_SEEDS,
         CYCLE_DECISION_DAYS, CYCLE_STEPS, P1_PRIMARY_METRIC,
         P1_CROSSFIT_THRESHOLD, P1_CROSSFIT_FOLDS, P1_DROP_WARMUP)
    _ = (compute_bands_and_labels, get_task_features, clip_and_scale,
         best_f1_threshold, walk_forward_splits, stratified_kfold_indices,
         cache_key, nn_meta_key, nn_seed_key, feature_class, apply_saved_pipeline,
         folds_consistency_pass, load_input, primary_score, all_scores,
         project1_add_realized_vol_Dn, project1_add_time_features,
         project1_core_features, drop_warmup_cycles)
    _ = build_cycle_record           # S2
    _ = standard_cycles              # S2 v3
    try:
        df_cycles
    except NameError:
        df_cycles = load_cycles_checkpoint()
        print("  ℹ️  df_cycles reloaded from checkpoint.")
except NameError as _ne:
    raise RuntimeError(f"❌ Missing {_ne}. Run S1_v3 → S2_v3 before S3_Train_v3.")

import os, time, warnings
import numpy as np
import pandas as pd
import joblib
from copy import deepcopy
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import CalibratedClassifierCV
warnings.filterwarnings("ignore")

_HAS_XGB = _HAS_CAT = False
try:
    import xgboost as xgb; _HAS_XGB = True
except Exception: pass
try:
    import catboost as cb; _HAS_CAT = True
except Exception: pass
if USE_NN_P1:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader

print("\n" + "=" * 78)
print(f"  🏋️ S3_Train_P1_v4.0 — TRAINING  (world: {RUN_TYPE})")
print(f"  Cache version : {CACHE_VERSION}")
print(f"  Primary metric: {PRIMARY_METRIC_NAME}")
print(f"  Threshold     : {'CROSS-FIT on train+val' if P1_CROSSFIT_THRESHOLD else 'inner-val (ASYMMETRIC)'}"
      f"  K={P1_CROSSFIT_FOLDS}")
print("=" * 78)


# ============================================================================
# A. MODEL SET + GRID
# ============================================================================
RUN_ONLY   = globals().get("RUN_ONLY", None)   # e.g. {"sigma": [0.5], "window": [4]}
PEEK_FRAME = globals().get("PEEK_FRAME", True)  # print the frame each task trains on


def _sel(grid, key):
    """Narrow one grid axis when RUN_ONLY is set.

    Parameters
    ----------
    grid : list
        The full axis from S1, e.g. SIGMA_GRID.
    key : str
        Axis name: "sigma", "window" or "features".

    Returns
    -------
    list
        The intersection with RUN_ONLY[key], or the full grid when RUN_ONLY
        does not constrain this axis.

    Notes
    -----
    RUN_ONLY narrows what is TRAINED, not what is cached or harvested. A
    smoke test therefore leaves the rest of the grid untrained rather than
    invalid.
    """
    if RUN_ONLY and key in RUN_ONLY and RUN_ONLY[key]:
        return [g for g in grid if g in RUN_ONLY[key]]
    return grid


_sigmas   = _sel(SIGMA_GRID, "sigma")
_windows  = _sel(SIGMA_WINDOW_GRID, "window")
_featcnts = _sel(ABLATION_FEATURE_COUNTS, "features")

ACTIVE_MODELS = [m for m in PROJECT1_MODELS
                 if not (m == "XGBoost" and not _HAS_XGB)
                 and not (m == "CatBoost" and not _HAS_CAT)]
_dropped = [m for m in PROJECT1_MODELS if m not in ACTIVE_MODELS]
if _dropped:
    print(f"  ⚠️  Unavailable, skipped: {_dropped}")

# Calibration only changes anything for a proper scoring rule: an isotonic or
# sigmoid map is MONOTONE, so it cannot alter an F1-optimal decision. Skipping
# it under metric='f1' saves ~3x the fits for an identical result.
P1_CALIBRATE = (P1_PRIMARY_METRIC == "brier")
CALIB_FOLDS  = 3

print(f"  Grid  : σ{_sigmas} × win{_windows} × feat{_featcnts}")
print(f"  Models: {ACTIVE_MODELS}  | calibrate={P1_CALIBRATE}")


# ============================================================================
# B. THE SHARED FRAME BUILDER
# ----------------------------------------------------------------------------
#  ONE definition of "the rows this task trains and is evaluated on". S6
#  imports this. If S3 and S6 filter differently by even one row, the
#  walk-forward index positions shift and Normal's per-fold thresholds stop
#  lining up with ML's cached OOS cycles — silently.
# ============================================================================
_daily_p1 = load_input(CLEAN_INPUT_PATH)
print(f"  Daily clean: {len(_daily_p1)} rows "
      f"({_daily_p1[DATE_COL].min().date()} → {_daily_p1[DATE_COL].max().date()})")


def p1_attach_features(frame, window, strict=False):
    """Config-dependent realized_vol_Dn + time features."""
    out = project1_add_realized_vol_Dn(frame, _daily_p1, window,
                                       cycle_days=CYCLE_DECISION_DAYS, strict=strict)
    out = project1_add_time_features(out)
    return out


def p1_build_task_frame(dfc, sigma, window, task, verbose=False):
    """df_cycles → the exact model-ready frame for ONE (config, task).

    Order matters and is fixed here so every consumer agrees:
        standard cycles → bands+labels → drop warm-up → attach features
        → drop rows with a non-finite whitelisted feature or label
        → sort by expiry_date → reset_index

    Returns (frame, feature_names, n_dropped_nan).
    """
    base = standard_cycles(dfc)
    b = compute_bands_and_labels(base, sigma, window, make_labels=True)
    if P1_DROP_WARMUP:
        b = drop_warmup_cycles(b, verbose=verbose)
    b = p1_attach_features(b, window)
    b = b.sort_values("expiry_date").reset_index(drop=True)

    tlabel = task["label_col"]
    num_cols = [c for c in b.columns if b[c].dtype.kind in "fi"]
    feats = get_task_features(task, num_cols)            # whitelist; hard-fails

    # drop, never impute. v2 called np.nan_to_num(..., nan=0.), which
    # turned a missing realized_vol into a volatility of exactly zero.
    ok = b[tlabel].notna() & np.isfinite(b[feats]).all(axis=1)
    n_drop = int((~ok).sum())
    b = b[ok].sort_values("expiry_date").reset_index(drop=True)
    return b, feats, n_drop


# ============================================================================
# C. PRE-FLIGHT
# ============================================================================
print("\n  🔎 Pre-flight: features exist and are populated at every config")
_pf_ok = True
for _sg in _sigmas[:1]:
    for _w in _windows:
        for _t in TASKS:
            try:
                _fr, _ft, _nd = p1_build_task_frame(df_cycles, _sg, _w, _t)
                _status = "✅" if len(_fr) >= 40 else "⚠️"
                if len(_fr) < 40: _pf_ok = False
                print(f"     {_status} σ={_sg} w={_w:<2} {_t['name']:<9} "
                      f"n={len(_fr):>4} feats={len(_ft):>2} dropped(NaN)={_nd}")
            except Exception as _e:
                _pf_ok = False
                print(f"     ❌ σ={_sg} w={_w} {_t['name']}: {type(_e).__name__}: {str(_e)[:90]}")
if not _pf_ok:
    raise RuntimeError(
        "❌ Pre-flight failed. Likely causes:\n"
        "   • S2 is not v3.0 (missing cycle_status / d1_date..d4_date)\n"
        "   • the clean daily file does not cover the cycle dates\n"
        "   • a window is so long that warm-up leaves too few cycles")
print("     ✅ Pre-flight passed")


# ============================================================================
# D. MODEL FACTORY
# ============================================================================
def make_model(name, params=None):
    """Build an estimator. `params` overrides the a-priori defaults.

    🔶 v4 When params is None the estimator is IDENTICAL to v3, so an untuned
    run reproduces the previous results exactly.
    """
    params = dict(params or {})
    if name == "LogisticRegression":
        kw = dict(C=1.0, class_weight="balanced",
                  max_iter=1000, random_state=RANDOM_STATE)
        kw.update(params)
        return LogisticRegression(**kw)
    if name == "SVM":
        #  probability=True runs SVC's own internal Platt calibration, and
        #  _wrap_calibrated adds another layer on top. That is slow but safe:
        #  turning it off would break predict_proba on the uncalibrated
        #  fallback paths.
        kw = dict(C=1.0, kernel="rbf", gamma="scale", probability=True,
                  class_weight="balanced", random_state=RANDOM_STATE)
        kw.update(params)
        return SVC(**kw)
    if name == "RandomForest":
        kw = dict(n_estimators=200, max_depth=6, min_samples_leaf=2,
                  class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
        kw.update(params)
        return RandomForestClassifier(**kw)
    if name == "XGBoost":
        kw = dict(n_estimators=200, max_depth=3, learning_rate=0.05,
                  subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                  tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE)
        kw.update(params)
        return xgb.XGBClassifier(**kw)
    if name == "CatBoost":
        return cb.CatBoostClassifier(iterations=200, depth=4, learning_rate=0.05,
                                     l2_leaf_reg=3, auto_class_weights="Balanced",
                                     verbose=False, random_seed=RANDOM_STATE, thread_count=2)
    raise ValueError(f"Unknown model '{name}' "
                     f"('NN_FF' is handled by train_nn(), not make_model()).")


def _wrap_calibrated(est, n_pos):
    """Sigmoid (Platt) calibration, cross-fitted inside the training data.

    Platt not isotonic: 2 parameters versus a step function, and at ~10-30
    positives isotonic overfits badly. cv=CALIB_FOLDS (not 'prefit') so the
    calibration set is never the model's own training rows.
    """
    if not P1_CALIBRATE:
        return est
    k = int(min(CALIB_FOLDS, max(2, n_pos)))
    if n_pos < 2:
        return est
    try:
        return CalibratedClassifierCV(est, method="sigmoid", cv=k)
    except Exception:
        return est


# ============================================================================
# E. NEURAL NET (only when P1_ENABLE_NN)
# ============================================================================
if USE_NN_P1:
    def _loader(X, y, bs, shuffle):
        """Wrap a matrix and labels in a torch DataLoader.

        Parameters
        ----------
        X : ndarray
            Feature matrix.
        y : ndarray
            Integer labels.
        bs : int
            Batch size.
        shuffle : bool
            Shuffle between epochs.

        Returns
        -------
        torch.utils.data.DataLoader
            Batches of (features, label).

        Notes
        -----
        A trailing batch of size one is avoided by shrinking the batch size by
        one, because BatchNorm cannot compute a variance from a single row.
        """
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                           torch.tensor(y, dtype=torch.long))
        sbs = bs - 1 if len(y) % bs == 1 else bs
        return DataLoader(ds, batch_size=max(2, sbs), shuffle=shuffle)

    def train_nn(Xtr, ytr, Xvl, yvl, seed, nf, hp=None):
        """Fit one neural network with early stopping on a held-out slice.

        Parameters
        ----------
        Xtr, ytr : ndarray
            Rows the network learns from.
        Xvl, yvl : ndarray
            Rows used ONLY to choose the stopping epoch.
        seed : int
            Torch and numpy seed for this network.
        nf : int
            Number of input features.
        hp : dict or None
            Hyperparameters from the search: lr, weight_decay, batch_size,
            h1, h2, dropout. None uses the pinned defaults from S1.

        Returns
        -------
        NiftyBinaryFF
            The network restored to its best-scoring epoch.

        Notes
        -----
        AdamW, class-weighted cross-entropy with label smoothing, gradient
        clipping at 1.0, at most 300 epochs, and early stopping after 30
        epochs without improvement on the validation slice.

        The validation slice is used ONLY for the stopping epoch. It is inside
        the fitting block, never the sealed test block.
        """
        hp = dict(hp or {})
        _lr  = float(hp.get("lr", 1e-3))
        _wd  = float(hp.get("weight_decay", 1e-3))
        _bs  = int(hp.get("batch_size", 16))
        _h1  = int(hp.get("h1", NN_H1))       # pinned by S1, never searched
        _h2  = int(hp.get("h2", NN_H2))
        _do  = float(hp.get("dropout", DROPOUT))
        torch.manual_seed(seed); np.random.seed(seed)
        m = NiftyBinaryFF(nf, do=_do, h1=_h1, h2=_h2).to(device)
        w = torch.tensor([1.0, 1.0], dtype=torch.float32).to(device)
        if len(np.unique(ytr)) == 2:
            c0, c1 = (ytr == 0).sum(), (ytr == 1).sum()
            w = torch.tensor([len(ytr) / (2 * max(c0, 1)), len(ytr) / (2 * max(c1, 1))],
                             dtype=torch.float32).to(device)
        opt = torch.optim.AdamW(m.parameters(), lr=_lr, weight_decay=_wd)
        tl, vl = _loader(Xtr, ytr, _bs, True), _loader(Xvl, yvl, _bs, False)
        best, best_state, bad = -1.0, None, 0
        for _ep in range(300):
            m.train()
            for xb, yb in tl:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                F.cross_entropy(m(xb), yb, weight=w, label_smoothing=0.05).backward()
                nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
            m.eval(); pr, la = [], []
            with torch.no_grad():
                for xb, yb in vl:
                    pr.extend(torch.softmax(m(xb.to(device)), -1)[:, 1].cpu().numpy())
                    la.extend(yb.numpy())
            sc = primary_score(la, np.array(pr), (np.array(pr) >= 0.5).astype(int))
            sc = -1.0 if not np.isfinite(sc) else sc
            if sc > best + 1e-4: best, best_state, bad = sc, deepcopy(m.state_dict()), 0
            else:
                bad += 1
                if bad >= 30: break
        if best_state is not None: m.load_state_dict(best_state)
        return m

    def nn_prob(models, X):
        """Average P(breach) across a set of seed-averaged networks.

        Parameters
        ----------
        models : sequence of NiftyBinaryFF
            One network per seed.
        X : ndarray
            Feature matrix to score.

        Returns
        -------
        ndarray of float
            Mean softmax probability of the positive class.

        Notes
        -----
        Seed averaging is variance reduction applied to the SELECTED model. It
        is not part of model selection, so it does not bias the comparison.
        """
        ps = []
        for m in models:
            m.eval()
            with torch.no_grad():
                ps.append(torch.softmax(m(torch.tensor(X, dtype=torch.float32).to(device)),
                                        -1).cpu().numpy()[:, 1])
        return np.mean(ps, axis=0)


# ============================================================================
# F. FEATURE SELECTION  (p<=k branch returns ALL — expected in Project 1)
# ============================================================================
def select_features(X_scaled, y, names, k):
    """Rank features by mutual information and keep the top k.

    Parameters
    ----------
    X_scaled : ndarray
        Scaled training matrix.
    y : ndarray
        Training labels.
    names : list of str
        Feature names, aligned to X_scaled's columns.
    k : int
        How many to keep.

    Returns
    -------
    (list of str, list of int, dict, dict)
        (selected names, their column indices, name → rank, name → MI score).

    Notes
    -----
    When p <= k this returns EVERY feature untouched, and that is the branch
    this study always takes. Nothing is selected, so the rank map is positional
    and must not be read as importance.
    """
    p = X_scaled.shape[1]
    if p <= k:
        idx = list(range(p))
        return list(names), idx, {names[i]: i + 1 for i in idx}, {}
    mi = mutual_info_classif(X_scaled, y, random_state=RANDOM_STATE)
    mi_map = {names[i]: float(mi[i]) for i in range(p)}
    order = np.argsort(mi)[::-1][:k]
    sel = sorted(order.tolist())
    return ([names[i] for i in sel], sel,
            {names[order[r]]: r + 1 for r in range(len(order))}, mi_map)


# ============================================================================
# G. THE CROSS-FIT CORE
# ============================================================================
def _fit_predict(model_name, Xa, ya, Xb_list, seed_offset=0, params=None,
                 n_seeds=None):
    """Fit on (Xa, ya); return P(breach) for each matrix in Xb_list.

    🔶 v4 `params` carries the hyperparameters chosen by the nested search on
    this fold's fitting block. None = the v3 a-priori defaults.
    """
    if model_name == "NN_FF":
        cut = max(4, int(len(ya) * 0.8))
        ns  = int(n_seeds if n_seeds is not None else N_SEEDS)
        ms = [train_nn(Xa[:cut], ya[:cut], Xa[cut:], ya[cut:],
                       RANDOM_STATE + seed_offset + s, Xa.shape[1], hp=params)
              for s in range(ns)]
        return [nn_prob(ms, Xb) for Xb in Xb_list], ms
    est = _wrap_calibrated(make_model(model_name, params), int(np.sum(ya)))
    est.fit(Xa, ya)
    return [est.predict_proba(Xb)[:, 1] for Xb in Xb_list], est


# ============================================================================
# F2. NESTED HYPERPARAMETER SEARCH
# ============================================================================
#  Called ONCE PER OUTER FOLD, on that fold's fitting block (train ∪ val) only.
#  The fold's test block is not passed in and cannot be reached from here.
#
#  Scaling lives INSIDE the search pipeline, so every inner fold refits its own
#  scaler. Fitting one scaler on the whole block and then cross-validating
#  would leak the block's own location and spread into each inner validation
#  split — a small effect, but the entire point of this study is that the
#  leakage controls are airtight.
_TUNE_LOG = []


def _tune_scoring():
    """The sklearn scoring string matching P1_PRIMARY_METRIC.

    Returns
    -------
    str
        "neg_brier_score" or "f1".

    Notes
    -----
    The hyperparameter search must optimise the same quantity the consistency
    gate and model selection use, or the search would tune for one objective
    and be judged on another.
    """
    return "neg_brier_score" if P1_PRIMARY_METRIC == "brier" else "f1"


def tune_sklearn(model_name, X_fit, y_fit, seed=RANDOM_STATE):
    """Randomised search on the fitting block. Returns (params, n_candidates)."""
    space = HYPERPARAM_SPACES.get(model_name)
    if not space:
        return None, 0
    n_pos = int(y_fit.sum())
    n_inner = int(min(P1_TUNE_INNER_FOLDS, max(2, n_pos)))
    if n_pos < 2 * n_inner:                      # too few positives to search
        return None, 0
    pipe = Pipeline([("sc", StandardScaler()),
                     ("clf", make_model(model_name))])
    grid = {f"clf__{k}": v for k, v in space.items()}
    n_comb = 1
    for v in space.values():
        n_comb *= len(v)
    n_iter = int(min(P1_TUNE_N_ITER, n_comb))
    try:
        rs = RandomizedSearchCV(
            pipe, grid, n_iter=n_iter,
            cv=StratifiedKFold(n_inner, shuffle=True, random_state=seed),
            scoring=_tune_scoring(), n_jobs=-1, random_state=seed,
            error_score=np.nan, refit=False)
        rs.fit(X_fit, y_fit)
        if not np.isfinite(rs.best_score_):
            return None, n_iter
        best = {k.replace("clf__", ""): v for k, v in rs.best_params_.items()}
        return best, n_iter
    except Exception as e:
        _TUNE_LOG.append(f"{model_name}: search failed ({type(e).__name__}), "
                         f"fell back to defaults")
        return None, 0


def tune_nn(X_fit, y_fit, seed=RANDOM_STATE):
    """Manual random search for NN_FF — sklearn cannot drive the torch loop.

    Searched with ONE seed; the winner is refit with all N_SEEDS by the caller.
    """
    if not USE_NN_P1:
        return None, 0
    n_pos = int(y_fit.sum())
    n_inner = int(min(P1_TUNE_INNER_FOLDS, max(2, n_pos)))
    if n_pos < 2 * n_inner:
        return None, 0
    rng = np.random.default_rng(seed)
    keys = list(NN_HYPERPARAM_SPACE)
    seen, cands = set(), []
    for _ in range(P1_TUNE_NN_N_ITER * 6):
        c = tuple(NN_HYPERPARAM_SPACE[k][rng.integers(len(NN_HYPERPARAM_SPACE[k]))]
                  for k in keys)
        if c not in seen:
            seen.add(c); cands.append(dict(zip(keys, c)))
        if len(cands) >= P1_TUNE_NN_N_ITER:
            break
    inner = stratified_kfold_indices(y_fit, n_inner, seed)
    if not inner:
        return None, 0
    best, best_sc = None, -np.inf
    for hp in cands:
        scs = []
        for a, b in inner:
            if len(np.unique(y_fit[a])) < 2 or len(a) < 10:
                continue
            Xa, Xb = X_fit[a].copy(), X_fit[b].copy()
            mu, sd = Xa.mean(0), Xa.std(0)
            sd[sd == 0] = 1.0
            Xa, Xb = (Xa - mu) / sd, (Xb - mu) / sd
            try:
                (p_b,), _ = _fit_predict("NN_FF", Xa, y_fit[a], [Xb],
                                         params=hp, n_seeds=1)
            except Exception:
                continue
            if P1_PRIMARY_METRIC == "brier":
                s = primary_score(y_fit[b], p_b, None)
            else:
                t, s = best_f1_threshold(y_fit[b], p_b)
                s = np.nan if t is None else s
            if np.isfinite(s):
                scs.append(s)
        if scs and np.mean(scs) > best_sc:
            best_sc, best = float(np.mean(scs)), hp
    return best, len(cands)


def tune_for_fold(model_name, X_fit, y_fit, seed=RANDOM_STATE):
    """Single entry point used by the outer fold loop."""
    if not P1_TUNE_HYPERPARAMS:
        return None, 0
    if model_name == "NN_FF":
        return tune_nn(X_fit, y_fit, seed)
    return tune_sklearn(model_name, X_fit, y_fit, seed)


def crossfit_block(X, y, feats, model_name, k_eff, seed=RANDOM_STATE, params=None):
    """Out-of-fold probabilities over an ENTIRE block.

    Every row gets a probability from a model that never saw it, so the whole
    block can be used to fit the decision threshold honestly. The scaler and
    the selector are refitted INSIDE each inner fold — fitting them once on the
    full block and cross-fitting only the model would leak the block's own
    distribution into its own scaling.

    Not leakage: the block is entirely in the past relative to the sealed test
    set, and the outer walk-forward is what enforces temporal honesty. The
    inner split only affects how well the threshold is estimated.

    Returns (oof_prob, mean_infold_score, n_inner_folds).
    """
    n = len(y)
    oof = np.full(n, np.nan)
    infold = []
    inner = stratified_kfold_indices(y, P1_CROSSFIT_FOLDS, seed)
    if not inner:
        return oof, np.nan, 0
    for a, b in inner:
        if len(np.unique(y[a])) < 2 or len(a) < 10:
            continue
        Xa, Xb = X[a].copy(), X[b].copy()
        Xa, Xb, _t, sc, cb_, ns = clip_and_scale(Xa, Xb, X[b].copy(), feats)
        _sn, si, _rk, _mi = select_features(Xa, y[a], feats, k_eff)
        (p_b, p_a), _m = _fit_predict(model_name, Xa[:, si], y[a],
                                      [Xb[:, si], Xa[:, si]], params=params)
        oof[b] = p_b
        # in-fold score at its OWN optimal cut — deliberately the optimistic
        # number, because the consistency gate measures in-fold vs out-of-fold
        if P1_PRIMARY_METRIC == "brier":
            infold.append(primary_score(y[a], p_a, None))
        else:
            t_a, f_a = best_f1_threshold(y[a], p_a)
            infold.append(f_a if t_a is not None else np.nan)
    m_in = float(np.nanmean(infold)) if len(infold) else np.nan
    return oof, m_in, len(inner)


# ============================================================================
# H. TRAIN ONE COMBINATION  →  cache dict
# ============================================================================
def train_combo(task, model_name, n_feat, sigma, window):
    """Train one (task, model, feature count, sigma, window) combination.

    Parameters
    ----------
    task : dict
        One entry of TASKS.
    model_name : str
        A member of PROJECT1_MODELS.
    n_feat : int
        The ablation budget passed to select_features.
    sigma : float
        Band multiplier.
    window : int
        Rolling window length in cycles.

    Returns
    -------
    dict or None
        The cache dict described in this cell's header, or None when the task
        frame has too few rows or no usable folds.

    Raises
    ------
    RuntimeError
        If any feature value is non-finite. Nothing is ever zero-filled.

    Notes
    -----
    Walk-forward folds are scored independently and the sealed test block of
    each fold is touched exactly once, after that fold's tuning and threshold
    fitting are complete.
    """
    tname, tlabel, tday = task["name"], task["label_col"], task["day"]

    v, raw_feats, n_drop = p1_build_task_frame(df_cycles, sigma, window, task)
    if len(v) < 40 or not raw_feats:
        return None
    y_all = v[tlabel].astype(int).values
    X_all = v[raw_feats].to_numpy(dtype=np.float64)

    # ── PROOF-OF-INPUT PRINT ────────────────────────────────────────────────
    #  Prints the last two rows of the frame the model is about to be fitted
    #  on, once per (task, sigma, window). This is the direct evidence that
    #  the features named in the whitelist are the ones actually reaching the
    #  estimator — useful when defending the pipeline, and cheap at one
    #  configuration. Set PEEK_FRAME = False to silence it.
    if globals().get("PEEK_FRAME", True):
        _pk = (tname, float(sigma), int(window))
        if _pk not in globals().setdefault("_PEEK_SEEN", set()):
            globals()["_PEEK_SEEN"].add(_pk)
            _cols = ([c for c in ("cycle_id", "expiry_date") if c in v.columns]
                     + list(raw_feats) + [tlabel])
            with pd.option_context("display.width", 250, "display.max_columns", 60):
                print(f"\n  ── FRAME {tname} | σ={sigma} win={window} | "
                      f"{len(v)} rows × {len(raw_feats)} features | "
                      f"y: {int(v[tlabel].sum())} breach / "
                      f"{int((v[tlabel] == 0).sum())} no-breach | dropped {n_drop}")
                print(v[_cols].tail(2).to_string(index=False))

    if not np.isfinite(X_all).all():
        raise RuntimeError(f"❌ non-finite features survived for {tname} — "
                           f"p1_build_task_frame should have dropped them.")
    k_eff = min(int(n_feat), len(raw_feats))

    splits = walk_forward_splits(len(v))
    if not splits:
        return None

    per_fold, audit_rows, fold_params = [], [], []
    oos_cid, oos_prob, oos_thr, oos_fold = [], [], [], []
    fold_notes = []

    for fold, (idx_tr, idx_vl, idx_te) in enumerate(splits, 1):
        # the fitting block is train ∪ val. NEVER test.
        fit_idx = np.concatenate([idx_tr, idx_vl])
        y_fit, X_fit = y_all[fit_idx], X_all[fit_idx]
        if len(np.unique(y_fit)) < 2 or int(y_fit.sum()) < 4:
            fold_notes.append(f"fold{fold}: skipped (only {int(y_fit.sum())} positives)")
            continue

        # NESTED SEARCH. Fitting block only — idx_te is not in scope here.
        _Xs = X_fit.copy()
        _mu, _sd = _Xs.mean(0), _Xs.std(0); _sd[_sd == 0] = 1.0
        hp, n_cand = tune_for_fold(model_name, (_Xs - _mu) / _sd, y_fit,
                                   RANDOM_STATE + fold)
        fold_params.append({"fold": int(fold), "n_candidates": int(n_cand),
                            "params": dict(hp) if hp else None})

        oof, infold_sc, n_inner = crossfit_block(X_fit, y_fit, raw_feats,
                                                 model_name, k_eff, params=hp)
        good = np.isfinite(oof)
        if good.sum() < 20 or len(np.unique(y_fit[good])) < 2:
            fold_notes.append(f"fold{fold}: skipped (cross-fit produced {int(good.sum())} usable rows)")
            continue

        # ── the fold threshold: SAME function, SAME block that Normal uses ──
        thr, _ = best_f1_threshold(y_fit[good], oof[good])
        if thr is None:
            fold_notes.append(f"fold{fold}: skipped (threshold undefined, <2 positives)")
            continue

        oof_pred  = (oof[good] >= thr).astype(int)
        oof_sc    = primary_score(y_fit[good], oof[good], oof_pred)
        oof_stats = all_scores(y_fit[good], oof[good], oof_pred)

        # ── the deployable fold model: trained on the FULL fitting block ──
        Xf = X_fit.copy(); Xt = X_all[idx_te].copy()
        Xf, Xt, _t, sc_f, cb_f, ns_f = clip_and_scale(Xf, Xt, X_all[idx_te].copy(), raw_feats)
        sn, si, rankmap, mimap = select_features(Xf, y_fit, raw_feats, k_eff)
        (p_te,), _m = _fit_predict(model_name, Xf[:, si], y_fit, [Xt[:, si]],
                                   params=hp)

        y_te = y_all[idx_te]
        te_pred = (p_te >= thr).astype(int)
        te_sc   = primary_score(y_te, p_te, te_pred)

        per_fold.append((float(infold_sc), float(oof_sc), float(te_sc)))
        # each test cycle carries ITS OWN fold's leak-free threshold
        oos_cid  += v.iloc[idx_te]["cycle_id"].astype(int).tolist()
        oos_prob += [float(x) for x in p_te]
        oos_thr  += [float(thr)] * len(idx_te)
        oos_fold += [int(fold)] * len(idx_te)

        for f in sn:
            audit_rows.append({
                "task": tname, "direction": task["direction"], "day": tday,
                "model": model_name, "sigma": float(sigma), "window": int(window),
                "n_features": int(n_feat), "fold": fold, "feature": f,
                "feature_class": feature_class(f),
                "perm_rank": rankmap.get(f, np.nan), "mi_score": mimap.get(f, np.nan)})

        fold_notes.append(
            f"fold{fold}: n_fit={len(fit_idx)} pos={int(y_fit.sum())} inner={n_inner} "
            f"thr={thr:.3f} infold={infold_sc:.3f} oof={oof_sc:.3f} test={te_sc:.3f} "
            f"| oof_brier={oof_stats['brier']:.4f} rel={oof_stats['reliability']:.4f}"
            f" | tuned={'defaults' if not hp else hp}")

    if not per_fold:
        return None

    mean_in  = float(np.nanmean([a for a, _, _ in per_fold]))
    mean_oof = float(np.nanmean([b for _, b, _ in per_fold]))
    mean_te  = float(np.nanmean([c for _, _, c in per_fold]))
    passed, _mv = folds_consistency_pass([(a, b) for a, b, _ in per_fold])

    # ── FINAL deployable model (LIVE ONLY) ──────────────────────────────────
    #  Trained on ALL cycles, with the threshold cross-fitted over all cycles.
    #  v2 trained on the first 80% and fitted the threshold on the last 20% —
    #  which is what createdonce S6 reused that threshold for the OOS
    #  comparison. It is now confined to live inference, where "all history"
    #  is exactly what Normal also gets.
    _Xa = X_all.copy()
    _mu2, _sd2 = _Xa.mean(0), _Xa.std(0); _sd2[_sd2 == 0] = 1.0
    hpF, _ncF = tune_for_fold(model_name, (_Xa - _mu2) / _sd2, y_all, RANDOM_STATE)
    oof_all, _in_all, _ = crossfit_block(X_all, y_all, raw_feats, model_name,
                                         k_eff, params=hpF)
    good_all = np.isfinite(oof_all)
    thrF, _ = best_f1_threshold(y_all[good_all], oof_all[good_all])
    if thrF is None:
        thrF = 0.5
    XA = X_all.copy()
    XA, _b, _t2, scalerF, clipsF, nsF = clip_and_scale(XA, X_all.copy(), X_all.copy(), raw_feats)
    snF, siF, _rk, _mi = select_features(XA, y_all, raw_feats, k_eff)
    (_pa,), mF = _fit_predict(model_name, XA[:, siF], y_all, [XA[:, siF]],
                              params=hpF)
    if model_name == "NN_FF":
        for s, m in enumerate(mF):
            try:
                sp = nn_seed_key(tname, s, n_feat, sigma, window)
                os.makedirs(os.path.dirname(sp), exist_ok=True)
                torch.save(m.state_dict(), sp)
            except Exception: pass
        mF = {"nn_state_dicts": [m.state_dict() for m in mF],
              "nf": len(siF), "n_seeds": int(N_SEEDS)}

    return {
        "task": tname, "direction": task["direction"], "day": tday,
        "model": model_name,                       # stays the NAME string
        "sigma": float(sigma), "window": int(window),
        "n_features": int(n_feat), "n_features_used": int(len(snF)),
        "cache_version": CACHE_VERSION, "world": RUN_TYPE,
        "tuned": bool(P1_TUNE_HYPERPARAMS),
        "fold_params": fold_params,          # per-fold chosen hyperparams
        "live_params": dict(hpF) if hpF else None,
        "feature_mode": P1_FEATURE_MODE, "primary_metric": P1_PRIMARY_METRIC,
        "crossfit_folds": int(P1_CROSSFIT_FOLDS), "calibrated": bool(P1_CALIBRATE),
        "n_cycles": int(len(v)), "n_dropped_nan": int(n_drop),
        "base_rate": float(y_all.mean()), "n_folds": len(per_fold),

        # per-fold triple, SAME SHAPE as v2 so Harvest needs no restructuring;
        # semantics are now (infold_score, oof_score, test_score)
        "per_fold": per_fold,
        "mean_infold_score": mean_in, "mean_oof_score": mean_oof, "mean_test_score": mean_te,
        # legacy aliases so nothing downstream KeyErrors
        "mean_inner_train_f1": mean_in, "mean_val_f1": mean_oof, "mean_test_f1": mean_te,
        "folds_consistency_pass": bool(passed),

        # out-of-sample predictions carry their OWN fold's threshold
        "oos_cycle_id": oos_cid, "oos_prob": oos_prob,
        "oos_thr": oos_thr, "oos_fold": oos_fold,

        # deployable pipeline — `threshold` is for LIVE INFERENCE ONLY
        "features": snF, "threshold": float(thrF), "threshold_scope": "live_only",
        "scaler": scalerF, "clip_bounds": clipsF, "no_scale_idx": nsF,
        "final_model": mF,
        "feature_audit": audit_rows, "fold_notes": fold_notes,
    }


# ============================================================================
# I. MAIN GRID LOOP (auto-resume)
# ============================================================================
for _d in [d for d in [globals().get("WORLD_DIR"), MODEL_CACHE_DIR,
                       globals().get("GRID_DIR")] if d]:
    os.makedirs(_d, exist_ok=True)
try:
    _probe = os.path.join(MODEL_CACHE_DIR, "_write_probe.tmp")
    with open(_probe, "w") as _fh: _fh.write("ok")
    os.remove(_probe)
except Exception as _e:
    raise RuntimeError(f"❌ Cannot write to MODEL_CACHE_DIR:\n     {MODEL_CACHE_DIR}\n"
                       f"   {type(_e).__name__}: {_e}\n"
                       f"   Google Drive is probably not mounted. Re-run S1.")

_combos = [(sg, w, nf, t, m)
           for sg in _sigmas for w in _windows for nf in _featcnts
           for t in TASKS for m in ACTIVE_MODELS]
print(f"\n  🚀 {len(_combos)} combinations "
      f"({len(_sigmas)}σ × {len(_windows)}win × {len(_featcnts)}feat "
      f"× {len(TASKS)}tasks × {len(ACTIVE_MODELS)}models)")

_t0 = time.time(); _done = _skip = _none = _err = 0
for _i, (sg, w, nf, task, mname) in enumerate(_combos, 1):
    ck = (nn_meta_key(task["name"], nf, sg, w) if mname == "NN_FF"
          else cache_key(task["name"], mname, nf, sg, w))
    if ENABLE_MODEL_CACHE and os.path.exists(ck):
        _skip += 1; continue
    try:
        res = train_combo(task, mname, nf, sg, w)
    except Exception as _e:
        _err += 1
        print(f"     ❌ [{_i}/{len(_combos)}] σ={sg} w={w} {task['name']} {mname}: "
              f"{type(_e).__name__}: {str(_e)[:110]}")
        continue
    if res is None:
        _none += 1
        print(f"     ⚪ [{_i}/{len(_combos)}] σ={sg} w={w} {task['name']} {mname}: "
              f"no usable folds")
        continue
    os.makedirs(os.path.dirname(ck), exist_ok=True)
    joblib.dump(res, ck)
    _done += 1
    print(f"     ✅ [{_i}/{len(_combos)}] σ={sg} w={w:<2} {task['name']:<9} {mname:<19} "
          f"folds={res['n_folds']} infold={res['mean_infold_score']:+.3f} "
          f"oof={res['mean_oof_score']:+.3f} test={res['mean_test_score']:+.3f} "
          f"{'PASS' if res['folds_consistency_pass'] else 'fail'} "
          f"n={res['n_cycles']} oos={len(res['oos_cycle_id'])}")

print("\n" + "=" * 78)
print(f"  ✅ S3_Train_P1_v4.0 COMPLETE  in {(time.time()-_t0)/60:.1f} min")
print(f"     trained={_done}  cached-skip={_skip}  no-folds={_none}  errors={_err}")
print(f"     Primary metric: {PRIMARY_METRIC_NAME} | threshold cross-fit on train+val")
print(f"     🔴 [L1] every OOS probability now carries its own fold's threshold "
      f"(`oos_thr`); `threshold` is live-only.")
print("     ➡️  NEXT: S3_Harvest → S3_Select")
print("=" * 78)


  🏋️ S3_Train_P1_v4.0 — TRAINING  (world: FULL)
  Cache version : v7CWFN1w4x2FMMBX1T1_V2SB
  Primary metric: Brier skill score
  Threshold     : CROSS-FIT on train+val  K=5
  Grid  : σ[0.5, 0.75, 1.0] × win[4, 8] × feat[13]
  Models: ['LogisticRegression', 'SVM', 'XGBoost', 'NN_FF']  | calibrate=True
  ──────────────────────────────────────────────────────────────
  📥 INPUT LOADED (read-only)
     Modified : 2026-09-14 20:35
     Rows×Cols: 1901 × 9
     Range    : 2019-01-01 → 2026-09-11
  ──────────────────────────────────────────────────────────────
  Daily clean: 1901 rows (2019-01-01 → 2026-09-11)

  🔎 Pre-flight: features exist and are populated at every config
     ✅ σ=0.5 w=4  upper_D2  n= 293 feats= 7 dropped(NaN)=0
     ✅ σ=0.5 w=4  lower_D2  n= 293 feats= 7 dropped(NaN)=0
     ✅ σ=0.5 w=4  upper_D3  n= 293 feats=10 dropped(NaN)=0
     ✅ σ=0.5 w=4  lower_D3  n= 293 feats=10 dropped(NaN)=0
     ✅ σ=0.5 w=4  upper_D4  n= 293 feats=13 dropped(NaN)=0
     ✅ σ=0.5 w=4  lower_D4  

In [45]:
# @title
# ============================================================================
# S3_Harvest_P1_v4.0 — REBUILD THE LEDGER FROM THE MODEL CACHE
# ============================================================================
#
# PURPOSE
# -------
# Inventory stage. Scans MODEL_CACHE_DIR, rebuilds the ledger from scratch,
# and RECOMPUTES the consistency flag from each cache's stored per-fold
# metrics using the CURRENT S1 thresholds. Trains nothing.
#
#   IN  : MODEL_CACHE_DIR/*.pkl
#   OUT : LEDGER_PATH  (master_results_<world>.xlsx)
#
# Because the flag is recomputed rather than read, changing a filter preset
# needs only Harvest → Select, never a retrain.
#
# THREE FILTERS EVERY CACHE FILE MUST PASS
# ----------------------------------------
# 1. VERSION. Only files whose name carries the current CACHE_VERSION are
#    read. Stale worlds are counted and ignored, never mixed.
# 2. GRID. Files for a (sigma, window) outside the active grid are skipped and
#    left on disk as the audit trail for earlier runs.
# 3. STRUCTURE. The filename is parsed BEFORE the pickle is opened, so a
#    corrupt dict cannot misreport what a file contains.
#
# FEATURE COUNTS: TWO COLUMNS, ONE OF THEM A TRAP
# -----------------------------------------------
#   n_features_budget   the ablation CAP from ABLATION_FEATURE_COUNTS. It is
#                       13 for every task and appears in every filename as
#                       "feat13". NOT a count.
#   n_features_used     the number of features actually fitted: 7 at D2, 10 at
#                       D3, 13 at D4.
#
# `n_features` is kept under its original name because it is part of the
# ledger key and the cache filename. Quote `n_features_used` anywhere a
# feature count appears in the write-up.
#
# When a cache does not carry its own used-count the value is NaN, counted and
# warned about — never filled in from the budget, which would report 13 as the
# count and is the exact confusion this column exists to prevent.
#
# THE oos_thr REQUIREMENT
# -----------------------
# Rows whose cache lacks `oos_thr` came from an older trainer and would let S6
# threshold out-of-sample predictions with a cut fitted on their own labels.
# They are flagged here and refused by S3_Select.
#
# RUN AFTER: S3_Train.  NEXT: S3_Select.
# ============================================================================

try:
    _ = (MODEL_CACHE_DIR, LEDGER_PATH, GRID_DIR, RUN_TYPE, TASKS,
         CACHE_VERSION, PROJECT1_MODELS, USE_NN_P1, ABLATION_FEATURE_COUNTS,
         SIGMA_GRID, SIGMA_WINDOW_GRID, MAX_SCORE_GAP, MIN_OOF_SCORE,
         SCORE_RATIO_MIN, FOLD_PASS_MIN_FRACTION, PRIMARY_METRIC_NAME)
    _ = (parse_cache_filename, folds_consistency_pass, consistency_pass)
except NameError as _ne:
    raise RuntimeError(f"❌ Missing {_ne}. Run S1_Config_P1_v3 before S3_Harvest.")

import os, glob, joblib, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

print("\n" + "=" * 78)
print(f"  📚 S3_Harvest_P1_v4.0 — REBUILD LEDGER  (world: {RUN_TYPE})")
print(f"  Cache version accepted: {CACHE_VERSION}")
print(f"  Primary metric        : {PRIMARY_METRIC_NAME}")
print("=" * 78)

_task_by_name = {t["name"]: t for t in TASKS}



# ============================================================================
#  TRUE FEATURE COUNT — read it, never infer it from the budget
# ----------------------------------------------------------------------------
#  S3_Train writes "n_features_used": int(len(snF)) — the length of the feature
#  list actually fitted. If a cache lacks it we return NaN rather than the
#  budget, so a missing value shows up as missing instead of as a wrong 13.
# ============================================================================
_nfu_missing = []


def _nfu_of(c, fn="?"):
    """Read a cache's true feature count, refusing to guess.

    Parameters
    ----------
    c : dict
        The unpickled cache dict.
    fn : str
        Filename, recorded when the value is missing.

    Returns
    -------
    int or float
        The stored n_features_used, or NaN when absent.

    Notes
    -----
    There is deliberately NO fallback to the ablation budget. The budget is 13
    for every task, so falling back would report 13 as the count — the one
    number this column exists to avoid.
    """
    v = c.get("n_features_used")
    try:
        v = int(v)
        if v > 0:
            return v
    except (TypeError, ValueError):
        pass
    _nfu_missing.append(str(fn))
    return np.nan


# ============================================================================
# STEP 1: SCAN THE CACHE
# ============================================================================
print("\n  📌 STEP 1: Scanning cache")
pkl_files = sorted(glob.glob(os.path.join(MODEL_CACHE_DIR, "*.pkl")))
print(f"     Files found: {len(pkl_files)}")

# GRID SCOPING.  Version-scoping alone is not enough.
#  CACHE_VERSION encodes the window-grid MODE ("full"/"single") and NOT the
#  grid VALUES, and does not encode SIGMA_GRID at all. So after editing
#      SIGMA_GRID    = [1.0, 0.5]
#      _WINDOW_GRIDS = {"full": [12, 16], ...}
#  every .pkl trained under a PREVIOUS grid — σ=1.5, window=4, window=8 — still
#  carries a matching CACHE_VERSION, still parses, and was still inventoried
#  here. S3_Select then chose the best config across that union, so a band
#  setting that is not in the active grid could win and be written into the
#  direction config. The bands ARE the labels, so every downstream probability
#  came from a configuration the user had already removed.
#
#  Identity is now scoped to the ACTIVE GRID as well as the version. Old files
#  are left on disk untouched — they are the audit trail for earlier results.
_GRID_S = {round(float(s), 2) for s in SIGMA_GRID}
_GRID_W = {int(w) for w in SIGMA_WINDOW_GRID}
print(f"     Active grid: σ{sorted(_GRID_S)} × win{sorted(_GRID_W)}")

rows, read_errors = [], []
n_wrong_version = n_unparseable = n_off_grid = 0
_off_grid_seen = {}
n_recomputed = n_frozen = 0
_seen_versions = {}

for fp in pkl_files:
    fn = os.path.basename(fp)

    # identity comes from the FILENAME, via S1's tested parser.
    meta = parse_cache_filename(fn)
    if meta is None:
        # not this CACHE_VERSION (or not a cache file at all)
        n_wrong_version += 1
        continue

    # reject configs outside the ACTIVE grid, before loading
    if (round(float(meta["sigma"]), 2) not in _GRID_S
            or int(meta["window"]) not in _GRID_W):
        n_off_grid += 1
        _k = f"σ{meta['sigma']}/w{meta['window']}"
        _off_grid_seen[_k] = _off_grid_seen.get(_k, 0) + 1
        continue

    try:
        c = joblib.load(fp)
    except Exception as e:
        read_errors.append((fn, f"{type(e).__name__}: {str(e)[:70]}")); continue
    if not isinstance(c, dict):
        read_errors.append((fn, "cache is not a dict")); continue

    # second, independent version check against the dict itself
    cv = c.get("cache_version", "?")
    _seen_versions[cv] = _seen_versions.get(cv, 0) + 1
    if cv != CACHE_VERSION:
        n_wrong_version += 1; continue

    t = _task_by_name.get(meta["task"], {})
    if not t:
        read_errors.append((fn, f"unknown task '{meta['task']}' — skipped")); continue

    # recompute consistency LIVE from per_fold, and record whether we
    # actually managed to. v2 fell back to the frozen flag inside a bare
    # `except: pass` while still printing "recomputed".
    pf = c.get("per_fold") or []
    pairs, src = [], "frozen"
    try:
        pairs = [(float(a), float(b)) for a, b, *_ in pf
                 if np.isfinite(float(a)) and np.isfinite(float(b))]
        if pairs:
            passed, mean_oof = folds_consistency_pass(pairs); src = "recomputed"
        else:
            passed, mean_oof = bool(c.get("folds_consistency_pass", False)), np.nan
    except Exception as e:
        passed = bool(c.get("folds_consistency_pass", False)); mean_oof = np.nan
        read_errors.append((fn, f"per_fold malformed ({type(e).__name__}) — using frozen flag"))
    n_recomputed += (src == "recomputed"); n_frozen += (src == "frozen")

    infold = float(np.nanmean([a for a, _, _ in pf])) if pf else np.nan
    test_s = float(np.nanmean([c_ for _, _, c_ in pf])) if pf else np.nan
    oof_s  = mean_oof if np.isfinite(mean_oof) else float(c.get("mean_oof_score", np.nan))

    rows.append({
        "task": meta["task"], "direction": t.get("direction", ""), "day": int(t.get("day", 0)),
        "model": meta["model"], "n_features": meta["n_features"],
        "sigma": round(float(meta["sigma"]), 2), "window": int(meta["window"]),
        "infold_score": infold, "oof_score": oof_s, "test_score": test_s,
        # legacy aliases — older sheets / helpers still read these
        "inner_train_f1": infold, "val_f1": oof_s, "test_f1": test_s,
        "folds_consistency_pass": bool(passed),
        "consistency_source": src, "n_folds": len(pf),
        #  NO fallback to meta["n_features"] — that is the ablation
        #  budget (13 everywhere), not a count. Absent means absent.
        "n_features_budget": int(meta["n_features"]),
        "n_features_used": _nfu_of(c, fn),
        "n_cycles": int(c.get("n_cycles", 0)),
        "n_dropped_nan": int(c.get("n_dropped_nan", 0)),
        "base_rate": float(c.get("base_rate", np.nan)),
        "n_oos": len(c.get("oos_cycle_id", []) or []),
        "has_oos_thr": bool(c.get("oos_thr")),          # must be True
        "primary_metric": str(c.get("primary_metric", "?")),
        "crossfit_folds": int(c.get("crossfit_folds", 0)),
        "calibrated": bool(c.get("calibrated", False)),
        "cache_version": cv, "cache_file": fp,
    })

led = pd.DataFrame(rows)
print(f"     Parsed rows            : {len(led)}")
print(f"     Wrong/absent version   : {n_wrong_version}   (accepted only {CACHE_VERSION})")
if n_off_grid:
    print(f"     Off-grid (skipped)     : {n_off_grid}   "
          + ", ".join(f"{k}×{v}" for k, v in sorted(_off_grid_seen.items())))
    print(f"                              left on disk as the audit trail for earlier runs")
if len(_seen_versions) > 1:
    print(f"     ⚠️  versions on disk    : {_seen_versions}  ← stale worlds present, ignored")
print(f"     Consistency recomputed : {n_recomputed}   frozen fallback: {n_frozen}"
      + ("   ⚠️ frozen rows do NOT reflect current thresholds" if n_frozen else ""))
if read_errors:
    print(f"     ⚠️  {len(read_errors)} read problem(s):")
    for fn, e in read_errors[:8]:
        print(f"         {fn[:60]} → {e}")
if led.empty:
    raise RuntimeError(
        f"❌ No usable cache rows at version {CACHE_VERSION}.\n"
        f"   Either S3_Train has not run under these settings, or you changed one\n"
        f"   of the five decisions in S1 (which deliberately starts a new cache\n"
        f"   world). Re-run S3_Train.")

# the whole point of the v3 train rewrite
_no_thr = led[~led["has_oos_thr"]]
if len(_no_thr):
    print(f"     ⚠️  {len(_no_thr)} row(s) have no `oos_thr` — these came from an "
          f"OLD S3_Train and would reintroduce the [L1] threshold leak in S6. "
          f"Delete those cache files and retrain.")

# de-duplicate defensively (should now be a no-op)
_key = ["task", "model", "n_features", "sigma", "window"]
_before = len(led)
led = led.sort_values("cache_file").drop_duplicates(_key, keep="last").reset_index(drop=True)
if _before != len(led):
    print(f"     ⚠️  de-duplicated {_before - len(led)} row(s) on {_key} — "
          f"investigate, this should not happen in v3")
led = led.sort_values(["direction", "day", "sigma", "window", "model"]).reset_index(drop=True)


# ============================================================================
#  REPORT THE TRUE FEATURE COUNT PER TASK
# ----------------------------------------------------------------------------
#  `n_features` / `n_features_budget` is the ablation CAP and is identical for
#  every task by design. The count that belongs in the write-up is
#  `n_features_used`: 7 at D2, 10 at D3, 13 at D4 under the v2 cumulative
#  spec, because each sheet carries one side's norm_dist plus realized_vol and
#  gap for every day up to its own, plus band_width_pct.
# ============================================================================
_nfc = (led.groupby("task")
        .agg(n_features_used_min=("n_features_used", "min"),
             n_features_used_max=("n_features_used", "max"),
             n_features_budget=("n_features_budget", "max"),
             rows=("task", "size")).reset_index())
print("\n     ── FEATURE COUNT PER TASK (used vs ablation budget) ──")
print(_nfc.to_string(index=False))

_spread = _nfc[_nfc["n_features_used_min"] != _nfc["n_features_used_max"]]
if len(_spread):
    print("     ⚠️  a task has rows with DIFFERENT used-counts — models within one "
          "task were fitted on different feature sets:")
    print(_spread.to_string(index=False))

if _nfu_missing:
    print(f"     ⚠️  {len(_nfu_missing)} cache file(s) carry no `n_features_used`; "
          f"reported as NaN, NOT as the budget:")
    for _f in sorted(set(_nfu_missing))[:8]:
        print(f"         {_f[:70]}")
    print("         These came from a pre-v4 S3_Train. Retrain them before "
          "quoting any feature count.")
else:
    print("     ✅ every cache carries its own n_features_used")
print("     ⚠️  quote n_features_used in the write-up, never n_features")


# ============================================================================
# STEP 2: WRITE THE LEDGER  (sole writer; rebuilt from scratch every run)
# ============================================================================
print("\n  📌 STEP 2: Writing ledger")
os.makedirs(os.path.dirname(LEDGER_PATH), exist_ok=True)
led.to_excel(LEDGER_PATH, index=False, engine="openpyxl")
print(f"     ✅ {LEDGER_PATH}  ({len(led)} rows)")

_cons = led[led["folds_consistency_pass"]]
print(f"     Consistent: {len(_cons)}/{len(led)}  "
      f"[gap≤{MAX_SCORE_GAP}, floor≥{MIN_OOF_SCORE}, ratio≥{SCORE_RATIO_MIN}, "
      f"≥{int(FOLD_PASS_MIN_FRACTION*100)}% folds]")


# ============================================================================
# STEP 3: PREDICTABILITY DIAGNOSTIC
# ============================================================================
print("\n  📌 STEP 3: Predictability by task")
print(f"     {'task':<10}{'n':>4}{'best oof':>10}{'best test':>11}{'cons':>6}  verdict")
for t in TASKS:
    sub = led[led["task"] == t["name"]]
    if sub.empty:
        print(f"     {t['name']:<10}{0:>4}{'—':>10}{'—':>11}{'—':>6}  NOT TRAINED"); continue
    b_oof = sub["oof_score"].max(); b_te = sub["test_score"].max()
    nc = int(sub["folds_consistency_pass"].sum())
    verdict = ("no consistent model" if nc == 0 else
               "weak" if b_oof < MIN_OOF_SCORE + 0.05 else "usable")
    print(f"     {t['name']:<10}{len(sub):>4}{b_oof:>10.3f}{b_te:>11.3f}{nc:>6}  {verdict}")


# ============================================================================
# STEP 4: CONSISTENCY SWEEP
# ----------------------------------------------------------------------------
#  v2 applied consistency_pass() to fold-AVERAGED metrics while the real gate
#  needs ≥FOLD_PASS_MIN_FRACTION of INDIVIDUAL folds to pass. Averages passing
#  does not imply 3-of-4 folds passing, so the table disagreed with the filter
#  it was meant to tune. The sweep now re-reads per_fold from each cache and
#  calls the genuine gate with the floor temporarily rebound.
# ============================================================================
print("\n  📌 STEP 4: Consistency sweep (the REAL gate, per-fold)")
_pf_cache = {}
for _, r in led.iterrows():
    try:
        c = joblib.load(r["cache_file"])
        _pf_cache[r["cache_file"]] = [(float(a), float(b)) for a, b, *_ in (c.get("per_fold") or [])]
    except Exception:
        _pf_cache[r["cache_file"]] = []

_floors = sorted({round(MIN_OOF_SCORE + d, 3) for d in (-0.10, -0.05, 0.0, 0.05, 0.10)})
_orig_floor = MIN_OOF_SCORE
sweep = []
for fl in _floors:
    MIN_OOF_SCORE = fl                                   # rebind for the gate
    n = sum(1 for _, r in led.iterrows()
            if _pf_cache.get(r["cache_file"]) and folds_consistency_pass(_pf_cache[r["cache_file"]])[0])
    sweep.append({"floor": fl, "n_pass": n, "pct": round(100 * n / max(len(led), 1), 1),
                  "is_current": fl == _orig_floor})
MIN_OOF_SCORE = _orig_floor                              # restore
_sw = pd.DataFrame(sweep)
for _, r in _sw.iterrows():
    print(f"     floor {r['floor']:>6.3f} → {int(r['n_pass']):>4} models pass "
          f"({r['pct']:>5.1f}%)" + ("   ← current" if r["is_current"] else ""))
print("     (gap and ratio held at their configured values; only the floor moves)")


# ============================================================================
# STEP 5: GAP REPORT
# ----------------------------------------------------------------------------
#  Expected comes from S1, never from what was found — otherwise a model that
#  failed for EVERY combination is absent from "expected" and the script
#  cheerfully reports a complete grid.
# ============================================================================
print("\n  📌 STEP 5: Grid gap report")
_expected_models = list(PROJECT1_MODELS)
_expected = [(t["name"], m, nf, round(float(sg), 2), int(w))
             for t in TASKS for m in _expected_models
             for nf in ABLATION_FEATURE_COUNTS
             for sg in SIGMA_GRID for w in SIGMA_WINDOW_GRID]
_present = set(zip(led["task"], led["model"], led["n_features"], led["sigma"], led["window"]))
_missing = [e for e in _expected if e not in _present]
print(f"     Expected {len(_expected)} | present {len(_present)} | missing {len(_missing)}")
if _missing:
    print(f"     Missing (first 15):")
    for e in _missing[:15]:
        print(f"       task={e[0]:<10} model={e[1]:<20} feat={e[2]} σ={e[3]} win={e[4]}")
    _by_model = pd.Series([e[1] for e in _missing]).value_counts()
    print(f"     Missing by model: {dict(_by_model)}")
    if (_by_model == len(_expected) / len(_expected_models)).any():
        print(f"     ⚠️  A model is missing for EVERY combination — it is probably "
              f"crashing in S3_Train. Check the error lines there.")
else:
    print("     ✅ Grid complete")

df_ledger = led
print("\n" + "=" * 78)
print("  ✅ S3_Harvest_P1_v4.0 COMPLETE")
print(f"     Ledger: {len(led)} rows | consistent: {len(_cons)} | "
      f"recomputed: {n_recomputed} | frozen: {n_frozen}")
print("     ➡️  NEXT: S3_Select")
print("=" * 78)


  📚 S3_Harvest_P1_v4.0 — REBUILD LEDGER  (world: FULL)
  Cache version accepted: v7CWFN1w4x2FMMBX1T1_V2SB
  Primary metric        : Brier skill score

  📌 STEP 1: Scanning cache
     Files found: 150
     Active grid: σ[0.5, 0.75, 1.0] × win[4, 8]
     Parsed rows            : 150
     Wrong/absent version   : 0   (accepted only v7CWFN1w4x2FMMBX1T1_V2SB)
     Consistency recomputed : 150   frozen fallback: 0

     ── FEATURE COUNT PER TASK (used vs ablation budget) ──
    task  n_features_used_min  n_features_used_max  n_features_budget  rows
lower_D2                    7                    7                 13    25
lower_D3                   10                   10                 13    25
lower_D4                   13                   13                 13    25
upper_D2                    7                    7                 13    25
upper_D3                   10                   10                 13    25
upper_D4                   13                   13                 13 

In [46]:
# @title
# ============================================================================
# S3_Select_P1_v4.0 — PICK THE WINNERS
# ============================================================================
#
# PURPOSE
# -------
# The ONLY place winners are chosen. Reads the ledger, keeps models that clear
# the consistency gate, then:
#   1. picks one (sigma, window) per DIRECTION, requiring D2+D3+D4 coverage;
#   2. picks the TOP-K ensemble members per task.
# No training. No ML-vs-Gaussian comparison — that is S6.
#
#   IN  : LEDGER_PATH
#   OUT : DIRECTION_CONFIG_PATH   the chosen band configuration per direction
#         BEST_MODELS_PATH        the top-k ensemble per task
#         FEATURE_AUDIT_PATH      a five-sheet audit workbook
#
# ⚠️ A DISCLOSED ASYMMETRY
# ------------------------
# Under P1_CONFIG_SELECTION_MODE = "ml_best", this cell chooses the shared
# (sigma, window) by ML's own out-of-fold score. That is leakage-free — the
# test block is never read — but it is a SELECTION BUDGET the Gaussian does
# not get: ML effectively picks the ground it fights on, and the Gaussian is
# then evaluated there.
#
# It is made explicit three ways: the mode can be switched off entirely, the
# chosen configuration's rank among all configurations is recorded, and the
# disclosure text is written into the JSON so it reaches the write-up instead
# of living in someone's memory.
#
# WHAT "SELECTED" MEANS
# ---------------------
# There is NOT one model per task. TOP_K members are kept and S6 scores a
# WEIGHTED ENSEMBLE of them, weighted by out-of-fold score. `single_best` is
# the rank-1 member and exists for interpretability only — quoting it as
# "the model for D3" misstates what produced the reported numbers.
#
# Selection sorts on out-of-fold score, then the smallest in-fold/out-of-fold
# gap, then fewest features. The TEST score is never a sort key.
#
# FEATURE COUNTS IN THE AUDIT WORKBOOK
# ------------------------------------
#   n_features_budget   the ablation cap (13 everywhere) — NOT a count
#   n_features_used     the real count, derived by counting distinct features
#                       per (task, model, sigma, window, fold) and asserted
#                       against get_task_features()
#
# Sheets B and C count how often a feature appeared in a TRAINED combination.
# Because no feature selection occurs, that is a training-run count, not an
# importance measure, and perm_rank is positional. Sheet D_read_me says so in
# the workbook itself; sheet E gives the per-task counts.
#
# RUN AFTER: S3_Harvest.  NEXT: S6_InferEval.
# ============================================================================

try:
    _ = (TASKS, GRID_DIR, LEDGER_PATH, MODEL_CACHE_DIR, RUN_TYPE, TOP_K,
         MAX_SCORE_GAP, MIN_OOF_SCORE, SCORE_RATIO_MIN, PRIMARY_METRIC_NAME,
         DIRECTION_CONFIG_PATH, BEST_MODELS_PATH, FEATURE_AUDIT_PATH,
         SIGMA_GRID, SIGMA_WINDOW_GRID)
    _ = (cache_key, nn_meta_key, select_top_k, feature_class)
except NameError as _ne:
    raise RuntimeError(f"❌ Missing {_ne}. Run S1_Config_P1_v4.0 → S3_Harvest before S3_Select.")

import os, json, glob, joblib, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

print("\n" + "=" * 78)
print(f"  🎯 S3_Select_P1_v4.0 — PICK WINNERS (world: {RUN_TYPE})")
print(f"  Selection metric: oof_score ({PRIMARY_METRIC_NAME}, cross-fitted on train+val)")
print("  Test is never read here, or anywhere in selection.")
print("=" * 78)


# ============================================================================
# CONFIG-SELECTION POLICY
# ----------------------------------------------------------------------------
#  "ml_best"  → pick (σ, window) by ML's mean oof_score per direction.
#               Highest headline ML performance, but ML chooses the config and
#               Normal does not. MUST be disclosed in the write-up.
#  "fixed"    → use P1_FIXED_SIGMA / P1_FIXED_WINDOW for both directions.
#               Neither contestant chooses. The cleanest control, and the one
#               a sceptical reviewer will ask for.
#  "median"   → the config whose ML oof_score is the MEDIAN across the grid.
#               A middle option: avoids cherry-picking the best cell without
#               committing to an arbitrary fixed one.
#  Whatever you choose, S6 Part A2 still reports EVERY config, so the headline
#  can always be checked against the full picture.
P1_CONFIG_SELECTION_MODE = "ml_best"      # "ml_best" | "fixed" | "median"
P1_FIXED_SIGMA  = 1.0
P1_FIXED_WINDOW = 8

REQUIRE_ALL_DAYS  = True
MIN_DAYS_FALLBACK = 2


# ============================================================================
# STEP 1: LOAD LEDGER + CONSISTENCY
# ============================================================================
print("\n  📌 STEP 1: Load ledger + consistency filter")
if not os.path.exists(LEDGER_PATH):
    raise FileNotFoundError(f"❌ {LEDGER_PATH} not found. Run S3_Harvest first.")
led = pd.read_excel(LEDGER_PATH)
print(f"     Ledger rows: {len(led)}")

# coerce dtypes BEFORE any comparison
for c in ("sigma", "window", "n_features", "day",
          "infold_score", "oof_score", "test_score", "val_f1", "test_f1"):
    if c in led.columns:
        led[c] = pd.to_numeric(led[c], errors="coerce")
if "sigma" in led.columns:
    led["sigma"] = led["sigma"].round(2)
if "folds_consistency_pass" in led.columns:
    led["folds_consistency_pass"] = (led["folds_consistency_pass"].astype(str)
                                     .str.strip().str.lower().isin(["true", "1", "yes"]))
_score = "oof_score" if "oof_score" in led.columns else "val_f1"

# refuse to select from caches that predate the threshold fix
if "has_oos_thr" in led.columns:
    _bad = led[~led["has_oos_thr"].astype(str).str.lower().isin(["true", "1", "yes"])]
    if len(_bad):
        raise RuntimeError(
            f"❌ {len(_bad)} ledger row(s) have no `oos_thr`. Those caches were written\n"
            f"   by an OLD S3_Train, and S6 would fall back to the single `threshold`\n"
            f"   fitted on the last 20% of ALL cycles — reintroducing the [L1] leak\n"
            f"   (~33% of OOS cycles thresholded on their own labels).\n"
            f"   Delete {MODEL_CACHE_DIR}/*.pkl and re-run S3_Train_v3.")

cons = led[led["folds_consistency_pass"] == True].copy()
print(f"     Consistent: {len(cons)}/{len(led)}  "
      f"[gap≤{MAX_SCORE_GAP}, floor≥{MIN_OOF_SCORE}, ratio≥{SCORE_RATIO_MIN}]")
if cons.empty:
    raise RuntimeError(
        "❌ No consistent models.\n"
        "   Options, in order of preference:\n"
        "     1) report the gap honestly — 'no model met the trustworthiness gate'\n"
        "        is a legitimate finding and is more defensible than relaxing until\n"
        "        something passes;\n"
        "     2) set P1_FILTER_PRESET='moderate' in S1, then re-run Harvest → Select\n"
        "        (no retrain needed — consistency is recomputed live);\n"
        "     3) train more of the grid (see the Harvest gap report).")


# ============================================================================
# STEP 2: DIRECTIONAL (sigma, window)
# ============================================================================
print(f"\n  📌 STEP 2: Band-config selection  [mode={P1_CONFIG_SELECTION_MODE}]")

best_day = (cons.sort_values(_score, ascending=False)
            .groupby(["direction", "sigma", "window", "day"], as_index=False)
            .first()[["direction", "sigma", "window", "day", _score, "test_score"]])


def _config_table(direction):
    """Summarise every (sigma, window) for one direction.

    Parameters
    ----------
    direction : {"upper", "lower"}
        Which band edge to summarise.

    Returns
    -------
    pandas.DataFrame
        One row per configuration: how many of D2/D3/D4 it covers, which ones,
        and the average and minimum out-of-fold score across the three days.

    Notes
    -----
    A day with no surviving model contributes 0.0 to both statistics, so a
    configuration cannot look good merely by covering fewer days.
    """
    sub = best_day[best_day["direction"] == direction]
    rows = []
    for (sg, w), g in sub.groupby(["sigma", "window"]):
        vmap = {int(r["day"]): float(r[_score]) for _, r in g.iterrows()}
        days = set(vmap) & {2, 3, 4}
        rows.append({"direction": direction, "sigma": round(float(sg), 2), "window": int(w),
                     "days_covered": len(days),
                     "days_present": ",".join(map(str, sorted(days))),
                     "avg_score": float(np.mean([vmap.get(d, 0.0) for d in (2, 3, 4)])),
                     "min_score": float(min([vmap.get(d, 0.0) for d in (2, 3, 4)]))})
    return pd.DataFrame(rows)


def pick_direction(direction):
    """Choose one band configuration for a direction.

    Parameters
    ----------
    direction : {"upper", "lower"}
        Which band edge to choose for.

    Returns
    -------
    (dict or None, pandas.DataFrame)
        The chosen configuration and the full comparison table. None when no
        configuration has any surviving model.

    Notes
    -----
    Configurations covering all three decision days are strongly preferred:
    without D2+D3+D4 the per-day results are not comparable to each other, and
    a warning says so. MIN_DAYS_FALLBACK controls how far coverage may drop
    before the direction is abandoned.

    Which score decides is set by P1_CONFIG_SELECTION_MODE — see the header's
    disclosed-asymmetry note.
    """
    cdf = _config_table(direction)
    if cdf.empty:
        return None, cdf

    full = cdf[cdf["days_covered"] == 3]
    if REQUIRE_ALL_DAYS and not full.empty:
        pool = full
    elif REQUIRE_ALL_DAYS:
        pool = cdf[cdf["days_covered"] >= MIN_DAYS_FALLBACK]
        if pool.empty: pool = cdf
        print(f"     ⚠️  {direction}: no config covers D2+D3+D4 — falling back to "
              f"≥{MIN_DAYS_FALLBACK} days. The headline for this direction is NOT "
              f"comparable across days.")
    else:
        pool = cdf

    ranked = pool.sort_values(["avg_score", "min_score", "days_covered"],
                              ascending=[False, False, False]).reset_index(drop=True)

    if P1_CONFIG_SELECTION_MODE == "fixed":
        hit = ranked[(ranked["sigma"] == round(float(P1_FIXED_SIGMA), 2))
                     & (ranked["window"] == int(P1_FIXED_WINDOW))]
        if hit.empty:
            raise RuntimeError(
                f"❌ P1_CONFIG_SELECTION_MODE='fixed' but σ={P1_FIXED_SIGMA} "
                f"w={P1_FIXED_WINDOW} has no consistent model for '{direction}'.\n"
                f"   Available: {ranked[['sigma','window','avg_score']].to_dict('records')[:6]}")
        best = hit.iloc[0]
        rank = int(ranked.index[(ranked["sigma"] == best["sigma"])
                                & (ranked["window"] == best["window"])][0]) + 1
    elif P1_CONFIG_SELECTION_MODE == "median":
        best = ranked.iloc[len(ranked) // 2]; rank = len(ranked) // 2 + 1
    else:                                    # "ml_best"
        best = ranked.iloc[0]; rank = 1
    return (best, rank, len(ranked)), cdf


DISCLOSURE = (
    "Config selection uses ML's out-of-fold score; Normal does not get to choose "
    "the (sigma, window) it is evaluated at. This is leakage-free (test is never "
    "read) but it is an unequal SELECTION budget favouring ML, and it is the "
    "mirror of the threshold budget that previously favoured Normal. Report the "
    "per-config table from S6 Part A2 alongside the headline, and state the "
    "chosen config's rank among all configs."
) if P1_CONFIG_SELECTION_MODE == "ml_best" else (
    "Config is fixed a priori; neither contestant selects it. This is the "
    "cleanest control and removes the selection-budget asymmetry entirely."
    if P1_CONFIG_SELECTION_MODE == "fixed" else
    "Config is the MEDIAN-ranked cell by ML's out-of-fold score — deliberately "
    "not the best, to blunt the selection-budget asymmetry."
)

direction_best_band_config, all_candidates = {}, []
for d in ["upper", "lower"]:
    res, cdf = pick_direction(d)
    all_candidates.append(cdf)
    if res is None:
        print(f"     ❌ {d}: no consistent config at all."); continue
    best, rank, n_cfg = res
    direction_best_band_config[d] = {
        "sigma": float(best["sigma"]), "window": int(best["window"]),
        "days_covered": int(best["days_covered"]), "days_present": str(best["days_present"]),
        "avg_score": float(best["avg_score"]), "min_score": float(best["min_score"]),
        "selection_mode": P1_CONFIG_SELECTION_MODE,
        "rank_among_configs": int(rank), "n_configs_considered": int(n_cfg),
        "selection_metric": _score, "objective": PRIMARY_METRIC_NAME,
        "max_score_gap": float(MAX_SCORE_GAP), "min_oof_score": float(MIN_OOF_SCORE),
        "score_ratio_min": float(SCORE_RATIO_MIN), "top_k": int(TOP_K),
        "training_subset": "standard cycles (D1..D4 + expiry), warm-up dropped",
        "world": RUN_TYPE,
        "DISCLOSURE": DISCLOSURE,
    }
    print(f"     🏆 {d.upper():5s}: σ={best['sigma']} w={int(best['window'])} "
          f"avg_{_score}={best['avg_score']:+.3f} days={best['days_present']} "
          f"| rank {rank}/{n_cfg}")

_missing_dirs = [d for d in ("upper", "lower") if d not in direction_best_band_config]
if _missing_dirs:
    print("\n     " + "⚠️ " * 14)
    print(f"     ⚠️  NO CONSISTENT MODELS for: {_missing_dirs}")
    print(f"     ⚠️  S6 will run Normal-only for that side. Report it as a gap —")
    print(f"     ⚠️  do NOT relax the filter just to fill the cell.")
    print("     " + "⚠️ " * 14)

if P1_CONFIG_SELECTION_MODE == "ml_best":
    print(f"\n     ⚠️  DISCLOSE: {DISCLOSURE}")

with open(DIRECTION_CONFIG_PATH, "w") as f:
    json.dump(direction_best_band_config, f, indent=2)
print(f"     ✅ {os.path.basename(DIRECTION_CONFIG_PATH)} "
      f"({len(direction_best_band_config)}/2 directions)")
if all_candidates:
    pd.concat(all_candidates, ignore_index=True).to_excel(
        os.path.join(GRID_DIR, f"direction_config_candidates_{RUN_TYPE}.xlsx"),
        index=False, engine="openpyxl")


# ============================================================================
# STEP 3: TOP-K MODELS PER TASK
# ============================================================================
#  A generous numeric-column probe so get_task_features() can be
#  asked "how many features does this task ask for?" without rebuilding a
#  frame. Every name the v2 whitelist can request must appear here, or the
#  whitelist would raise on a missing feature.
TRUE_FEAT_PROBE = sorted({
    f.format(d=d, side=sd)
    for d in (1, 2, 3, 4) for sd in ("upper", "lower")
    for f in ("norm_dist_{side}_D{d}", "realized_vol_D{d}", "gap_D{d}",
              "dist_to_{side}_D{d}", "days_left_D{d}", "sqrt_dl_frac_D{d}")
} | {"band_width_pct", "mu", "sigma"})

print("\n  📌 STEP 3: Top-K ensemble per task")
best_models_by_task, skipped = {}, []
for task in TASKS:
    tn, td = task["name"], task["direction"]
    if td not in direction_best_band_config:
        skipped.append(tn); continue
    sg = round(float(direction_best_band_config[td]["sigma"]), 2)
    w  = int(direction_best_band_config[td]["window"])
    sub = cons[(cons["task"] == tn) & (cons["sigma"] == sg) & (cons["window"] == w)].copy()
    if sub.empty:
        print(f"     ⚠️ {tn}: no consistent model at σ={sg} w={w} → skipped")
        skipped.append(tn); continue

    top = select_top_k(sub, k=TOP_K)
    models = []
    for rank, (_, r) in enumerate(top.iterrows(), 1):
        mn, nf = str(r["model"]), int(r["n_features"])
        # NN caches live under a different filename scheme
        cf = (nn_meta_key(tn, nf, sg, w) if mn == "NN_FF" else cache_key(tn, mn, nf, sg, w))
        if not os.path.exists(cf) and isinstance(r.get("cache_file"), str):
            cf = r["cache_file"]
        #  nf is the BUDGET and must stay in the cache key. The true
        #  count lives in the cache dict as `n_features_used` (cell 8 wrote it);
        #  fall back to the ledger column, then to the whitelist length.
        _nfu = r.get("n_features_used")
        if not (isinstance(_nfu, (int, float)) and np.isfinite(_nfu) and _nfu > 0):
            _nfu = np.nan
        if not np.isfinite(_nfu) and os.path.exists(cf) and mn != "NN_FF":
            try:
                _nfu = float(joblib.load(cf).get("n_features_used", np.nan))
            except Exception:
                _nfu = np.nan
        if not np.isfinite(_nfu):
            _nfu = float(len(get_task_features(task, TRUE_FEAT_PROBE)))
        models.append({"rank": rank, "task": tn, "model": mn,
                       "n_features": nf,                    # ABLATION BUDGET (cache key)
                       "n_features_used": int(_nfu),        # 🔴 the real count
                       "oof_score": float(r[_score]),
                       "val_f1": float(r[_score]),          # legacy alias for S6 weights
                       "test_score": float(r.get("test_score", np.nan)),
                       "cache_file": cf, "cache_exists": bool(os.path.exists(cf))})
    _miss = [m["model"] for m in models if not m["cache_exists"]]
    if _miss:
        print(f"     ⚠️ {tn}: cache file missing for {_miss} — S6 will skip them")

    best_models_by_task[tn] = {
        "task": tn, "direction": td, "day": int(task["day"]),
        "sigma": float(sg), "window": int(w),
        "top_k_requested": int(TOP_K), "top_k_actual": len(models),
        "n_consistent": int(len(sub)),
        "single_best": models[0] if models else None,
        "models": models,
        "ensemble_note": ("Reported ML predictions are a TOP-K ENSEMBLE, not the "
                          "single_best model. Quote single_best only for "
                          "interpretability, never as the headline result."),
    }
    print(f"     {tn:9s} σ={sg} w={w} [{len(sub)} cons → {len(models)} used] "
          f"| best={models[0]['model']}({models[0]['oof_score']:+.3f}) "
          f"| feats={models[0]['n_features_used']} (budget {models[0]['n_features']}) "
          f"| types={len({m['model'] for m in models})}")

with open(BEST_MODELS_PATH, "w") as f:
    json.dump(best_models_by_task, f, indent=2)
print(f"     ✅ {os.path.basename(BEST_MODELS_PATH)} "
      f"({len(best_models_by_task)}/{len(TASKS)} tasks)")
if skipped:
    print(f"     ⚠️ skipped: {skipped}")


# ============================================================================
# STEP 4: FEATURE AUDIT (5 sheets)
# ============================================================================
print("\n  📌 STEP 4: Feature-selection audit")
audit_rows = []
for fp in glob.glob(os.path.join(MODEL_CACHE_DIR, "*.pkl")):
    try:
        c = joblib.load(fp)
        if c.get("cache_version") != CACHE_VERSION:      # version-scoped
            continue
        audit_rows.extend(c.get("feature_audit", []) or [])
    except Exception:
        continue
raw = pd.DataFrame(audit_rows)

if raw.empty:
    print("     ⚠️ no feature_audit rows at this cache version")
else:
    if "feature_class" not in raw.columns:
        raw["feature_class"] = raw["feature"].map(feature_class)
    raw["sigma"] = pd.to_numeric(raw["sigma"], errors="coerce").round(2)

    #  cell 8 stamps `n_features` = the ablation BUDGET (13) on every
    #  audit row. Rename it so nobody reads it as a count, and derive the TRUE
    #  count by counting distinct features inside each (task, model, σ, w, fold)
    #  group — the audit lists one row per feature actually put into the model.
    raw = raw.rename(columns={"n_features": "n_features_budget"})
    _grp = ["task", "model", "n_features_budget", "sigma", "window", "fold"]
    raw["n_features_used"] = raw.groupby(_grp)["feature"].transform("nunique")

    #  the winner key still uses the BUDGET, because that is what the cache
    #  filename and the ledger row are keyed on. Only the reporting changes.
    winner_keys = {(tn, m["model"], int(m["n_features"]),
                    round(float(dm["sigma"]), 2), int(dm["window"]))
                   for tn, dm in best_models_by_task.items() for m in dm["models"]}
    raw["is_winner"] = raw.apply(
        lambda r: (r["task"], str(r["model"]), int(r["n_features_budget"]),
                   round(float(r["sigma"]), 2), int(r["window"])) in winner_keys, axis=1)
    win = raw[raw["is_winner"]].copy()

    if not win.empty:
        freq = (win.groupby(["feature", "feature_class"])
                .agg(times_selected=("feature", "size"), avg_mi=("mi_score", "mean"),
                     tasks=("task", "nunique")).reset_index()
                .sort_values("times_selected", ascending=False))
        tot = len(win)
        cat = (win.groupby("feature_class")
               .agg(total_selections=("feature", "size"),
                    distinct_features=("feature", "nunique"),
                    tasks=("task", "nunique")).reset_index())
        cat["pct_of_selections"] = (cat["total_selections"] / tot * 100).round(1)
        cat = cat.sort_values("total_selections", ascending=False)
    else:
        freq = pd.DataFrame(); cat = pd.DataFrame()

    # ------------------------------------------------------------------
    #  PER-TASK FEATURE COUNT, cross-checked against the whitelist
    # ------------------------------------------------------------------
    _cnt = (raw.groupby("task")
            .agg(n_features_used=("n_features_used", "max"),
                 n_features_budget=("n_features_budget", "max"),
                 distinct_features=("feature", "nunique")).reset_index())
    _exp = {}
    for _t in TASKS:
        try:
            _exp[_t["name"]] = len(get_task_features(_t, TRUE_FEAT_PROBE))
        except Exception:
            _exp[_t["name"]] = np.nan
    _cnt["expected_from_whitelist"] = _cnt["task"].map(_exp)
    _cnt["agrees"] = (_cnt["n_features_used"] == _cnt["expected_from_whitelist"])
    _cnt["features"] = _cnt["task"].map(
        lambda t: ", ".join(sorted(raw.loc[raw.task == t, "feature"].unique())))

    print("\n     ── TRUE FEATURE COUNT PER TASK ──")
    print(_cnt[["task", "n_features_used", "n_features_budget",
                "expected_from_whitelist", "agrees"]].to_string(index=False))

    _dis = _cnt[~_cnt["agrees"] & _cnt["expected_from_whitelist"].notna()]
    if len(_dis):
        raise RuntimeError(
            "❌ [NFEAT] the features actually trained on do not match the "
            "whitelist for:\n" + _dis[["task", "n_features_used",
                                       "expected_from_whitelist"]].to_string(index=False)
            + "\n   Do NOT report these counts until this is explained.")
    print("     ✅ every task's trained features match get_task_features()")
    print("     ⚠️  quote n_features_used, NEVER n_features_budget, as 'the "
          "number of features'")

    note = pd.DataFrame([
        {"column / sheet": "n_features_budget",
         "what it means": "ABLATION_FEATURE_COUNTS — the CAP passed to "
                          "select_features(). Set to the size of the largest task "
                          "(13) so the p<=k branch always fires and NO feature "
                          "selection happens. It is part of the cache filename and "
                          "the ledger key, which is why it is 13 on every row.",
         "safe to quote as a feature count?": "NO — it is a budget, not a count"},
        {"column / sheet": "n_features_used",
         "what it means": "the number of DISTINCT features actually put into the "
                          "model for that (task, model, sigma, window, fold). "
                          "7 at D2, 10 at D3, 13 at D4 under the v2 cumulative "
                          "spec, and asserted against get_task_features().",
         "safe to quote as a feature count?": "YES — this is the real count"},
        {"column / sheet": "B_feature_frequency / times_selected",
         "what it means": "how often a feature appeared in a TRAINED combination. "
                          "Because no selection occurs, this counts training runs, "
                          "not importance.",
         "safe to quote as a feature count?": "NO — not an importance measure"},
        {"column / sheet": "C_category_predictability",
         "what it means": "the same counts grouped by feature class. Same caveat.",
         "safe to quote as a feature count?": "NO — not an importance measure"},
        {"column / sheet": "perm_rank",
         "what it means": "positional only, for the same reason.",
         "safe to quote as a feature count?": "NO"},
    ])

    with pd.ExcelWriter(FEATURE_AUDIT_PATH, engine="openpyxl") as xw:
        raw.sort_values(["task", "model", "sigma", "window", "fold"]).to_excel(
            xw, sheet_name="A_raw_ledger", index=False)
        freq.to_excel(xw, sheet_name="B_feature_frequency", index=False)
        cat.to_excel(xw, sheet_name="C_category_predictability", index=False)
        note.to_excel(xw, sheet_name="D_read_me", index=False)
        _cnt.to_excel(xw, sheet_name="E_feature_count_per_task", index=False)
    print(f"     ✅ {os.path.basename(FEATURE_AUDIT_PATH)} "
          f"(raw {len(raw)} | features {len(freq)} | classes {len(cat)})")

df_direction_config = direction_best_band_config
df_best_models = best_models_by_task

print("\n" + "=" * 78)
print("  ✅ S3_Select_P1_v4.0 COMPLETE")
print(f"     Winners: {len(best_models_by_task)}/{len(TASKS)} tasks | skipped: {len(skipped)}")
print(f"     Config mode: {P1_CONFIG_SELECTION_MODE}"
      + ("   ⚠️ disclose the selection-budget asymmetry"
         if P1_CONFIG_SELECTION_MODE == "ml_best" else ""))
print("     ➡️  NEXT: S6_InferEval")
print("=" * 78)


  🎯 S3_Select_P1_v4.0 — PICK WINNERS (world: FULL)
  Selection metric: oof_score (Brier skill score, cross-fitted on train+val)
  Test is never read here, or anywhere in selection.

  📌 STEP 1: Load ledger + consistency filter
     Ledger rows: 150
     Consistent: 88/150  [gap≤0.15, floor≥0.02, ratio≥0.4]

  📌 STEP 2: Band-config selection  [mode=ml_best]
     🏆 UPPER: σ=0.5 w=4 avg_oof_score=+0.474 days=2,3,4 | rank 1/6
     🏆 LOWER: σ=0.5 w=4 avg_oof_score=+0.473 days=2,3,4 | rank 1/6

     ⚠️  DISCLOSE: Config selection uses ML's out-of-fold score; Normal does not get to choose the (sigma, window) it is evaluated at. This is leakage-free (test is never read) but it is an unequal SELECTION budget favouring ML, and it is the mirror of the threshold budget that previously favoured Normal. Report the per-config table from S6 Part A2 alongside the headline, and state the chosen config's rank among all configs.
     ✅ direction_best_band_config_FULL.json (2/2 directions)

  📌 STEP 3: To

In [47]:
# @title
# ============================================================================
# S6_InferEval_P1_v6.0 — THE COMPARISON, AND TODAY'S RECOMMENDATION
# ============================================================================
#
# PURPOSE
# -------
# Score the trained models out of sample against the analytical baselines and
# produce the live recommendation. This is where ML meets the Gaussian.
#
#   PART A   the headline out-of-sample comparison at the selected config
#   PART A2  the same comparison at EVERY config that was trained
#   PART B   today's HOLD / EXIT recommendation for the open cycle
#
#   IN  : BEST_MODELS_PATH, DIRECTION_CONFIG_PATH, the model cache, df_cycles
#   OUT : backtest_artifact_<world>.json  (read by S7 and S8)
#         project1_mlcore_vs_normal.xlsx
#
# THREE CONTESTANTS, THE SAME ROWS
# --------------------------------
#   ML-core     a weighted TOP-K ensemble of the selected models
#   Normal      the parameter-free Gaussian from S1
#   Student-t   the same closed form with a fat-tailed distribution, which
#               tests whether the Gaussian's tail assumption is what costs it
#
# THE THRESHOLD RULE — THE STUDY'S MAIN LEAKAGE CONTROL
# -----------------------------------------------------
# Every out-of-sample ML probability is thresholded with `oos_thr`, the cut
# from the fold that produced that prediction. A single cached `threshold`
# fitted on the last portion of ALL cycles would threshold a large share of
# out-of-sample rows on their own labels. This cell REFUSES to run on a cache
# that lacks oos_thr.
#
# The Gaussian's threshold is fitted on all history, which is the analytical
# counterpart of the same rule: both contestants fit their cut on data that
# excludes the row being scored.
#
# STALE-CACHE REFUSAL
# -------------------
# Every cache is checked against the current CACHE_VERSION before it is
# scored. A mismatch raises, because scoring with models trained under
# different settings would mix configurations silently. The fix is to re-run
# S3_Train → S3_Harvest → S3_Select; the old cache is left on disk as the
# audit trail.
#
# PART B AND THE D1 CASE
# ----------------------
# The day loop runs from D2 to the current day, so on D1 it never executes.
# There is no model at D1 by design — TASKS starts at D2 — but the BAND exists
# and is worth reporting, so a guard derives it and prints the Gaussian view
# alone. Without that guard the cell fails on roughly one trading day in five.
#
# Dist% is computed from the band and the close directly rather than read from
# a feature column, so it cannot silently become NaN when the feature set
# changes.
#
# RUN AFTER: S3_Select.  NEXT: S7_Significance → S8_Charts.
# ============================================================================

try:
    _ = (TASKS, RUN_TYPE, RANDOM_STATE, RESULTS_DIR, MODEL_CACHE_DIR, LEDGER_PATH,
         DIRECTION_CONFIG_PATH, BEST_MODELS_PATH, CLEAN_INPUT_PATH,
         DATE_COL, CLOSE_COL, CUTOFF_DATE, GRID_DIR, SIGMA_GRID, SIGMA_WINDOW_GRID,
         BAND_SIGMA, SIGMA_WINDOW, TOP_K, CACHE_VERSION, CYCLE_DECISION_DAYS,
         CYCLE_STEPS, P1_PRIMARY_METRIC, PRIMARY_METRIC_NAME, ECONOMIC_THRESHOLD,
         P1_DECISION_RULE)
    _ = (compute_asymmetric_bands_and_labels, normal_breach_probs, best_f1_threshold,
         ensemble_predict, apply_saved_pipeline, is_expiry, load_input, cache_key,
         nn_meta_key, select_top_k, binary_payoff_vec, walk_forward_splits,
         all_scores, primary_score, brier_decomposition, reliability_table,
         f1_at, drop_warmup_cycles, _norm_cdf)
    _ = build_cycle_record; _ = add_rolling_cycle_features; _ = standard_cycles   # S2
    _ = p1_build_task_frame; _ = p1_attach_features                              # S3_Train
    try:
        df_cycles
    except NameError:
        df_cycles = load_cycles_checkpoint()
except NameError as _ne:
    raise RuntimeError(f"❌ Missing {_ne}. Run S1_Config_v4 → S2_CycleBuild → "
                       f"S3_Train_v4 (or S3_Defs_Only_v4 for live-only) → "
                       f"S3_Harvest → S3_Select first.")

import os, json, glob, joblib, warnings, datetime, math
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
np.random.seed(RANDOM_STATE)

os.makedirs(RESULTS_DIR, exist_ok=True)
print("\n" + "=" * 78)
print(f"  🎯 S6_InferEval_P1_v3.0 — ML-core vs NORMAL (+Student-t) + LIVE  ({RUN_TYPE})")
print(f"     Results → {RESULTS_DIR}")
print("=" * 78)

with open(DIRECTION_CONFIG_PATH) as f: dir_cfg = json.load(f)
with open(BEST_MODELS_PATH)     as f: day_models = json.load(f)
# early, cheap staleness signal: the cache paths in the ledger carry
# the CACHE_VERSION of the run that wrote them. Warn now rather than letting a
# mismatch surface deep inside live inference.
_led_paths = [m.get("cache_file", "") for t in day_models.values()
              for m in t.get("models", [])]
_led_stale = [p for p in _led_paths if p and CACHE_VERSION not in p]
if _led_stale:
    print(f"  ⚠️  LEDGER STALENESS: {len(_led_stale)} of {len(_led_paths)} entries in "
          f"{os.path.basename(BEST_MODELS_PATH)}\n"
          f"      do not carry the current CACHE_VERSION ({CACHE_VERSION}).\n"
          f"      e.g. {os.path.basename(_led_stale[0])}\n"
          f"      Re-run S3_Train → S3_Harvest → S3_Select before trusting any ML output.")

# ── Student-t baseline ──────────────────────────────────────────────────────
P1_ENABLE_STUDENT_T = True
STUDENT_T_DF        = 4          # ν; ν>2 required for a finite variance
N_BOOT              = 5000


_GRID_S = [float(x) for x in SIGMA_GRID]
_GRID_W = [int(x) for x in SIGMA_WINDOW_GRID]
print(f"  Active grid → σ{_GRID_S} × win{_GRID_W}")


def _dir_band(direction):
    """Resolve the band configuration S3_Select chose for one direction.

    Parameters
    ----------
    direction : {"upper", "lower"}
        Which edge to look up.

    Returns
    -------
    (float, int, bool)
        (sigma, window, has_ml_models). The flag is False when no model
        survived selection for that direction, in which case the Gaussian is
        reported alone.

    Notes
    -----
    The configuration is also validated against the ACTIVE grid. CACHE_VERSION
    encodes the window-grid MODE, not its values, and does not encode
    SIGMA_GRID at all — so a stale direction-config file can point at a
    configuration that is no longer being trained. That is caught here rather
    than surfacing as a missing cache much later.
    """
    c = dir_cfg.get(direction)
    if c and ("sigma" in c) and ("window" in c):
        s, w_ = float(c["sigma"]), int(c["window"])
        # STALE DIRECTION CONFIG.
        #
        # CACHE_VERSION encodes the window-grid MODE ("full"/"single") and NOT
        # the grid VALUES, and does not encode SIGMA_GRID at all. So editing
        #     SIGMA_GRID   = [1.0, 0.5]
        #     _WINDOW_GRIDS = {"full": [12, 16], ...}
        # leaves CACHE_VERSION byte-identical. Nothing is invalidated: the
        # per-config .pkl files still match, theguard still passes, and
        # direction_config.json — written by an EARLIER S3_Select when σ=1.5
        # and w=4 were in the grid — is loaded and used verbatim.
        #
        # The band levels, and therefore the LABELS and every probability, then
        # come from a configuration that is not in the current grid. Silent.
        if s not in _GRID_S or w_ not in _GRID_W:
            raise RuntimeError(
                f"❌ [E4] STALE DIRECTION CONFIG for {direction.upper()}.\n"
                f"   {os.path.basename(DIRECTION_CONFIG_PATH)} selects "
                f"σ={s} window={w_}\n"
                f"   but the ACTIVE grid is σ{_GRID_S} × win{_GRID_W}.\n"
                f"   That file was written by an earlier S3_Select run under a\n"
                f"   different grid. CACHE_VERSION does not encode grid VALUES\n"
                f"   (only the 'full'/'single' mode), so editing SIGMA_GRID or\n"
                f"   _WINDOW_GRIDS invalidates nothing automatically.\n"
                f"   Fix: re-run S3_Harvest → S3_Select to rebuild the ledger\n"
                f"   and direction config for the current grid. No retraining\n"
                f"   is needed — per-config models are keyed by σ and window in\n"
                f"   their filenames and remain valid.")
        return s, w_, True
    print(f"  ⚠️ No ML config for {direction.upper()} → default σ={BAND_SIGMA}, "
          f"w={SIGMA_WINDOW} (Normal-only that side).")
    return float(BAND_SIGMA), int(SIGMA_WINDOW), False


_u_sig, _u_win, _u_has = _dir_band("upper")
_l_sig, _l_win, _l_has = _dir_band("lower")
print(f"  Config → Upper σ={_u_sig} w={_u_win} {'(ML)' if _u_has else '(Normal-only)'} | "
      f"Lower σ={_l_sig} w={_l_win} {'(ML)' if _l_has else '(Normal-only)'}")
for _d, _c in dir_cfg.items():
    if _c.get("selection_mode") == "ml_best":
        print(f"  ⚠️ {_d}: config chosen by ML's own score, rank "
              f"{_c.get('rank_among_configs')}/{_c.get('n_configs_considered')} — "
              f"see Part A2 and the DISCLOSURE in the config JSON.")

RUN_STAMP = os.path.basename(RESULTS_DIR)
_daily_p1 = load_input(CLEAN_INPUT_PATH)


# ============================================================================
# SHARED STATISTICS  (defined ONCE here; S7 imports them — v2 had two copies)
# ============================================================================
def paired_bootstrap_diff(y, a_pred, b_pred, n_boot=N_BOOT, seed=RANDOM_STATE,
                          stat="f1", a_prob=None, b_prob=None):
    """Paired bootstrap on a metric DIFFERENCE (contestant A − contestant B).

    Resamples CYCLES and applies the SAME index to both contestants, so the
    pairing is preserved. The metric is recomputed inside every draw from the
    resampled data (F1 is not the average of per-item F1s).

    🔴 The p-value uses the (1 + k)/(1 + B) correction and splits exact ties
    between the tails. v2 used 2*min((d<=0).mean(), (d>=0).mean()), which
    (a) can report a literal p = 0.0000 from 5000 draws — the smallest
    defensible value is 2/(B+1) ≈ 0.0004 — and (b) counted every exact-zero
    draw in BOTH tails, so a distribution that was half zeros and half +0.2
    returned p = 1.0.
    """
    y = np.asarray(y, int)
    n = len(y)

    def _m(idx, pred, prob):
        """Score one bootstrap resample under the chosen statistic.

        Parameters
        ----------
        idx : ndarray
            Row positions drawn for this resample.
        pred : array-like of int
            Hard predictions for every row.
        prob : array-like of float
            Probabilities for every row.

        Returns
        -------
        float
            F1 or the primary score on the resampled rows.
        """
        if stat == "f1":
            return f1_at(y[idx], np.asarray(pred, float)[idx], 0.5)
        return primary_score(y[idx], np.asarray(prob, float)[idx],
                             np.asarray(pred, int)[idx])

    obs = _m(np.arange(n), a_pred, a_prob) - _m(np.arange(n), b_pred, b_prob)
    if n < 8 or y.sum() < 2:
        return dict(obs=float(obs), lo=float("-inf"), hi=float("inf"),
                    p=1.0, n=n, note="insufficient_n")

    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    n_degen = 0
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        if y[idx].sum() == 0:                      # degenerate draw
            n_degen += 1
            idx = rng.integers(0, n, n)
        diffs[b] = _m(idx, a_pred, a_prob) - _m(idx, b_pred, b_prob)

    lo, hi = np.percentile(diffs, [2.5, 97.5])
    n_lt = int((diffs < 0).sum()); n_gt = int((diffs > 0).sum())
    n_eq = int(n_boot - n_lt - n_gt)
    le = n_lt + n_eq / 2.0; ge = n_gt + n_eq / 2.0
    p = min(1.0, 2.0 * (1.0 + min(le, ge)) / (1.0 + n_boot))
    return dict(obs=float(obs), lo=float(lo), hi=float(hi), p=float(p), n=n,
                n_ties=n_eq, n_degenerate=n_degen, note="")


def benjamini_hochberg(pvals, alpha=0.05):
    """BH step-up + monotone adjusted p. Verified against a reference."""
    p = np.asarray(pvals, float); m = len(p)
    if m == 0:
        return np.array([], bool), np.array([])
    order = np.argsort(p)
    passed = p[order] <= alpha * (np.arange(1, m + 1) / m)
    kmax = np.where(passed)[0].max() + 1 if passed.any() else 0
    sig = np.zeros(m, bool)
    if kmax > 0:
        sig[order[:kmax]] = True
    q = np.empty(m); prev = 1.0
    for i in range(m - 1, -1, -1):
        prev = min(prev, p[order[i]] * m / (i + 1)); q[order[i]] = prev
    return sig, q


def student_t_breach_probs(frame, task, nu=STUDENT_T_DF):
    """The Normal baseline with a fat-tailed t CDF of the SAME variance.

    A t_ν has variance ν/(ν-2), so z is rescaled by sqrt((ν-2)/ν) to keep the
    two baselines on an identical scale — the ONLY thing that differs is the
    tail shape. That is exactly the comparison Part 10.3 needs:
      t ≫ Normal → the Gaussian's errors are structured; ML has room.
      t ≈ Normal → residual moves are noise here; no model on these inputs wins.
    """
    try:
        from scipy import stats as _st
    except Exception:
        return np.full(len(frame), np.nan)
    tdir, tday = task["direction"], int(task["day"])
    sig_col  = "sigma_upper_rolling" if tdir == "upper" else "sigma_lower_rolling"
    band_col = "band_upper" if tdir == "upper" else "band_lower"

    def _n(c):
        """Read one column as a float array, NaN-filled when absent.

        Parameters
        ----------
        c : str
            Column name.

        Returns
        -------
        ndarray of float
            The column's values, or an all-NaN array of the right length.
        """
        return (pd.to_numeric(frame[c], errors="coerce").to_numpy(float)
                if c in frame.columns else np.full(len(frame), np.nan))

    band, sg, ref = _n(band_col), _n(sig_col), _n(f"d{tday}_close")
    cs = _n("expected_cycle_days")
    cs = np.where(np.isfinite(cs) & (cs > 0), cs, float(CYCLE_STEPS))
    dl = cs + 1.0 - float(tday)
    with np.errstate(divide="ignore", invalid="ignore"):
        sa = sg * np.sqrt(np.clip(dl / cs, 0, None))
        z  = np.log(band / ref) / sa
        zs = z * math.sqrt((nu - 2.0) / nu)
        p  = np.where(np.isfinite(zs),
                      (1.0 - _st.t.cdf(zs, df=nu)) if tdir == "upper" else _st.t.cdf(zs, df=nu),
                      np.nan)
    ok = np.isfinite(band) & np.isfinite(ref) & (ref > 0) & np.isfinite(sg) & (sg > 0)
    return np.where(ok, p, np.nan)


# ============================================================================
# OOS BUILDERS
# ============================================================================
def analytic_oos_perfold(frame, task, splits, prob_fn):
    """B1-FAIR: threshold fitted per fold on TRAIN+VAL, applied to that fold's
    sealed TEST block. Identical function, identical block, identical objective
    to the cross-fit ML threshold in S3_Train v3 — so neither contestant has a
    data-budget advantage."""
    y_all = frame[task["label_col"]].astype(int).values
    pv    = prob_fn(frame, task)
    cid   = frame["cycle_id"].astype(int).values
    pred, prob = {}, {}
    n_skipped = 0
    for (idx_tr, idx_vl, idx_te) in splits:
        fit_idx = np.concatenate([idx_tr, idx_vl])              # never test
        ok = np.isfinite(pv[fit_idx])
        if ok.sum() < 10 or y_all[fit_idx][ok].sum() < 2:
            n_skipped += len(idx_te); continue
        thr, _ = best_f1_threshold(y_all[fit_idx][ok], pv[fit_idx][ok])
        if thr is None:
            n_skipped += len(idx_te); continue
        for i in idx_te:
            if not np.isfinite(pv[i]):
                continue
            prob[int(cid[i])] = float(pv[i])
            pred[int(cid[i])] = int(pv[i] >= thr)
    return pred, prob, n_skipped


def ml_oos_ensemble(models_list, sigma, window):
    """Ensemble the cached out-of-sample probabilities.

    🔴 [L1] Uses `oos_thr` — the threshold of the fold that produced each
    prediction — NOT the single cache-level `threshold`, which is fitted on the
    last 20% of all cycles and overlaps ~33% of the OOS set.
    """
    per_model = []
    for m in models_list:
        cf = m.get("cache_file")
        if (not cf or not os.path.exists(cf)) and "task" in m:
            cf = (nn_meta_key(m["task"], int(m["n_features"]), sigma, window)
                  if m["model"] == "NN_FF"
                  else cache_key(m["task"], m["model"], int(m["n_features"]), sigma, window))
        if not cf or not os.path.exists(cf):
            continue
        try: c = joblib.load(cf)
        except Exception: continue
        cids = c.get("oos_cycle_id") or []
        prob = c.get("oos_prob") or []
        thrs = c.get("oos_thr")
        if not cids:
            continue
        if not thrs or len(thrs) != len(cids):
            raise RuntimeError(
                f"❌ [L1] {os.path.basename(cf)} has no per-cycle `oos_thr`.\n"
                f"   Falling back to the cache-level `threshold` would reintroduce\n"
                f"   the leak (it is fitted on the last 20% of ALL cycles, which\n"
                f"   overlaps ~33% of the OOS set). Delete the cache and re-run\n"
                f"   S3_Train_P1_v3.")
        per_model.append({
            "prob": {int(a): float(b) for a, b in zip(cids, prob)},
            "thr":  {int(a): float(b) for a, b in zip(cids, thrs)},
            "w":    float(m.get("oof_score", m.get("val_f1", 0.5))),
            "name": m.get("model", "?")})
    if not per_model:
        return {}, {}, "?"
    common = sorted(set.intersection(*[set(pm["prob"]) for pm in per_model]))
    prob_map, pred_map = {}, {}
    for cid in common:
        ps = [pm["prob"][cid] for pm in per_model]
        ts = [pm["thr"][cid] for pm in per_model]         # 🔴 per-fold, per-model
        ws = [max(pm["w"], 0.01) for pm in per_model]
        pred, prob, *_ = ensemble_predict(ps, ts, ws)
        prob_map[cid] = float(prob); pred_map[cid] = int(pred)
    return prob_map, pred_map, "+".join(sorted({pm["name"] for pm in per_model}))


def task_frame_and_splits(task):
    """🔶 [ALIGN] the SAME frame S3_Train used — same function, same order."""
    sg, w = ((_u_sig, _u_win) if task["direction"] == "upper" else (_l_sig, _l_win))
    fr, feats, n_drop = p1_build_task_frame(df_cycles, sg, w, task)
    return fr, walk_forward_splits(len(fr)), sg, w


# ============================================================================
# ██  PART A — the headline comparison  ██
# ============================================================================
print("\n" + "█" * 78)
print("  PART A — ML-core vs NORMAL (out-of-sample; both thresholds on train+val)")
print("█" * 78)

preds_store, comparison, payoff_diff = {}, [], {}

for task in TASKS:
    tn, tlabel = task["name"], task["label_col"]
    frame, splits, sg, w = task_frame_and_splits(task)
    bidx = frame.set_index("cycle_id")

    nm_pred, nm_prob, _sk = analytic_oos_perfold(frame, task, splits, normal_breach_probs)
    st_pred, st_prob = ({}, {})
    if P1_ENABLE_STUDENT_T:
        st_pred, st_prob, _ = analytic_oos_perfold(frame, task, splits, student_t_breach_probs)

    ml_prob, ml_pred, mname = ({}, {}, "?")
    if tn in day_models:
        dm = day_models[tn]
        ml_prob, ml_pred, mname = ml_oos_ensemble(dm["models"], dm["sigma"], dm["window"])

    if not ml_pred:
        cids = sorted(nm_pred)
        row = {"task": tn, "n": len(cids), "ml_f1": np.nan, "winner": "Normal only"}
        if cids:
            y = bidx.loc[cids][tlabel].astype(int).values
            row["normal_f1"] = round(f1_at(y, np.array([nm_pred[c] for c in cids], float), 0.5), 4)
        comparison.append(row)
        print(f"     {tn:9s}  ML=N/A  → Normal only  (n={len(cids)})")
        continue

    cids = sorted(set(ml_pred) & set(nm_pred))
    if P1_ENABLE_STUDENT_T and st_pred:
        cids = sorted(set(cids) & set(st_pred))
    if not cids:
        comparison.append({"task": tn, "n": 0, "winner": "no overlap"}); continue

    y    = bidx.loc[cids][tlabel].astype(int).values
    mlpr = np.array([ml_prob[c] for c in cids]); mlpd = np.array([ml_pred[c] for c in cids])
    nmpr = np.array([nm_prob[c] for c in cids]); nmpd = np.array([nm_pred[c] for c in cids])
    stpr = np.array([st_prob.get(c, np.nan) for c in cids])
    stpd = np.array([st_pred.get(c, 0) for c in cids])

    S_ml = all_scores(y, mlpr, mlpd); S_nm = all_scores(y, nmpr, nmpd)
    S_st = all_scores(y, stpr, stpd) if P1_ENABLE_STUDENT_T else {}

    # the winner comes from the bootstrap CI, not a fixed dead-band
    bs = paired_bootstrap_diff(y, mlpd, nmpd, stat="f1")
    winner = "TIE" if (bs["lo"] <= 0 <= bs["hi"]) else ("ML" if bs["obs"] > 0 else "Normal")

    comparison.append({
        "task": tn, "n": len(y), "base_rate": round(S_ml["base_rate"], 4),
        "ml_f1": round(S_ml["f1"], 4), "normal_f1": round(S_nm["f1"], 4),
        "t_f1": round(S_st.get("f1", np.nan), 4) if S_st else np.nan,
        "delta_f1": round(S_ml["f1"] - S_nm["f1"], 4),
        "ci_low": round(bs["lo"], 4), "ci_high": round(bs["hi"], 4),
        "p_boot": round(bs["p"], 5), "winner": winner,
        "ml_brier_skill": round(S_ml["brier_skill"], 4),
        "normal_brier_skill": round(S_nm["brier_skill"], 4),
        "t_brier_skill": round(S_st.get("brier_skill", np.nan), 4) if S_st else np.nan,
        "ml_brier": round(S_ml["brier"], 5), "normal_brier": round(S_nm["brier"], 5),
        "normal_reliability": round(S_nm["reliability"], 5),
        "normal_resolution": round(S_nm["resolution"], 5),
        "ml_model": mname, "sigma": sg, "window": w,
    })
    payoff_diff[tn] = (binary_payoff_vec(y, mlpd) - binary_payoff_vec(y, nmpd)).tolist()
    preds_store[tn] = {
        "y_true": y.tolist(), "cycle_ids": [int(c) for c in cids],
        "ml_prob": mlpr.tolist(), "ml_pred": mlpd.tolist(),
        "normal_prob": nmpr.tolist(), "normal_pred": nmpd.tolist(),
        "t_prob": [None if not np.isfinite(v) else float(v) for v in stpr],
        "t_pred": [int(v) for v in stpd],
        "n": int(len(y)), "ml_model": mname, "sigma": float(sg), "window": int(w),
        "scores": {"ml": S_ml, "normal": S_nm, "student_t": S_st},
    }
    em = {"ML": "🟢", "Normal": "🔴", "TIE": "🟡"}[winner]
    print(f"     {em} {tn:9s} F1 ML={S_ml['f1']:.3f} Nm={S_nm['f1']:.3f} "
          f"t={S_st.get('f1', float('nan')):.3f} Δ={bs['obs']:+.3f} "
          f"[{bs['lo']:+.3f},{bs['hi']:+.3f}] p={bs['p']:.4f} → {winner}  (n={len(y)})")

df_cmp = pd.DataFrame(comparison)


# ── Student-t verdict: the Part 10.3 discriminator ──────────────────────────
if P1_ENABLE_STUDENT_T and "t_f1" in df_cmp.columns and df_cmp["t_f1"].notna().any():
    _dt = (df_cmp["t_brier_skill"] - df_cmp["normal_brier_skill"]).dropna()
    print("\n  ── STUDENT-t DIAGNOSTIC " + "─" * 52)
    print(f"     mean Brier-skill  t − Normal = {_dt.mean():+.4f}  "
          f"(t wins in {int((_dt > 0).sum())}/{len(_dt)} tasks)")
    if _dt.mean() > 0.01:
        print("     → The fat tail HELPS. The Gaussian's errors are structured, so a")
        print("       learned model has genuine room to improve. A ML≈Normal tie is")
        print("       then a SMALL-SAMPLE result, not a Bayes-floor result.")
    else:
        print("     → The fat tail does NOT help. The residual move from the decision")
        print("       day to expiry looks like noise on these inputs, so the Gaussian")
        print("       is near-optimal and no model restricted to them can win. That is")
        print("       a legitimate, publishable finding — and it says the next study")
        print("       needs richer information, not a better function.")
    print("  " + "─" * 74)


# ============================================================================
# ██  PART A2 — per-config robustness  ██
# ============================================================================
print("\n" + "█" * 78)
print("  PART A2 — every (σ, window), so the headline can be checked")
print("█" * 78)

_ledger = pd.read_excel(LEDGER_PATH) if os.path.exists(LEDGER_PATH) else pd.DataFrame()
if not _ledger.empty:
    for _c in ("sigma", "window", "n_features", "oof_score", "val_f1", "test_score"):
        if _c in _ledger.columns:
            _ledger[_c] = pd.to_numeric(_ledger[_c], errors="coerce")
    _ledger["sigma"] = _ledger["sigma"].round(2)
    if "folds_consistency_pass" in _ledger.columns:
        _ledger["folds_consistency_pass"] = (_ledger["folds_consistency_pass"]
                                             .astype(str).str.strip().str.lower()
                                             .isin(["true", "1", "yes"]))

percfg = []
for sg in SIGMA_GRID:
    for w in SIGMA_WINDOW_GRID:
        for task in TASKS:
            tn, tlabel = task["name"], task["label_col"]
            try:
                fr, feats, _nd = p1_build_task_frame(df_cycles, sg, w, task)
            except Exception:
                continue
            sp = walk_forward_splits(len(fr)); bidx = fr.set_index("cycle_id")
            nm_pred, nm_prob, _ = analytic_oos_perfold(fr, task, sp, normal_breach_probs)
            mlp, mlq = {}, {}
            if not _ledger.empty:
                sub = _ledger[(_ledger["task"] == tn) & (_ledger["sigma"] == round(float(sg), 2))
                              & (_ledger["window"] == int(w))
                              & (_ledger["folds_consistency_pass"] == True)]
                if not sub.empty:
                    top = select_top_k(sub, k=TOP_K)
                    ml = [{"task": tn, "model": str(r["model"]),
                           "n_features": int(r["n_features"]),
                           "oof_score": float(r.get("oof_score", r.get("val_f1", 0.5))),
                           "cache_file": r.get("cache_file")}
                          for _, r in top.iterrows()]
                    try:
                        mlq, mlp, _ = ml_oos_ensemble(ml, float(sg), int(w))
                    except RuntimeError:
                        mlq, mlp = {}, {}
            cids = sorted(set(mlp) & set(nm_pred)) if mlp else sorted(nm_pred)
            if not cids:
                continue
            y = bidx.loc[cids][tlabel].astype(int).values
            nf1 = f1_at(y, np.array([nm_pred[c] for c in cids], float), 0.5)
            mf1 = (f1_at(y, np.array([mlp[c] for c in cids], float), 0.5) if mlp else np.nan)
            percfg.append({"task": tn, "direction": task["direction"], "day": task["day"],
                           "sigma": round(float(sg), 2), "window": int(w),
                           "ml_f1": round(mf1, 4) if np.isfinite(mf1) else np.nan,
                           "normal_f1": round(nf1, 4),
                           "delta_f1": round(mf1 - nf1, 4) if np.isfinite(mf1) else np.nan,
                           "base_rate": round(float(y.mean()), 4), "n": len(y)})
df_percfg = pd.DataFrame(percfg)
if not df_percfg.empty:
    _d = df_percfg["delta_f1"].dropna()
    print(f"  Cells: {len(df_percfg)} | ML>Normal {int((_d > 0).sum())} | "
          f"Normal>ML {int((_d < 0).sum())} | mean Δ {_d.mean():+.4f}")
    print("  ⚠️ F1 is base-rate sensitive and the base rate CHANGES with σ "
          "(the band IS the label),")
    print("     so ΔF1 is comparable WITHIN a cell but not ACROSS σ levels.")


# ============================================================================
# BH ACROSS TASKS  — 2 label families, not 6 independent tests
# ============================================================================
_sig = df_cmp[df_cmp["p_boot"].notna()].copy() if "p_boot" in df_cmp.columns else pd.DataFrame()
if not _sig.empty:
    s, q = benjamini_hochberg(_sig["p_boot"].values)
    _sig["q_BH_all6"] = np.round(q, 5); _sig["sig_BH_all6"] = s
    for fam in ("upper", "lower"):
        m = _sig["task"].str.startswith(fam)
        if m.any():
            sf, qf = benjamini_hochberg(_sig.loc[m, "p_boot"].values)
            _sig.loc[m, "q_BH_family"] = np.round(qf, 5)
            _sig.loc[m, "sig_BH_family"] = sf
    print(f"\n  BH: {int(_sig['sig_BH_all6'].sum())}/{len(_sig)} significant "
          f"treating 6 cells as 6 tests; "
          f"{int(_sig.get('sig_BH_family', pd.Series(dtype=bool)).sum())} within "
          f"the 2 label families.")
    print("  ⚠️ upper_D2/D3/D4 share an IDENTICAL label vector (the label is a")
    print("     cycle-level property), as do lower_*. There are ~2 independent")
    print("     families, not 6. BH stays valid under this positive dependence but")
    print("     becomes conservative — report the family column, not just k/6.")


# ============================================================================
# OUTPUTS
# ============================================================================
prov = pd.DataFrame([
    {"key": "script", "value": "S6_InferEval_P1_v3.0"},
    {"key": "feature_mode", "value": P1_FEATURE_MODE},
    {"key": "primary_metric", "value": f"{P1_PRIMARY_METRIC} ({PRIMARY_METRIC_NAME})"},
    {"key": "ml_threshold", "value": "per-fold `oos_thr`, cross-fitted on train+val [L1 FIXED]"},
    {"key": "normal_threshold", "value": "per-fold on train+val, same function, same block"},
    {"key": "student_t_df", "value": STUDENT_T_DF if P1_ENABLE_STUDENT_T else "disabled"},
    {"key": "tie_rule", "value": "bootstrap 95% CI contains 0"},
    {"key": "selected_upper", "value": f"sigma={_u_sig}, window={_u_win}"},
    {"key": "selected_lower", "value": f"sigma={_l_sig}, window={_l_win}"},
    {"key": "config_selection", "value": json.dumps({k: v.get("selection_mode")
                                                     for k, v in dir_cfg.items()})},
    {"key": "cache_version", "value": CACHE_VERSION},
    {"key": "run_stamp", "value": RUN_STAMP},
    {"key": "random_state", "value": RANDOM_STATE},
    {"key": "generated", "value": datetime.datetime.now().strftime("%d-%b-%Y %H:%M")},
])

_rel = []
for tn, d in preds_store.items():
    for who, pk in [("ml", "ml_prob"), ("normal", "normal_prob"), ("student_t", "t_prob")]:
        pr = [np.nan if v is None else v for v in d[pk]]
        rt = reliability_table(d["y_true"], pr)
        if len(rt):
            rt.insert(0, "contestant", who); rt.insert(0, "task", tn); _rel.append(rt)
df_rel = pd.concat(_rel, ignore_index=True) if _rel else pd.DataFrame()

P1_XLSX = os.path.join(RESULTS_DIR, "project1_mlcore_vs_normal.xlsx")
with pd.ExcelWriter(P1_XLSX, engine="openpyxl") as xw:
    df_cmp.to_excel(xw, sheet_name="Headline", index=False)
    df_percfg.to_excel(xw, sheet_name="Per_config", index=False)
    (_sig if not _sig.empty else pd.DataFrame({"note": ["no ML tasks"]})
     ).to_excel(xw, sheet_name="Significance", index=False)
    (df_rel if not df_rel.empty else pd.DataFrame({"note": ["none"]})
     ).to_excel(xw, sheet_name="Calibration", index=False)
    prov.to_excel(xw, sheet_name="Provenance", index=False)
print(f"\n  ✅ Workbook: {P1_XLSX}")

artifact = {
    "project": "project1", "script": "S6_InferEval_P1_v3.0",
    "run_type": RUN_TYPE, "run_stamp": RUN_STAMP, "cache_version": CACHE_VERSION,
    "feature_mode": P1_FEATURE_MODE, "primary_metric": P1_PRIMARY_METRIC,
    "student_t_df": STUDENT_T_DF if P1_ENABLE_STUDENT_T else None,
    "results_dir": RESULTS_DIR, "ledger_path": LEDGER_PATH,
    "direction_config": dir_cfg,
    "comparison": df_cmp.to_dict("records"),
    "per_config": df_percfg.to_dict("records"),
    "predictions": preds_store, "payoff_diff": payoff_diff,
}
ARTIFACT_PATH = os.path.join(RESULTS_DIR, f"backtest_artifact_{RUN_TYPE}.json")
with open(ARTIFACT_PATH, "w") as f:
    json.dump(artifact, f, indent=2, default=str)
print(f"  ✅ Artifact : {ARTIFACT_PATH}")

print("\n" + "=" * 78)
print("  ===== PROJECT 1 HEADLINE =====")
print(f"  {'Task':<10}{'Normal':>9}{'ML':>9}{'t':>9}{'Δ(ML-Nm)':>10}{'95% CI':>19}{'winner':>9}")
for _, r in df_cmp.iterrows():
    if pd.isna(r.get("ml_f1")):
        print(f"  {r['task']:<10}{r.get('normal_f1', float('nan')):>9.3f}"
              f"{'—':>9}{'—':>9}{'—':>10}{'—':>19}{'Normal only':>9}"); continue
    _ci = f"[{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]"
    _tf = r["t_f1"] if pd.notna(r["t_f1"]) else float("nan")
    print(f"  {r['task']:<10}{r['normal_f1']:>9.3f}{r['ml_f1']:>9.3f}{_tf:>9.3f}"
          f"{r['delta_f1']:>+10.3f}{_ci:>19}{r['winner']:>9}")
print("=" * 78)


# ============================================================================
# ██  PART B — LIVE  ██
# ============================================================================
print("\n" + "█" * 78)
print("  PART B — LIVE INFERENCE (current cycle)")
print("█" * 78)

daily = load_input(CLEAN_INPUT_PATH)
_alld = set(daily[DATE_COL].dt.normalize())
daily["is_expiry"] = daily[DATE_COL].apply(lambda d: int(is_expiry(d, CUTOFF_DATE, _alld)))
_exp_idx = daily.index[daily["is_expiry"] == 1].tolist()

if not _exp_idx or _exp_idx[-1] == len(daily) - 1:
    print("  ℹ️ No in-progress cycle (the last day is an expiry). Live skipped.")
else:
    cur_ne = daily.iloc[_exp_idx[-1] + 1:].copy()
    cur_ne = cur_ne[cur_ne["is_expiry"] == 0].reset_index(drop=True)
    cur_day = len(cur_ne)
    if cur_day == 0:
        print("  ℹ️ No non-expiry rows after the last expiry. Live skipped.")
    elif cur_day > CYCLE_DECISION_DAYS:
        print(f"  ⚠️ {cur_day} days since the last expiry (> {CYCLE_DECISION_DAYS}). "
              f"An expiry was probably missed — live skipped rather than guessed.")
    else:
        last_close = float(daily[CLOSE_COL].iloc[-1])
        print(f"  Current cycle: D{cur_day} | last close {last_close:,.2f} | "
              f"{daily[DATE_COL].max().date()}")
        cur_rec = build_cycle_record(cur_ne, expiry_row=None, cycle_id=int(1e9),
                                     expected_cycle_days=CYCLE_STEPS)
        print(f"     days_left_D{cur_day} = {cur_rec[f'days_left_D{cur_day}']:.0f}  "
              f"(training parity ✅)")

        hist = standard_cycles(df_cycles)
        live_df = pd.concat([hist, pd.DataFrame([cur_rec])], ignore_index=True)
        live_df = add_rolling_cycle_features(live_df)
        assert live_df.index[-1] == len(live_df) - 1, "live row must be last"

        # score EVERY elapsed decision day, not only today.
        #  At D3 this yields D2 and D3. There is deliberately NO D1 model:
        #  DAYS = [2, 3, 4] because band_upper is BUILT from d1_close, so the
        #  D1 distance to the band is exactly k*sigma for every cycle and
        #  carries no cycle-specific information. D1 is shown as context only.
        #
        #  These earlier-day numbers are RETROSPECTIVE: they are what today's
        #  deployed model says about D2's data, not a record of what was
        #  emitted on D2. Labelled as such in the table and the JSON.
        recs = {}
        day_rows = []          # one record per (day, direction) for the table
        for task in [t for t in TASKS if 2 <= t["day"] <= cur_day]:
            tn, td = task["name"], task["direction"]
            # PER-DIRECTION window. v2 used the UPPER window for both
            # legs, so with different windows the lower leg was scored live on
            # features built the wrong way — a pure train/live skew.
            sg, w = ((_u_sig, _u_win) if td == "upper" else (_l_sig, _l_win))
            lb = compute_asymmetric_bands_and_labels(live_df.copy(), sg, w, sg, w,
                                                     make_labels=False)
            lb = p1_attach_features(lb, w, strict=True)
            row = lb.iloc[[-1]].copy()

            bU, bL = float(row["band_upper"].iloc[0]), float(row["band_lower"].iloc[0])
            npv = float(normal_breach_probs(row, task, strict=True)[0])

            # per-day state. The BANDS are fixed at D1 for the whole
            # cycle; what moves is the close, and therefore the distance to
            # each band. Distance in sigma units is the comparable quantity.
            _d   = int(task["day"])
            _g   = lambda col: (float(row[col].iloc[0])
                                if col in row.columns
                                and pd.notna(row[col].iloc[0]) else float("nan"))
            _cl  = _g(f"d{_d}_close")
            _dt  = row[f"d{_d}_date"].iloc[0] if f"d{_d}_date" in row.columns else None
            #  dist_to_*_Dn was DROPPED from the v2 feature set, so
            #  reading it here returned NaN and the Dist% column printed blank.
            #  Compute it directly from the band and the close — same definition
            #  the old column had, and independent of whatever is whitelisted.
            _bnd  = bU if task["direction"] == "upper" else bL
            _dist = ((_bnd - _cl) / _cl if task["direction"] == "upper"
                     else (_cl - _bnd) / _cl) if _cl == _cl and _cl else float("nan")
            _nrm  = _g(f"norm_dist_{task['direction']}_D{_d}")
            _dl   = _g(f"days_left_D{_d}")

            hb = compute_asymmetric_bands_and_labels(hist.copy(), sg, w, sg, w)
            hb = drop_warmup_cycles(hb, verbose=False)
            yh  = hb[task["label_col"]].astype(int).values
            nhp = normal_breach_probs(hb, task, strict=False)
            _ok = np.isfinite(nhp)
            nthr, _ = best_f1_threshold(yh[_ok], nhp[_ok])        # all history
            if nthr is None: nthr = 0.5
            n_act = "EXIT" if npv >= nthr else "HOLD"

            if tn not in day_models:
                print(f"     {tn}: ⚠️ no ML → Normal only → {n_act} (p={npv:.1%}, thr={nthr:.2f})")
                recs[tn] = {"action": n_act, "source": "normal_only",
                            "normal_prob": npv, "band_upper": bU, "band_lower": bL}
                day_rows.append(dict(day=_d, direction=task["direction"], date=_dt,
                                     close=_cl, band_upper=bU, band_lower=bL,
                                     dist=_dist, norm_dist=_nrm, days_left=_dl,
                                     ml_prob=float("nan"), ml_action="n/a",
                                     normal_prob=npv, normal_action=n_act,
                                     econ_action="n/a", source="normal_only",
                                     retrospective=bool(_d < cur_day)))
                continue

            probs, thrs, ws = [], [], []
            for m in day_models[tn]["models"]:
                cf = m["cache_file"]
                if not os.path.exists(cf): continue
                c = joblib.load(cf)
                # STALE-LEDGER GUARD.
                # Part A builds its filenames with cache_key(), which embeds
                # CACHE_VERSION, so it cannot load a foreign config. This path
                # instead trusts `cache_file` recorded in the ledger by a PRIOR
                # S3_Select run, and checked only that the file exists. After a
                # config change the old .pkl files are still on disk, so live
                # inference silently scored with models trained under different
                # settings — the exact mixing CACHE_VERSION exists to prevent.
                _cv = c.get("cache_version")
                if _cv != CACHE_VERSION:
                    raise RuntimeError(
                        f"❌ [E3] STALE MODEL CACHE for {tn}/{m['model']}.\n"
                        f"   cache file : {os.path.basename(cf)}\n"
                        f"   trained as : {_cv}\n"
                        f"   config now : {CACHE_VERSION}\n"
                        f"   The ledger ({os.path.basename(BEST_MODELS_PATH)}) was written by an\n"
                        f"   earlier run and points at models built under different settings.\n"
                        f"   Scoring with them would mix configurations.\n"
                        f"   Fix: re-run S3_Train → S3_Harvest → S3_Select, which rewrites\n"
                        f"   the ledger for {CACHE_VERSION}. Do NOT delete the old cache —\n"
                        f"   it is the audit trail for the previous results.")
                feats = c["features"]
                miss = [f for f in feats if f not in row.columns]
                if miss:
                    raise RuntimeError(f"❌ LIVE feature parity error {tn}/{m['model']}: "
                                       f"missing {miss[:5]} — fix S2/S0, do NOT zero-fill.")
                nanf = [f for f in feats if not np.isfinite(pd.to_numeric(row[f].iloc[0],
                                                                         errors='coerce'))]
                if nanf:
                    raise RuntimeError(f"❌ LIVE NaN in core feature(s) {tn}/{m['model']}: "
                                       f"{nanf[:5]} — do NOT zero-fill.")
                Xs = apply_saved_pipeline(row[feats].to_numpy(float), c["clip_bounds"],
                                          c["scaler"], c.get("no_scale_idx", []),
                                          strict=True, feature_names=feats)
                mdl = c.get("final_model")
                if mdl is None or not hasattr(mdl, "predict_proba"):
                    print(f"     ⚠️ {tn}/{m['model']}: no usable estimator — skipped"); continue
                probs.append(float(mdl.predict_proba(Xs)[0, 1]))
                # cache `threshold` is cross-fitted over ALL cycles in
                # S3_Train v3 — the live-only counterpart of Normal's
                # all-history threshold. Symmetric.
                thrs.append(float(c.get("threshold", 0.5)))
                ws.append(float(m.get("oof_score", m.get("val_f1", 0.5))))

            if not probs:
                print(f"     {tn}: ⚠️ no usable ML → Normal fallback → {n_act}")
                recs[tn] = {"action": n_act, "source": "normal_fallback"}; continue

            pred, prob, vote, note = ensemble_predict(probs, thrs, ws)
            act = "EXIT" if pred else "HOLD"
            econ = "EXIT" if prob >= ECONOMIC_THRESHOLD else "HOLD"
            if P1_DECISION_RULE == "economic":
                act = econ
            print(f"     {tn}: ML {act} (p={prob:.1%}, {vote}, {note}) | "
                  f"Normal {n_act} (p={npv:.1%}) | economic-rule {econ} "
                  f"(p*={ECONOMIC_THRESHOLD:.2f}) → "
                  f"{'AGREE' if act == n_act else 'DISAGREE'}")
            recs[tn] = {"action": act, "ml_prob": prob, "normal_prob": npv,
                        "normal_action": n_act, "economic_action": econ,
                        "source": "ml_ensemble", "band_upper": bU, "band_lower": bL,
                        "sigma": sg, "window": w, "day": _d,
                        "close": _cl, "dist_to_band": _dist,
                        "dist_in_sigma": _nrm, "days_left": _dl,
                        "retrospective": bool(_d < cur_day)}
            day_rows.append(dict(day=_d, direction=task["direction"], date=_dt,
                                 close=_cl, band_upper=bU, band_lower=bL,
                                 dist=_dist, norm_dist=_nrm, days_left=_dl,
                                 ml_prob=prob, ml_action=act,
                                 normal_prob=npv, normal_action=n_act,
                                 econ_action=econ, source="ml_ensemble",
                                 retrospective=bool(_d < cur_day)))

        # ── 🔴 D1 GUARD ─────────────────────────────────────────────────────
        #  The loop above iterates days 2..cur_day, so on D1 it never executes
        #  and bU / bL / sg / w are never bound. Every line below reads them, so
        #  S6 died with "NameError: name 'bU' is not defined" on roughly one
        #  trading day in five — whenever a new cycle had only its entry day.
        #  There is no model at D1 by design, but the BAND still exists and is
        #  worth reporting, so derive it from the upper config and carry on.
        if not day_rows:
            sg, w = _u_sig, _u_win
            _lb0 = compute_asymmetric_bands_and_labels(live_df.copy(), sg, w, sg, w,
                                                       make_labels=False)
            _r0  = _lb0.iloc[[-1]]
            bU   = float(_r0["band_upper"].iloc[0])
            bL   = float(_r0["band_lower"].iloc[0])
            print("\n  ℹ️  D1 is the entry day: the band is BUILT from today's close, "
                  "so no model is scored.")

        # ── CYCLE-TO-DATE TABLE ──────────────────────────────────
        print(f"\n  ── CYCLE TO DATE  (bands are FIXED at D1 for the whole cycle) ──")
        print(f"     Band upper {bU:>12,.2f}   Band lower {bL:>12,.2f}   "
              f"sigma={sg} window={w}")
        _d1c  = float(cur_ne[CLOSE_COL].iloc[0])
        _d1dt = cur_ne[DATE_COL].iloc[0]
        print(f"\n     {'Day':<8}{'Date':<12}{'Close':>10}{'Dist%':>9}{'Dist/σ':>8}"
              f"{'DL':>4}  {'ML p':>7} {'Nm p':>7}  {'ML':<5}{'Nm':<5}{'':<3}")
        print("     " + "-" * 77)
        print(f"     {'D1':<8}{str(_d1dt.date()):<12}{_d1c:>10,.2f}"
              f"{'—':>9}{'—':>8}{'—':>4}  {'—':>7} {'—':>7}  "
              f"{'—':<5}{'—':<5}   entry: band is BUILT from this close,")
        print(f"     {'':<8}{'':<12}{'':>10}{'':>9}{'':>8}{'':>4}  {'':>7} {'':>7}  "
              f"{'':<5}{'':<5}   so D1 has no model by design")
        for r in sorted(day_rows, key=lambda x: (x["day"], x["direction"])):
            _tag = "  ← today" if not r["retrospective"] else "  (retro)"
            _dd  = r["date"].date() if hasattr(r["date"], "date") else str(r["date"])[:10]
            _lab = f"D{r['day']} {'up' if r['direction'] == 'upper' else 'dn'}"
            print(f"     {_lab:<8}{str(_dd):<12}"
                  f"{r['close']:>10,.2f}{100*r['dist']:>8.2f}%{r['norm_dist']:>8.2f}"
                  f"{r['days_left']:>4.0f}  "
                  f"{('%.1f%%' % (100*r['ml_prob'])) if r['ml_prob'] == r['ml_prob'] else '  n/a':>7} "
                  f"{100*r['normal_prob']:>6.1f}%  "
                  f"{r['ml_action']:<5}{r['normal_action']:<5}{_tag}")
        print("     " + "-" * 77)
        print("     Dist%  = distance from that day's close to ITS band "
              "(upper rows → upper band, lower rows → lower band)")
        print("     Dist/σ = the same distance in cycle-volatility units — the "
              "comparable figure across days")
        print("     (retro) = today's deployed model re-scored on that day's data; "
              "NOT what was emitted that day")

        ua = recs.get(f"upper_D{cur_day}", {}).get("action", "N/A")
        la = recs.get(f"lower_D{cur_day}", {}).get("action", "N/A")
        print(f"\n  🎯 RECOMMENDATION (D{cur_day}):")
        if not day_rows:
            print("     — no recommendation on D1. The band is defined by today's "
                  "close; the first decision is at D2.")
        else:
            print("     ✅ HOLD both legs" if (ua == "HOLD" and la == "HOLD") else
                  "     🚨 EXIT entire straddle" if (ua == "EXIT" and la == "EXIT") else
                  "     ⚠️ EXIT CALL leg (upper breach expected)" if ua == "EXIT" else
                  "     ⚠️ EXIT PUT leg (lower breach expected)" if la == "EXIT" else
                  f"     Upper={ua} | Lower={la}")

        logp = os.path.join(RESULTS_DIR, f"live_recommendation_{RUN_TYPE}.json")
        with open(logp, "w") as f:
            json.dump({"date": str(daily[DATE_COL].max().date()), "day": cur_day,
                       "nifty": last_close, "decision_rule": P1_DECISION_RULE,
                       "economic_threshold": ECONOMIC_THRESHOLD,
                       "band_upper": bU, "band_lower": bL,
                       "sigma": sg, "window": w,
                       "d1_close": _d1c, "d1_date": str(_d1dt.date()),
                       "cycle_to_date": day_rows,      # every elapsed day
                       "recommendations": recs}, f, indent=2, default=str)
        print(f"     📝 {logp}")

print("\n" + "=" * 78)
print("  ✅ S6_InferEval_P1_v6.0 COMPLETE")
print("     ➡️  NEXT: S7_Significance  then  S8_Charts")
print("=" * 78)


  🎯 S6_InferEval_P1_v3.0 — ML-core vs NORMAL (+Student-t) + LIVE  (FULL)
     Results → /content/drive/MyDrive/LSTM/Project1_MLcore_vs_Normal/output_binary/FULL/results/run_20260914_203539
  Active grid → σ[0.5, 0.75, 1.0] × win[4, 8]
  Config → Upper σ=0.5 w=4 (ML) | Lower σ=0.5 w=4 (ML)
  ⚠️ upper: config chosen by ML's own score, rank 1/6 — see Part A2 and the DISCLOSURE in the config JSON.
  ⚠️ lower: config chosen by ML's own score, rank 1/6 — see Part A2 and the DISCLOSURE in the config JSON.
  ──────────────────────────────────────────────────────────────
  📥 INPUT LOADED (read-only)
     Modified : 2026-09-14 20:35
     Rows×Cols: 1901 × 9
     Range    : 2019-01-01 → 2026-09-11
  ──────────────────────────────────────────────────────────────

██████████████████████████████████████████████████████████████████████████████
  PART A — ML-core vs NORMAL (out-of-sample; both thresholds on train+val)
██████████████████████████████████████████████████████████████████████████████
    

In [48]:
# @title
# ============================================================================
# S7_Significance_P1_v4.0 — STATISTICAL TESTS
# ============================================================================
#
# PURPOSE
# -------
# Decide whether any difference S6 reported is real or is sampling noise.
# Pure statistics on saved numbers: no model runs, no re-prediction.
#
#   IN  : backtest_artifact_<world>.json  (written by S6)
#   OUT : significance_<world>.xlsx
#
# WHAT IS TESTED
# --------------
#   PRIMARY    paired bootstrap on the F1 difference, ML minus Gaussian
#   SECONDARY  paired bootstrap on the Brier-skill difference, which is
#              threshold-free and therefore has more power at this sample size
#   THIRD      Student-t against the Gaussian: does a fat tail help at all?
#   ECONOMIC   paired bootstrap on the mean per-cycle payoff difference
#   PLUS       Brier decomposition, ML/Gaussian error correlation, and the
#              multiple-testing correction
#
# WHY THE BOOTSTRAP IS PAIRED
# ---------------------------
# Both contestants score the SAME cycles. Resampling them independently would
# discard that pairing and inflate the variance of the difference, making a
# real gap look like noise.
#
# THREE PROPERTIES OF THE p-VALUES
# --------------------------------
# 1. The (1+k)/(1+B) correction is applied, so the smallest reportable value
#    from B resamples is 2/(B+1), never a literal 0.0000. A p of exactly zero
#    in a thesis table is an unfalsifiable claim.
# 2. Resamples with a difference of exactly zero are not counted in both
#    tails, which would otherwise let a distribution that is half exact-zero
#    and half strongly favourable report p = 1.0.
# 3. Resamples with no positive labels make F1 undefined; they are redrawn
#    once and counted rather than silently treated as a zero difference.
#
# A degenerate case — fewer than about eight usable cycles — returns an
# infinite confidence interval and is marked `insufficient_n`, rather than
# returning lo == hi == observed, which would read as significant whenever the
# observed difference was not exactly zero.
#
# ⚠️ THE SIX TASKS ARE NOT SIX INDEPENDENT HYPOTHESES
# ---------------------------------------------------
# upper_D2, upper_D3 and upper_D4 share an IDENTICAL label vector, because the
# breach label is a CYCLE-level property and only the features differ by day.
# The same holds for the three lower tasks. There are roughly TWO independent
# families, not six.
#
# Benjamini-Hochberg stays VALID under this positive dependence but becomes
# conservative, and "BH-significant k of 6" would read as six independent
# confirmations. This cell therefore reports the correction BOTH ways and
# seeds the bootstrap per task, so the three upper tasks do not draw an
# identical index stream against an identical y.
#
# Raw, unrounded p-values feed the correction. BH thresholds for m = 6 are
# 0.0083, 0.0167, 0.025 and so on, so rounding first can flip a borderline
# hypothesis. Rounding is display-only.
#
# RUN AFTER: S6_InferEval.  NEXT: S8_Charts.
# ============================================================================

try:
    _ = (RESULTS_DIR, RUN_TYPE, RANDOM_STATE, TASKS, PRIMARY_METRIC_NAME,
         PAYOFF_BREACH_CORRECT, PAYOFF_NOBREACH_WRONG, ECONOMIC_THRESHOLD)
    _ = (paired_bootstrap_diff, benjamini_hochberg)      # defined once in S6
    _ = (all_scores, brier_decomposition, brier_skill_score, f1_at, log_loss_safe)
except NameError as _ne:
    raise RuntimeError(f"❌ Missing {_ne}. Run S1_Config_P1_v4.0 → … → S6 first.")

import os, json, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

ALPHA  = 0.05
N_BOOT = 5000

print("\n" + "=" * 78)
print(f"  📊 S7_Significance_P1_v4.0  ({RUN_TYPE})   B={N_BOOT}, α={ALPHA}")
print("=" * 78)

_cands = sorted(glob.glob(os.path.join(os.path.dirname(RESULTS_DIR), "run_*",
                                       f"backtest_artifact_{RUN_TYPE}.json")))
ARTIFACT_PATH = os.path.join(RESULTS_DIR, f"backtest_artifact_{RUN_TYPE}.json")
if not os.path.exists(ARTIFACT_PATH):
    if not _cands:
        raise FileNotFoundError("❌ No S6 artifact found. Run S6_InferEval first.")
    ARTIFACT_PATH = _cands[-1]
    print(f"  ℹ️ Using the most recent artifact: {ARTIFACT_PATH}")
with open(ARTIFACT_PATH) as f:
    art = json.load(f)
preds = art.get("predictions", {})
payoff = art.get("payoff_diff", {})
if not preds:
    raise RuntimeError("❌ Artifact has no predictions — every task was Normal-only.")
print(f"  Tasks with ML predictions: {list(preds)}")


def _arr(v, dtype=float):
    """Convert a JSON list to a float array, mapping null to NaN.

    Parameters
    ----------
    v : list
        Values from the S6 artifact; JSON null becomes None in Python.
    dtype : type
        Target dtype.

    Returns
    -------
    ndarray
        The values with None replaced by NaN, so a missing score cannot be
        read as a real zero.
    """
    return np.array([np.nan if x is None else x for x in v], dtype=dtype)


# ============================================================================
# 1. PRIMARY — F1 difference, and SECONDARY — Brier-skill difference
# ============================================================================
print("\n  📌 1-2: ML vs Normal   (paired bootstrap over CYCLES)")
rows = []
for i, (tn, d) in enumerate(preds.items()):
    y   = np.asarray(d["y_true"], int)
    mlp, mlq = _arr(d["ml_pred"], int), _arr(d["ml_prob"])
    nmp, nmq = _arr(d["normal_pred"], int), _arr(d["normal_prob"])
    # seed per task — v1 reused one seed, coupling the three tasks that
    # share a label vector into literally identical draws
    seed = RANDOM_STATE + 1000 * i

    b_f1 = paired_bootstrap_diff(y, mlp, nmp, N_BOOT, seed, stat="f1")
    b_bs = paired_bootstrap_diff(y, mlp, nmp, N_BOOT, seed + 1, stat="primary",
                                 a_prob=mlq, b_prob=nmq)
    S_ml, S_nm = all_scores(y, mlq, mlp), all_scores(y, nmq, nmp)

    rows.append({
        "task": tn, "n": int(d["n"]), "base_rate": round(S_ml["base_rate"], 4),
        "family": "upper" if tn.startswith("upper") else "lower",
        "ml_f1": round(S_ml["f1"], 4), "normal_f1": round(S_nm["f1"], 4),
        "f1_diff": round(b_f1["obs"], 4),
        "f1_ci_low": round(b_f1["lo"], 4), "f1_ci_high": round(b_f1["hi"], 4),
        "f1_p_raw": b_f1["p"],
        "ml_brier_skill": round(S_ml["brier_skill"], 4),
        "normal_brier_skill": round(S_nm["brier_skill"], 4),
        "bss_diff": round(b_bs["obs"], 4),
        "bss_ci_low": round(b_bs["lo"], 4), "bss_ci_high": round(b_bs["hi"], 4),
        "bss_p_raw": b_bs["p"],
        "ml_log_loss": round(S_ml["log_loss"], 4),
        "normal_log_loss": round(S_nm["log_loss"], 4),
        "note": b_f1.get("note", ""), "n_ties": b_f1.get("n_ties", 0),
        "n_degenerate_draws": b_f1.get("n_degenerate", 0),
    })
    print(f"     {tn:<10} ΔF1={b_f1['obs']:+.3f} [{b_f1['lo']:+.3f},{b_f1['hi']:+.3f}] "
          f"p={b_f1['p']:.4f} | ΔBSS={b_bs['obs']:+.3f} p={b_bs['p']:.4f}"
          + (f"  ⚠️{b_f1['note']}" if b_f1.get("note") else ""))

df = pd.DataFrame(rows)
# the floor is 2/(B+1); a p of exactly 0 is not attainable from B draws
print(f"\n     Smallest attainable p at B={N_BOOT}: {2.0/(N_BOOT+1):.5f} "
      f"(v1 could report 0.00000)")


# ============================================================================
# 3. STUDENT-t vs NORMAL — the Part 10.3 discriminator
# ============================================================================
print("\n  📌 3: Student-t vs Normal — is there exploitable tail structure?")
t_rows = []
for i, (tn, d) in enumerate(preds.items()):
    if not d.get("t_prob"): continue
    y = np.asarray(d["y_true"], int)
    tq, tp = _arr(d["t_prob"]), _arr(d["t_pred"], int)
    nq, npd = _arr(d["normal_prob"]), _arr(d["normal_pred"], int)
    if not np.isfinite(tq).any(): continue
    b = paired_bootstrap_diff(y, tp, npd, N_BOOT, RANDOM_STATE + 500 + i,
                              stat="primary", a_prob=tq, b_prob=nq)
    t_rows.append({"task": tn, "t_minus_normal": round(b["obs"], 4),
                   "ci_low": round(b["lo"], 4), "ci_high": round(b["hi"], 4),
                   "p_raw": b["p"],
                   "t_brier_skill": round(brier_skill_score(y, tq), 4),
                   "normal_brier_skill": round(brier_skill_score(y, nq), 4)})
df_t = pd.DataFrame(t_rows)
if not df_t.empty:
    _m = df_t["t_minus_normal"].mean()
    _w = int((df_t["t_minus_normal"] > 0).sum())
    print(f"     mean Δ(t − Normal) = {_m:+.4f}; t wins {_w}/{len(df_t)} tasks")
    print("     → " + ("Fat tails HELP → the Gaussian's errors are STRUCTURED, so a "
                       "learned\n       model has genuine room. An ML≈Normal tie is then a "
                       "SMALL-SAMPLE\n       result, not a Bayes-floor one."
                       if _m > 0.01 else
                       "Fat tails do NOT help → the residual move looks like NOISE on "
                       "these\n       inputs. The Gaussian is near-optimal and no model "
                       "restricted to them\n       can win. Publishable as a null result; "
                       "the follow-on study needs\n       richer information, not a better "
                       "function."))


# ============================================================================
# 4. ECONOMIC — payoff difference
# ============================================================================
print("\n  📌 4: Economic view (mean per-cycle payoff difference)")
pay_rows = []
for i, (tn, v) in enumerate(payoff.items()):
    a = np.asarray(v, float)
    if len(a) < 8:
        pay_rows.append({"task": tn, "n": len(a), "mean_payoff_diff": float(np.nanmean(a)),
                         "ci_low": -np.inf, "ci_high": np.inf, "p_raw": 1.0,
                         "note": "insufficient_n"}); continue
    rng = np.random.default_rng(RANDOM_STATE + 2000 + i)
    dd = np.array([a[rng.integers(0, len(a), len(a))].mean() for _ in range(N_BOOT)])
    lo, hi = np.percentile(dd, [2.5, 97.5])
    n_lt, n_gt = int((dd < 0).sum()), int((dd > 0).sum())
    n_eq = N_BOOT - n_lt - n_gt
    p = min(1.0, 2.0 * (1.0 + min(n_lt + n_eq / 2, n_gt + n_eq / 2)) / (1.0 + N_BOOT))
    pay_rows.append({"task": tn, "n": len(a), "mean_payoff_diff": round(float(a.mean()), 4),
                     "ci_low": round(lo, 4), "ci_high": round(hi, 4), "p_raw": p, "note": ""})
    print(f"     {tn:<10} Δpayoff={a.mean():+.3f} [{lo:+.3f},{hi:+.3f}] p={p:.4f}")
df_pay = pd.DataFrame(pay_rows)
print(f"     (payoff matrix: correct-exit {PAYOFF_BREACH_CORRECT}, held-into-breach "
      f"{PAYOFF_NOBREACH_WRONG} → economic cut p* = {ECONOMIC_THRESHOLD:.4f})")


# ============================================================================
# 5. MULTIPLICITY
# ============================================================================
print("\n  📌 5: Benjamini-Hochberg (on RAW p-values)")
for frame, col, label in [(df, "f1_p_raw", "F1"), (df, "bss_p_raw", "Brier-skill"),
                          (df_pay, "p_raw", "payoff")]:
    if frame.empty or col not in frame.columns: continue
    s, q = benjamini_hochberg(frame[col].values, ALPHA)      # not rounded
    frame[f"q_BH_all"] = np.round(q, 5); frame[f"sig_BH_all"] = s
    if "family" in frame.columns:
        for fam in frame["family"].unique():
            m = frame["family"] == fam
            sf, qf = benjamini_hochberg(frame.loc[m, col].values, ALPHA)
            frame.loc[m, "q_BH_family"] = np.round(qf, 5)
            frame.loc[m, "sig_BH_family"] = sf
    n_all = int(s.sum())
    n_fam = int(frame["sig_BH_family"].sum()) if "sig_BH_family" in frame.columns else n_all
    print(f"     {label:<12} significant: {n_all}/{len(frame)} treating all cells as "
          f"one family | {n_fam}/{len(frame)} within label families")

print("\n     ⚠️ MULTIPLICITY CAVEAT — state this in the write-up:")
print("        The breach label is a CYCLE-level property, so upper_D2/D3/D4 share an")
print("        IDENTICAL label vector, and so do lower_*. There are ~2 independent")
print("        label families, not 6 independent tests. BH remains VALID under this")
print("        positive dependence (PRDS) but is CONSERVATIVE, and 'k of 6 significant'")
print("        would read as six independent confirmations. Report the family column.")


# ============================================================================
# 6. CALIBRATION DECOMPOSITION — which explanation does the evidence support?
# ============================================================================
print("\n  📌 6: Brier decomposition   BS = reliability − resolution + uncertainty")
dec_rows = []
for tn, d in preds.items():
    y = np.asarray(d["y_true"], int)
    for who, key in [("ML", "ml_prob"), ("Normal", "normal_prob"), ("Student-t", "t_prob")]:
        if not d.get(key): continue
        q = _arr(d[key])
        if not np.isfinite(q).any(): continue
        dd = brier_decomposition(y, q)
        dec_rows.append({"task": tn, "contestant": who, **{k: round(v, 5) for k, v in dd.items()},
                         "log_loss": round(log_loss_safe(y, q), 4)})
df_dec = pd.DataFrame(dec_rows)
if not df_dec.empty:
    piv = df_dec.pivot_table(index="contestant", values=["brier", "reliability", "resolution"],
                             aggfunc="mean").round(5)
    print(piv.to_string())
    _nm = df_dec[df_dec.contestant == "Normal"]
    if len(_nm):
        rel, res = _nm["reliability"].mean(), _nm["resolution"].mean()
        print(f"\n     Normal: reliability={rel:.5f}  resolution={res:.5f}")
        if rel > 0.5 * res and rel > 1e-3:
            print("     → Normal is systematically MIS-CALIBRATED (reliability is large")
            print("       relative to resolution). There IS structure to exploit, which")
            print("       supports explanation A/B, not the Bayes-noise explanation C.")
        else:
            print("     → Normal is well calibrated and resolution is low for everyone.")
            print("       That points to explanation C: the residual move is close to")
            print("       unpredictable from price and volatility alone.")


# ============================================================================
# 7. ERROR CORRELATION — could a hybrid help at all?
# ============================================================================
print("\n  📌 7: ML vs Normal error correlation")
err_rows = []
for tn, d in preds.items():
    y = np.asarray(d["y_true"], int)
    em = (_arr(d["ml_pred"], int) != y).astype(int)
    en = (_arr(d["normal_pred"], int) != y).astype(int)
    r = float(np.corrcoef(em, en)[0, 1]) if em.std() > 0 and en.std() > 0 else np.nan
    err_rows.append({"task": tn, "n": len(y), "err_corr": round(r, 4),
                     "ml_err_rate": round(em.mean(), 4), "normal_err_rate": round(en.mean(), 4),
                     "both_wrong": int(((em == 1) & (en == 1)).sum()),
                     "only_ml_wrong": int(((em == 1) & (en == 0)).sum()),
                     "only_normal_wrong": int(((em == 0) & (en == 1)).sum())})
df_err = pd.DataFrame(err_rows)
print(df_err.to_string(index=False))
_mc = df_err["err_corr"].mean()
print(f"     mean error correlation = {_mc:.3f} → "
      + ("errors are highly correlated; a hybrid or ensemble of the two cannot help."
         if _mc > 0.6 else
         "errors are partly independent; a hybrid could in principle add value "
         "(though S6 blending was rejected on variance grounds)."))


# ============================================================================
# 8. WRITE
# ============================================================================
OUT = os.path.join(RESULTS_DIR, f"significance_{RUN_TYPE}.xlsx")
notes = pd.DataFrame([
    ("script", "S7_Significance_P1_v4.0"),
    ("bootstrap draws", N_BOOT),
    ("p-value", "(1 + k)/(1 + B), ties split between tails [S1 FIXED]"),
    ("smallest attainable p", round(2.0 / (N_BOOT + 1), 5)),
    ("BH input", "RAW p-values; rounding is display-only [S3 FIXED]"),
    ("multiplicity", "upper_* and lower_* each share ONE label vector → ~2 "
                     "independent families, not 6 tests [S4]"),
    ("degenerate draws", "resamples with zero positives are redrawn and counted [S5]"),
    ("threshold uncertainty", "NOT propagated — thresholds are held fixed inside the "
                              "bootstrap, so variance is understated for both "
                              "contestants, more so for ML. Disclose."),
    ("artifact", ARTIFACT_PATH),
], columns=["key", "value"])

with pd.ExcelWriter(OUT, engine="openpyxl") as xw:
    df.to_excel(xw, sheet_name="Primary_F1_and_Brier", index=False)
    (df_t if not df_t.empty else pd.DataFrame({"note": ["disabled"]})
     ).to_excel(xw, sheet_name="Student_t_vs_Normal", index=False)
    df_pay.to_excel(xw, sheet_name="Economic_payoff", index=False)
    df_dec.to_excel(xw, sheet_name="Brier_decomposition", index=False)
    df_err.to_excel(xw, sheet_name="Error_correlation", index=False)
    notes.to_excel(xw, sheet_name="Method_notes", index=False)
print(f"\n  ✅ {OUT}")

df_significance = df
print("\n" + "=" * 78)
print("  ✅ S7_Significance_P1_v4.0 COMPLETE")
print("     ➡️  NEXT: S8_Charts")
print("=" * 78)


  📊 S7_Significance_P1_v4.0  (FULL)   B=5000, α=0.05
  Tasks with ML predictions: ['upper_D2', 'lower_D2', 'upper_D3', 'lower_D3', 'upper_D4', 'lower_D4']

  📌 1-2: ML vs Normal   (paired bootstrap over CYCLES)
     upper_D2   ΔF1=+0.000 [-0.042,+0.046] p=0.9970 | ΔBSS=-0.049 p=0.1980
     lower_D2   ΔF1=+0.005 [-0.045,+0.056] p=0.8280 | ΔBSS=+0.012 p=0.7838
     upper_D3   ΔF1=-0.086 [-0.168,-0.007] p=0.0316 | ΔBSS=+0.011 p=0.7706
     lower_D3   ΔF1=+0.025 [-0.008,+0.069] p=0.1888 | ΔBSS=-0.005 p=0.8718
     upper_D4   ΔF1=+0.006 [-0.051,+0.065] p=0.8178 | ΔBSS=-0.028 p=0.5047
     lower_D4   ΔF1=+0.015 [-0.040,+0.072] p=0.6257 | ΔBSS=-0.036 p=0.1096

     Smallest attainable p at B=5000: 0.00040 (v1 could report 0.00000)

  📌 3: Student-t vs Normal — is there exploitable tail structure?
     mean Δ(t − Normal) = -0.0238; t wins 0/6 tasks
     → Fat tails do NOT help → the residual move looks like NOISE on these
       inputs. The Gaussian is near-optimal and no model restricted to 

In [49]:
# @title
# ============================================================================
# S8_Charts_P1_v2.0 — FIGURES
# ============================================================================
#
# PURPOSE
# -------
# Render every figure the write-up uses. Reads the S6 artifact, the S7
# significance workbook and the ledger, and computes nothing that changes a
# result — every number plotted already exists in a saved file, so a chart can
# never disagree with the table it came from.
#
#   OUT : charts_project1_<world>.pdf  plus one PNG per figure
#
# THE FIGURES
# -----------
#   1  Headline: F1 and Brier skill, ML vs Gaussian vs Student-t, per task
#   2  Bootstrap forest plot — the F1 difference with its 95% interval, which
#      is the honest picture of how much power this sample size buys
#   3  Reliability diagrams — the calibration evidence
#   4  Brier decomposition — reliability against resolution, per contestant
#   5  Confusion matrices, ML against the Gaussian
#   6  ROC and Precision-Recall. PR is the more honest curve here
#   7  sigma x window heatmaps — ML out-of-fold score, consistency rate,
#      and the F1 difference
#   8  Error-overlap bars: do the two contestants fail on the SAME cycles?
#
# DESIGN RULES
# ------------
# * One colour per contestant, used consistently in every figure.
# * Every panel showing F1 also shows n and the base rate, because F1 is
#   base-rate sensitive and the base rate CHANGES with sigma — the band IS the
#   label. So a difference in F1 is comparable within a configuration but not
#   across sigma levels.
# * Figure 1 labels each task with its TRUE feature count (7 at D2, 10 at D3,
#   13 at D4), resolved from best_models.json's n_features_used, then the
#   ledger, then the whitelist. An unresolved count prints "?" rather than a
#   number that might be wrong. The `n_features` column is the ablation
#   BUDGET — 13 for every task — and is never plotted.
# * Permutation ranks are NOT plotted. ABLATION_FEATURE_COUNTS sits at or
#   above the largest feature set, so select_features() returns every feature
#   and any "importance" ordering would be positional, not real.
#
# RUN AFTER: S6_InferEval → S7_Significance.
# ============================================================================

try:
    _ = (RESULTS_DIR, RUN_TYPE, TASKS, LEDGER_PATH, PRIMARY_METRIC_NAME,
         SIGMA_GRID, SIGMA_WINDOW_GRID, MIN_OOF_SCORE)
    _ = (reliability_table, brier_decomposition, f1_at)
except NameError as _ne:
    raise RuntimeError(f"❌ Missing {_ne}. Run S1_Config_P1_v4.0 → … → S6 → S7 first.")

import os, json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "font.size": 9,
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
})

C = {"ml": "#2563eb", "normal": "#dc2626", "t": "#059669", "muted": "#94a3b8"}
LBL = {"ml": "ML-core", "normal": "Normal (Gaussian)", "t": f"Student-t"}

print("\n" + "=" * 78)
print(f"  📈 S8_Charts_P1_v2.0  ({RUN_TYPE})")
print("=" * 78)

ART = os.path.join(RESULTS_DIR, f"backtest_artifact_{RUN_TYPE}.json")
if not os.path.exists(ART):
    _c = sorted(glob.glob(os.path.join(os.path.dirname(RESULTS_DIR), "run_*",
                                       f"backtest_artifact_{RUN_TYPE}.json")))
    if not _c:
        raise FileNotFoundError("❌ No S6 artifact. Run S6_InferEval first.")
    ART = _c[-1]; print(f"  ℹ️ using {ART}")
with open(ART) as f:
    art = json.load(f)
preds = art.get("predictions", {})
cmp_df = pd.DataFrame(art.get("comparison", []))
cfg_df = pd.DataFrame(art.get("per_config", []))
if not preds:
    raise RuntimeError("❌ Artifact has no ML predictions — nothing to chart.")
TASK_ORDER = [t["name"] for t in TASKS if t["name"] in preds]

SIG = os.path.join(RESULTS_DIR, f"significance_{RUN_TYPE}.xlsx")
sig_df = pd.read_excel(SIG, sheet_name="Primary_F1_and_Brier") if os.path.exists(SIG) else pd.DataFrame()


# ============================================================================
#  TRUE FEATURE COUNT PER TASK — for the axis labels
# ----------------------------------------------------------------------------
#  Preference order, most authoritative first:
#    1. best_models.json → models[].n_features_used   (S3_Select v3.1)
#    2. the ledger's n_features_used column            (S3_Harvest)
#    3. len(get_task_features(task, probe))            (the whitelist itself)
#  NEVER n_features / n_features_budget — that is the ablation cap (13), the
#  same for every task, and quoting it would overstate D2 and D3 by 6 and 3.
# ============================================================================
NFEAT_USED = {}
try:
    if "BEST_MODELS_PATH" in dir() and os.path.exists(BEST_MODELS_PATH):
        with open(BEST_MODELS_PATH) as f:
            for _tn, _dm in json.load(f).items():
                _v = [m.get("n_features_used") for m in _dm.get("models", [])
                      if m.get("n_features_used")]
                if _v:
                    NFEAT_USED[_tn] = int(max(_v))
except Exception:
    pass
if len(NFEAT_USED) < len(TASK_ORDER):
    try:
        _led = pd.read_csv(LEDGER_PATH) if str(LEDGER_PATH).endswith(".csv") \
               else pd.read_excel(LEDGER_PATH)
        if "n_features_used" in _led.columns:
            for _tn, _g in _led.groupby("task"):
                _v = pd.to_numeric(_g["n_features_used"], errors="coerce").dropna()
                if len(_v) and _tn not in NFEAT_USED:
                    NFEAT_USED[str(_tn)] = int(_v.max())
    except Exception:
        pass
if len(NFEAT_USED) < len(TASK_ORDER) and "get_task_features" in dir():
    _probe = sorted({
        f.format(d=d, side=sd)
        for d in (1, 2, 3, 4) for sd in ("upper", "lower")
        for f in ("norm_dist_{side}_D{d}", "realized_vol_D{d}", "gap_D{d}",
                  "dist_to_{side}_D{d}", "days_left_D{d}", "sqrt_dl_frac_D{d}")
    } | {"band_width_pct", "mu", "sigma"})
    for _t in TASKS:
        if _t["name"] in TASK_ORDER and _t["name"] not in NFEAT_USED:
            try:
                NFEAT_USED[_t["name"]] = len(get_task_features(_t, _probe))
            except Exception:
                pass

_nf_txt = {t: (f"{NFEAT_USED[t]}f" if t in NFEAT_USED else "?f") for t in TASK_ORDER}
print("  feature count per task (TRUE, not the ablation budget): "
      + ", ".join(f"{t}={_nf_txt[t]}" for t in TASK_ORDER))
if any(v == "?f" for v in _nf_txt.values()):
    print("  ⚠️  some counts unresolved — re-run S3_Select v3.1 so "
          "best_models.json carries n_features_used")


def _a(d, k, dtype=float):
    """Read one array from the S6 artifact, mapping null to NaN.

    Parameters
    ----------
    d : dict
        A per-task block of the artifact.
    k : str
        Key to read.
    dtype : type
        Target dtype.

    Returns
    -------
    ndarray
        The values, with JSON nulls as NaN and an empty array when absent.
    """
    return np.array([np.nan if v is None else v for v in d.get(k, [])], dtype=dtype)


PDF = os.path.join(RESULTS_DIR, f"charts_project1_{RUN_TYPE}.pdf")
pdf = PdfPages(PDF)
_saved = []


def _finish(fig, name):
    """Lay out a figure, append it to the PDF and save it as a PNG.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        The completed figure.
    name : str
        Stem for the PNG filename, e.g. "01_headline".

    Returns
    -------
    None
        The figure is closed; its filename is recorded for the summary.
    """
    fig.tight_layout()
    pdf.savefig(fig)
    p = os.path.join(RESULTS_DIR, f"chart_{name}_{RUN_TYPE}.png")
    fig.savefig(p, bbox_inches="tight")
    plt.close(fig); _saved.append(os.path.basename(p))


# ============================================================================
# 1. HEADLINE — F1 and Brier skill side by side
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
x = np.arange(len(TASK_ORDER)); w = 0.26
for ax, (mk, title) in zip(axes, [("f1", "F1 (threshold-dependent)"),
                                  ("brier_skill", "Brier skill (threshold-free)")]):
    for j, who in enumerate(["normal", "ml", "t"]):
        vals = []
        for tn in TASK_ORDER:
            s = preds[tn]["scores"].get({"ml": "ml", "normal": "normal", "t": "student_t"}[who], {})
            vals.append(s.get(mk, np.nan) if s else np.nan)
        ax.bar(x + (j - 1) * w, vals, w, label=LBL[who], color=C[who], alpha=.88)
    ax.axhline(0, color="k", lw=.8)
    ax.set_xticks(x); ax.set_xticklabels(TASK_ORDER, rotation=20)
    ax.set_title(title, fontweight="bold"); ax.legend(fontsize=8)
_br = [preds[t]["scores"]["ml"].get("base_rate", np.nan) for t in TASK_ORDER]
_n = [preds[t]["n"] for t in TASK_ORDER]
#  the feature count on the axis is n_features_used, not the budget
axes[0].set_xticklabels([f"{t}\n{_nf_txt[t]}, n={n}, base={b:.0%}"
                         for t, n, b in zip(TASK_ORDER, _n, _br)],
                        rotation=20, fontsize=7)
axes[1].set_xticklabels([f"{t}\n{_nf_txt[t]}" for t in TASK_ORDER],
                        rotation=20, fontsize=7)
fig.suptitle("ML-core vs Normal vs Student-t — identical inputs", fontweight="bold")
_finish(fig, "01_headline")

# ============================================================================
# 2. FOREST PLOT — ΔF1 with bootstrap CI. The honest view of statistical power.
# ============================================================================
src = sig_df if not sig_df.empty else cmp_df
if not src.empty and {"f1_diff", "f1_ci_low"}.issubset(src.columns) or \
   {"delta_f1", "ci_low"}.issubset(src.columns):
    dcol = "f1_diff" if "f1_diff" in src.columns else "delta_f1"
    lcol = "f1_ci_low" if "f1_ci_low" in src.columns else "ci_low"
    hcol = "f1_ci_high" if "f1_ci_high" in src.columns else "ci_high"
    s = src.dropna(subset=[dcol]).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8.5, 0.6 * len(s) + 2.2))
    yy = np.arange(len(s))
    for i, r in s.iterrows():
        lo, hi = r[lcol], r[hcol]
        crosses = (lo <= 0 <= hi)
        ax.plot([lo, hi], [i, i], color=C["muted"] if crosses else C["ml"], lw=2.4)
        ax.plot(r[dcol], i, "o", color=C["muted"] if crosses else C["ml"], ms=7)
    ax.axvline(0, color=C["normal"], ls="--", lw=1.2)
    ax.set_yticks(yy); ax.set_yticklabels(s["task"])
    ax.set_xlabel("ΔF1  (ML − Normal).  Grey = CI contains 0 → tie")
    ax.set_title("Paired bootstrap, 95% CI — resampling cycles", fontweight="bold")
    ax.invert_yaxis()
    _w = float(np.nanmean(s[hcol] - s[lcol]))
    ax.text(0.01, 0.02, f"mean CI width = {_w:.3f}\nAn effect smaller than this is "
                        f"undetectable at n≈{int(s['n'].mean()) if 'n' in s else 0}.",
            transform=ax.transAxes, fontsize=8, va="bottom",
            bbox=dict(fc="#fef3c7", ec="#d97706", alpha=.9))
    _finish(fig, "02_forest_deltaF1")

# ============================================================================
# 3. RELIABILITY DIAGRAMS — the calibration evidence
# ============================================================================
nc = min(3, len(TASK_ORDER)); nr = int(np.ceil(len(TASK_ORDER) / nc))
fig, axes = plt.subplots(nr, nc, figsize=(4.3 * nc, 3.9 * nr), squeeze=False)
for k, tn in enumerate(TASK_ORDER):
    ax = axes[k // nc][k % nc]
    y = np.asarray(preds[tn]["y_true"], int)
    ax.plot([0, 1], [0, 1], "--", color="#64748b", lw=1, label="perfect")
    for who, key in [("normal", "normal_prob"), ("ml", "ml_prob"), ("t", "t_prob")]:
        q = _a(preds[tn], key)
        if not np.isfinite(q).any(): continue
        rt = reliability_table(y, q, n_bins=6)
        if len(rt):
            ax.plot(rt["mean_pred"], rt["obs_freq"], "o-", color=C[who], ms=4,
                    lw=1.6, label=LBL[who])
    ax.set_title(tn, fontsize=9, fontweight="bold")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("predicted P(breach)"); ax.set_ylabel("observed frequency")
    if k == 0: ax.legend(fontsize=7)
for k in range(len(TASK_ORDER), nr * nc):
    axes[k // nc][k % nc].axis("off")
fig.suptitle("Reliability — above the diagonal = under-predicting breaches",
             fontweight="bold")
_finish(fig, "03_reliability")

# ============================================================================
# 4. BRIER DECOMPOSITION — reliability vs resolution
# ============================================================================
rows = []
for tn in TASK_ORDER:
    y = np.asarray(preds[tn]["y_true"], int)
    for who, key in [("normal", "normal_prob"), ("ml", "ml_prob"), ("t", "t_prob")]:
        q = _a(preds[tn], key)
        if not np.isfinite(q).any(): continue
        rows.append({"task": tn, "who": who, **brier_decomposition(y, q)})
dec = pd.DataFrame(rows)
if not dec.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
    for j, (mk, ttl, good) in enumerate([
            ("reliability", "Reliability  (miscalibration — LOWER is better)", "lower"),
            ("resolution", "Resolution  (discrimination — HIGHER is better)", "higher")]):
        ax = axes[j]; xx = np.arange(len(TASK_ORDER)); w = .26
        for i, who in enumerate(["normal", "ml", "t"]):
            sub = dec[dec.who == who].set_index("task").reindex(TASK_ORDER)
            ax.bar(xx + (i - 1) * w, sub[mk].values, w, label=LBL[who], color=C[who], alpha=.88)
        ax.set_xticks(xx); ax.set_xticklabels(TASK_ORDER, rotation=20, fontsize=8)
        ax.set_title(ttl, fontweight="bold", fontsize=10)
        if j == 0: ax.legend(fontsize=8)
    fig.suptitle("Brier decomposition:  BS = reliability − resolution + uncertainty\n"
                 "Large reliability for Normal ⇒ exploitable structure. "
                 "Low resolution for all ⇒ Bayes floor.", fontweight="bold", fontsize=10)
    _finish(fig, "04_brier_decomposition")

# ============================================================================
# 5. CONFUSION MATRICES
# ============================================================================
fig, axes = plt.subplots(2, len(TASK_ORDER), figsize=(2.55 * len(TASK_ORDER), 5.4),
                         squeeze=False)
for j, tn in enumerate(TASK_ORDER):
    y = np.asarray(preds[tn]["y_true"], int)
    for i, (who, key) in enumerate([("ml", "ml_pred"), ("normal", "normal_pred")]):
        p = _a(preds[tn], key, int); ax = axes[i][j]
        cm = np.array([[int(((y == a) & (p == b)).sum()) for b in (0, 1)] for a in (0, 1)])
        ax.imshow(cm, cmap="Blues" if who == "ml" else "Reds", aspect="equal")
        for a in range(2):
            for b in range(2):
                ax.text(b, a, cm[a, b], ha="center", va="center", fontsize=10,
                        fontweight="bold",
                        color="white" if cm[a, b] > cm.max() / 2 else "black")
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["No", "Breach"], fontsize=7)
        ax.set_yticklabels(["No", "Breach"], fontsize=7)
        ax.grid(False)
        if i == 0: ax.set_title(tn, fontsize=8, fontweight="bold")
        if j == 0: ax.set_ylabel(f"{LBL[who]}\ntrue", fontsize=8)
        if i == 1: ax.set_xlabel("predicted", fontsize=7)
fig.suptitle("Confusion matrices", fontweight="bold")
_finish(fig, "05_confusion")

# ============================================================================
# 6. ROC + PRECISION-RECALL
# ============================================================================
try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
    fig, axes = plt.subplots(2, len(TASK_ORDER), figsize=(2.9 * len(TASK_ORDER), 6.0),
                             squeeze=False)
    for j, tn in enumerate(TASK_ORDER):
        y = np.asarray(preds[tn]["y_true"], int)
        base = y.mean()
        for who, key in [("normal", "normal_prob"), ("ml", "ml_prob"), ("t", "t_prob")]:
            q = _a(preds[tn], key)
            m = np.isfinite(q)
            if m.sum() < 5 or len(np.unique(y[m])) < 2: continue
            fpr, tpr, _ = roc_curve(y[m], q[m])
            axes[0][j].plot(fpr, tpr, color=C[who], lw=1.6,
                            label=f"{LBL[who]} {auc(fpr, tpr):.2f}")
            pr, rc, _ = precision_recall_curve(y[m], q[m])
            axes[1][j].plot(rc, pr, color=C[who], lw=1.6,
                            label=f"AP {average_precision_score(y[m], q[m]):.2f}")
        axes[0][j].plot([0, 1], [0, 1], "--", color="#94a3b8", lw=.9)
        axes[1][j].axhline(base, ls="--", color="#94a3b8", lw=.9)
        axes[0][j].set_title(tn, fontsize=8, fontweight="bold")
        axes[0][j].legend(fontsize=6, loc="lower right")
        axes[1][j].legend(fontsize=6, loc="upper right")
        axes[1][j].set_xlabel("recall", fontsize=7)
        if j == 0:
            axes[0][j].set_ylabel("TPR (ROC)", fontsize=8)
            axes[1][j].set_ylabel("precision (PR)", fontsize=8)
    fig.suptitle("ROC and Precision-Recall.  PR is the honest curve for a rare "
                 "positive class — the dashed line is the base rate.", fontweight="bold",
                 fontsize=10)
    _finish(fig, "06_roc_pr")
except Exception as e:
    print(f"  ⚠️ ROC/PR skipped: {e}")

# ============================================================================
# 7. σ × WINDOW HEATMAPS
# ============================================================================
led = pd.read_excel(LEDGER_PATH) if os.path.exists(LEDGER_PATH) else pd.DataFrame()
if not led.empty:
    for c in ("sigma", "window", "oof_score", "val_f1"):
        if c in led.columns: led[c] = pd.to_numeric(led[c], errors="coerce")
    led["sigma"] = led["sigma"].round(2)
    if "folds_consistency_pass" in led.columns:
        led["folds_consistency_pass"] = (led["folds_consistency_pass"].astype(str)
                                         .str.lower().isin(["true", "1", "yes"]))
    sc = "oof_score" if "oof_score" in led.columns else "val_f1"
    sgs = sorted(led["sigma"].dropna().unique()); wns = sorted(led["window"].dropna().unique())

    panels = [(f"best {sc} (consistent only)",
               lambda s, w: led[(led.sigma == s) & (led.window == w)
                                & led.folds_consistency_pass][sc].max(), "viridis"),
              ("consistency pass rate",
               lambda s, w: led[(led.sigma == s) & (led.window == w)]
                            ["folds_consistency_pass"].mean(), "Greens")]
    if not cfg_df.empty and "delta_f1" in cfg_df.columns:
        cfg_df["sigma"] = pd.to_numeric(cfg_df["sigma"], errors="coerce").round(2)
        panels.append(("mean ΔF1  (ML − Normal)",
                       lambda s, w: cfg_df[(cfg_df.sigma == s)
                                           & (cfg_df.window == w)]["delta_f1"].mean(), "RdBu_r"))

    fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 3.6), squeeze=False)
    for k, (ttl, fn, cmap) in enumerate(panels):
        M = np.array([[fn(s, w) for w in wns] for s in sgs], float)
        ax = axes[0][k]
        vmax = np.nanmax(np.abs(M)) if "Δ" in ttl else None
        im = ax.imshow(M, cmap=cmap, aspect="auto",
                       **({"vmin": -vmax, "vmax": vmax} if vmax else {}))
        for i in range(len(sgs)):
            for j in range(len(wns)):
                if np.isfinite(M[i, j]):
                    ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8)
        ax.set_xticks(range(len(wns))); ax.set_xticklabels([int(w) for w in wns])
        ax.set_yticks(range(len(sgs))); ax.set_yticklabels(sgs)
        ax.set_xlabel("window"); ax.set_ylabel("sigma")
        ax.set_title(ttl, fontsize=9, fontweight="bold"); ax.grid(False)
        fig.colorbar(im, ax=ax, shrink=.85)
    fig.suptitle("Config robustness. ⚠️ The base rate changes with sigma (the band IS "
                 "the label),\nso ΔF1 is comparable WITHIN a cell but not ACROSS sigma "
                 "levels.", fontweight="bold", fontsize=9)
    _finish(fig, "07_heatmaps")

# ============================================================================
# 8. ERROR OVERLAP — can a hybrid help?
# ============================================================================
rows = []
for tn in TASK_ORDER:
    y = np.asarray(preds[tn]["y_true"], int)
    em = (_a(preds[tn], "ml_pred", int) != y).astype(int)
    en = (_a(preds[tn], "normal_pred", int) != y).astype(int)
    rows.append({"task": tn, "both": int(((em == 1) & (en == 1)).sum()),
                 "only_ml": int(((em == 1) & (en == 0)).sum()),
                 "only_nm": int(((em == 0) & (en == 1)).sum()),
                 "neither": int(((em == 0) & (en == 0)).sum())})
eo = pd.DataFrame(rows).set_index("task")
fig, ax = plt.subplots(figsize=(8.6, 4.0))
btm = np.zeros(len(eo))
for col, colr, lab in [("neither", "#22c55e", "both correct"),
                       ("only_nm", C["normal"], "only Normal wrong"),
                       ("only_ml", C["ml"], "only ML wrong"),
                       ("both", "#334155", "both wrong")]:
    ax.bar(eo.index, eo[col], bottom=btm, color=colr, label=lab, alpha=.9)
    btm += eo[col].values
ax.set_ylabel("cycles"); ax.legend(fontsize=8, ncol=2)
ax.set_title("Error overlap — a large 'both wrong' block means the two contestants "
             "fail on the SAME cycles,\nso no hybrid of them can help.",
             fontweight="bold", fontsize=9)
plt.setp(ax.get_xticklabels(), rotation=20)
_finish(fig, "08_error_overlap")

pdf.close()
print(f"\n  ✅ Combined PDF: {PDF}")
print(f"  ✅ {len(_saved)} PNGs: {', '.join(_saved)}")
print("\n" + "=" * 78)
print("  ✅ S8_Charts_P1_v2.0 COMPLETE")
print("     Figures 3 and 4 are the ones that answer Part 10.3 — whether the")
print("     Gaussian is mis-calibrated (room to learn) or at the Bayes floor.")
print("=" * 78)


  📈 S8_Charts_P1_v2.0  (FULL)
  feature count per task (TRUE, not the ablation budget): upper_D2=7f, lower_D2=7f, upper_D3=10f, lower_D3=10f, upper_D4=13f, lower_D4=13f

  ✅ Combined PDF: /content/drive/MyDrive/LSTM/Project1_MLcore_vs_Normal/output_binary/FULL/results/run_20260914_203539/charts_project1_FULL.pdf
  ✅ 8 PNGs: chart_01_headline_FULL.png, chart_02_forest_deltaF1_FULL.png, chart_03_reliability_FULL.png, chart_04_brier_decomposition_FULL.png, chart_05_confusion_FULL.png, chart_06_roc_pr_FULL.png, chart_07_heatmaps_FULL.png, chart_08_error_overlap_FULL.png

  ✅ S8_Charts_P1_v2.0 COMPLETE
     Figures 3 and 4 are the ones that answer Part 10.3 — whether the
     Gaussian is mis-calibrated (room to learn) or at the Bayes floor.
